# Đánh giá khả năng nhận dạng rò rỉ nước trên đường ống trên cơ sở phân tích âm thanh và ứng dụng mạng học sâu với đường ống PVC áp lực thấp — **v7**

**Acoustic Leak Identification in Low-Pressure PVC Water Distribution Pipes Using Deep Convolutional Neural Networks**

---

### Mục tiêu / Objective

Notebook này là **pipeline nghiên cứu hoàn chỉnh, chạy được trên Kaggle (GPU T4 x2)**, phục vụ trực tiếp việc viết bài báo khoa học:

1. **Đọc & kiểm kê dữ liệu** hiện trường (`leak/` vs `noleak/`, ống PVC, áp suất 1–2 bar), trích mã bản ghi `Rxxx` và gộp mọi tệp cùng bản ghi.
2. **Phân tích tín hiệu (EDA)** ở **mức bản ghi**: PSD Welch, năng lượng theo dải, AUC đơn biến theo dải — mọi họ kiểm định đều **hiệu chỉnh Holm**.
3. **Tiền xử lý**: khử DC, lọc thông cao 20 Hz, notch 50 Hz × 6 hài, tái lấy mẫu, chuẩn hoá đỉnh theo tệp, frame 2 s chồng 50%.
4. **5 kiến trúc**: `LeakCNN2D` (log-Mel), `LeakCNN1D` (sóng thô), `TransferCNN` (ResNet18 ImageNet), `MultiBranchCNN` (Mel + MFCC + STFT dải thấp), `CRNN` (CNN + BiGRU + attention).
5. **Giao thức chống rò rỉ dữ liệu**: CV phân tầng trên **bảng bản ghi** + assertion cứng; **lựa chọn lồng** (epoch, ngưỡng) chỉ trên validation-trong.
6. **CV lặp** (mặc định 3 phân hoạch khác nhau) để đo độ bất định do phân hoạch/khởi tạo, và **trung bình OOF qua các lần lặp** làm ước lượng ổn định nhất.
7. **Thống kê đúng đơn vị**: bootstrap theo cụm bản ghi, McNemar mức bản ghi + Holm, bootstrap **ghép cặp** cho ΔAUC.
8. **Baseline cổ điển** (đặc trưng phổ + thống kê log-Mel) so sánh **ghép cặp** với CNN; **ablation** cấu hình tín hiệu với lựa chọn trên validation-trong.
9. **Diễn giải**: Grad-CAM, occlusion theo dải, t-SNE tô theo bản ghi; **kiểm tra confound** điểm khảo sát.
10. **Xuất bảng LaTeX/Markdown, hình 300 dpi, gói mô hình TorchScript** kèm manifest và giới hạn đã đo.

### Điểm mới của v7 (chi tiết ở mục 16)

| Mã | Sửa đổi |
|---|---|
| **B1** | Grad-CAM của CRNN làm v6 **dừng giữa chừng** (cuDNN RNN backward ở chế độ eval) → mục 10–16 không chạy. Đã sửa. |
| **B2** | "Không chọn checkpoint trong warm-up" giờ **được thực thi** (v6 chỉ khai báo; LeakCNN1D bị chọn ở epoch ~0). |
| **B3** | Không còn tự hạ `low_band_hz` về 1000 Hz dựa trên một dải "mạnh nhất" không có ý nghĩa sau hiệu chỉnh đa so sánh. |
| **B5–B11** | `DEPLOY_MODEL` dùng trước khi định nghĩa; ghi đè `sel_df`; lọc sai baseline RMS; đoạn Results trộn số liệu hai mô hình; kết luận từ Grad-CAM/occlusion suy biến; tương thích matplotlib mới. |
| **S1** | CV lặp nhiều phân hoạch (mặc định 3 seed). |
| **S2** | Quy tắc gộp mức bản ghi **đăng ký trước** (`p_mean`, xác suất thô); Platt và lựa chọn trên validation-trong chỉ là phân tích độ nhạy. |
| **S3** | Baseline mạnh hơn (thống kê log-Mel) + so sánh **ghép cặp** thay cho "CI có chồng lấn không". |
| **S4** | Hiệu chỉnh Holm cho mọi kiểm định theo dải; giả thuyết "dải thấp" được kiểm định bằng CI bootstrap của hiệu số. |
| **A1** | Ablation bật mặc định: bỏ FIR, loss cân bằng theo bản ghi, 16 kHz, frame 1 s/4 s, bỏ notch/high-pass. |
| **D1** | Gán nhãn theo **từ**, bỏ khoá `"rr"`; dò thư mục dữ liệu báo lỗi rõ khi `cfg.data_root` sai. |

> **Cách dùng nhanh / Quick start**
> 1. `Add Data` → chọn dataset chứa hai thư mục `leak/` và `noleak/`; đặt đúng `cfg.data_root` (mục 1.2).
> 1b. **Kiểm tra cell 2.3b**: nó xác nhận mã bản ghi `Rxxx` được trích đúng và báo mâu thuẫn nhãn. Nếu tên tệp không theo dạng `Rxxx`, chỉnh `cfg.record_regex`.
> 1c. **Nếu có thông tin điểm khảo sát**: điền `site_map_template.csv` (notebook tự sinh ở mục 9.6), đặt `cfg.site_source="csv"` và `cfg.run_loso_full=True`. Đây là thí nghiệm quan trọng nhất còn thiếu.
> 2. `Settings` → `Accelerator` = **GPU T4 x2**, bật **Internet** (tải trọng số ImageNet cho TransferCNN).
> 3. `Run All` (~45–60 phút với cấu hình mặc định). Không tìm thấy dataset → notebook **tự sinh dữ liệu mô phỏng** (SYNTHETIC DEMO MODE).
> 4. Chạy thử nhanh: đặt biến môi trường `LEAK_SMOKE=1` (3 epoch, 2 fold).


## 0. Cơ sở đối chứng từ tài liệu WoS/Scopus / Literature baseline

Các mốc đối chứng chính được dùng để định vị đóng góp của nghiên cứu (chi tiết trong tài liệu kèm theo):

| # | Công bố | Vật liệu ống / Áp suất | Biểu diễn | Mô hình | Kết quả báo cáo |
|---|---------|------------------------|-----------|---------|-----------------|
| 1 | Shukla & Piratla, *Automation in Construction* (2020) | PVC/gang, phòng thí nghiệm | Tín hiệu gia tốc thô | 1D-CNN | Acc ≈ 0.95–0.99 |
| 2 | Cody et al., *J. Comput. Civ. Eng.* 34(2), 2020 | Mạng cấp nước thực | Hydroacoustic spectrogram | Deep Autoencoder (bán giám sát) | AUC ≈ 0.99 (controlled) |
| 3 | Guo et al., *J. Water Resour. Plan. Manage.* 147(2), 2021 | Ống kim loại | Time–frequency map | TF-CNN | Acc ≈ 0.97 |
| 4 | Ahmad et al., *Sensors* 22(4):1562, 2022 | Thép, 7 & 13 bar | Wavelet scalogram (Morse CWT) | CNN + Conv-Autoencoder | Acc 96.6–98.4% |
| 5 | Zhang et al., *Struct. Health Monit.* 22(4), 2023 | Mạng thông minh | Tín hiệu áp suất/âm | CNN | Acc ≈ 0.96 |
| 6 | Wu et al., *Process Saf. Environ. Prot.* (2023) | Ống cấp nước | Đặc trưng thời gian–tần số | CNN/ML | (dùng đối chứng phương pháp) |
| 7 | Ma et al., *J. Water Resour. Plan. Manage.* 150(6), 2024 | Giám sát liên tục | Logarithmic spectrogram | CNN | Acc > 0.95 |
| 8 | Wang et al., *Water Research* (2025) | Mạng cấp nước thực | Spectrogram | Interpretable DL | — |
| 9 | Li et al., *Water* 17(8):1115, 2025 | Thép DN50 | STFT + tín hiệu 1D | MobileNetV3 + Mamba (SSM) | Acc 98.3% |
| 10 | Zhang et al., *Water* 18(12):1528, 2026 | 5 thành phố (4.170 mẫu) | STFT 224×224 | Incep-ResNet + SE | Acc 96.6% |

**Khoảng trống nghiên cứu được khai thác trong bài báo này:**

- Hầu hết công bố dùng **ống kim loại** và **áp suất cao (≥ 5 bar)**; ống **PVC áp lực 1–2 bar** hầu như vắng mặt. Vật lý lan truyền khác hẳn: sóng trong ống nhựa suy giảm mạnh, tốc độ thấp (~350–500 m/s), năng lượng rò rỉ tập trung ở **dải tần rất thấp (< 800 Hz, thường 100–500 Hz)** — theo Brennan, Muggleton, Hunaidi và cộng sự.
- Nhiều công bố chia train/test **ngẫu nhiên ở mức frame**, gây **rò rỉ dữ liệu** giữa các frame cùng một bản ghi → chỉ số lạc quan giả tạo. Notebook này bắt buộc `StratifiedGroupKFold` theo bản ghi và **kiểm tra cứng** không giao nhau.
- Rất ít công bố đánh giá ở **mức bản ghi** (điều mà kỹ sư vận hành thực sự cần) và **kiểm định thống kê** sự khác biệt giữa các mô hình.


## 1. Thiết lập môi trường / Environment setup


In [ ]:
# =========================================================
# 1.1 Imports & cài đặt phụ thuộc / Imports and dependencies
# =========================================================
import os, sys, gc, json, math, re, time, random, shutil, hashlib, warnings, subprocess, itertools, threading
import unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict

warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

def _pip_install(pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
    if r.returncode != 0:
        print(f"  !! pip install thất bại cho {pkgs} (mã {r.returncode}) - lỗi sẽ lộ ra ở bước sau.")
    return r.returncode == 0

try:
    import soundfile as sf
except Exception:
    _pip_install(["soundfile"]); import soundfile as sf

try:
    import librosa
except Exception:
    _pip_install(["librosa"]); import librosa
import librosa.display

import numpy as np
import pandas as pd
import scipy.signal as sps
import scipy.stats as spstats

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import gridspec
from IPython.display import display, Markdown

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio

from sklearn.model_selection import (StratifiedGroupKFold, LeaveOneGroupOut, GroupShuffleSplit,
                                     StratifiedKFold, StratifiedShuffleSplit)
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                             precision_recall_curve, confusion_matrix, accuracy_score,
                             precision_recall_fscore_support, matthews_corrcoef,
                             balanced_accuracy_score)
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier

# ---- API AMP tương thích nhiều phiên bản PyTorch --------------------------
# [V6-T4] v5 dùng torch.cuda.amp.* (đã deprecated từ torch 2.x). Bọc lại một lần.
def amp_autocast(device, enabled: bool):
    try:
        return torch.amp.autocast(device_type=torch.device(device).type, enabled=bool(enabled))
    except Exception:
        return torch.cuda.amp.autocast(enabled=bool(enabled))

def make_grad_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=bool(enabled))
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=bool(enabled))

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("PyTorch     :", torch.__version__)
print("torchaudio  :", torchaudio.__version__)
print("librosa     :", librosa.__version__)
print("CUDA avail  :", torch.cuda.is_available())
N_GPU = torch.cuda.device_count()
print("GPU count   :", N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {p.name} | {p.total_memory/1024**3:.1f} GB | SM {p.major}.{p.minor}")


In [ ]:
# =========================================================
# 1.2 Cấu hình trung tâm / Central configuration
#     -> Chỉnh mọi tham số thí nghiệm tại đây.
#     [V6] Các thay đổi so với v5 được đánh dấu [V6-...]; [V7-...] là thay đổi của v7.
#     Tổng hợp đầy đủ ở mục 16.
# =========================================================
@dataclass
class CFG:
    # ---- Dữ liệu / Data -------------------------------------------------
    data_root: str = "/kaggle/input/datasets/nguyennguyenkhac/tapdulieutt-8k-v7-2/field_rec_8k_V7.2"            # để trống = tự dò trong /kaggle/input
    # [V7-D1] Khớp theo TỪ (ranh giới "_", ".", chữ số), không còn khớp chuỗi con.
    #          Bỏ khoá "rr": nó khớp nhầm mọi tên chứa hai chữ r liền nhau.
    leak_keys: tuple = ("leak", "ro_ri", "rori")
    noleak_keys: tuple = ("noleak", "no_leak", "non_leak", "nonleak", "normal",
                          "background", "khong_ro_ri", "khongroli", "no-leak")
    audio_ext: tuple = (".wav", ".flac", ".ogg", ".mp3", ".WAV")

    # ---- [V6-P2] Gán nhãn & kiểm tra nghiêm ------------------------------
    # Thư mục gần nhất được ưu tiên TRƯỚC tên tệp (tên tệp có thể sai), và mọi
    # chuỗi được chuẩn hoá Unicode để 'rò_rỉ' khớp 'ro_ri'.
    label_source_priority: str = "folder_then_filename"   # "folder_then_filename" | "filename_then_folder"
    unicode_fold: bool = True
    strict_checks: bool = True        # True -> DỪNG ngay khi có mâu thuẫn nhãn / định danh bản ghi

    # ---- MÃ BẢN GHI / recording identifier -------------------------------
    # TỐI QUAN TRỌNG: đây là đơn vị nhóm của cross-validation VÀ là đơn vị
    # đánh giá mức bản ghi. Nếu một bản ghi Rxxx bị cắt thành nhiều tệp
    # (R001_seg01.wav, R001_seg02.wav...) mà lấy TÊN TỆP làm mã bản ghi thì các
    # đoạn của cùng một bản ghi sẽ rơi vào các fold khác nhau -> rò rỉ dữ liệu.
    record_id_mode: str = "regex"        # "regex" | "filename" | "parent_dir"
    record_regex: str = r"(R\d{1,4})"    # nhóm 1 = mã bản ghi, ví dụ R001, R12
    derivative_markers: tuple = ("_aug", "_augment", "_events_removed",
                                 "_clean", "_denois", "_copy", "_dup")
    report_derivatives: bool = True

    # ---- [V6-P0] ĐIỂM KHẢO SÁT / acquisition site ------------------------
    # Không có thông tin điểm khảo sát thì KHÔNG THỂ loại trừ khả năng mô hình
    # đang học "dấu vân tay của vị trí thu" thay vì học hiện tượng rò rỉ.
    # Nếu có, hãy cung cấp CSV hai cột `record_id,site` (hoặc regex trên đường dẫn)
    # để notebook chạy được kiểm định leave-one-site-out ở mục 9.6.
    site_source: str = "none"        # "none" | "csv" | "regex"
    site_csv: str = ""               # ví dụ "/kaggle/input/meta/site_map.csv"
    site_regex: str = ""             # ví dụ r"site[_\-]?(\w+)" trên đường dẫn
    site_regex_group: int = 1

    # ---- Học đa thể hiện / Multiple-instance learning --------------------
    # Nhãn được gán ở MỨC BẢN GHI. Nếu tiếng rò rỉ chỉ nghe được từng đoạn thì
    # nhãn "leak" gán cho MỌI frame của bản ghi là NHÃN NHIỄU -> bài toán MIL.
    # LƯU Ý [V6-M3]: đây là GIẢ THIẾT cần kiểm chứng, không phải sự thật đã biết.
    # Mục 8.7 đo trực tiếp tỉ lệ thể hiện dương; nếu tỉ lệ đó ~1 thì giả thiết
    # KHÔNG được dữ liệu ủng hộ và phải bỏ khung MIL khi viết bài.
    mil_mode: str = "sil"            # "sil" = mỗi frame mang nhãn túi (mặc định)
                                     # "bag" = hàm mất mát tính ở mức túi
    mil_pooling: str = "lse"         # "lse" | "max" | "mean" | "attention"
    mil_lse_r: float = 5.0
    bag_size: int = 16

    # ---- [V6-P1-T3] Tiêu chí chọn epoch ----------------------------------
    # v5 dùng AUC MỨC BẢN GHI trên validation-trong; chỉ số này BÃO HOÀ ở 1.0000
    # sau vài epoch (validation-trong chỉ vài chục bản ghi) nên việc chọn epoch gần
    # như ngẫu nhiên. Mặc định mới: log-loss trên validation-trong (không bão hoà).
    select_metric: str = "inner_logloss"   # inner_logloss | inner_frame_auc | inner_record_auc
    criteria_ablation: bool = True         # lưu song song cả 3 tiêu chí (KHÔNG tốn thêm lượt huấn luyện)
    select_min_records: int = 6
    min_epochs: int = 8                    # số epoch tối thiểu trước khi cho phép dừng sớm
    min_epochs_after_lr_peak: bool = True   # [V6-T1] không chọn checkpoint trong pha warm-up
    select_smooth_w: int = 3               # làm mịn tiêu chí theo cửa sổ trượt (chống nhiễu chọn epoch)
    # [V7-B2] v6 KHAI BÁO "không chọn checkpoint trong warm-up" nhưng mã vẫn chọn từ epoch 0
    #         (log v6: "chọn ep1", LeakCNN1D chọn trung bình epoch 0.8). v7 THỰC THI ràng buộc.
    select_after_lr_peak: bool = True

    # ---- Tham số tín hiệu / Signal parameters ---------------------------
    target_sr: int = 8000
    highpass_hz: float = 20.0      # khử trôi DC / hạ tần rung nền
    notch_mains: bool = True       # khử hài lưới điện 50 Hz (Việt Nam)
    mains_hz: float = 50.0
    mains_harmonics: int = 6
    notch_q: float = 30.0
    # [V7] Chuẩn hoá đỉnh theo TỪNG TỆP. 100 = max|x| (giống v6, tương thích notebook suy luận);
    #      ví dụ 99.9 = phân vị bền với xung nhiễu. Mô hình phổ đều chuẩn hoá theo mẫu nên
    #      tham số này hầu như chỉ ảnh hưởng đặc trưng RMS.
    peak_norm_pct: float = 100.0

    frame_sec: float = 2.0         # độ dài 1 frame phân loại
    hop_sec: float = 1.0           # bước trượt (chồng lấn 50%)
    min_frames_per_file: int = 1

    # ---- Biểu diễn thời gian-tần số / Time-frequency representation -----
    n_fft: int = 1024              # 128 ms @ 8 kHz
    hop_length: int = 256          # 32 ms
    n_mels: int = 64
    f_min: float = 20.0
    f_max: float = 4000.0
    n_mfcc: int = 40
    # [V7-B3] v6 tự hạ low_band_hz 2800 -> 1000 Hz dựa trên MỘT dải "mạnh nhất" (283-413 Hz,
    #         không còn ý nghĩa sau hiệu chỉnh đa so sánh) và cắt mất dải 1.3-2.7 kHz mà chính
    #         Grad-CAM của LeakCNN2D dùng (~2.5 kHz). v7: CỐ ĐỊNH trước (pre-registered).
    #         Nếu bật auto, dải được mở để bao TOP-3 dải mang thông tin, không bao giờ hạ thấp.
    low_band_hz: float = 2800.0
    low_band_auto: bool = False
    mfcc_norm: str = "per_coeff"   # [V6-T7] "per_coeff" (CMVN) | "global"
    specaug_before_norm: bool = True   # [V6-T6] che TRƯỚC khi chuẩn hoá (che = im lặng)

    # ---- Huấn luyện / Training -----------------------------------------
    n_splits: int = 5
    epochs: int = 24               # [V6-T1] trần epoch
    sched_horizon_epochs: int = 24  # [V6-T1] lịch LR hoàn tất trong ngần này epoch
    lr_schedule: str = "cosine"    # [V6-T1] "cosine" (warmup + cosine về ~0) | "onecycle"
    warmup_frac: float = 0.10      # tỉ lệ epoch dành cho warm-up
    batch_size: int = 64           # mỗi GPU (mỗi fold chạy trọn trên một GPU)
    lr: float = 3e-4
    record_balanced_loss: bool = False  # [V7] trọng số 1/n_frame(bản ghi): mỗi BẢN GHI đóng góp như nhau
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    early_stop_patience: int = 8
    grad_clip: float = 5.0
    amp: bool = True
    # [V6-T4] v5 tự tắt AMP toàn cục khi thấy 3/11.3M gradient không hữu hạn ở một
    # batch chẩn đoán. GradScaler vốn đã tự bỏ qua step đó -> mặc định GIỮ AMP.
    amp_policy: str = "keep"       # "keep" | "force_off"
    num_workers: int = 2
    seed: int = 42
    # [V7-S1] CV LẶP: mỗi seed = một phân hoạch fold MỚI + khởi tạo trọng số mới.
    #         Seed đầu là lần chạy chính (checkpoint, diễn giải, triển khai); trung bình OOF qua
    #         các lần lặp là ước lượng ỔN ĐỊNH NHẤT và vẫn trung thực (mỗi dự đoán đến từ mô
    #         hình chưa từng thấy bản ghi đó).
    seeds: tuple = (42, 43, 44)
    deterministic_run: bool = False  # [V6-T5] True -> chạy tuần tự + cudnn.deterministic

    # ---- Tăng cường dữ liệu / Augmentation (chỉ áp dụng cho train) ------
    # [V6-T2] ĐÃ BỎ aug_gain_p/aug_gain_db: front-end chuẩn hoá theo từng mẫu nên
    # mọi phép nhân gain bị triệt tiêu HOÀN TOÀN (đo được sai khác 1.2e-7).
    aug_noise_p: float = 0.5
    aug_noise_snr_db: tuple = (10.0, 30.0)
    aug_fir_p: float = 0.5         # [V6] đáp ứng kênh ngẫu nhiên: bền vững với điểm khảo sát/cảm biến
    aug_fir_taps: int = 48
    aug_shift_p: float = 0.5
    aug_polarity_waveform_only: bool = True   # [V6] đảo cực là no-op với phổ biên độ
    specaug_freq_mask: int = 8
    specaug_time_mask: int = 8

    # ---- Cách chia fold ---------------------------------------------------
    # Mỗi bản ghi chỉ mang MỘT nhãn nên nhóm là "thuần nhãn": chia phân tầng trực
    # tiếp trên BẢNG BẢN GHI rồi trải về frame cho kết quả cân bằng chính xác.
    fold_mode: str = "record_stratified"   # "record_stratified" | "frame_group"
                                           # (leave-one-site-out xem mục 9.6)

    # ---- Giao thức LỰA CHỌN (chống lạc quan do selection bias) -----------
    nested_selection: bool = True
    inner_frac: float = 0.25        # tỉ lệ bản ghi của train dành cho validation-trong
    selection_ablation: bool = True

    # ---- Đa GPU & tốc độ / Multi-GPU and throughput ----------------------
    # KHÔNG dùng nn.DataParallel: nó chia batch làm đôi nên thống kê BatchNorm
    # chỉ phản ánh một nửa dữ liệu. Thay bằng chạy song song các FOLD trên 2 GPU.
    parallel_folds: bool = True
    max_parallel_folds: int = 2     # số fold chạy đồng thời (= số GPU dùng cho hàng đợi fold)
    gpu_resident: bool = True
    gpu_augment: bool = True
    gpu_resident_max_gb: float = 6.0
    ckpt_fp16: bool = True

    # ---- Mô hình cần đánh giá / Models to benchmark ----------------------
    models: tuple = ("LeakCNN2D", "LeakCNN1D", "TransferCNN", "MultiBranchCNN", "CRNN")
    transfer_backbone: str = "resnet18"   # hoặc 'mobilenet_v3_small'
    transfer_min_size: int = 128          # [V7] phóng log-Mel lên >=128 -> bản đồ 4x4 thay vì 2x2
    cam_preference: tuple = ("LeakCNN2D", "CRNN", "MultiBranchCNN")

    # ---- [V6] Thống kê suy luận ------------------------------------------
    # Hàng nghìn frame chỉ đến từ vài chục BẢN GHI độc lập: mọi khoảng tin cậy phải
    # bootstrap theo CỤM bản ghi, nếu không CI sẽ hẹp hơn thực tế gần một bậc độ lớn.
    bootstrap_n: int = 2000
    bootstrap_resample_unit: str = "record"   # "record" (đúng) | "frame" (chỉ để so sánh)
    # [V7-S2] Giảm "ngã rẽ phân tích": quy tắc gộp mức bản ghi được ĐĂNG KÝ TRƯỚC (p_mean trên
    #         xác suất THÔ). Bản có hiệu chỉnh Platt và bản chọn quy tắc trên validation-trong
    #         vẫn được in ra, nhưng chỉ là phân tích độ nhạy. (v6: Platt làm AUC bản ghi của
    #         MultiBranchCNN giảm 0.856 -> 0.801.)
    calibrate_aggregation: bool = False
    record_agg: str = "p_mean"                # p_mean | p_med | p_p90 | frac_pos
    agg_selection: str = "fixed"              # "fixed" (đăng ký trước) | "inner" (chọn trên validation-trong)
    ensemble_matrix: bool = True              # dựng ma trận [K fold x N frame] cho mô hình triển khai

    # ---- [V6-M11] Baseline cổ điển & ablation tuỳ chọn -------------------
    run_baseline_models: bool = True    # LogReg/SVM/GBM trên đặc trưng dải tần - câu hỏi "CNN có thêm giá trị?"
    baseline_models: tuple = ("logreg", "svm", "gbm")
    # [V7-S3] thêm bộ đặc trưng MẠNH hơn: thống kê log-Mel (mean+std theo thời gian của 64 dải,
    #         đã trừ mức trung bình -> bất biến với gain, giống front-end của CNN).
    baseline_feature_sets: tuple = ("spectral13", "logmel_stats")
    run_confound_checks: bool = True    # kiểm tra confound mức năng lượng + leave-one-site-out
    run_loso_full: bool = False         # [V6-P0] chạy hẳn leave-one-site-out (tốn thêm 1 chu kỳ)
    run_leakage_ablation: bool = True
    # [V7-A1] Ablation cấu hình tín hiệu/huấn luyện, BẬT mặc định (mỗi biến thể = 1 chu kỳ CV của
    #         MỘT mô hình). Biến thể được "chọn" bằng AUC trên validation-TRONG -> không thiên lệch.
    run_signal_ablation: bool = True
    ablation_model: str = "LeakCNN2D"         # cố định TRƯỚC, không chọn theo kết quả
    ablation_variants: tuple = ("no_fir", "rec_balanced", "sr16k", "frame4s", "frame1s",
                                "no_notch", "no_highpass")

    # ---- Đầu ra / Outputs -----------------------------------------------
    out_dir: str = "/kaggle/working"
    fig_dpi: int = 300
    save_models: bool = True

    # ---- Chế độ thử nhanh / Smoke test ----------------------------------
    smoke_test: bool = False


cfg = CFG()
if os.environ.get("LEAK_SMOKE") == "1":
    cfg.smoke_test = True
# [V7] Cho phép chạy ngoài Kaggle mà không sửa mã
if os.environ.get("LEAK_DATA") is not None:
    cfg.data_root = os.environ["LEAK_DATA"]
if os.environ.get("LEAK_OUT"):
    cfg.out_dir = os.environ["LEAK_OUT"]
elif not Path("/kaggle").exists():
    cfg.out_dir = str(Path("./leak_outputs").resolve())

if cfg.smoke_test:
    print("!! SMOKE TEST MODE: epochs=3, folds=2 - chỉ dùng để kiểm tra pipeline.")
    cfg.epochs, cfg.sched_horizon_epochs, cfg.n_splits = 3, 3, 2
    cfg.early_stop_patience, cfg.min_epochs = 99, 1
    cfg.num_workers, cfg.bootstrap_n = 0, 200
    cfg.seeds = (cfg.seed, cfg.seed + 1)              # vẫn chạy thử nhánh CV lặp
    cfg.ablation_variants = ("no_fir", "sr16k")       # vẫn chạy thử nhánh ablation
    cfg.baseline_models = ("logreg",)
    cfg.ensemble_matrix = False

# [V6-T1] Đỉnh lịch LR - dùng để chặn việc chọn checkpoint trong pha warm-up.
SCHED_TOTAL = int(cfg.sched_horizon_epochs)
if cfg.lr_schedule == "cosine":
    LR_PEAK_EPOCH = max(1, int(math.ceil(cfg.warmup_frac * SCHED_TOTAL)))
else:
    LR_PEAK_EPOCH = max(1, int(math.ceil(0.25 * SCHED_TOTAL)))
MIN_EPOCHS_EFF = (max(cfg.min_epochs, LR_PEAK_EPOCH + 2)
                  if cfg.min_epochs_after_lr_peak else cfg.min_epochs)
# [V7-B2] Epoch sớm nhất được phép CHỌN làm checkpoint (thực thi trong train_one_fold)
SELECT_FROM_EPOCH = int(LR_PEAK_EPOCH) if cfg.select_after_lr_peak else 0
assert cfg.epochs > SELECT_FROM_EPOCH, "cfg.epochs phải lớn hơn epoch đỉnh LR"

# [V6-T5] Chạy tất định cho lần chốt số: tuần tự + cudnn.deterministic
if cfg.deterministic_run:
    cfg.parallel_folds = False
    print("!! cfg.deterministic_run=True -> chạy TUẦN TỰ, cudnn.deterministic=True (chậm hơn ~2x).")
if not torch.cuda.is_available():
    cfg.parallel_folds = False
if cfg.amp_policy == "force_off":
    cfg.amp = False
    print("!! cfg.amp_policy='force_off' -> huấn luyện fp32.")

# Thư mục đầu ra
OUT = Path(cfg.out_dir)
FIG_DIR   = OUT / "figures";  FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR   = OUT / "tables";   TAB_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR  = OUT / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUT / "cache";    CACHE_DIR.mkdir(parents=True, exist_ok=True)

FRAME_LEN = int(cfg.frame_sec * cfg.target_sr)
HOP_LEN   = int(cfg.hop_sec   * cfg.target_sr)

print(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cfg).items()},
                 indent=2, ensure_ascii=False))
print(f"\nFRAME_LEN = {FRAME_LEN} samples ({cfg.frame_sec}s) | HOP = {HOP_LEN} samples")
print(f"Lịch LR     : {cfg.lr_schedule} trên {SCHED_TOTAL} epoch | đỉnh ở epoch {LR_PEAK_EPOCH} "
      f"| min_epochs hiệu dụng {MIN_EPOCHS_EFF} | chỉ chọn checkpoint từ epoch {SELECT_FROM_EPOCH}")
print(f"CV lặp      : {len(cfg.seeds)} lần (seeds {list(cfg.seeds)}) | quy tắc gộp: "
      f"{cfg.record_agg} ({cfg.agg_selection}) | Platt: {cfg.calibrate_aggregation}")
print(f"Thư mục ra  : {OUT}")

# ---- Cảnh báo cấu hình dễ gây mất thông tin ----
if cfg.f_min > 100:
    print(f"\n!! CẢNH BÁO: cfg.f_min = {cfg.f_min:.0f} Hz. Thang Mel sẽ KHÔNG NHÌN THẤY gì")
    print("   dưới mức đó. Hãy xem mục 3.5 (AUC theo từng dải) trước khi quyết định.")
if cfg.frame_sec < 1.5:
    print(f"\n!! LƯU Ý: cửa sổ {cfg.frame_sec}s cho {int(cfg.frame_sec*cfg.target_sr/cfg.hop_length)} "
          f"khung STFT - khá ngắn để ước lượng ổn định một thành phần dải hẹp.")
if cfg.site_source == "none":
    print("\n!! CHƯA CÓ THÔNG TIN ĐIỂM KHẢO SÁT (cfg.site_source='none').")
    print("   Vì vậy KHÔNG thể loại trừ confound 'mô hình học vị trí thu'.")
    print("   Mục 9.6 sẽ đo các dấu hiệu confound gián tiếp (mức năng lượng, baseline dải tần).")


In [ ]:
# =========================================================
# 1.3 Tái lập & phong cách hình vẽ / Reproducibility and plot style
#     [V6-T5] Thêm chế độ tất định và bộ sinh số ngẫu nhiên CỤC BỘ cho từng fold.
# =========================================================
def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        # benchmark=True nhanh hơn nhưng thuật toán conv không tất định -> kết quả
        # đổi giữa các lần chạy. Chỉ dùng khi KHÔNG cần con số chốt.
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    return int(seed)

def make_generator(device, seed: int):
    # [V6-T5] Mọi phép ngẫu nhiên lúc huấn luyện nên dùng generator RIÊNG của fold
    # thay vì RNG toàn cục (RNG toàn cục bị set_seed của thread khác ghi đè).
    g = torch.Generator(device=torch.device(device).type)
    g.manual_seed(int(seed) % (2 ** 32))
    return g

set_seed(cfg.seed, deterministic=cfg.deterministic_run)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": cfg.fig_dpi, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": ":",
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.facecolor": "white",
})
# Bảng màu nhất quán, an toàn với người mù màu / colour-blind safe palette
PAL = {"leak": "#C8553D", "noleak": "#2E6F95", "acc": "#3F7D20",
       "aux": "#8C6BB1", "warn": "#E0A458", "grey": "#6E7B8B"}
MODEL_COLORS = {"LeakCNN2D": "#2E6F95", "LeakCNN1D": "#C8553D", "TransferCNN": "#3F7D20",
                "MultiBranchCNN": "#8C6BB1", "CRNN": "#E0A458"}

FIG_INDEX = {}
def savefig(fig, name, caption=""):
    # Lưu hình ở cả PNG (xem nhanh) và PDF (đưa vào manuscript)
    n = len(FIG_INDEX) + 1
    stem = f"Fig{n:02d}_{name}"
    fig.savefig(FIG_DIR / f"{stem}.png")
    fig.savefig(FIG_DIR / f"{stem}.pdf")
    FIG_INDEX[stem] = caption
    plt.show()
    if caption:
        display(Markdown(f"**Figure {n}.** {caption}"))
    plt.close(fig)
    return stem

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| seed:", cfg.seed, "| deterministic:", cfg.deterministic_run)


## 2. Đọc và kiểm kê dữ liệu / Data discovery and inventory

Notebook tự dò cấu trúc thư mục trong `/kaggle/input`. Quy ước nhãn:

- `leak` → **1** (có rò rỉ)
- `noleak` / `no_leak` / `normal` / `background` → **0** (không rò rỉ)

Mỗi tệp `.wav` được coi là **một bản ghi hiện trường độc lập** (`record_id`). Đây chính là **đơn vị nhóm (group)** trong cross-validation — mọi frame sinh ra từ cùng một bản ghi luôn nằm cùng một phía train/validation.


In [ ]:
# =========================================================
# 2.1 Dò dataset / Auto-discover dataset root
#     [V6-P2] Chuẩn hoá Unicode + ưu tiên THƯ MỤC trước tên tệp khi gán nhãn.
#     [V7-D1] Khớp nhãn theo TỪ (ranh giới "_", ".", chữ số) thay vì chuỗi con; dò thư mục
#             dữ liệu chọn thư mục SÂU NHẤT chứa cả leak/ và noleak/, và báo rõ khi
#             cfg.data_root không tồn tại (v6 âm thầm rơi về quét cả /kaggle/input/datasets).
# =========================================================
def _norm_token(s: str) -> str:
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return s.lower().replace("-", "_").replace(" ", "_")

def _key_regex(keys):
    ks = sorted({_norm_token(k) for k in keys}, key=len, reverse=True)
    # khoá phải đứng ở đầu từ; sau nó là hết chuỗi, "_", "." hoặc chữ số (leak01, leak_02)
    return re.compile(r"(?:^|[_.])(?:" + "|".join(map(re.escape, ks)) + r")(?=$|[_.\d])")

_NOLEAK_RE = _key_regex(cfg.noleak_keys)
_LEAK_RE = _key_regex(cfg.leak_keys)

def _label_of_token(s):
    s = _norm_token(s)
    if _NOLEAK_RE.search(s):        # kiểm tra 'noleak' TRƯỚC 'leak'
        return 0
    if _LEAK_RE.search(s):
        return 1
    return None

def _label_tokens(p: Path):
    # Thứ tự xét: thư mục chứa tệp (nếu ưu tiên thư mục) -> tên tệp -> các thư mục cha
    parts = [_norm_token(x) for x in p.parts]
    toks = list(reversed(parts))
    if cfg.label_source_priority.startswith("folder"):
        toks = [_norm_token(p.parent.name)] + toks
    return toks

def _label_from_path(p: Path):
    # Trả về (nhãn, token quyết định) để truy vết được nguồn gán nhãn.
    for s in _label_tokens(p):
        lab = _label_of_token(s)
        if lab is not None:
            return lab, s
    return None, None

# --- tự kiểm tra bộ gán nhãn (chạy mỗi lần, tốn vài micro giây) ---
for _t, _want in [("leak", 1), ("noleak", 0), ("no_leak", 0), ("No-Leak", 0), ("R001_leak01", 1),
                  ("R12_noleak_03", 0), ("rò_rỉ", 1), ("khong_ro_ri", 0), ("normal", 0),
                  ("current_error", None), ("field_rec_8k_V7.2", None), ("leakage", None)]:
    assert _label_of_token(_t) == _want, f"bộ gán nhãn sai với '{_t}': {_label_of_token(_t)} != {_want}"

def _dirs_with_both(base: Path, max_depth=6):
    # Mọi thư mục (tính cả base) có thư mục con mang nhãn leak VÀ thư mục con mang nhãn noleak.
    out = []
    if not base.exists():
        return out
    stack = [(base, 0)]
    while stack:
        d, depth = stack.pop()
        try:
            kids = [k for k in d.iterdir() if k.is_dir()]
        except Exception:
            continue
        labs = {_label_of_token(k.name) for k in kids}
        if {0, 1}.issubset(labs):
            out.append(d)
        if depth < max_depth:
            stack += [(k, depth + 1) for k in kids]
    return sorted(out, key=lambda p: (-len(p.parts), str(p)))

def _audio_in(root: Path):
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix in cfg.audio_ext)

def discover_root(explicit: str = ""):
    if explicit:
        ex = Path(explicit)
        if ex.exists():
            files = _audio_in(ex)
            if files and {0, 1}.issubset({_label_from_path(p)[0] for p in files}):
                return ex, files
            print(f"!! cfg.data_root = {ex} tồn tại nhưng KHÔNG chứa đủ hai lớp leak/noleak.")
        else:
            print(f"!! cfg.data_root = {ex} KHÔNG TỒN TẠI -> tự dò trong /kaggle/input.")
    cands = []
    for base in [Path("/kaggle/input"), Path("./data")]:
        cands += _dirs_with_both(base)
    if not cands:
        return None, []
    if len(cands) > 1:
        # ưu tiên ứng viên có tên trùng với phần cuối của cfg.data_root
        tail = Path(explicit).name.lower() if explicit else ""
        cands.sort(key=lambda p: (0 if tail and p.name.lower() == tail else 1, -len(p.parts), str(p)))
        print(f"!! Tìm thấy {len(cands)} thư mục chứa leak/noleak; dùng thư mục đầu tiên. Danh sách:")
        for c in cands[:10]:
            print("     -", c)
    root = cands[0]
    return root, _audio_in(root)

DATA_ROOT, AUDIO_FILES = discover_root(cfg.data_root)
SYNTHETIC = DATA_ROOT is None

if SYNTHETIC:
    print("!! KHÔNG tìm thấy dataset leak/noleak -> chuyển sang SYNTHETIC DEMO MODE.")
    print("!! No leak/noleak dataset found -> running in SYNTHETIC DEMO MODE.")
else:
    print("Dataset root :", DATA_ROOT)
    print("Audio files  :", len(AUDIO_FILES))


In [ ]:
# =========================================================
# 2.2 Chế độ mô phỏng / Synthetic demo data
#     Chỉ chạy khi KHÔNG tìm thấy dataset thật. Mô phỏng đặc tính vật lý
#     của ống PVC áp lực thấp: rò rỉ = nhiễu băng hẹp 150-600 Hz cộng nền,
#     không rò = nhiễu hồng + hài lưới điện 50 Hz.
#     [V6-P2] Nếu không xoá được thư mục cũ (file đang bị giữ / OneDrive),
#     dùng thư mục mới thay vì làm chết cả notebook.
# =========================================================
def _synthesise_dataset(root="/kaggle/working/synthetic_demo", n_rec_per_class=7,
                        files_per_rec=2, dur=24.0, sr=8000, seed=0, witness=0.4):
    # Mô phỏng ĐÚNG cấu trúc dữ liệu thực tế:
    #   - mỗi BẢN GHI Rxxx được cắt thành nhiều TỆP  (R001_seg01.wav, R001_seg02.wav)
    #   - trong bản ghi nhãn leak, tiếng rò rỉ chỉ hiện diện ~witness thời lượng
    #     (phần còn lại bị nhiễu nền/transient che) -> nhãn frame là NHÃN NHIỄU
    rng = np.random.default_rng(seed)
    root = Path(root)
    if root.exists():
        try:
            shutil.rmtree(root)
        except Exception as e:
            root = Path(str(root) + "_" + time.strftime("%H%M%S"))
            print(f"  !! không xoá được thư mục mô phỏng cũ ({type(e).__name__}) "
                  f"-> dùng thư mục mới: {root.name}")
    for cls in ("leak", "noleak"):
        (root / cls).mkdir(parents=True, exist_ok=True)
    n = int(dur * sr); t = np.arange(n) / sr

    def pink(n):
        w = rng.standard_normal(n); X = np.fft.rfft(w)
        f = np.fft.rfftfreq(n, 1/sr); f[0] = f[1]
        return np.fft.irfft(X / np.sqrt(f), n)

    rid = 0
    for cls in ("leak", "noleak"):
        for _ in range(n_rec_per_class):
            rid += 1
            rec_name = f"R{rid:03d}"
            f0 = rng.uniform(180, 550); bw = rng.uniform(60, 160)   # cố định trong 1 bản ghi
            for seg in range(1, files_per_rec + 1):
                x = pink(n); x /= (np.std(x) + 1e-12)
                for h in range(1, 4):                                # hài lưới 50 Hz
                    x += 0.05 * rng.uniform(.4, 1.) * np.sin(2*np.pi*cfg.mains_hz*h*t + rng.uniform(0, 6.28))
                if cls == "leak":
                    b, a = sps.butter(4, [max(20., f0-bw)/(sr/2), min(sr/2-1., f0+bw)/(sr/2)], btype="band")
                    leak = sps.lfilter(b, a, rng.standard_normal(n))
                    leak /= (np.std(leak) + 1e-12)
                    # mặt nạ hiện diện: chỉ ~witness thời lượng nghe được tiếng rò
                    mask = np.zeros(n)
                    n_int = int(rng.integers(2, 5))
                    for _ in range(n_int):
                        L = int(witness * n / n_int)
                        s0 = int(rng.integers(0, max(1, n - L)))
                        mask[s0:s0+L] = 1.0
                    mask = np.convolve(mask, np.hanning(int(.2*sr))/np.sum(np.hanning(int(.2*sr))), "same")
                    x = x + leak * mask * (10 ** (rng.uniform(-2, 8) / 20.0))
                for _ in range(rng.integers(2, 6)):                  # transient xe cộ
                    s0 = int(rng.integers(0, n - sr))
                    x[s0:s0+sr] += 3.0 * np.exp(-np.arange(sr)/(sr*.05)) * rng.standard_normal(sr)
                x = x / (np.max(np.abs(x)) + 1e-9) * 0.9
                sf.write(root / cls / f"{rec_name}_seg{seg:02d}.wav", x.astype(np.float32), sr)
    return root

if SYNTHETIC:
    DATA_ROOT = _synthesise_dataset(root=str(OUT / "synthetic_demo"))   # [V7] không ghi cứng /kaggle
    AUDIO_FILES = [p for p in DATA_ROOT.rglob("*") if p.suffix in cfg.audio_ext]
    print("Synthetic dataset :", DATA_ROOT, "|", len(AUDIO_FILES), "files")


In [ ]:
# =========================================================
# 2.3 Bảng kiểm kê + TRÍCH MÃ BẢN GHI Rxxx + ĐIỂM KHẢO SÁT
#     Mã bản ghi là đơn vị nhóm của cross-validation VÀ đơn vị đánh giá
#     mức bản ghi. Trích sai -> rò rỉ dữ liệu ở mọi bước sau.
# =========================================================
_REC_RE = re.compile(cfg.record_regex, re.IGNORECASE)

def extract_record_id(path: Path):
    if cfg.record_id_mode == "filename":
        return path.stem, "filename"
    if cfg.record_id_mode == "parent_dir":
        return path.parent.name, "parent_dir"
    m = _REC_RE.search(path.stem)
    if m: return m.group(1).upper(), "regex:stem"
    m = _REC_RE.search(str(path.parent))
    if m: return m.group(1).upper(), "regex:parent"
    return path.stem, "fallback:stem"

def is_derivative(stem: str):
    low = stem.lower()
    return any(k in low for k in cfg.derivative_markers)

# ---- [V6-P0] Nạp bản đồ điểm khảo sát (nếu có) ------------------------
SITE_MAP, site_map_rows = {}, 0
if cfg.site_source == "csv":
    _sp = Path(cfg.site_csv)
    assert _sp.exists(), f"cfg.site_csv không tồn tại: {_sp}"
    _sm = pd.read_csv(_sp)
    assert {"record_id", "site"}.issubset(_sm.columns), "CSV điểm khảo sát cần 2 cột: record_id, site"
    SITE_MAP = {str(a).strip().upper(): str(b).strip() for a, b in
                zip(_sm.record_id, _sm.site)}
    site_map_rows = len(SITE_MAP)
    print(f"Đã nạp bản đồ điểm khảo sát: {site_map_rows} bản ghi -> {len(set(SITE_MAP.values()))} điểm")

def extract_site(path: Path, rid: str):
    if cfg.site_source == "csv":
        return SITE_MAP.get(str(rid).upper(), None)
    if cfg.site_source == "regex" and cfg.site_regex:
        m = re.search(cfg.site_regex, str(path), re.IGNORECASE)
        if m:
            return m.group(cfg.site_regex_group).upper()
    return None

rows = []
for p_ in sorted(AUDIO_FILES):
    lab, tok = _label_from_path(p_)
    if lab is None:
        continue
    try:
        info = sf.info(str(p_))
        dur, sr_native, ch = info.duration, info.samplerate, info.channels
    except Exception as e:
        print("  [bỏ qua]", p_.name, e); continue
    rid, how = extract_record_id(p_)
    rows.append(dict(path=str(p_), file_stem=p_.stem, record_id=rid, rid_source=how,
                     label_token=tok, derivative=is_derivative(p_.stem),
                     site=extract_site(p_, rid),
                     label=lab, cls="leak" if lab == 1 else "noleak",
                     duration_s=dur, sr_native=sr_native, channels=ch, folder=p_.parent.name))

files = pd.DataFrame(rows).sort_values(["cls", "record_id", "file_stem"]).reset_index(drop=True)
assert len(files) > 0, "Không đọc được tệp âm thanh nào / no audio files parsed"
files.to_csv(OUT / "manifest_files.csv", index=False)

n_fallback = int((files.rid_source == "fallback:stem").sum())
print(f"Số tệp                 : {len(files)}")
print(f"Số mã bản ghi (Rxxx)   : {files.record_id.nunique()}")
print(f"Trích mã bằng          : {files.rid_source.value_counts().to_dict()}")
print(f"Nguồn gán nhãn         : {files.label_token.value_counts().head(6).to_dict()}")
if n_fallback:
    print(f"!! {n_fallback} tệp KHÔNG khớp regex '{cfg.record_regex}' -> tạm lấy tên tệp làm mã.")
    display(files.loc[files.rid_source == 'fallback:stem', ['file_stem', 'cls']].head(15))
files.head(8)


In [ ]:
# =========================================================
# 2.3b KIỂM TRA CHẤT LƯỢNG ĐỊNH DANH - [V6-P2] DỪNG THẬT nếu có mâu thuẫn
# =========================================================
problems, warnings_ = [], []

# (1) Một mã bản ghi mang HAI nhãn khác nhau -> mâu thuẫn nhãn, phải sửa dữ liệu
conf = files.groupby("record_id").cls.nunique()
conflicts = conf[conf > 1].index.tolist()
if conflicts:
    problems.append(f"{len(conflicts)} mã bản ghi có nhãn mâu thuẫn")
    print("!! MÂU THUẪN NHÃN - cùng một mã bản ghi xuất hiện ở cả leak và noleak:")
    display(files[files.record_id.isin(conflicts)][["record_id", "cls", "file_stem", "label_token"]])
    print("   Hãy sửa dữ liệu hoặc chỉnh cfg.record_regex trước khi chạy tiếp.\n")

# (2) Một mã bản ghi lấy từ HAI nguồn khác nhau -> định danh không đáng tin
if files.groupby("record_id").rid_source.nunique().max() > 1:
    warnings_.append("có bản ghi được trích mã bằng nhiều cách khác nhau")
    print("!! ĐỊNH DANH KHÔNG NHẤT QUÁN - cùng một mã bản ghi đến từ nhiều nguồn trích:")
    display(files[files.record_id.isin(
        files.groupby("record_id").rid_source.nunique().pipe(lambda s: s[s > 1]).index)]
        [["record_id", "rid_source", "file_stem"]])

# (3) Phân bố số tệp trên mỗi mã bản ghi
fpr = files.groupby("record_id").size()
multi = fpr[fpr > 1]
print(f"Số tệp / bản ghi: trung vị {int(fpr.median())}, min {fpr.min()}, max {fpr.max()}")
print(f"Bản ghi gồm NHIỀU tệp: {len(multi)} / {len(fpr)}")

# (4) Định lượng mức rò rỉ SẼ xảy ra nếu nhóm theo TÊN TỆP (cách cũ)
n_by_file = files.file_stem.nunique(); n_by_rec = files.record_id.nunique()
if n_by_file > n_by_rec:
    inflate = n_by_file / n_by_rec
    print(f"\n>> Nếu nhóm theo TÊN TỆP: {n_by_file} 'bản ghi giả' thay vì {n_by_rec} bản ghi thật "
          f"(thổi phồng {inflate:.2f}x số đơn vị độc lập).")
    print(f">> {len(multi)} bản ghi sẽ bị xé lẻ giữa các fold -> RÒ RỈ DỮ LIỆU.")
    print(">> Đây chính là con số đáng đưa vào phần Discussion của bài báo.")

# (5) Tệp dẫn xuất (augment / đã xử lý) - phải cùng nhóm với bản ghi mẹ
if cfg.report_derivatives and files.derivative.any():
    d = files[files.derivative]
    print(f"\n{len(d)} tệp có dấu hiệu là bản DẪN XUẤT ({', '.join(cfg.derivative_markers)}):")
    display(d[["record_id", "file_stem", "cls"]].head(12))
    n_bad_deriv = int((~d.record_id.isin(files.record_id)).sum())
    print(f"   -> Tất cả đã cùng mã bản ghi với tệp mẹ (số tệp lệch nhóm: {n_bad_deriv}) -> KHÔNG gây rò rỉ.")
    print("   -> Nhưng chúng làm tăng số frame của bản ghi đó: cân nhắc loại khỏi tập validation.")

# (6) Bản ghi trên mỗi lớp - quyết định số fold tối đa
rec_lab_chk = files.groupby("record_id").cls.first()
print(f"\nSố BẢN GHI ĐỘC LẬP theo lớp: {rec_lab_chk.value_counts().to_dict()}")
n_min = int(rec_lab_chk.value_counts().min())
print(f"-> Số fold tối đa khả dụng: {n_min}")
if n_min < 5:
    warnings_.append(f"lớp hiếm chỉ có {n_min} bản ghi")
    print(f"!! Lớp hiếm chỉ có {n_min} bản ghi độc lập. Mọi khoảng tin cậy sẽ rất rộng;")
    print("   phải nêu rõ trong phần Limitations của bài báo.")

# (7) [V6-P0] Kiểm tra điểm khảo sát
HAS_SITE = files.site.notna().any()
if HAS_SITE:
    n_site = files.site.nunique()
    m = files.groupby("record_id").site.first()
    print(f"\nĐIỂM KHẢO SÁT: {n_site} điểm cho {len(m)} bản ghi.")
    display(pd.crosstab(m, files.groupby('record_id').cls.first()).rename_axis('site'))
    lab_per_site = files.groupby("site").cls.nunique()
    if (lab_per_site < 2).any():
        print("!! CẢNH BÁO CONFOUND: có điểm khảo sát CHỈ chứa một lớp ->")
        print("   mô hình có thể phân biệt ĐIỂM KHẢO SÁT thay vì phân biệt rò rỉ.")
        print("   Bắt buộc chạy mục 9.6 (leave-one-site-out) và nêu rõ trong bài báo.")
else:
    print("\n!! KHÔNG có cột điểm khảo sát -> mục 9.6 chỉ chạy được kiểm tra gián tiếp")
    print("   (mức năng lượng, baseline dải tần). Xem cảnh báo ở mục 15/16.")

if problems:
    msg = "CẦN XỬ LÝ TRƯỚC KHI CHẠY TIẾP: " + "; ".join(problems)
    print("\n=== " + msg + " ===")
    if cfg.strict_checks:
        raise ValueError(msg + " -- sửa dữ liệu, hoặc đặt cfg.strict_checks=False để bỏ qua.")
elif warnings_:
    print("\n=== CẢNH BÁO (không chặn): " + "; ".join(warnings_) + " ===")
else:
    print("\n✓ Mã bản ghi nhất quán, không có mâu thuẫn nhãn.")


In [ ]:
# =========================================================
# 2.3c Bảng kiểm kê MỨC BẢN GHI (đơn vị thống kê thật sự)
# =========================================================
manifest = files.copy()          # mức TỆP - dùng để nạp âm thanh

_agg = dict(cls=("cls", "first"), label=("label", "first"),
            n_files=("file_stem", "size"), duration_s=("duration_s", "sum"),
            n_derivative=("derivative", "sum"),
            sr=("sr_native", lambda s: sorted(set(s))[0]))
if HAS_SITE:
    _agg["site"] = ("site", "first")
records = (files.groupby("record_id").agg(**_agg).reset_index())
records.to_csv(OUT / "manifest_records.csv", index=False)

summary = (records.groupby("cls")
           .agg(n_records=("record_id", "nunique"), n_files=("n_files", "sum"),
                total_min=("duration_s", lambda s: s.sum()/60),
                mean_s=("duration_s", "mean"), min_s=("duration_s", "min"),
                max_s=("duration_s", "max")).round(2))
display(Markdown("### Table 1. Recording-level inventory"))
display(summary)
print(f"\nBản ghi độc lập : {len(records)}   <- ĐƠN VỊ THỐNG KÊ của mọi kết luận")
print(f"Tệp âm thanh    : {len(files)}")
print(f"Tổng thời lượng : {records.duration_s.sum()/60:.1f} phút")
print(f"Tần số lấy mẫu  : {sorted(files.sr_native.unique())}")
if HAS_SITE:
    print(f"Điểm khảo sát   : {records.site.nunique()}")
summary.to_csv(TAB_DIR / "table1_inventory.csv")


In [ ]:
# =========================================================
# 2.4 Hình 1: Tổng quan dữ liệu / Figure 1: dataset overview
# =========================================================
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))

cnt = records.cls.value_counts()
ax[0].bar(cnt.index, cnt.values, color=[PAL[c] for c in cnt.index], width=.55)
for i, v in enumerate(cnt.values):
    ax[0].text(i, v, str(v), ha="center", va="bottom", fontweight="bold")
ax[0].set_title("(a) Independent recordings per class"); ax[0].set_ylabel("count")

for c in ["noleak", "leak"]:
    d = records.loc[records.cls == c, "duration_s"]
    ax[1].hist(d, bins=min(15, max(4, len(d)//2)), alpha=.65, label=c, color=PAL[c])
ax[1].set_title("(b) Recording duration"); ax[1].set_xlabel("seconds"); ax[1].legend()

tot = records.groupby("cls").duration_s.sum() / 60
ax[2].bar(tot.index, tot.values, color=[PAL[c] for c in tot.index], width=.55)
for i, v in enumerate(tot.values):
    ax[2].text(i, v, f"{v:.1f}", ha="center", va="bottom", fontweight="bold")
ax[2].set_title("(c) Total duration"); ax[2].set_ylabel("minutes")

fig.suptitle("Dataset overview — field recordings, low-pressure PVC pipes (1–2 bar)", y=1.06)
savefig(fig, "dataset_overview",
        "Field dataset overview: number of independent recordings, duration distribution and "
        "total duration per class.")


## 3. Phân tích tín hiệu thăm dò / Exploratory signal analysis

Mục tiêu: **chứng minh bằng dữ liệu** rằng dải tần mang thông tin rò rỉ trên ống PVC áp lực thấp nằm ở vùng tần số thấp, làm cơ sở cho thiết kế bộ lọc và thang Mel ở các bước sau (đây là nội dung Mục *Materials & Signal characterisation* của bài báo).


In [ ]:
# =========================================================
# 3.1 Nạp & tiền xử lý một tệp / Load and pre-process one file
#     [V6] Tham số hoá high-pass / notch để phục vụ ablation tiền xử lý (mục 9.7).
# =========================================================
_HP_CACHE, _NOTCH_CACHE = {}, {}

def _hp_coeffs(sr, fc):
    key = (int(sr), float(fc))
    if key not in _HP_CACHE:
        _HP_CACHE[key] = sps.butter(4, float(fc) / (sr / 2), btype="high")
    return _HP_CACHE[key]

def design_notches(sr, f0, n_harm, q):
    key = (int(sr), float(f0), int(n_harm), float(q))
    if key not in _NOTCH_CACHE:
        out = []
        for h in range(1, n_harm + 1):
            f = f0 * h
            if f < sr / 2 - 10:
                out.append(sps.iirnotch(f / (sr / 2), q))
        _NOTCH_CACHE[key] = out
    return _NOTCH_CACHE[key]

_NOTCHES = design_notches(cfg.target_sr, cfg.mains_hz, cfg.mains_harmonics, cfg.notch_q)
_HP_B, _HP_A = _hp_coeffs(cfg.target_sr, cfg.highpass_hz)

LOAD_REPAIRED = []      # các tệp phải sửa NaN/Inf khi đọc

def load_audio(path, sr=None, apply_filters=True, highpass=None, notch=None):
    # highpass=None -> dùng cfg.highpass_hz ; notch=None -> dùng cfg.notch_mains
    sr = sr or cfg.target_sr
    hp = cfg.highpass_hz if highpass is None else float(highpass)
    do_notch = cfg.notch_mains if notch is None else bool(notch)
    try:
        x, _ = librosa.load(str(path), sr=sr, mono=True)
    except Exception as e:
        # librosa từ chối tệp có mẫu NaN/Inf ("Audio buffer is not finite everywhere").
        y, sr0 = sf.read(str(path), always_2d=True, dtype="float32")
        n_bad = int((~np.isfinite(y)).sum())
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).mean(axis=1)
        x = librosa.resample(y, orig_sr=sr0, target_sr=sr) if sr0 != sr else y
        LOAD_REPAIRED.append(dict(path=str(path), n_nonfinite=n_bad, reason=type(e).__name__))
        print(f"  !! {Path(path).name}: {n_bad} mẫu NaN/Inf đã được vá về 0.")
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    x = x - np.mean(x)                                    # khử DC
    if apply_filters:
        if hp and hp > 0:
            b, a = _hp_coeffs(sr, hp)
            x = sps.filtfilt(b, a, x).astype(np.float32)   # thông cao, pha bằng 0
        if do_notch:
            for b, a in design_notches(sr, cfg.mains_hz, cfg.mains_harmonics, cfg.notch_q):
                x = sps.filtfilt(b, a, x).astype(np.float32)   # khử hài 50 Hz
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    # [V7] chuẩn hoá đỉnh theo TỪNG TỆP (v6 ghi nhầm là "theo bản ghi").
    #      peak_norm_pct=100 -> max|x| như v6; <100 -> phân vị, bền với một xung nhiễu đơn lẻ.
    m = (float(np.max(np.abs(x))) if cfg.peak_norm_pct >= 100
         else float(np.percentile(np.abs(x), cfg.peak_norm_pct)))
    if m > 1e-9:
        x = x / m
    return x.astype(np.float32)

t0 = time.time()
_probe = load_audio(manifest.path.iloc[0])
print(f"Kiểm thử nạp 1 tệp: {len(_probe)/cfg.target_sr:.1f}s, {time.time()-t0:.2f}s wall")
print("LƯU Ý TRIỂN KHAI: chuẩn hoá đỉnh theo TOÀN BỘ tệp là phép xử lý OFFLINE")
print("(không nhân quả). Pipeline này vì vậy chạy theo lô, không phải thời gian thực.")


In [ ]:
# =========================================================
# 3.2 Hình 2: Dạng sóng & spectrogram đại diện / waveform + spectrogram
# =========================================================
ex = {c: manifest[manifest.cls == c].iloc[0] for c in ["leak", "noleak"]}
fig = plt.figure(figsize=(13, 6.2))
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1.5], hspace=.35, wspace=.18)

for j, c in enumerate(["noleak", "leak"]):
    x = load_audio(ex[c].path)[: int(8 * cfg.target_sr)]
    t = np.arange(len(x)) / cfg.target_sr
    a0 = fig.add_subplot(gs[0, j]); a0.plot(t, x, lw=.4, color=PAL[c])
    a0.set_title(f"{c.upper()} — waveform ({ex[c].record_id})"); a0.set_xlabel("time (s)")
    a0.set_ylabel("amplitude"); a0.set_ylim(-1.05, 1.05)

    S = librosa.amplitude_to_db(np.abs(librosa.stft(x, n_fft=cfg.n_fft,
                                                    hop_length=cfg.hop_length)), ref=np.max)
    a1 = fig.add_subplot(gs[1, j])
    im = librosa.display.specshow(S, sr=cfg.target_sr, hop_length=cfg.hop_length,
                                  x_axis="time", y_axis="linear", ax=a1, cmap="magma")
    a1.set_ylim(0, 2000); a1.set_title(f"{c.upper()} — STFT spectrogram (0–2 kHz)")
    fig.colorbar(im, ax=a1, format="%+2.0f dB", pad=.01)

savefig(fig, "waveform_spectrogram",
        "Representative waveform and time-frequency spectrum of the two classes on field recordings.")


In [ ]:
# =========================================================
# 3.3 PSD trung bình theo lớp (Welch) + năng lượng theo dải tần
#     [V6-M1] GỘP THEO BẢN GHI TRƯỚC KHI KIỂM ĐỊNH.
#     Kiểm định trên TỆP khi đơn vị thống kê là BẢN GHI (các tệp của một bản ghi là các
#     đoạn của cùng một lần thu) là pseudo-replication -> p-value thiên lệch về "có ý nghĩa".
# =========================================================
TRAPZ = getattr(np, "trapezoid", None) or getattr(np, "trapz")

def holm(p):
    # Hiệu chỉnh Holm-Bonferroni (định nghĩa sớm để dùng cho kiểm định dải tần ở mục 3).
    p = np.asarray(p, dtype=float); o = np.argsort(p); m = len(p); adj = np.empty(m)
    run = 0.0
    for r, i in enumerate(o):
        v = (m - r) * p[i]; run = max(run, v); adj[i] = min(1.0, run)
    return adj

BANDS = [(20,100), (100,250), (250,500), (500,800), (800,1500), (1500,2500), (2500,4000)]
BAND_NAMES = [f"{a}-{b}" for a, b in BANDS]

def welch_psd(x, sr=None, nper=2048):
    sr = sr or cfg.target_sr
    f, P = sps.welch(x, fs=sr, nperseg=min(nper, len(x)), noverlap=None, scaling="density")
    return f, P

def band_energies(f, P):
    tot = TRAPZ(P, f) + 1e-20
    return np.array([TRAPZ(P[(f >= a) & (f < b)], f[(f >= a) & (f < b)]) / tot
                     for a, b in BANDS])

psd_store, band_rows = {"leak": [], "noleak": []}, []
psd_by_record, rec_cls_map = defaultdict(list), {}
N_REC_ANALYSE = 400 if not cfg.smoke_test else 20       # số BẢN GHI đưa vào phân tích phổ

# Chọn bản ghi theo lớp (không phải theo tệp) để số lượng hai lớp cân bằng
_rec_order = (records.sort_values(["cls", "record_id"])
              .groupby("cls", group_keys=False).head(max(2, N_REC_ANALYSE // 2)))
_files_by_rec = manifest.groupby("record_id")
FREQ_AXIS, skipped = None, []

for _, r in _rec_order.iterrows():
    rid = r.record_id
    grp = _files_by_rec.get_group(rid)
    Ps = []
    for pth in grp.path:
        x = load_audio(pth)
        if len(x) < 2048:
            skipped.append(rid); continue
        f, P = welch_psd(x)
        if FREQ_AXIS is None:
            FREQ_AXIS = f
        elif len(f) != len(FREQ_AXIS):
            skipped.append(rid); continue
        Ps.append(P)
    if not Ps:
        continue
    P_rec = np.mean(np.vstack(Ps), axis=0)          # <-- gộp TRONG bản ghi trước
    psd_store[r.cls].append(P_rec)
    psd_by_record[rid].append(P_rec); rec_cls_map[rid] = r.cls
    be = band_energies(FREQ_AXIS, P_rec)
    band_rows.append(dict(record_id=rid, cls=r.cls, site=(r.site if HAS_SITE else None),
                          **{n: v for n, v in zip(BAND_NAMES, be)}))

band_df = pd.DataFrame(band_rows)
band_df.to_csv(TAB_DIR / "band_energy_per_record.csv", index=False)

def paired_site_pvalue(band, min_pairs=4):
    # [V6-P0] Kiểm định GHÉP CẶP theo điểm khảo sát: chỉ dùng các điểm có CẢ HAI lớp.
    # Đây là kiểm định duy nhất loại được confound "khác biệt giữa các vị trí thu":
    # nó so sánh leak vs noleak TRONG CÙNG một điểm khảo sát.
    if not HAS_SITE:
        return np.nan
    d = band_df[["site", "cls", band]].dropna()
    piv = d.pivot_table(index="site", columns="cls", values=band, aggfunc="mean")
    if not {"leak", "noleak"}.issubset(set(piv.columns)):
        return np.nan
    piv = piv.dropna()
    if len(piv) < min_pairs:
        return np.nan
    try:
        return float(spstats.wilcoxon(piv["leak"], piv["noleak"], zero_method="zsplit").pvalue)
    except Exception:
        return np.nan

print(f"Phân tích PSD cho {len(band_df)} BẢN GHI "
      f"({int((band_df.cls=='leak').sum())} leak / {int((band_df.cls=='noleak').sum())} no-leak).")
print(f"ĐƠN VỊ THỐNG KÊ của mọi kiểm định từ đây: BẢN GHI (n={len(band_df)}), không phải tệp.")
if skipped:
    print(f"!! Bỏ qua {len(set(skipped))} bản ghi quá ngắn/lệch lưới tần số: {sorted(set(skipped))[:10]}")
if len(band_df) < len(records):
    print(f"!! Độ phủ phân tích phổ: {len(band_df)}/{len(records)} bản ghi "
          f"({100*len(band_df)/len(records):.0f}%) - nêu rõ trong bài báo.")


In [ ]:
# =========================================================
# 3.4 Hình 3: PSD theo lớp + tỉ lệ năng lượng dải + kiểm định Mann-Whitney
#     [V6-M1] Kiểm định ở MỨC BẢN GHI (n = số bản ghi), kèm kiểm định thay thế
#     có tính đến cặp "cùng điểm khảo sát" khi có thông tin site.
# =========================================================
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))

for c in ["noleak", "leak"]:
    A = np.vstack(psd_store[c]); med = np.median(A, 0)
    lo, hi = np.percentile(A, 25, 0), np.percentile(A, 75, 0)
    ax[0].semilogy(FREQ_AXIS, med, color=PAL[c], lw=1.6, label=f"{c} (median, n={A.shape[0]})")
    ax[0].fill_between(FREQ_AXIS, lo, hi, color=PAL[c], alpha=.18)
ax[0].set_xlim(0, 2000); ax[0].set_xlabel("frequency (Hz)")
ax[0].set_ylabel("PSD (V$^2$/Hz)"); ax[0].legend()
ax[0].set_title("(a) Welch PSD per class\n(one curve = one recording; files averaged within recording)")

ratio = (np.median(np.vstack(psd_store["leak"]), 0) /
         (np.median(np.vstack(psd_store["noleak"]), 0) + 1e-20))
ax[1].plot(FREQ_AXIS, 10*np.log10(ratio + 1e-20), color=PAL["aux"], lw=1.4)
ax[1].axhline(0, color="k", lw=.8)
ax[1].set_xlim(0, 2000); ax[1].set_xlabel("frequency (Hz)")
ax[1].set_ylabel("leak / no-leak (dB)")
ax[1].set_title("(b) Spectral contrast")

pos = np.arange(len(BAND_NAMES)); w = .38
pv, pv_site = [], []
for i, n in enumerate(BAND_NAMES):
    a = band_df.loc[band_df.cls == "leak", n]; b = band_df.loc[band_df.cls == "noleak", n]
    try:
        pv.append(spstats.mannwhitneyu(a, b, alternative="two-sided").pvalue)
    except Exception:
        pv.append(np.nan)
    pv_site.append(paired_site_pvalue(n) if HAS_SITE else np.nan)
    ax[2].bar(i - w/2, a.mean(), w, yerr=a.sem(), color=PAL["leak"], capsize=2)
    ax[2].bar(i + w/2, b.mean(), w, yerr=b.sem(), color=PAL["noleak"], capsize=2)
pv_holm = holm(np.nan_to_num(np.asarray(pv, dtype=float), nan=1.0))   # [V7-S4]
for i, p in enumerate(pv_holm):
    if p < .05:
        ax[2].text(i, ax[2].get_ylim()[1]*.92, "*" if p >= .01 else "**",
                   ha="center", fontweight="bold")
ax[2].set_xticks(pos); ax[2].set_xticklabels(BAND_NAMES, rotation=45, ha="right")
ax[2].set_ylabel("relative band energy"); ax[2].set_xlabel("band (Hz)")
ax[2].set_title("(c) Relative band energy\n(Holm-adjusted * p<0.05, ** p<0.01; n = recordings)")
ax[2].legend(handles=[matplotlib.patches.Patch(color=PAL["leak"], label="leak"),
                      matplotlib.patches.Patch(color=PAL["noleak"], label="noleak")],
             fontsize=8, loc="upper right")

savefig(fig, "psd_band_analysis",
        "Spectral characteristics of the two classes, computed PER RECORDING: (a) median Welch PSD "
        "with interquartile range (one observation = one recording, not one file); (b) leak/no-leak "
        "spectral ratio; (c) relative band energy with Mann-Whitney U tests at recording level.")

band_stats = pd.DataFrame({
    "band_Hz": BAND_NAMES,
    "leak_mean": [band_df.loc[band_df.cls=="leak", n].mean() for n in BAND_NAMES],
    "noleak_mean": [band_df.loc[band_df.cls=="noleak", n].mean() for n in BAND_NAMES],
    "p_MannWhitney_record": pv,
    "p_holm": pv_holm,
    "p_paired_site": pv_site,
}).round(5)
band_stats["significant_holm"] = band_stats.p_holm < .05
band_stats["n_records"] = len(band_df)
display(Markdown(f"### Table 2. Band-energy significance tests "
                 f"(unit of analysis = recording, n={len(band_df)})"))
display(band_stats)
band_stats.to_csv(TAB_DIR / "table2_band_energy_stats.csv", index=False)
print(f"Đơn vị thống kê: BẢN GHI (n = {len(band_df)}). Cột p_holm đã hiệu chỉnh {len(BAND_NAMES)} "
      f"kiểm định đồng thời (Holm).")
print(f"Số dải có ý nghĩa SAU hiệu chỉnh: {int(band_stats.significant_holm.sum())}/{len(BAND_NAMES)}")


### 3.5 Dải tần nào thực sự mang thông tin? — đo TRƯỚC khi huấn luyện

Câu hỏi quyết định hai tham số `cfg.f_min` và `cfg.low_band_hz`: **thông tin phân biệt
leak/no-leak nằm ở dải tần nào?**

Cell này trả lời **không cần mô hình nào cả**. Với mỗi bản ghi, lấy năng lượng tương đối
trong từng dải hẹp rồi tính AUC mức bản ghi của **riêng đặc trưng một chiều đó**. Dải nào
có AUC xa 0,5 nhất là dải mang thông tin.

> Đây là kiểm tra nên chạy **trước** mọi quyết định về thang Mel. Nếu `cfg.f_min` được đặt
> cao hơn dải mang thông tin, mô hình **không bao giờ nhìn thấy** phần dữ liệu quan trọng
> nhất — và không kiến trúc nào cứu được điều đó.
>
> **[V6]** Kết quả cell này còn dùng để **kiểm định giả thuyết vật lý** "năng lượng rò rỉ
> tập trung ở dải rất thấp" (xem mục 3.5b). Nếu số liệu không ủng hộ, phần Discussion phải
> viết đúng như vậy.


In [ ]:
# =========================================================
# 3.5 AUC mức bản ghi của từng dải tần (không dùng mô hình)
#     Đơn vị thống kê: BẢN GHI. Kèm CI bootstrap theo cụm + hiệu chỉnh Holm [V7-S4]
#     để tránh kết luận quá mạnh trên vài chục quan sát.
# =========================================================
_recs = sorted(psd_by_record)
_P = np.vstack([np.mean(np.vstack(psd_by_record[r]), axis=0) for r in _recs])
_y = np.array([1 if rec_cls_map[r] == "leak" else 0 for r in _recs])
print(f"Phân tích {len(_recs)} bản ghi ({int(_y.sum())} leak / {int(len(_y)-_y.sum())} no-leak)")

edges = np.unique(np.round(np.geomspace(20, min(4000, cfg.target_sr/2 - 1), 15)).astype(int))
FINE = list(zip(edges[:-1], edges[1:]))
tot = TRAPZ(_P, FREQ_AXIS, axis=1) + 1e-20
rows = []
for a, b in FINE:
    m = (FREQ_AXIS >= a) & (FREQ_AXIS < b)
    if m.sum() < 2: continue
    e = TRAPZ(_P[:, m], FREQ_AXIS[m], axis=1) / tot
    auc = roc_auc_score(_y, e) if len(np.unique(_y)) > 1 else np.nan
    try:
        pv = float(spstats.mannwhitneyu(e[_y == 1], e[_y == 0], alternative="two-sided").pvalue)
    except Exception:
        pv = np.nan
    # CI bootstrap 95% theo CỤM = lấy mẫu lại BẢN GHI (ở đây mỗi hàng đã là 1 bản ghi,
    # nên bootstrap theo hàng chính là bootstrap theo cụm).
    rng = np.random.default_rng(7)
    bs = []
    for _ in range(400):
        i = rng.integers(0, len(_y), len(_y))
        if len(np.unique(_y[i])) < 2: continue
        bs.append(roc_auc_score(_y[i], e[i]))
    lo, hi = (float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))) if bs else (np.nan, np.nan)
    rows.append(dict(band=f"{a}-{b}", f_lo=a, f_hi=b, auc=auc, auc_lo=lo, auc_hi=hi,
                     huong="leak higher" if auc > .5 else "no-leak higher",
                     suc_phan_biet=abs(auc - .5) * 2, p=pv))
band_auc = pd.DataFrame(rows)
band_auc["p_holm"] = holm(band_auc.p.fillna(1.0).values)      # [V7-S4] họ kiểm định đồng thời
band_auc.round(4).to_csv(TAB_DIR / "table2b_band_informativeness.csv", index=False)
display(Markdown("### Table 2b. Informativeness of each frequency band (model-free, n = recordings)"))
display(band_auc.sort_values("suc_phan_biet", ascending=False).round(4).head(8))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.9))
xs = np.arange(len(band_auc))
cols = [PAL["leak"] if a > .5 else PAL["noleak"] for a in band_auc.auc]
ax[0].bar(xs, band_auc.auc - .5, bottom=.5, color=cols, width=.7)
ax[0].errorbar(xs, band_auc.auc, yerr=[band_auc.auc - band_auc.auc_lo,
                                       band_auc.auc_hi - band_auc.auc],
               fmt="none", ecolor="k", elinewidth=.8, capsize=2)
ax[0].axhline(.5, color="k", lw=1)
ax[0].set_xticks(xs); ax[0].set_xticklabels(band_auc.band, rotation=60, ha="right", fontsize=7)
ax[0].set_ylabel("Recording-level AUC of that band alone")
ax[0].set_title("(a) Which bands separate leak from no-leak?\n(error bars = 95% CI)")
if cfg.f_min > FINE[0][0]:
    cut = float(np.searchsorted(band_auc.f_hi.values, cfg.f_min)) - .5
    ax[0].axvspan(-.5, cut, color="k", alpha=.12)
    ax[0].text(max(cut/2, 0.0), ax[0].get_ylim()[1],
               f"cut by f_min={cfg.f_min:.0f} Hz", ha="center", va="top", fontsize=8)

ax[1].semilogx(band_auc.f_lo, band_auc.suc_phan_biet, marker="o", lw=1.6, color=PAL["aux"])
ax[1].axvline(cfg.f_min, color=PAL["leak"], ls="--", lw=1.4, label=f"f_min = {cfg.f_min:.0f} Hz")
ax[1].axvline(cfg.f_max, color=PAL["warn"], ls="--", lw=1.4, label=f"f_max = {cfg.f_max:.0f} Hz")
ax[1].set_xlabel("frequency (Hz, log scale)"); ax[1].set_ylabel("|2·AUC − 1|")
ax[1].legend(fontsize=8); ax[1].set_title("(b) Discriminative power vs the Mel range in use")
savefig(fig, "band_informativeness",
        "Discriminative power of each frequency band, measured DIRECTLY from the data as the "
        "recording-level AUC of that band energy alone (no model involved), with 95% bootstrap " 
        "confidence intervals over recordings.")

best = band_auc.loc[band_auc.suc_phan_biet.idxmax()]
lost = band_auc[band_auc.f_hi <= cfg.f_min]
print(f"\nDải phân biệt mạnh nhất: {best.band} Hz (AUC {best.auc:.3f} "
      f"[{best.auc_lo:.3f}, {best.auc_hi:.3f}], {best.huong}; p={best.p:.3f}, p_holm={best.p_holm:.3f})")
print(f"Số dải có ý nghĩa SAU hiệu chỉnh Holm: {int((band_auc.p_holm < .05).sum())}/{len(band_auc)}"
      " -> nếu bằng 0, KHÔNG dải đơn lẻ nào đủ bằng chứng để quyết định thiết kế.")
if len(lost):
    j = lost.suc_phan_biet.idxmax()
    print(f"Phần bị cắt bởi f_min = {cfg.f_min:.0f} Hz gồm {len(lost)} dải; mạnh nhất trong đó: "
          f"{lost.loc[j,'band']} Hz (sức phân biệt {lost.suc_phan_biet.max():.3f})")
    if lost.suc_phan_biet.max() > 0.15:
        print(f"!! ĐANG CẮT BỎ THÔNG TIN CÓ GIÁ TRỊ -> mục 3.5b sẽ tự hạ cfg.f_min.")
    else:
        print("   Mức mất mát không đáng kể.")
else:
    print("f_min hiện tại không cắt bỏ dải nào đã phân tích.")


### 3.5b Quyết định dải tần dựa trên SỐ LIỆU, không dựa trên giả thuyết

Mục 3.5 đo sức phân biệt của từng dải **trước khi dùng mô hình**. Cell dưới đây dùng
chính số đo đó để:

1. **Kiểm định giả thuyết "năng lượng rò rỉ tập trung ở dải rất thấp (< 800 Hz)"** bằng
   **CI bootstrap theo bản ghi** của hiệu số *sức phân biệt trung bình dải cao − dải thấp*.
   Ba kết luận có thể có: *ủng hộ*, *không ủng hộ*, hoặc **chưa phân định được** (CI chứa 0).
   v6 so sánh hai con số điểm mà không có CI, nên đã phát biểu mạnh hơn bằng chứng.
2. **Không tự hạ** `cfg.low_band_hz`. v6 đã hạ 2800 → 1000 Hz chỉ dựa trên *một* dải "mạnh
   nhất" (283–413 Hz, p = 0,034, không còn ý nghĩa sau hiệu chỉnh Holm) và cắt mất dải
   1,3–2,7 kHz mà chính Grad-CAM của LeakCNN2D sử dụng. v7 giữ giá trị đăng ký trước;
   nếu bật `cfg.low_band_auto`, dải chỉ được **mở rộng** để bao top-3 dải mang thông tin.


In [ ]:
# =========================================================
# 3.5b QUYẾT ĐỊNH theo số liệu, có KHOẢNG TIN CẬY
#       [V7-S4] Kiểm định giả thuyết dải thấp bằng CI bootstrap (theo bản ghi) của hiệu số
#               sức phân biệt trung bình: dải cao (>= 800 Hz) - dải thấp (<= 800 Hz).
#       [V7-B3] KHÔNG tự hạ low_band_hz; chỉ mở rộng nếu cfg.low_band_auto=True.
# =========================================================
LOW_HZ = 800.0
_ba = band_auc.reset_index(drop=True).copy()
_lo_idx = np.where(_ba.f_hi.values <= LOW_HZ)[0]
_hi_idx = np.where(_ba.f_lo.values >= LOW_HZ)[0]
_tot = TRAPZ(_P, FREQ_AXIS, axis=1) + 1e-20
_E = np.column_stack([
    TRAPZ(_P[:, (FREQ_AXIS >= a) & (FREQ_AXIS < b)], FREQ_AXIS[(FREQ_AXIS >= a) & (FREQ_AXIS < b)], axis=1) / _tot
    for a, b in zip(_ba.f_lo.values, _ba.f_hi.values)])

def _disc_power(E, y):
    return np.array([abs(2.0 * roc_auc_score(y, E[:, j]) - 1.0) for j in range(E.shape[1])])

def _region_delta(E, y):
    d = _disc_power(E, y)
    return float(d[_hi_idx].mean() - d[_lo_idx].mean())

# Các dải của 3.5 cách đều trên thang LOG -> trung bình không trọng số = trọng số theo log-tần số
# (v6 lấy trọng số theo độ rộng Hz, khiến các dải cao rộng áp đảo phép so sánh).
s_lo = float(_ba.suc_phan_biet.values[_lo_idx].mean()) if len(_lo_idx) else float("nan")
s_hi = float(_ba.suc_phan_biet.values[_hi_idx].mean()) if len(_hi_idx) else float("nan")
delta_obs = s_hi - s_lo

_rng = np.random.default_rng(2024); _bs = []
for _ in range(200 if cfg.smoke_test else 1000):
    i = _rng.integers(0, len(_y), len(_y))
    if len(np.unique(_y[i])) < 2:
        continue
    _bs.append(_region_delta(_E[i], _y[i]))
d_lo, d_hi = (np.percentile(_bs, [2.5, 97.5]) if len(_bs) else (np.nan, np.nan))
best = _ba.loc[_ba.suc_phan_biet.idxmax()]

print("=== KIỂM ĐỊNH GIẢ THUYẾT DẢI TẦN (đơn vị: bản ghi, n = %d) ===" % len(_y))
print(f"Sức phân biệt trung bình |2AUC-1| | dải <= {LOW_HZ:.0f} Hz : {s_lo:.3f} ({len(_lo_idx)} dải)")
print(f"Sức phân biệt trung bình |2AUC-1| | dải >= {LOW_HZ:.0f} Hz : {s_hi:.3f} ({len(_hi_idx)} dải)")
print(f"Hiệu số (cao - thấp) = {delta_obs:+.3f}, CI 95% bootstrap theo bản ghi [{d_lo:+.3f}, {d_hi:+.3f}]")
print(f"Dải mạnh nhất quan sát được: {best.band} Hz (AUC {best.auc:.3f}, {best.huong}, "
      f"p_holm = {best.p_holm:.3f})")

if np.isfinite(d_lo) and d_lo > 0:
    narrative = "low_band_not_supported"
    verdict = "Dữ liệu KHÔNG ủng hộ giả thuyết dải thấp: dải cao phân biệt TỐT HƠN (CI > 0)."
elif np.isfinite(d_hi) and d_hi < 0:
    narrative = "low_band_supported"
    verdict = "Dữ liệu ỦNG HỘ giả thuyết dải thấp: dải thấp phân biệt tốt hơn (CI < 0)."
else:
    narrative = "inconclusive"
    verdict = ("CHƯA PHÂN ĐỊNH ĐƯỢC: CI của hiệu số chứa 0. Với số bản ghi hiện có, dữ liệu KHÔNG "
               "đủ để nói thông tin nằm ở dải thấp hay dải cao.")
LOWBAND_HYPOTHESIS_SUPPORTED = narrative == "low_band_supported"
print("KẾT LUẬN:", verdict)
if narrative == "inconclusive":
    print("  => Trong bài báo: KHÔNG lặp lại giả thuyết 'năng lượng rò rỉ ống nhựa < 800 Hz' như sự")
    print("     thật, và cũng KHÔNG khẳng định điều ngược lại. Phát biểu đúng: sức phân biệt phân tán")
    print("     trên nhiều dải, không dải đơn lẻ nào có ý nghĩa sau hiệu chỉnh đa so sánh.")
elif narrative == "low_band_not_supported":
    print("  => Discussion PHẢI viết rằng trên tập dữ liệu này, dấu hiệu phân biệt KHÔNG tập trung")
    print("     ở dải rất thấp như kỳ vọng lý thuyết cho ống nhựa.")

# ---- Dải cho nhánh STFT dải thấp: đăng ký trước, chỉ được MỞ RỘNG ------------------
_top = _ba.nlargest(3, "suc_phan_biet")
_need = float(_top.f_hi.max())
print(f"\nTop-3 dải mang thông tin: {', '.join(_top.band)} Hz -> cần phủ tới {_need:.0f} Hz")
if cfg.low_band_auto:
    _new = float(min(cfg.f_max, max(cfg.low_band_hz, np.ceil(_need / 100.0) * 100.0)))
    if _new > cfg.low_band_hz + 1e-6:
        print(f"[V7] cfg.low_band_hz: {cfg.low_band_hz:.0f} -> {_new:.0f} Hz (mở rộng để bao top-3 dải)")
        cfg.low_band_hz = _new
    if len(lost) and lost.suc_phan_biet.max() > 0.15:
        _fmin_new = float(min(300.0, max(0.0, np.floor(lost.f_lo.min() / 10.0) * 10.0)))
        if _fmin_new < cfg.f_min:
            print(f"[V7] cfg.f_min: {cfg.f_min:.0f} -> {_fmin_new:.0f} Hz (tránh cắt mất thông tin)")
            cfg.f_min = _fmin_new
elif _need > cfg.low_band_hz:
    print(f"!! cfg.low_band_hz = {cfg.low_band_hz:.0f} Hz KHÔNG phủ hết top-3 dải. Giá trị được giữ vì là "
          "tham số đăng ký trước; nêu trong Limitations hoặc bật cfg.low_band_auto rồi chạy lại.")
else:
    print(f"✓ cfg.low_band_hz = {cfg.low_band_hz:.0f} Hz phủ trọn top-3 dải mang thông tin.")

INFO_BAND_LO, INFO_BAND_HI = float(best.f_lo), float(best.f_hi)
BAND_NARRATIVE = narrative
BAND_TEST = dict(s_lo=s_lo, s_hi=s_hi, delta=delta_obs, ci=[float(d_lo), float(d_hi)], verdict=narrative)
print(f"\nDải dùng cho nhánh STFT dải thấp: 0-{cfg.low_band_hz:.0f} Hz | f_min = {cfg.f_min:.0f} Hz")


## 4. Tiền xử lý và cắt frame / Pre-processing and framing

Mỗi tệp được cắt thành các frame `cfg.frame_sec` giây với bước trượt `cfg.hop_sec` (mặc định 2 s, chồng lấn 50%).

> **Điểm mấu chốt về tính chính xác khoa học:** mọi frame đều mang theo `record_id` của bản
> ghi mẹ. Chỉ số này được dùng làm **group** trong cross-validation. Nếu chia ngẫu nhiên ở
> mức frame, các frame chồng lấn của cùng một bản ghi sẽ xuất hiện ở cả train lẫn validation
> → **rò rỉ dữ liệu** và chỉ số bị thổi phồng (mục 9.3 đo trực tiếp mức thổi phồng này).
>
> **Hệ quả thứ hai:** số frame lớn KHÔNG làm tăng cỡ mẫu hiệu dụng. Với ICC cao và khoảng
> một trăm frame mỗi bản ghi, hệ số phóng đại phương sai (design effect = 1 + (m − 1)·ICC)
> lên tới hàng chục, nên **cỡ mẫu hiệu dụng chỉ cỡ vài lần số bản ghi**. Mục 8.2 in ra ICC,
> design effect và cỡ mẫu hiệu dụng **đo được**; mọi khoảng tin cậy mức frame đều bootstrap
> theo cụm bản ghi (mục 7.2b).


In [ ]:
# =========================================================
# 4.1 Cắt frame và lưu cache / Frame extraction with on-disk cache
#     [V6-P2] Khoá cache tính theo NỘI DUNG manifest (kích thước + thời điểm sửa của
#     từng tệp), không phải theo số lượng tệp -> không còn nguy cơ dùng cache cũ.
#     [V6] Tham số hoá frame_sec/hop/filter để phục vụ ablation (mục 9.7).
# =========================================================
def _manifest_fingerprint(mf):
    h = hashlib.sha1()
    for _, r in mf.sort_values("path").iterrows():
        try:
            st = os.stat(r.path)
            meta = f"{st.st_size}|{int(st.st_mtime)}"
        except Exception:
            meta = "NA"
        h.update(f"{r.path}|{meta}|{r.record_id}|{int(r.label)}|".encode("utf-8"))
    h.update(json.dumps(dict(sr=cfg.target_sr, hp=cfg.highpass_hz, notch=bool(cfg.notch_mains),
                             mh=cfg.mains_hz, mharm=cfg.mains_harmonics, q=cfg.notch_q,
                             rx=cfg.record_regex, prio=cfg.label_source_priority),
                        sort_keys=True).encode("utf-8"))
    return h.hexdigest()[:12]

MANIFEST_FP = _manifest_fingerprint(manifest)

def build_frames(frame_sec=None, hop_sec=None, highpass=None, notch=None, apply_filters=True):
    fl = int((frame_sec or cfg.frame_sec) * cfg.target_sr)
    hl = int((hop_sec or cfg.hop_sec) * cfg.target_sr)
    metas, chunks = [], []
    for _, r in manifest.iterrows():
        x = load_audio(r.path, apply_filters=apply_filters, highpass=highpass, notch=notch)
        if not np.isfinite(x).all():
            BAD_FILES.append(dict(record_id=r.record_id, n_nonfinite=int((~np.isfinite(x)).sum()),
                                  path=r.path))
            x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if len(x) < fl:                       # đệm nếu bản ghi quá ngắn
            x = np.pad(x, (0, fl - len(x)))
        starts = list(range(0, len(x) - fl + 1, hl))
        if len(starts) < cfg.min_frames_per_file:
            starts = [0]
        for s in starts:
            seg = x[s:s + fl]
            chunks.append(seg.astype(np.float32))
            metas.append(dict(record_id=r.record_id, label=int(r.label), cls=r.cls,
                              path=r.path, start=s, start_s=s / cfg.target_sr,
                              rms=float(np.sqrt(np.mean(seg ** 2))),
                              zcr=float(np.mean(np.abs(np.diff(np.sign(seg))) > 0))))
    return np.stack(chunks).astype(np.float32), pd.DataFrame(metas)

def get_frames(tag="base", frame_sec=None, hop_sec=None, highpass=None, notch=None,
               apply_filters=True):
    # Trả về (X, frames_df) có cache riêng cho từng cấu hình (phục vụ ablation).
    fl = int((frame_sec or cfg.frame_sec) * cfg.target_sr)
    hl = int((hop_sec or cfg.hop_sec) * cfg.target_sr)
    key = (f"{MANIFEST_FP}_{tag}_{fl}_{hl}_{highpass}_{notch}_{int(apply_filters)}"
           f"_sr{int(cfg.target_sr)}_pn{cfg.peak_norm_pct:g}")          # [V7] khoá có sr
    xp, mp = CACHE_DIR / f"frames_{key}.npy", CACHE_DIR / f"frames_{key}.csv"
    if xp.exists() and mp.exists():
        print(f"Nạp cache / loaded cache [{tag}]:", xp.name)
        return np.load(xp, mmap_mode="r"), pd.read_csv(mp)
    t0 = time.time()
    X, F = build_frames(frame_sec=frame_sec, hop_sec=hop_sec,
                        highpass=highpass, notch=notch, apply_filters=apply_filters)
    np.save(xp, X); F.to_csv(mp, index=False)
    print(f"Cắt frame [{tag}] xong trong {time.time()-t0:.1f}s -> {xp.name}")
    return np.load(xp, mmap_mode="r"), F

BAD_FILES = []
X_ALL, frames = get_frames("base")
FRAME_SEC_BASE, HOP_SEC_BASE = cfg.frame_sec, cfg.hop_sec

# ---- Kiểm tra toàn vẹn số học (bắt buộc) / numerical sanity check ----
_chk = np.asarray(X_ALL[: min(len(X_ALL), 4096)])
assert np.isfinite(_chk).all(), "Vẫn còn NaN/Inf trong frame - kiểm tra lại tệp nguồn."
if BAD_FILES:
    print(f"!! {len(BAD_FILES)} tệp chứa mẫu NaN/Inf, đã được vá về 0:")
    display(pd.DataFrame(BAD_FILES))
_const = int(np.sum(np.asarray(X_ALL[: min(len(X_ALL), 4096)]).std(axis=1) < 1e-8))
if _const:
    print(f"!! {_const} frame câm/hằng số trong 4096 frame đầu - đã xử lý an toàn ở front-end.")

y_all = frames.label.values.astype(np.float32)
groups_all = frames.record_id.values

print(f"Tổng số frame / total frames : {len(frames)}  (shape {X_ALL.shape}, "
      f"{X_ALL.nbytes/1024**2:.0f} MB)")
display(frames.groupby("cls").agg(n_frames=("label", "size"),
                                  n_records=("record_id", "nunique"),
                                  mean_rms=("rms", "mean")).round(4))
_nfr = frames.groupby("record_id").size()
print(f"Frame / bản ghi: trung vị {int(_nfr.median())} (min {_nfr.min()}, max {_nfr.max()}) "
      f"-> bản ghi dài có trọng số lớn hơn trong hàm mất mát (xem ablation rec_balanced, mục 9.7)")
print(f"Design effect ước lượng "
      f"1+(m-1)*ICC; xem mục 8.2 để biết giá trị thực đo được.")
frames.head(3)


In [ ]:
# =========================================================
# 4.2 Hình 4: Phân bố frame và đặc trưng mức frame
#     Figure 4: frame distribution and frame-level descriptors
# =========================================================
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))

fc = frames.cls.value_counts()
ax[0].bar(fc.index, fc.values, color=[PAL[c] for c in fc.index], width=.55)
for i, v in enumerate(fc.values): ax[0].text(i, v, str(v), ha="center", va="bottom")
ax[0].set_title("(a) Frames per class")

per_rec = frames.groupby(["record_id", "cls"]).size().reset_index(name="n")
for c in ["noleak", "leak"]:
    ax[1].hist(per_rec.loc[per_rec.cls == c, "n"], bins=12, alpha=.65, color=PAL[c], label=c)
ax[1].set_title("(b) Frames per recording"); ax[1].legend()

for c in ["noleak", "leak"]:
    ax[2].hist(20*np.log10(frames.loc[frames.cls == c, "rms"] + 1e-9),
               bins=40, alpha=.6, color=PAL[c], label=c, density=True)
ax[2].set_xlabel("frame RMS (dBFS)"); ax[2].set_title("(c) Frame level (RMS)"); ax[2].legend()

for c in ["noleak", "leak"]:
    ax[3].hist(frames.loc[frames.cls == c, "zcr"], bins=40, alpha=.6,
               color=PAL[c], label=c, density=True)
ax[3].set_xlabel("zero-crossing rate"); ax[3].set_title("(d) ZCR"); ax[3].legend()

savefig(fig, "frame_distribution",
        "Frame-level statistics: class balance, frames per recording, RMS-level distribution and "
        "zero-crossing rate (ZCR).")

# Cảnh báo mất cân bằng / class-imbalance warning
r_imb = fc.max() / fc.min()
print(f"Tỉ lệ mất cân bằng frame / frame imbalance ratio = {r_imb:.2f}"
      + ("  -> sẽ dùng class weights trong hàm mất mát." if r_imb > 1.2 else "  -> cân bằng tốt."))


## 5. Biểu diễn đặc trưng và kiến trúc mạng / Feature front-ends and CNN architectures

Thiết kế then chốt: **biểu diễn thời gian–tần số được đặt bên trong mô hình** và tính trên GPU.
Nhờ đó (i) mọi mô hình dùng chung một nguồn dữ liệu trả về dạng sóng thô, (ii) SpecAugment thực
hiện trực tiếp trên GPU, và (iii) Grad-CAM truy vết ngược được về đúng ảnh log-Mel mà mô hình nhìn thấy.

| Mô hình | Đầu vào | Ý tưởng | Tham chiếu tương đương |
|---|---|---|---|
| `LeakCNN2D` | log-Mel 64 dải | CNN 4 khối + Squeeze-Excitation, pooling mean+max | Ma et al. 2024; Zhang et al. 2026 |
| `LeakCNN1D` | dạng sóng (chuẩn hoá theo mẫu) | Bộ lọc học được trên miền thời gian (kernel rộng 80) | Shukla & Piratla 2020 |
| `TransferCNN` | log-Mel → 3 kênh, phóng ≥ 128×128 | Fine-tune ResNet18 (ImageNet) | Li et al. 2025 |
| `MultiBranchCNN` | log-Mel + MFCC + STFT 0–`low_band_hz` | Hợp nhất đa biểu diễn | Zhang et al. 2026 (multi-scale) |
| `CRNN` | log-Mel → BiGRU + attention | Mô hình hoá tính dừng theo thời gian | Cody et al. 2020 |

**Các nguyên tắc ở tầng đặc trưng:**

1. **Chuẩn hoá theo TỪNG MẪU** ở mọi mô hình (v7: cả `LeakCNN1D`) → không mô hình nào dùng được
   "độ to" tuyệt đối của tín hiệu, vốn khác nhau giữa hai lớp và có thể là confound.
2. **SpecAugment áp TRƯỚC chuẩn hoá**, giá trị che = mức im lặng của mẫu, **mặt nạ riêng cho
   từng mẫu** (v6 dùng một mặt nạ chung cho cả batch). `MultiBranchCNN`: mỗi nhánh có
   SpecAugment riêng (v6 *viết* là dùng chung cho 3 nhánh nhưng *mã* chỉ che nhánh Mel).
3. **MFCC chuẩn hoá theo từng hệ số (CMVN)**.
4. **Nhánh STFT dải thấp giữ 0–2800 Hz** (đăng ký trước; xem mục 3.5b).


In [ ]:
# =========================================================
# 5.1 Front-end: log-Mel / MFCC / STFT dải thấp (chạy trên GPU)
#     [V6-T6] SpecAugment áp TRƯỚC chuẩn hoá, giá trị che = mức im lặng của mẫu.
#     [V6-T7] MFCC chuẩn hoá theo từng hệ số (CMVN).
#     [V7] Mặt nạ SpecAugment RIÊNG cho từng mẫu; tham số front-end đọc từ cfg LÚC KHỞI TẠO
#          (v6 gắn cứng lúc định nghĩa lớp -> ablation 16 kHz không đổi được sr/n_fft).
# =========================================================
def _fp32_ctx(x):
    # Buộc front-end chạy fp32 ngay cả khi bật AMP: STFT/Mel của tín hiệu đã chuẩn hoá
    # đỉnh có thể vượt dải biểu diễn của fp16 (65504) -> inf -> NaN lan xuống toàn mạng.
    return amp_autocast(x, False)

def _safe_norm(m):
    # Chuẩn hoá theo từng mẫu (không dùng thống kê toàn tập -> không rò rỉ dữ liệu):
    #  - nan_to_num trước khi tính thống kê
    #  - clamp_min độ lệch chuẩn: frame câm/hằng số có sd = 0 -> phép chia 0/0
    m = torch.nan_to_num(m, nan=0.0, posinf=0.0, neginf=-80.0)
    mu = m.mean(dim=(2, 3), keepdim=True)
    sd = m.std(dim=(2, 3), keepdim=True).clamp_min(1e-3)
    return torch.nan_to_num((m - mu) / sd, nan=0.0, posinf=0.0, neginf=0.0)

def _safe_norm_percoeff(m):
    # CMVN: chuẩn hoá theo TỪNG hệ số trên trục thời gian.
    m = torch.nan_to_num(m, nan=0.0, posinf=0.0, neginf=-80.0)
    mu = m.mean(dim=3, keepdim=True)
    sd = m.std(dim=3, keepdim=True).clamp_min(1e-3)
    return torch.nan_to_num((m - mu) / sd, nan=0.0, posinf=0.0, neginf=0.0)

class SpecAugment(nn.Module):
    # Che ngẫu nhiên dải tần / khoảng thời gian. CHỈ khi training.
    # Giá trị che = mức nhỏ nhất của chính mẫu đó ("im lặng"), áp TRƯỚC chuẩn hoá.
    # [V7] mỗi mẫu một mặt nạ riêng (v6: một mặt nạ chung cho cả batch -> kém đa dạng).
    def __init__(self, f=None, t=None, n=2, freq=True):
        super().__init__()
        self.f = cfg.specaug_freq_mask if f is None else int(f)
        self.t = cfg.specaug_time_mask if t is None else int(t)
        self.n, self.freq = int(n), bool(freq)
        self.generator = None      # generator riêng của fold (tất định theo fold)
    def _band_mask(self, B, L, W, device):
        g = self.generator
        w = torch.randint(0, W + 1, (B,), generator=g)
        s0 = (torch.rand(B, generator=g) * (L - w).clamp_min(1).float()).long()
        ar = torch.arange(L)
        return ((ar[None, :] >= s0[:, None]) & (ar[None, :] < (s0 + w)[:, None])).to(device)
    def forward(self, s):
        if not self.training or (self.f <= 0 and self.t <= 0):
            return s
        B, C, Fq, Tm = s.shape
        fill = s.amin(dim=(2, 3), keepdim=True).detach()
        mask = torch.zeros(B, 1, Fq, Tm, dtype=torch.bool, device=s.device)
        for _ in range(self.n):
            if self.freq and self.f > 0 and Fq > self.f:
                mask = mask | self._band_mask(B, Fq, self.f, s.device)[:, None, :, None]
            if self.t > 0 and Tm > self.t:
                mask = mask | self._band_mask(B, Tm, self.t, s.device)[:, None, None, :]
        return torch.where(mask, fill, s)

def set_specaug_generators(model, generator):
    # Gán generator riêng cho mọi SpecAugment trong mô hình -> tất định theo fold.
    n = 0
    for m in model.modules():
        if isinstance(m, SpecAugment):
            m.generator = generator; n += 1
    return n

class LogMelFrontend(nn.Module):
    def __init__(self, sr=None, n_fft=None, hop=None, n_mels=None, f_min=None, f_max=None,
                 specaug=None):
        super().__init__()
        sr = sr or cfg.target_sr; n_fft = n_fft or cfg.n_fft; hop = hop or cfg.hop_length
        n_mels = n_mels or cfg.n_mels
        f_min = cfg.f_min if f_min is None else f_min
        f_max = min(cfg.f_max if f_max is None else f_max, sr / 2)
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop, n_mels=n_mels,
            f_min=f_min, f_max=f_max, power=2.0, center=True)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80.0)
        self.specaug = specaug
    def wave_to_db(self, x):            # dB thô, chưa che, chưa chuẩn hoá
        with _fp32_ctx(x):
            return self.to_db(self.mel(x.float())).unsqueeze(1)
    def forward(self, x):               # x: [B, T]
        s = self.wave_to_db(x)
        if self.specaug is not None:
            s = self.specaug(s)         # TRƯỚC khi chuẩn hoá
        return _safe_norm(s)

class MFCCFrontend(nn.Module):
    def __init__(self, sr=None, n_mfcc=None, specaug=None):
        super().__init__()
        sr = sr or cfg.target_sr
        self.mfcc = torchaudio.transforms.MFCC(
            sample_rate=sr, n_mfcc=n_mfcc or cfg.n_mfcc, log_mels=True,
            melkwargs=dict(n_fft=cfg.n_fft, hop_length=cfg.hop_length, n_mels=cfg.n_mels,
                           f_min=cfg.f_min, f_max=min(cfg.f_max, sr / 2)))
        self.specaug = specaug
    def forward(self, x):
        with _fp32_ctx(x):
            m = self.mfcc(x.float()).unsqueeze(1)
        if self.specaug is not None:
            m = self.specaug(m)
        if cfg.mfcc_norm == "per_coeff":
            return _safe_norm_percoeff(m)
        return _safe_norm(m)

class LowBandSTFTFrontend(nn.Module):
    # Chỉ giữ 0 - low_band_hz (tham số đăng ký trước, xem mục 3.5b).
    def __init__(self, sr=None, n_fft=None, hop=None, fmax=None, specaug=None):
        super().__init__()
        sr = sr or cfg.target_sr; n_fft = n_fft or cfg.n_fft; hop = hop or cfg.hop_length
        fmax = min(cfg.low_band_hz if fmax is None else fmax, sr / 2)
        self.spec = torchaudio.transforms.Spectrogram(n_fft=n_fft, hop_length=hop, power=2.0)
        self.kmax = max(2, int(fmax / (sr / n_fft)) + 1)
        self.specaug = specaug
    def forward(self, x):
        with _fp32_ctx(x):
            s = self.spec(x.float())[:, :self.kmax, :]
            s = torch.log(s.clamp_min(1e-10)).unsqueeze(1)
        if self.specaug is not None:
            s = self.specaug(s)
        return _safe_norm(s)

# Kiểm tra kích thước tensor / sanity check on shapes
with torch.no_grad():
    _x = torch.from_numpy(np.asarray(X_ALL[:2])).float()
    print("waveform      :", tuple(_x.shape))
    print("log-Mel       :", tuple(LogMelFrontend()(_x).shape))
    print("MFCC (CMVN)   :", tuple(MFCCFrontend()(_x).shape))
    print("low-band STFT :", tuple(LowBandSTFTFrontend()(_x).shape),
          f"| giữ 0-{cfg.low_band_hz:.0f} Hz")
    # SpecAugment: mỗi mẫu một mặt nạ, và chỉ hoạt động khi training
    _sa = SpecAugment(f=8, t=8).train(); _sa.generator = torch.Generator().manual_seed(0)
    _s = torch.randn(4, 1, 64, 63); _o = _sa(_s.clone())
    _masked = (_o != _s).flatten(1).float().mean(1)
    assert (_masked > 0).any(), "SpecAugment không che gì"
    assert torch.equal(SpecAugment().eval()(_s.clone()), _s), "SpecAugment che cả khi eval"
    print("SpecAugment   : tỉ lệ ô bị che theo mẫu =", [round(float(v), 3) for v in _masked])


In [ ]:
# =========================================================
# 5.2 Khối dựng cơ bản / Building blocks
# =========================================================
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ch, max(4, ch // r)), nn.ReLU(inplace=True),
                                nn.Linear(max(4, ch // r), ch), nn.Sigmoid())
    def forward(self, x):
        w = self.fc(x.mean(dim=(2, 3))).unsqueeze(-1).unsqueeze(-1)
        return x * w

class ConvBlock2D(nn.Module):
    def __init__(self, cin, cout, pool=(2, 2), p_drop=0.1, se=True):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        self.se = SEBlock(cout) if se else nn.Identity()
        self.pool = nn.MaxPool2d(pool)
        self.drop = nn.Dropout2d(p_drop)
    def forward(self, x):
        return self.drop(self.pool(self.se(self.body(x))))

class Head(nn.Module):
    def __init__(self, cin, hidden=128, p=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cin, hidden), nn.BatchNorm1d(hidden),
                                 nn.ReLU(inplace=True), nn.Dropout(p), nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

def gap_gmp(x):
    # Gộp trung bình + cực đại: giữ cả mức nền lẫn đỉnh phổ cục bộ
    return torch.cat([x.mean(dim=(2, 3)), x.amax(dim=(2, 3))], dim=1)


In [ ]:
# =========================================================
# 5.3 Mô hình 1 - LeakCNN2D: CNN chuyên biệt trên log-Mel
#     [V6] Bỏ SpecAugment bên ngoài (đã chuyển vào trong front-end, trước chuẩn hoá).
# =========================================================
class LeakCNN2D(nn.Module):
    uses_mel_cam = True
    uses_raw_waveform = False
    cam_usable = True          # bản đồ đặc trưng 8x7 -> Grad-CAM có ý nghĩa
    def __init__(self, widths=(32, 64, 128, 256), p=0.1):
        super().__init__()
        self.specaug = SpecAugment()
        self.front = LogMelFrontend(specaug=self.specaug)
        blocks, cin = [], 1
        for i, w in enumerate(widths):
            # Ở khối cuối chỉ pool theo trục tần số để giữ độ phân giải thời gian
            pool = (2, 2) if i < len(widths) - 1 else (2, 1)
            blocks.append(ConvBlock2D(cin, w, pool=pool, p_drop=p)); cin = w
        self.blocks = nn.Sequential(*blocks)
        self.head = Head(cin * 2)
    def spec(self, x):  return self.front(x)
    def features(self, x):
        return self.blocks(self.front(x))
    def embed(self, x): return gap_gmp(self.features(x))
    def forward(self, x): return self.head(self.embed(x))
    def cam_layer(self):  return self.blocks[-1].body

# =========================================================
# 5.4 Mô hình 2 - LeakCNN1D: học trực tiếp trên dạng sóng
#     LƯU Ý [V6]: đây là mô hình DUY NHẤT mà việc đảo cực có tác dụng
#     (phổ biên độ |STFT|^2 bất biến với đảo cực -> với 4 mô hình còn lại
#      đảo cực là phép vô tác dụng).
# =========================================================
class LeakCNN1D(nn.Module):
    uses_mel_cam = False
    uses_raw_waveform = True
    def __init__(self, ch=(64, 128, 256, 512)):
        super().__init__()
        c1, c2, c3, c4 = ch
        self.net = nn.Sequential(
            nn.Conv1d(1, c1, kernel_size=80, stride=4, padding=38, bias=False),  # ~10 ms
            nn.BatchNorm1d(c1), nn.ReLU(inplace=True), nn.MaxPool1d(4),
            nn.Conv1d(c1, c2, 3, padding=1, bias=False), nn.BatchNorm1d(c2),
            nn.ReLU(inplace=True), nn.MaxPool1d(4), nn.Dropout(0.1),
            nn.Conv1d(c2, c3, 3, padding=1, bias=False), nn.BatchNorm1d(c3),
            nn.ReLU(inplace=True), nn.MaxPool1d(4), nn.Dropout(0.1),
            nn.Conv1d(c3, c4, 3, padding=1, bias=False), nn.BatchNorm1d(c4),
            nn.ReLU(inplace=True))
        self.head = Head(c4 * 2)
    def embed(self, x):
        # [V7] chuẩn hoá theo TỪNG MẪU như các front-end phổ: loại bỏ tín hiệu "độ to" (mức
        # năng lượng khác nhau giữa hai lớp ~ confound) để so sánh công bằng với 4 mô hình kia.
        x = x.float()
        x = (x - x.mean(-1, keepdim=True)) / x.std(-1, keepdim=True).clamp_min(1e-4)
        h = self.net(x.unsqueeze(1))
        return torch.cat([h.mean(-1), h.amax(-1)], dim=1)
    def forward(self, x): return self.head(self.embed(x))

# =========================================================
# 5.5 Mô hình 3 - TransferCNN: fine-tune backbone ImageNet
#     [V6] Ghi lại trạng thái pretrained để bảng kết quả không che giấu việc
#     âm thầm rơi về khởi tạo ngẫu nhiên khi Kaggle tắt Internet.
# =========================================================
class TransferCNN(nn.Module):
    uses_mel_cam = True
    uses_raw_waveform = False
    cam_usable = False         # ResNet18 hạ mẫu 32 lần -> bản đồ 4x4 (v7) vẫn quá thô cho Grad-CAM
    def __init__(self, backbone=cfg.transfer_backbone, pretrained=True):
        super().__init__()
        import torchvision
        self.specaug = SpecAugment()
        self.front = LogMelFrontend(specaug=self.specaug)
        self.backbone_name = backbone
        self.pretrained = False
        net = None
        if pretrained:
            try:
                if backbone == "resnet18":
                    net = torchvision.models.resnet18(weights="IMAGENET1K_V1")
                else:
                    net = torchvision.models.mobilenet_v3_small(weights="IMAGENET1K_V1")
                self.pretrained = True
            except Exception as e:
                print(f"  [TransferCNN] không tải được trọng số ImageNet ({type(e).__name__}) "
                      f"-> khởi tạo NGẪU NHIÊN. Kết quả transfer learning KHÔNG còn hiệu lực.")
                net = None
        if net is None:
            net = (torchvision.models.resnet18(weights=None) if backbone == "resnet18"
                   else torchvision.models.mobilenet_v3_small(weights=None))
        if backbone == "resnet18":
            self.feat = nn.Sequential(*list(net.children())[:-2]); cout = 512
        else:
            self.feat = net.features; cout = net.classifier[0].in_features
        self.head = Head(cout * 2)
    def spec(self, x): return self.front(x)
    def features(self, x):
        s = self.front(x).repeat(1, 3, 1, 1)   # 1 -> 3 kênh
        # [V7] v6 đưa ảnh 64x63 vào ResNet18 -> bản đồ cuối 2x2, gần như mất cấu trúc T-F.
        ms = int(cfg.transfer_min_size)
        if s.shape[-1] < ms or s.shape[-2] < ms:
            s = F.interpolate(s, size=(max(ms, s.shape[-2]), max(ms, s.shape[-1])),
                              mode="bilinear", align_corners=False)
        return self.feat(s)
    def embed(self, x): return gap_gmp(self.features(x))
    def forward(self, x): return self.head(self.embed(x))
    def cam_layer(self): return self.feat[-1]


In [ ]:
# =========================================================
# 5.6 Mô hình 4 - MultiBranchCNN: hợp nhất log-Mel + MFCC + STFT dải thấp
#     [V7-B6] v6 VIẾT là SpecAugment dùng chung cho 3 nhánh nhưng mã chỉ che nhánh Mel.
#     v7: mỗi nhánh có SpecAugment riêng, áp TRƯỚC chuẩn hoá (MFCC: chỉ che thời gian,
#     vì che theo trục hệ số cepstral không có ý nghĩa vật lý).
# =========================================================
class Branch(nn.Module):
    def __init__(self, widths=(24, 48, 96)):
        super().__init__()
        b, cin = [], 1
        for w in widths:
            b.append(ConvBlock2D(cin, w, pool=(2, 2), p_drop=.1, se=True)); cin = w
        self.body = nn.Sequential(*b); self.out_dim = cin * 2
    def forward(self, s): return gap_gmp(self.body(s))

class MultiBranchCNN(nn.Module):
    uses_mel_cam = True
    uses_raw_waveform = False
    cam_usable = True          # nhánh Mel: bản đồ 8x7
    def __init__(self):
        super().__init__()
        self.specaug = SpecAugment()
        self.specaug_mfcc = SpecAugment(freq=False)
        self.specaug_low = SpecAugment()
        self.f_mel = LogMelFrontend(specaug=self.specaug)
        self.f_mfcc = MFCCFrontend(specaug=self.specaug_mfcc)
        self.f_low = LowBandSTFTFrontend(specaug=self.specaug_low)
        self.b_mel, self.b_mfcc, self.b_low = Branch(), Branch(), Branch()
        d = self.b_mel.out_dim + self.b_mfcc.out_dim + self.b_low.out_dim
        self.fuse = nn.Sequential(nn.Linear(d, 256), nn.BatchNorm1d(256),
                                  nn.ReLU(inplace=True), nn.Dropout(.3))
        self.head = Head(256)
    def spec(self, x): return self.f_mel(x)
    def embed(self, x):
        # Mỗi front-end tự áp SpecAugment của nó (chỉ khi training, trước chuẩn hoá).
        z = torch.cat([self.b_mel(self.f_mel(x)),
                       self.b_mfcc(self.f_mfcc(x)),
                       self.b_low(self.f_low(x))], dim=1)
        return self.fuse(z)
    def forward(self, x): return self.head(self.embed(x))
    def cam_layer(self): return self.b_mel.body[-1].body

# =========================================================
# 5.7 Mô hình 5 - CRNN: CNN + BiGRU + attention pooling
# =========================================================
class CRNN(nn.Module):
    uses_mel_cam = True
    uses_raw_waveform = False
    cam_usable = True          # c3 giữ trục thời gian, bản đồ 8x15
    def __init__(self, widths=(32, 64, 128), rnn_hidden=128):
        super().__init__()
        self.specaug = SpecAugment()
        self.front = LogMelFrontend(specaug=self.specaug)
        self.c1 = ConvBlock2D(1, widths[0], pool=(2, 2))
        self.c2 = ConvBlock2D(widths[0], widths[1], pool=(2, 2))
        self.c3 = ConvBlock2D(widths[1], widths[2], pool=(2, 1))   # giữ trục thời gian
        fbins = cfg.n_mels // 8       # 3 lần maxpool (2,2) trên trục tần số
        self.rnn = nn.GRU(widths[2] * fbins, rnn_hidden, num_layers=1,
                          batch_first=True, bidirectional=True)
        self.attn = nn.Sequential(nn.Linear(2 * rnn_hidden, 64), nn.Tanh(), nn.Linear(64, 1))
        self.head = Head(2 * rnn_hidden)
    def spec(self, x): return self.front(x)
    def features(self, x): return self.c3(self.c2(self.c1(self.front(x))))
    def embed(self, x):
        h = self.features(x)                       # [B, C, F, T]
        B, C, Fq, T = h.shape
        h = h.permute(0, 3, 1, 2).reshape(B, T, C * Fq)
        o, _ = self.rnn(h)                         # [B, T, 2H]
        a = torch.softmax(self.attn(o), dim=1)     # trọng số chú ý theo thời gian
        return (o * a).sum(1)
    def forward(self, x): return self.head(self.embed(x))
    def cam_layer(self): return self.c3.body

MODEL_ZOO = {"LeakCNN2D": LeakCNN2D, "LeakCNN1D": LeakCNN1D, "TransferCNN": TransferCNN,
             "MultiBranchCNN": MultiBranchCNN, "CRNN": CRNN}

def build_model(name, **kw): return MODEL_ZOO[name](**kw)

def model_uses_raw_waveform(name):
    return bool(getattr(MODEL_ZOO[name], "uses_raw_waveform", False))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

rows = []
with torch.no_grad():
    xb = torch.from_numpy(np.asarray(X_ALL[:4])).float()
    for n in list(cfg.models):
        try:
            m = build_model(n).eval()
            out = m(xb)
        except Exception as e:
            # [V6] Không để một kiến trúc hỏng (ví dụ thiếu torchvision) làm chết cả notebook:
            # loại khỏi danh sách kèm cảnh báo rõ ràng, và ghi lại vào bảng.
            print(f"!! BỎ QUA {n}: {type(e).__name__}: {e}")
            print(f"   -> loại {n} khỏi cfg.models cho toàn bộ phần còn lại.")
            cfg.models = tuple(x for x in cfg.models if x != n)
            rows.append(dict(model=n, params=np.nan, params_M=np.nan, pretrained=None,
                             out_shape=f"LỖI: {type(e).__name__}"))
            continue
        rows.append(dict(model=n, params=count_params(m), params_M=round(count_params(m)/1e6, 3),
                         pretrained=(getattr(m, "pretrained", None) if n == "TransferCNN" else None),
                         out_shape=str(tuple(out.shape))))
        del m
param_table = pd.DataFrame(rows)
display(Markdown("### Table 3. Model complexity"))
display(param_table)
param_table.to_csv(TAB_DIR / "table3_model_complexity.csv", index=False)
assert len(cfg.models) > 0, "Không còn mô hình nào chạy được - kiểm tra môi trường (torchvision?)."
if "TransferCNN" in cfg.models:
    _pt = param_table.set_index("model").pretrained.get("TransferCNN", None)
    if _pt is not True:
        print("!! TransferCNN KHÔNG dùng trọng số ImageNet -> mọi kết luận về 'transfer learning'")
        print("   là không hợp lệ. Bật Internet trong Settings của Kaggle rồi chạy lại.")


In [ ]:
# =========================================================
# 5.8 Hình 5: Biểu diễn đầu vào của các mô hình / input representations
# =========================================================
idx_leak   = int(frames.index[frames.cls == "leak"][len(frames[frames.cls=="leak"])//2])
idx_noleak = int(frames.index[frames.cls == "noleak"][len(frames[frames.cls=="noleak"])//2])
xb = torch.from_numpy(np.asarray(X_ALL[[idx_noleak, idx_leak]])).float()

with torch.no_grad():
    mel  = LogMelFrontend()(xb).squeeze(1).numpy()
    mfcc = MFCCFrontend()(xb).squeeze(1).numpy()
    low  = LowBandSTFTFrontend()(xb).squeeze(1).numpy()

fig, ax = plt.subplots(2, 3, figsize=(13.5, 5.6))
for r, (nm, c) in enumerate(zip(["noleak", "leak"], ["noleak", "leak"])):
    for j, (M, title, ylab) in enumerate([
            (mel[r],  "log-Mel (64 bands, 20–4000 Hz)", "mel band"),
            (mfcc[r], f"MFCC ({cfg.n_mfcc} coeff.)", "coefficient"),
            (low[r],  f"log-STFT 0–{int(cfg.low_band_hz)} Hz", "freq bin")]):
        im = ax[r, j].imshow(M, aspect="auto", origin="lower", cmap="magma")
        ax[r, j].set_title(f"{nm.upper()} — {title}", fontsize=9)
        ax[r, j].set_xlabel("time frame"); ax[r, j].set_ylabel(ylab)
        fig.colorbar(im, ax=ax[r, j], pad=.01)
fig.tight_layout()
savefig(fig, "input_representations",
        "The three input representations used in this study (per-sample normalised), shown for one "
        "no-leak frame and one leak frame.")


## 6. Giao thức đánh giá chống rò rỉ dữ liệu / Leakage-free evaluation protocol

Notebook phân tầng **trên BẢNG BẢN GHI** rồi trải về frame (`cfg.fold_mode = "record_stratified"`):

- Mỗi `Rxxx` chỉ mang MỘT nhãn nên nhóm là "thuần nhãn"; phân tầng trực tiếp trên danh sách bản
  ghi giữ tỉ lệ lớp giữa các fold gần như **chính xác** (bảng 4 in biên độ dao động thực tế).
- `StratifiedGroupKFold` (`cfg.fold_mode = "frame_group"`) cân bằng theo **số frame**, nên khi số
  frame mỗi bản ghi khác nhau, tỉ lệ lớp giữa các fold có thể lệch đáng kể.

**4 assertion cứng** được kiểm tra cho **mọi** phân hoạch (kể cả các lần lặp CV) và notebook
**dừng ngay** nếu vi phạm:

| # | Điều kiện | Ý nghĩa |
|---|---|---|
| 1 | `set(groups[tr]) ∩ set(groups[va]) = ∅` | Không bản ghi nào nằm ở cả hai phía |
| 2 | Mỗi bản ghi xuất hiện **đúng một lần** ở validation | Không fold nào lặp/thiếu bản ghi |
| 3 | `len(tr) + len(va) = len(frames)` | Phủ hết frame |
| 4 | Validation có **đủ hai lớp** | Không fold nào chỉ toàn một lớp |

Số fold tự điều chỉnh theo số bản ghi của lớp hiếm nhất; quá ít bản ghi → `LeaveOneGroupOut`.

> **[V7] CV lặp:** `build_splits(seed)` sinh một phân hoạch mới cho mỗi seed trong `cfg.seeds`.
> Chia đúng theo bản ghi **chưa đủ**: vì số bản ghi độc lập nhỏ, mọi khoảng tin cậy mức frame
> **phải** bootstrap theo **cụm bản ghi**, nếu không CI hẹp hơn thực tế gần một bậc độ lớn.


In [ ]:
# =========================================================
# 6.1 Thiết lập cross-validation — PHÂN TẦNG TRÊN BẢNG BẢN GHI
#     [V7-S1] Tách thành hàm build_splits(seed) để CV lặp (mỗi seed một phân hoạch mới);
#             check_splits() áp 4 assertion cứng cho MỌI phân hoạch.
# =========================================================
rec_lab = frames.groupby("record_id").label.first()
rec_to_frames = frames.groupby("record_id").indices
REC_IDS = rec_lab.index.values
REC_Y = rec_lab.values.astype(int)
n_rec_min = pd.Series(REC_Y).value_counts().min()
K = int(min(cfg.n_splits, n_rec_min))
USE_LOGO = K < 2

def _expand(rec_subset):
    return np.sort(np.concatenate([rec_to_frames[r] for r in rec_subset]))

def build_splits(seed):
    if USE_LOGO:
        return [(_expand(np.delete(REC_IDS, i)), _expand(REC_IDS[[i]])) for i in range(len(REC_IDS))]
    if cfg.fold_mode == "record_stratified":
        skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=int(seed))
        return [(_expand(REC_IDS[a]), _expand(REC_IDS[b]))
                for a, b in skf.split(np.zeros(len(REC_IDS)), REC_Y)]
    sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=int(seed))
    return list(sgkf.split(np.zeros(len(frames)), y_all, groups_all))

def check_splits(splits):
    seen = set()
    for k, (tr, va) in enumerate(splits):
        gtr, gva = set(groups_all[tr]), set(groups_all[va])
        assert not (gtr & gva), f"DATA LEAKAGE ở fold {k}: {gtr & gva}"
        assert len(np.unique(y_all[va])) == 2 or USE_LOGO, f"Fold {k} thiếu một lớp ở validation"
        assert len(tr) + len(va) == len(frames), f"Fold {k} không phủ hết frame"
        if not USE_LOGO:
            assert not (seen & gva), f"Bản ghi lặp ở hai fold validation: {seen & gva}"
            seen |= gva
    if not USE_LOGO:
        assert seen == set(REC_IDS), "Có bản ghi không xuất hiện ở validation lần nào"
    return True

if USE_LOGO:
    print("!! Quá ít bản ghi -> LeaveOneGroupOut.")
elif K < cfg.n_splits:
    print(f"!! Giảm số fold {cfg.n_splits} -> {K} (lớp hiếm chỉ có {n_rec_min} bản ghi).")

SPLITS = build_splits(cfg.seed)
check_splits(SPLITS)
print(f"✓ {len(SPLITS)} fold | mỗi bản ghi xuất hiện ĐÚNG MỘT LẦN ở validation | "
      f"không có bản ghi nào ở cả hai phía")

split_tab = pd.DataFrame([
    dict(fold=k, n_train_frames=len(tr), n_val_frames=len(va),
         n_train_rec=len(set(groups_all[tr])), n_val_rec=len(set(groups_all[va])),
         val_leak_REC=int(sum(rec_lab[r] for r in set(groups_all[va]))),
         val_leak_rate_rec=round(float(np.mean([rec_lab[r] for r in set(groups_all[va])])), 3),
         val_leak_rate_frame=round(float(y_all[va].mean()), 3))
    for k, (tr, va) in enumerate(SPLITS)])
display(Markdown("### Table 4. Cross-validation partitioning"))
display(split_tab)
spread = split_tab.val_leak_rate_rec.max() - split_tab.val_leak_rate_rec.min()
print(f"Biên độ dao động tỉ lệ lớp giữa các fold (mức bản ghi): {spread:.3f}")
print("  (< 0.10 là tốt; giá trị lớn làm phương sai giữa các fold tăng mạnh)")
split_tab.to_csv(TAB_DIR / "table4_cv_splits.csv", index=False)


In [ ]:
# =========================================================
# 6.2 Hình 6: Trực quan hoá phân chia CV / CV partition map
# =========================================================
order = np.argsort(groups_all, kind="stable")
rec_ids = pd.unique(groups_all[order])
rec_pos = {r: i for i, r in enumerate(rec_ids)}
M = np.full((len(SPLITS), len(rec_ids)), np.nan)
for k, (tr, va) in enumerate(SPLITS):
    for r in set(groups_all[tr]): M[k, rec_pos[r]] = 0
    for r in set(groups_all[va]): M[k, rec_pos[r]] = 1

fig, ax = plt.subplots(2, 1, figsize=(13, 3.6), sharex=True,
                       gridspec_kw=dict(height_ratios=[3, .6], hspace=.15))
ax[0].imshow(M, aspect="auto", cmap=matplotlib.colors.ListedColormap([PAL["grey"], PAL["warn"]]),
             interpolation="nearest")
ax[0].set_yticks(range(len(SPLITS))); ax[0].set_yticklabels([f"fold {k}" for k in range(len(SPLITS))])
ax[0].set_ylabel("CV fold"); ax[0].grid(False)
ax[0].set_title("Recording-grouped CV partition (grey = train, orange = validation)")

lab_row = np.array([[rec_lab[r] for r in rec_ids]])
ax[1].imshow(lab_row, aspect="auto",
             cmap=matplotlib.colors.ListedColormap([PAL["noleak"], PAL["leak"]]), interpolation="nearest")
ax[1].set_yticks([0]); ax[1].set_yticklabels(["label"]); ax[1].set_xlabel("recording index")
ax[1].grid(False)
savefig(fig, "cv_partition",
        "Cross-validation partition map: within a fold every recording belongs to either train or "
        "validation, eliminating leakage across folds.")


## 7. Bộ máy huấn luyện / Training engine

Phần này gồm bốn thành phần, theo thứ tự:

- **7.0 Chẩn đoán NaN** — truy vết nguồn sinh giá trị không hữu hạn trước khi huấn luyện.
- **7.1 Dữ liệu & tăng cường** — hai đường: DataLoader cổ điển và đường **thường trú GPU** (nhanh hơn, cùng phép toán).
- **7.2 Lựa chọn lồng** — tách validation-trong để chọn epoch / ngưỡng / phép gộp, **không đụng vào fold báo cáo**.
- **7.3–7.5 Huấn luyện** — AMP, chạy song song hai GPU theo fold.


### 7.0 CHẨN ĐOÁN — truy vết nguồn sinh NaN / NaN provenance diagnostic

Cell này chạy trong vài giây và **chỉ đích danh khâu sinh giá trị không hữu hạn**, thay vì để lỗi nổ ra giữa vòng huấn luyện. Nó kiểm tra ba tầng:

1. **Dữ liệu** — quét toàn bộ frame tìm NaN/Inf và frame câm (hằng số).
2. **Lượt truyền xuôi** — với từng kiến trúc, đo số phần tử không hữu hạn ở đầu ra front-end và ở logit, **ở cả hai chế độ fp32 và AMP (fp16)**.
3. **Lượt truyền ngược** — kiểm tra gradient.

Nếu NaN **chỉ xuất hiện khi bật AMP**, cell sẽ **tự đặt `cfg.amp = False`** và nói rõ lý do.


In [ ]:
# =========================================================
# 7.0 CHẨN ĐOÁN NaN - chạy trước khi huấn luyện
#     [V6-T4] ĐỔI CHÍNH SÁCH: v5 tự tắt AMP toàn cục chỉ vì 3/11.3M gradient không
#     hữu hạn ở MỘT batch chẩn đoán - trong khi GradScaler vốn đã tự bỏ qua step đó.
#     Nay: chỉ tắt AMP khi lỗi xuất hiện ở fp32 (nguyên nhân dữ liệu/kiến trúc),
#     còn lỗi chỉ ở fp16 thì giữ AMP và đếm số step bị bỏ qua.
# =========================================================
def _nf(t):
    return int((~torch.isfinite(t)).sum().item())

print("=== 1. Toàn vẹn dữ liệu / data integrity ===")
bad_idx, amax, n_const = [], 0.0, 0
for s0 in range(0, len(X_ALL), 2048):
    blk = np.asarray(X_ALL[s0:s0 + 2048])
    fin = np.isfinite(blk)
    bad_idx += (s0 + np.where(~fin.all(axis=1))[0]).tolist()
    amax = max(amax, float(np.abs(np.where(fin, blk, 0.0)).max()))
    n_const += int(np.sum(blk.std(axis=1) < 1e-8))
print(f"  frame chứa NaN/Inf : {len(bad_idx)} / {len(X_ALL)}")
print(f"  frame câm (hằng số): {n_const}")
print(f"  biên độ lớn nhất   : {amax:.4f}")
if bad_idx:
    print("  Bản ghi liên quan:", sorted(set(frames.record_id.values[bad_idx]))[:10])
if LOAD_REPAIRED:
    print(f"  {len(LOAD_REPAIRED)} tệp đã phải vá khi đọc:")
    display(pd.DataFrame(LOAD_REPAIRED))

print("\n=== 2-3. Truyền xuôi / truyền ngược theo từng kiến trúc ===")
_rng_d = np.random.default_rng(0)
probe = np.sort(_rng_d.choice(len(X_ALL), min(64, len(X_ALL)), replace=False))
xb_d = torch.from_numpy(np.asarray(X_ALL[probe])).float().to(DEVICE)
yb_d = torch.from_numpy(y_all[probe]).float().to(DEVICE)

diag_rows = []
for name in cfg.models:
    for amp_on in ([False, True] if DEVICE.type == "cuda" else [False]):
        set_seed(cfg.seed, deterministic=cfg.deterministic_run)
        m = build_model(name).to(DEVICE); m.train()
        set_specaug_generators(m, make_generator("cpu", cfg.seed))
        try:
            with amp_autocast(DEVICE, amp_on):
                spec_bad = _nf(m.spec(xb_d)) if hasattr(m, "spec") else -1
                spec_max = float(m.spec(xb_d).abs().max()) if hasattr(m, "spec") else float("nan")
                logit = m(xb_d)
                loss = F.binary_cross_entropy_with_logits(logit, yb_d)
            make_grad_scaler(amp_on).scale(loss).backward()
            gbad = sum(_nf(q.grad) for q in m.parameters() if q.grad is not None)
            diag_rows.append(dict(model=name, amp=amp_on, spec_nonfinite=spec_bad,
                                  spec_absmax=round(spec_max, 2), logit_nonfinite=_nf(logit),
                                  logit_absmax=round(float(torch.nan_to_num(logit).abs().max()), 3),
                                  loss=round(float(loss), 5), grad_nonfinite=gbad, error=""))
        except Exception as e:
            diag_rows.append(dict(model=name, amp=amp_on, spec_nonfinite=-1, spec_absmax=np.nan,
                                  logit_nonfinite=-1, logit_absmax=np.nan, loss=np.nan,
                                  grad_nonfinite=-1, error=f"{type(e).__name__}: {e}"))
        del m; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

diag_df = pd.DataFrame(diag_rows)
display(Markdown("### NaN diagnostic table"))
display(diag_df)
diag_df.to_csv(TAB_DIR / "nan_diagnostic.csv", index=False)

def _is_bad(r):
    return (r.spec_nonfinite > 0 or r.logit_nonfinite > 0 or r.grad_nonfinite > 0
            or not np.isfinite(r.loss) or r.error != "")
bad32 = diag_df[diag_df.amp == False].apply(_is_bad, axis=1)
bad16 = (diag_df[diag_df.amp == True].apply(_is_bad, axis=1)
         if (diag_df.amp == True).any() else pd.Series(dtype=bool))
print()
if bad32.any():
    cfg.amp = False
    print("KẾT LUẬN: có giá trị không hữu hạn NGAY Ở fp32 -> nguyên nhân ở dữ liệu hoặc")
    print("          kiến trúc. ĐÃ TẮT AMP để an toàn; xem cột spec_nonfinite vs logit_nonfinite.")
elif len(bad16) and bad16.any():
    if cfg.amp_policy == "force_off":
        cfg.amp = False
        print("KẾT LUẬN: bất thường chỉ ở fp16 và cfg.amp_policy='force_off' -> TẮT AMP.")
    else:
        n_bad16 = int(bad16.sum())
        print(f"KẾT LUẬN: {n_bad16} cấu hình có giá trị không hữu hạn CHỈ khi bật AMP (fp16).")
        print("          GIỮ AMP: GradScaler tự phát hiện và BỎ QUA đúng những step đó")
        print("          (số step bị bỏ qua được đếm và in ở mục 7.5). Nếu muốn tắt hẳn,")
        print("          đặt cfg.amp_policy='force_off'.")
else:
    print("KẾT LUẬN: không phát hiện giá trị không hữu hạn ở cả fp32 lẫn AMP. Sẵn sàng huấn luyện.")


### 7.1 Dữ liệu và tăng cường / Data feeding and augmentation

Hai đường dẫn dữ liệu, **cùng phép toán, khác cách vận chuyển**:

| Đường | Khi nào | Đặc điểm |
|---|---|---|
| `DataLoader` cổ điển | `cfg.gpu_resident = False`, hoặc dữ liệu quá lớn so với VRAM | Augmentation numpy trên CPU worker. Mỗi worker **phải được gieo hạt riêng**. |
| **Thường trú GPU** | mặc định | Toàn bộ frame nằm sẵn trên VRAM, không worker, không copy host→device; augmentation theo lô trên GPU. |

**[V6-T2] Sửa danh sách augmentation — chỉ giữ những phép THỰC SỰ có tác dụng:**

| Phép | Tác dụng thật? | Ghi chú |
|---|---|---|
| Nhiễu Gauss theo SNR | **Có** | Làm thay đổi *hình dạng* phổ (nâng nền nhiễu) — không bị chuẩn hoá triệt tiêu |
| Đáp ứng kênh ngẫu nhiên (FIR) | **Có** (mới thêm) | Mô phỏng khác biệt cảm biến / khớp nối / điểm khảo sát — đúng loại bất biến mà bài toán cần |
| Dịch vòng thời gian | **Có** | Thay đổi nội dung cửa sổ |
| Đảo cực | **Chỉ với `LeakCNN1D`** | `|STFT|²` bất biến với đảo cực → vô tác dụng với 4 mô hình phổ |
| **Gain ±6 dB** | **KHÔNG — đã bỏ** | Front-end chuẩn hoá theo từng mẫu ⇒ mọi phép nhân gain bị triệt tiêu hoàn toàn (đo được sai khác 1.2e−7 giữa −6 dB và +6 dB) |

> Hai đường cho kết quả **thống kê tương đương** nhưng **không bit-identical**. Để có con số
> chốt, đặt `cfg.deterministic_run = True` (chạy tuần tự + `cudnn.deterministic=True`).


In [ ]:
# =========================================================
# 7.1a Đường cổ điển: Dataset + DataLoader (dự phòng)
#      [V6-T2] Bỏ gain (no-op), thêm đáp ứng kênh ngẫu nhiên, đảo cực chỉ khi có tác dụng.
# =========================================================
def rand_fir(rng, taps):
    # Đáp ứng xung ngẫu nhiên được làm trơn -> thay đổi ĐỘ DỐC PHỔ nhưng không đổi nhãn.
    # Đây là augmentation quan trọng nhất cho bài toán ít điểm khảo sát: nó buộc mô hình
    # bất biến với "dấu vân tay" của cảm biến/khớp nối/vị trí thu.
    taps = int(taps) if int(taps) % 2 == 1 else int(taps) + 1
    h = rng.standard_normal(taps) * np.hanning(taps)
    return (h / (np.linalg.norm(h) + 1e-12)).astype(np.float32)

class WaveDataset(Dataset):
    def __init__(self, X, y, idx, augment=False, seed=0, use_polarity=False):
        self.X, self.y, self.idx, self.augment = X, y, np.asarray(idx), augment
        self.use_polarity = bool(use_polarity)
        self.seed = int(seed)
        self.rng = np.random.default_rng(self.seed)
    def __len__(self): return len(self.idx)
    def _aug(self, x):
        r = self.rng
        if r.random() < cfg.aug_fir_p:
            x = sps.fftconvolve(x, rand_fir(r, cfg.aug_fir_taps), mode="same").astype(np.float32)
        if r.random() < cfg.aug_noise_p:
            snr = r.uniform(*cfg.aug_noise_snr_db)
            p_sig = float(np.mean(x ** 2)) + 1e-12
            x = x + r.standard_normal(len(x)).astype(np.float32) * np.sqrt(p_sig / (10 ** (snr / 10.0)))
        if r.random() < cfg.aug_shift_p:
            x = np.roll(x, int(r.integers(0, len(x))))
        if self.use_polarity and r.random() < 0.5:
            x = -x                      # chỉ có tác dụng với mô hình dạng sóng thô
        return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    def __getitem__(self, i):
        j = self.idx[i]
        x = np.asarray(self.X[j], dtype=np.float32).copy()
        if self.augment: x = self._aug(x)
        return torch.from_numpy(x), torch.tensor(self.y[j], dtype=torch.float32), int(j)

def _worker_init(worker_id):
    # SỬA LỖI: không có hàm này, mọi worker nhân bản cùng một trạng thái RNG
    # và sinh ra CÙNG một dãy số ngẫu nhiên.
    info = torch.utils.data.get_worker_info()
    if info is not None:
        info.dataset.rng = np.random.default_rng(int(info.seed) % (2 ** 32))

def _loader_kw(seed):
    return dict(num_workers=cfg.num_workers, pin_memory=(DEVICE.type == "cuda"),
                worker_init_fn=_worker_init, persistent_workers=cfg.num_workers > 0)

def make_eval_loader(idx, seed=0, batch_size=None):
    bs = batch_size or cfg.batch_size * 2
    return DataLoader(WaveDataset(X_ALL, y_all, idx, augment=False, seed=seed),
                      batch_size=bs, shuffle=False, **_loader_kw(seed))

def make_loaders(tr_idx, va_idx, seed=0, batch_size=None, use_polarity=False):
    bs = batch_size or cfg.batch_size
    dl_tr = DataLoader(WaveDataset(X_ALL, y_all, tr_idx, augment=True, seed=seed,
                                   use_polarity=use_polarity),
                       batch_size=bs, shuffle=True, drop_last=len(tr_idx) > bs * 2,
                       **_loader_kw(seed))
    return dl_tr, make_eval_loader(va_idx, seed)
print("Đường DataLoader cổ điển sẵn sàng.")


In [ ]:
# =========================================================
# 7.1b Đường nhanh: dữ liệu thường trú GPU + augmentation theo lô
#      [V6-T2] Cùng phép toán với đường CPU (bỏ gain, thêm FIR, đảo cực có điều kiện).
# =========================================================
_RESIDENT, _RES_LOCK, _BUILD_LOCK = {}, threading.Lock(), threading.Lock()

DATA_GB = X_ALL.nbytes / 1024 ** 3
GPU_RESIDENT = cfg.gpu_resident and DATA_GB <= cfg.gpu_resident_max_gb
if cfg.gpu_resident and not GPU_RESIDENT:
    print(f"!! Dữ liệu {DATA_GB:.1f} GB > ngưỡng {cfg.gpu_resident_max_gb} GB "
          f"-> quay về DataLoader cổ điển.")

def get_resident(device):
    key = str(device)
    with _RES_LOCK:
        if key not in _RESIDENT:
            arr = np.ascontiguousarray(np.asarray(X_ALL, dtype=np.float32))
            _RESIDENT[key] = torch.from_numpy(arr).to(device)
            print(f"  [fast] nạp {arr.nbytes/1024**2:.0f} MB frame lên {device}")
    return _RESIDENT[key]

def _fir_taps(n):
    return n if n % 2 == 1 else n + 1

def gpu_augment(xb, g, use_polarity=False):
    # Cùng phân phối với WaveDataset._aug:
    #   đáp ứng kênh ngẫu nhiên p=aug_fir_p | nhiễu Gauss SNR U(10,30) dB p=.5
    #   dịch vòng p=.5 | đảo cực p=.5 (chỉ khi có tác dụng)
    B, T = xb.shape; dev = xb.device
    rnd = lambda n: torch.rand(n, device=dev, generator=g)

    m = (rnd(B) < cfg.aug_fir_p)
    if bool(m.any()):
        taps = _fir_taps(cfg.aug_fir_taps)
        h = torch.randn(B, 1, taps, device=dev, generator=g)
        h = h * torch.hann_window(taps, periodic=False, device=dev)   # phổ mượt, không kiểu comb
        h = h / h.norm(dim=2, keepdim=True).clamp_min(1e-12)
        delta = torch.zeros(B, 1, taps, device=dev)
        delta[:, 0, taps // 2] = 1.0                                  # mẫu không chọn -> giữ nguyên
        h = torch.where(m.view(B, 1, 1), h, delta)
        # Lọc RIÊNG cho từng mẫu: đặt batch vào trục kênh rồi dùng grouped conv.
        xb = F.conv1d(xb.view(1, B, T), h, padding=taps // 2, groups=B).view(B, T)

    m = (rnd(B) < cfg.aug_noise_p).float()
    lo, hi = cfg.aug_noise_snr_db
    snr = rnd(B) * (hi - lo) + lo
    sigma = torch.sqrt(((xb ** 2).mean(1) + 1e-12) / (10 ** (snr / 10.0))) * m
    xb = xb + torch.randn(xb.shape, device=dev, generator=g) * sigma[:, None]

    m = rnd(B) < cfg.aug_shift_p
    if bool(m.any()):
        sh = torch.randint(0, T, (B,), device=dev, generator=g) * m
        idx = (torch.arange(T, device=dev)[None, :] - sh[:, None]) % T
        xb = torch.gather(xb, 1, idx)

    if use_polarity:
        xb = xb * torch.where(rnd(B) < 0.5, -1.0, 1.0)[:, None]
    return torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)

class DeviceBatches:
    def __init__(self, idx, device, bs, shuffle, augment, seed, drop_last=False,
                 use_polarity=False):
        self.idx = torch.as_tensor(np.asarray(idx), dtype=torch.long, device=device)
        self.y = torch.as_tensor(y_all, dtype=torch.float32, device=device)
        self.X = get_resident(device)
        self.device, self.bs, self.shuffle = device, bs, shuffle
        self.augment, self.drop_last = augment, drop_last
        self.use_polarity = bool(use_polarity)
        self.g = torch.Generator(device=device); self.g.manual_seed(int(seed))
    def __len__(self):
        n = len(self.idx)
        return max(1, n // self.bs) if self.drop_last else (n + self.bs - 1) // self.bs
    def __iter__(self):
        order = (self.idx[torch.randperm(len(self.idx), device=self.device, generator=self.g)]
                 if self.shuffle else self.idx)
        n = len(order); last = n - (n % self.bs) if self.drop_last else n
        for s in range(0, max(last, 1), self.bs):
            j = order[s:s + self.bs]
            if len(j) == 0: continue
            xb = self.X.index_select(0, j)
            if self.augment:
                xb = gpu_augment(xb, self.g, use_polarity=self.use_polarity)
            yield xb, self.y.index_select(0, j), j

def make_batches(tr_idx, va_idx, in_idx, seed=0, device=None, batch_size=None,
                 use_polarity=False):
    # Trả về ba nguồn batch: train / validation-trong / validation-ngoài
    device = device or DEVICE
    bs = batch_size or cfg.batch_size
    if GPU_RESIDENT:
        tr = DeviceBatches(tr_idx, device, bs, True, cfg.gpu_augment, seed,
                           drop_last=len(tr_idx) > bs * 2, use_polarity=use_polarity)
        vin = DeviceBatches(in_idx, device, bs * 2, False, False, seed + 1)
        vout = DeviceBatches(va_idx, device, bs * 2, False, False, seed + 2)
        return tr, vin, vout
    dl_tr, _ = make_loaders(tr_idx, va_idx, seed=seed, use_polarity=use_polarity)
    dl_in = make_eval_loader(in_idx, seed)
    dl_out = make_eval_loader(va_idx, seed)
    return dl_tr, dl_in, dl_out
print("Đường thường trú GPU:", GPU_RESIDENT, "| augment GPU:", cfg.gpu_augment and GPU_RESIDENT)

# ---------------------------------------------------------------------
# Túi (bag) cho học đa thể hiện - [V6-M3] MÃ NÀY KHÔNG CHẠY Ở CẤU HÌNH MẶC ĐỊNH.
# Chỉ bật khi mục 8.7 CHỨNG MINH được tiền đề MIL (tỉ lệ thể hiện dương << 1).
# Mục 8.7 đo tỉ lệ đó trên dữ liệu thật và in kết luận.
# ---------------------------------------------------------------------
FRAMES_BY_REC = frames.groupby("record_id").indices

class BagBatches:
    def __init__(self, idx, device, n_bags, bag_size, augment, seed, use_polarity=False):
        sel = set(np.asarray(idx).tolist())
        self.recs, self.labels = [], []
        for rid, fidx in FRAMES_BY_REC.items():
            keep = np.array([i for i in fidx if i in sel], dtype=np.int64)
            if len(keep) == 0: continue
            self.recs.append(keep); self.labels.append(float(y_all[keep[0]]))
        self.labels = np.asarray(self.labels, dtype=np.float32)
        self.device, self.n_bags, self.bag_size = device, max(1, n_bags), bag_size
        self.augment, self.use_polarity = augment, bool(use_polarity)
        self.X = get_resident(device)
        self.g = torch.Generator(device=device); self.g.manual_seed(int(seed))
        self.rng = np.random.default_rng(int(seed))
    def __len__(self):
        return max(1, len(self.recs) // self.n_bags)
    def __iter__(self):
        order = self.rng.permutation(len(self.recs))
        for s in range(0, len(order) - self.n_bags + 1, self.n_bags):
            sel = order[s:s + self.n_bags]
            picks = np.stack([self.rng.choice(self.recs[i], self.bag_size,
                                              replace=len(self.recs[i]) < self.bag_size)
                              for i in sel])
            j = torch.as_tensor(picks.reshape(-1), dtype=torch.long, device=self.device)
            xb = self.X.index_select(0, j)
            if self.augment:
                xb = gpu_augment(xb, self.g, use_polarity=self.use_polarity)
            xb = xb.view(len(sel), self.bag_size, -1)
            yb = torch.as_tensor(self.labels[sel], dtype=torch.float32, device=self.device)
            yield xb, yb, picks

class MILWrapper(nn.Module):
    # Gộp logit của các frame trong một túi thành logit mức bản ghi.
    #   lse | max | mean | attention (Ilse et al., 2018)
    def __init__(self, base, pooling=None, r=None):
        super().__init__()
        self.base = base
        self.pooling = pooling or cfg.mil_pooling
        self.r = r or cfg.mil_lse_r
        self.attn = None
    def _ensure_attn(self, d, device):
        if self.attn is None:
            self.attn = nn.Sequential(nn.Linear(d, 64), nn.Tanh(), nn.Linear(64, 1)).to(device)
    def forward(self, x):                       # x: [B, K, T]
        B, Kb, T = x.shape
        flat = x.reshape(B * Kb, T)
        if self.pooling == "attention":
            z = self.base.embed(flat)
            self._ensure_attn(z.shape[1], z.device)
            inst = self.base.head(z).reshape(B, Kb)
            a = torch.softmax(self.attn(z).reshape(B, Kb), dim=1)
            return (a * inst).sum(1), inst
        inst = self.base(flat).reshape(B, Kb)
        if self.pooling == "max":
            bag = inst.amax(1)
        elif self.pooling == "mean":
            bag = inst.mean(1)
        else:                                   # lse
            bag = (torch.logsumexp(self.r * inst, dim=1) - math.log(Kb)) / self.r
        return bag, inst

MIL_ON = (cfg.mil_mode == "bag")
if MIL_ON and not GPU_RESIDENT:
    print("!! mil_mode='bag' cần GPU_RESIDENT -> lùi về 'sil'.")
    MIL_ON = False
print("Chế độ huấn luyện:", "MIL mức túi (" + cfg.mil_pooling + ")" if MIL_ON
      else "SIL - mỗi frame mang nhãn bản ghi (mặc định)")
if MIL_ON:
    print("!! ĐANG BẬT MIL. Chỉ hợp lệ nếu mục 8.7 cho thấy tỉ lệ thể hiện dương << 1.")


### 7.2 Lựa chọn lồng — sửa ba kênh lạc quan / Nested selection

**Vấn đề.** Chọn epoch dừng sớm, ngưỡng quyết định và phép gộp mức bản ghi **trên chính tập sẽ báo cáo** làm chỉ số lạc quan một cách có hệ thống. Mô phỏng trên dữ liệu ngẫu nhiên thuần (AUC thật = 0,500) cho thấy độ lớn:

| Nguồn lạc quan | n nhỏ | n trung bình |
|---|---|---|
| Early stopping tốt nhất trong 40 epoch | 0,590 (n=200) | 0,540 (n=1000) |
| Chọn 1 trong 4 phép gộp mức bản ghi | +0,133 (20 bản ghi) | +0,072 (80 bản ghi) |
| Ngưỡng Youden tự chọn | +0,043 accuracy (n=200) | +0,010 (n=3000) |

Độ lớn này **cùng bậc với chính hiệu ứng rò rỉ dữ liệu** mà nghiên cứu đang phê phán.

**Cách sửa.** Tách từ phần train của mỗi fold ra một **validation-trong** (vẫn nhóm theo bản ghi). Validation-trong quyết định *mọi lựa chọn*; fold validation-ngoài **chỉ để báo cáo**.

Notebook ghi lại **cả hai** con số ở mỗi lần chạy — chọn lồng (trung thực) và chọn trên tập báo cáo (thiên lệch) — nên thầy có ngay một bảng đối chứng nữa cho bài báo, không tốn thêm lần huấn luyện nào.


In [ ]:
# =========================================================
# 7.2a Tách validation-trong (nhóm theo bản ghi)
# =========================================================
def record_auc_from_frames(idx, y, p, min_records=None):
    # Gộp TRUNG BÌNH - quy tắc cố định, định trước.
    min_records = cfg.select_min_records if min_records is None else min_records
    y = np.asarray(y); p = np.asarray(p)
    if len(np.unique(y)) < 2: return np.nan
    df = pd.DataFrame(dict(r=frames.record_id.values[np.asarray(idx)], y=y, p=p))
    g = df.groupby("r").agg(y=("y", "first"), p=("p", "mean"))
    if g.y.nunique() < 2 or len(g) < min_records: return np.nan
    return float(roc_auc_score(g.y, g.p))

def inner_split(tr_idx, seed, frac=None):
    # Tách validation-TRONG bằng phân tầng trên DANH SÁCH BẢN GHI của phần train.
    frac = frac or cfg.inner_frac
    recs = pd.unique(groups_all[tr_idx])
    ylab = np.array([int(rec_lab[r]) for r in recs])
    if len(recs) < 4 or len(np.unique(ylab)) < 2:
        return None
    n_test = max(2, int(round(frac * len(recs))))
    if n_test >= len(recs) - 1:
        return None
    try:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=n_test, random_state=int(seed))
        a, b = next(sss.split(np.zeros(len(recs)), ylab))
    except Exception:
        return None
    if len(np.unique(ylab[a])) < 2 or len(np.unique(ylab[b])) < 2:
        return None
    f_in = set(np.concatenate([rec_to_frames[r] for r in recs[b]]).tolist())
    tr_idx = np.asarray(tr_idx)
    mask = np.array([i in f_in for i in tr_idx])
    return tr_idx[~mask], tr_idx[mask]

NESTED_OK = []
for k, (tr, va) in enumerate(SPLITS):
    sp = inner_split(tr, cfg.seed + k)
    NESTED_OK.append(sp is not None)
    if sp is not None:
        a, b = sp
        assert not (set(groups_all[a]) & set(groups_all[b])), f"rò rỉ inner ở fold {k}"
        assert not (set(groups_all[b]) & set(groups_all[va])), f"inner chạm fold ngoài ở fold {k}"

if cfg.nested_selection and not all(NESTED_OK):
    print(f"!! {NESTED_OK.count(False)}/{len(SPLITS)} fold không tách nổi validation-trong "
          f"(quá ít bản ghi) -> các fold đó dùng lựa chọn không lồng, có ghi chú.")
print(f"Lựa chọn lồng: {cfg.nested_selection} | tỉ lệ validation-trong: {cfg.inner_frac}")

# [V6-T3] Kích thước thật của validation-trong quyết định ĐỘ NHIỄU của việc chọn epoch.
_in_sizes = []
for k, (tr, va) in enumerate(SPLITS):
    sp = inner_split(tr, cfg.seed + k)
    if sp is not None:
        in_rec = pd.unique(groups_all[sp[1]])
        _in_sizes.append(len(in_rec))
if _in_sizes:
    print(f"Validation-trong: {int(np.mean(_in_sizes))} bản ghi/fold "
          f"(min {min(_in_sizes)}, max {max(_in_sizes)}).")
    if np.mean(_in_sizes) < 15:
        print("!! Ít bản ghi -> AUC mức bản ghi trên tập này sẽ BÃO HOÀ ở 1.0 sau vài epoch.")
        print("   Vì vậy tiêu chí chọn epoch mặc định của v6 là log-loss (không bão hoà):")
        print(f"   cfg.select_metric = '{cfg.select_metric}'.")


In [ ]:
# =========================================================
# 7.2b Chỉ số đánh giá + THỐNG KÊ SUY LUẬN ĐÚNG ĐƠN VỊ (BẢN GHI)
#      [V6-S1/S2] Toàn bộ CI dùng bootstrap theo CỤM bản ghi. Bootstrap theo frame
#      cho CI hẹp hơn thực tế gần một bậc độ lớn (mục 8.2 đo trực tiếp hệ số này).
# =========================================================
def eer_from_roc(fpr, tpr):
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)

def youden_threshold(y, p):
    if len(np.unique(np.asarray(y))) < 2: return 0.5
    fpr, tpr, thr = roc_curve(y, p)
    thr = np.where(np.isfinite(thr), thr, 1.0)
    return float(thr[int(np.argmax(tpr - fpr))])

def sanitize_probs(p, name=""):
    p = np.asarray(p, dtype=float)
    bad = ~np.isfinite(p)
    if bad.any():
        print(f"  !! {int(bad.sum())}/{p.size} xác suất không hữu hạn"
              f"{(' (' + name + ')') if name else ''} -> thay bằng 0.5. Xem mục 7.0.")
        p = np.where(bad, 0.5, p)
    return np.clip(p, 0.0, 1.0)

def safe_bce(y, p):
    pc = np.clip(sanitize_probs(p), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return float(-np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc)))

def compute_metrics(y, p, thr=0.5):
    y = np.asarray(y).astype(int); p = sanitize_probs(p)
    yh = (p >= thr).astype(int)
    out = dict(threshold=float(thr), accuracy=accuracy_score(y, yh),
               balanced_acc=balanced_accuracy_score(y, yh))
    pr, rc, f1, _ = precision_recall_fscore_support(y, yh, average="binary", zero_division=0)
    out.update(precision=pr, recall=rc, f1=f1,
               mcc=matthews_corrcoef(y, yh) if len(set(y)) > 1 else np.nan)
    if len(set(y)) > 1:
        out["auc"] = roc_auc_score(y, p); out["ap"] = average_precision_score(y, p)
        fpr, tpr, _ = roc_curve(y, p); out["eer"] = eer_from_roc(fpr, tpr)
    else:
        out.update(auc=np.nan, ap=np.nan, eer=np.nan)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    out.update(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
               specificity=tn / (tn + fp + 1e-9), fpr=fp / (fp + tn + 1e-9),
               miss_rate=fn / (fn + tp + 1e-9))
    return out

# ---------------------------------------------------------------------
# Bootstrap theo CỤM BẢN GHI - hàm trung tâm của mọi khoảng tin cậy trong v6
# ---------------------------------------------------------------------
def _rec_index(rec_ids):
    rec_ids = np.asarray(rec_ids)
    uniq = np.unique(rec_ids)
    return uniq, {r: np.where(rec_ids == r)[0] for r in uniq}

def cluster_bootstrap_ci(rec_ids, y, p, fn=None, n=2000, alpha=0.05, seed=0):
    fn = fn or roc_auc_score
    y = np.asarray(y); p = np.asarray(p)
    uniq, idx_by = _rec_index(rec_ids)
    if len(uniq) < 3: return dict(lo=np.nan, hi=np.nan, n_valid=0, n_clusters=len(uniq))
    rng = np.random.default_rng(seed); vals = []
    for _ in range(int(n)):
        pick = rng.integers(0, len(uniq), len(uniq))
        ii = np.concatenate([idx_by[uniq[k]] for k in pick])
        if len(np.unique(y[ii])) < 2: continue
        try: vals.append(fn(y[ii], p[ii]))
        except Exception: pass
    if not vals: return dict(lo=np.nan, hi=np.nan, n_valid=0, n_clusters=len(uniq))
    return dict(lo=float(np.percentile(vals, 100*alpha/2)),
                hi=float(np.percentile(vals, 100*(1-alpha/2))),
                n_valid=len(vals), n_clusters=int(len(uniq)), n=len(y))

def naive_frame_bootstrap_ci(y, p, fn=None, n=2000, alpha=0.05, seed=0):
    # CHỈ để ĐO mức hẹp giả tạo của cách làm cũ (không dùng để báo cáo).
    fn = fn or roc_auc_score
    y = np.asarray(y); p = np.asarray(p)
    rng = np.random.default_rng(seed); vals = []
    for _ in range(int(n)):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        vals.append(fn(y[i], p[i]))
    if not vals: return dict(lo=np.nan, hi=np.nan)
    return dict(lo=float(np.percentile(vals, 100*alpha/2)),
                hi=float(np.percentile(vals, 100*(1-alpha/2))))

def paired_cluster_delta_ci(rec_ids, y, pa, pb, fn=None, n=2000, alpha=0.05, seed=0):
    # CI GHÉP CẶP cho Delta = fn(a) - fn(b), lấy mẫu lại BẢN GHI (giữ ghép cặp).
    fn = fn or roc_auc_score
    y = np.asarray(y); pa = np.asarray(pa); pb = np.asarray(pb)
    uniq, idx_by = _rec_index(rec_ids)
    rng = np.random.default_rng(seed); vals = []
    for _ in range(int(n)):
        pick = rng.integers(0, len(uniq), len(uniq))
        ii = np.concatenate([idx_by[uniq[k]] for k in pick])
        if len(np.unique(y[ii])) < 2: continue
        try: vals.append(fn(y[ii], pa[ii]) - fn(y[ii], pb[ii]))
        except Exception: pass
    if not vals: return dict(lo=np.nan, hi=np.nan, p_two_sided=np.nan)
    lo, hi = float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))
    pv = float(2 * min(np.mean(np.asarray(vals) <= 0), np.mean(np.asarray(vals) >= 0)))
    return dict(lo=lo, hi=hi, p_two_sided=min(1.0, pv), n_valid=len(vals))

def mcnemar_test(y_true, pred_a, pred_b):
    # Dùng được ở MỌI đơn vị (frame hoặc BẢN GHI). Chỉ có ý nghĩa suy luận khi
    # đơn vị là BẢN GHI.
    y = np.asarray(y_true).astype(int)
    a = np.asarray(pred_a).astype(int) == y
    b = np.asarray(pred_b).astype(int) == y
    n01 = int(np.sum(a & ~b)); n10 = int(np.sum(~a & b))
    if n01 + n10 == 0: return dict(n01=0, n10=0, stat=np.nan, p=1.0)
    if n01 + n10 < 25:
        p = float(spstats.binomtest(min(n01, n10), n01 + n10, .5).pvalue); stat = np.nan
    else:
        stat = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
        p = float(1 - spstats.chi2.cdf(stat, 1))
    return dict(n01=n01, n10=n10, stat=stat, p=p)

def holm(p):
    p = np.asarray(p, dtype=float); o = np.argsort(p); m = len(p); adj = np.empty(m)
    run = 0.0
    for r, i in enumerate(o):
        v = (m - r) * p[i]; run = max(run, v); adj[i] = min(1.0, run)
    return adj

def wilcoxon_min_p(k):
    # [V6-S2] Với k fold, p nhỏ nhất mà Wilcoxon signed-rank CÓ THỂ đạt là 2/2**k.
    # k=5 -> 0.0625: KHÔNG BAO GIỜ bác bỏ được ở mức 0.05. Phải in ra để không
    # ai trích dẫn p-value vô nghĩa.
    return float(2.0 / (2 ** int(k)))

# ---------------------------------------------------------------------
# Phân rã phương sai + ICC ([V6] chuyển lên đây để mục 8.2 dùng được design effect)
# ---------------------------------------------------------------------
def _normal_scores(p):
    p = np.asarray(p, dtype=float)
    r = spstats.rankdata(p, method="average")
    return spstats.norm.ppf((r - 0.5) / len(p))

def design_effect(icc, m_bar):
    return float(1.0 + (m_bar - 1.0) * float(icc))

def variance_components(idx, y, p):
    # Phân rã phương sai điểm chuẩn tắc SAU KHI trừ trung bình theo lớp, nên sb2 là
    # phương sai GIỮA CÁC BẢN GHI cùng lớp (không lẫn hiệu ứng lớp).
    z = _normal_scores(p); y = np.asarray(y).astype(int)
    rec = frames.record_id.values[np.asarray(idx)]
    d = pd.DataFrame(dict(rec=rec, y=y, z=z))
    mu = d.groupby("y").z.mean()
    d["r"] = d.z - d.y.map(mu)
    g = d.groupby("rec").r
    n_i = g.size().values
    m_i = g.mean().values
    within = float(((d.r - d.rec.map(g.mean())) ** 2).sum() / max(len(d) - len(n_i), 1))
    k = len(n_i); N = n_i.sum()
    n0 = (N - (n_i ** 2).sum() / N) / max(k - 1, 1)
    msb = float((n_i * (m_i - m_i.mean()) ** 2).sum() / max(k - 1, 1))
    between = max((msb - within) / max(n0, 1e-9), 0.0)
    return dict(mu=float(mu.get(1, 0) - mu.get(0, 0)), sb2=between, se2=within,
                icc=between / (between + within + 1e-12), K_eff=float(n0),
                n_rec=k, n_frames=int(N), m_bar=float(N / max(k, 1)),
                sat=float(np.mean((np.asarray(p) < 1e-3) | (np.asarray(p) > 1 - 1e-3))))

# ---------------------------------------------------------------------
# [V6-T9] Hiệu chỉnh xác suất Platt, CHỈ fit trên validation-TRONG (lồng nhau)
# ---------------------------------------------------------------------
def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def platt_fit(p_in, y_in):
    z = _logit(p_in).reshape(-1, 1); y = np.asarray(y_in).astype(int)
    if len(np.unique(y)) < 2: return (1.0, 0.0)
    lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000).fit(z, y)
    return (float(lr.coef_[0, 0]), float(lr.intercept_[0]))

def platt_apply(p, ab):
    a, b = ab
    return 1.0 / (1.0 + np.exp(-(a * _logit(p) + b)))

# ---------------------------------------------------------------------
# [V6-T3] Tiêu chí chọn epoch - quy ước: GIÁ TRỊ LỚN HƠN LÀ TỐT HƠN
# ---------------------------------------------------------------------
def _safe_auc(y, p):
    y = np.asarray(y)
    if len(np.unique(y)) < 2: return np.nan
    return float(roc_auc_score(y, p))

SELECTION_CRITERIA = {
    # log-loss: KHÔNG bão hoà, ổn định trên tập nhỏ -> mặc định của v6
    "inner_logloss":    lambda idx, y, p: (-safe_bce(y, p)) if len(np.unique(y)) > 1 else np.nan,
    "inner_frame_auc":  lambda idx, y, p: _safe_auc(y, p),
    "inner_record_auc": lambda idx, y, p: record_auc_from_frames(idx, y, p),
}
CRITERION_LABEL = {"inner_logloss": "log-loss (thấp hơn = tốt hơn)",
                   "inner_frame_auc": "AUC mức frame",
                   "inner_record_auc": "AUC mức bản ghi (dễ bão hoà)"}
def record_auc_simple(oof, agg=None):
    # [V7] AUC mức bản ghi của một vector OOF (gộp theo quy tắc đăng ký trước).
    agg = agg or cfg.record_agg
    mk = ~np.isnan(oof)
    df = pd.DataFrame(dict(r=groups_all[mk], y=y_all[mk], p=oof[mk]))
    fn = {"p_mean": "mean", "p_med": "median"}.get(agg, "mean")
    g = df.groupby("r").agg(y=("y", "first"), p=("p", fn))
    return (float(roc_auc_score(g.y, g.p)) if g.y.nunique() > 1 else np.nan), g

print("Metrics OK | bootstrap theo cụm bản ghi | tiêu chí chọn:", list(SELECTION_CRITERIA))


In [ ]:
# =========================================================
# 7.3 Huấn luyện một fold / Single-fold training engine
#   [V6-T1] Lịch LR hoàn tất trong cfg.sched_horizon_epochs; không chọn checkpoint
#           trong pha warm-up (MIN_EPOCHS_EFF >= đỉnh LR + 2).
#   [V6-T3] Chọn epoch theo tiêu chí KHÔNG bão hoà, có làm mịn; LƯU SONG SONG cả ba
#           tiêu chí -> ablation tiêu chí chọn miễn phí (không tốn thêm lượt huấn luyện).
#   [V6-T9] Fit Platt trên validation-trong, lưu cả xác suất thô và đã hiệu chỉnh.
#   [V7-B2] THỰC THI "chỉ chọn checkpoint từ epoch SELECT_FROM_EPOCH" (v6 chỉ khai báo).
#   [V7]    Tuỳ chọn trọng số mức bản ghi (cfg.record_balanced_loss).
# =========================================================
@torch.no_grad()
def predict(model, batches, device):
    model.eval(); P, Y, I = [], [], []
    for xb, yb, ib in batches:
        xb = xb.to(device, non_blocking=True)
        with amp_autocast(device, cfg.amp and device.type == "cuda"):
            logit = model(xb)
        pr = torch.sigmoid(logit.float())
        pr = torch.nan_to_num(pr, nan=0.5, posinf=1.0, neginf=0.0).clamp(0.0, 1.0)
        P.append(pr.cpu().numpy())
        Y.append(yb.cpu().numpy() if torch.is_tensor(yb) else np.asarray(yb))
        I.append(ib.cpu().numpy() if torch.is_tensor(ib) else np.asarray(ib))
    return np.concatenate(P), np.concatenate(Y), np.concatenate(I)

def _snapshot(model):
    return {k: (v.detach().half() if (cfg.ckpt_fp16 and v.is_floating_point()) else v.detach()).cpu().clone()
            for k, v in model.state_dict().items()}

def _make_scheduler(opt, steps_per_epoch):
    total_steps = max(1, int(steps_per_epoch) * int(cfg.sched_horizon_epochs))
    if cfg.lr_schedule == "cosine":
        warm = max(1, int(cfg.warmup_frac * total_steps))
        def lr_fn(step):
            if step < warm:
                return float(step + 1) / float(warm)
            prog = min(1.0, (step - warm) / max(1, total_steps - warm))
            return 1e-3 + (1 - 1e-3) * 0.5 * (1 + math.cos(math.pi * prog))
        return torch.optim.lr_scheduler.LambdaLR(opt, lr_fn), total_steps
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg.lr * 5, total_steps=total_steps, pct_start=.25,
        div_factor=10, final_div_factor=100)
    return sch, total_steps

def train_one_fold(model_name, tr_idx, va_idx, fold, device=None, verbose=True,
                   seed_base=None, tag=""):
    device = device or DEVICE
    seed_base = cfg.seed if seed_base is None else int(seed_base)
    fold_seed = seed_base + fold
    # [V6] Không phụ thuộc NESTED_OK[fold] nữa: nhờ vậy có thể chạy được cả
    # leave-one-site-out và các bộ fold khác (ablation) mà không lệch chỉ số.
    use_nested = bool(cfg.nested_selection)
    sp = inner_split(tr_idx, seed_base + fold) if use_nested else None
    if sp is None:
        tr_fit, in_idx, use_nested = tr_idx, va_idx, False   # thoái lui: chọn trên tập ngoài
    else:
        tr_fit, in_idx = sp

    with _BUILD_LOCK:                     # khởi tạo trọng số phải tất định
        set_seed(fold_seed, deterministic=cfg.deterministic_run)
        model = build_model(model_name).to(device)
    # [V6-T5] generator RIÊNG của fold cho SpecAugment (không dùng RNG toàn cục)
    set_specaug_generators(model, make_generator("cpu", fold_seed * 7919 + 13))
    use_pol = model_uses_raw_waveform(model_name)

    dl_tr, dl_in, dl_out = make_batches(tr_fit, va_idx, in_idx, seed=fold_seed,
                                        device=device, use_polarity=use_pol)
    if MIL_ON:
        n_bags = max(2, cfg.batch_size // cfg.bag_size)
        dl_tr = BagBatches(tr_fit, device, n_bags, cfg.bag_size,
                           cfg.gpu_augment, fold_seed, use_polarity=use_pol)
        trainer = MILWrapper(model).to(device)
        with torch.no_grad():
            _d = next(iter(dl_tr))[0][:1]
            trainer(_d)
        bag_y = dl_tr.labels
        pos, neg = float(bag_y.sum()), float(len(bag_y) - bag_y.sum())
    else:
        trainer = model
        pos = float(y_all[tr_fit].sum()); neg = float(len(tr_fit) - pos)

    W_ALL = None
    if cfg.record_balanced_loss and not MIL_ON:
        # mỗi BẢN GHI đóng góp tổng trọng số như nhau -> khớp với đơn vị đánh giá
        _nrec = frames.groupby("record_id").size()
        _w = (1.0 / frames.record_id.map(_nrec).values).astype(np.float32)
        _w = _w / float(_w[np.asarray(tr_fit)].mean())
        W_ALL = torch.as_tensor(_w, device=device)
        _rl = pd.Series(y_all[tr_fit]).groupby(groups_all[tr_fit]).first()
        pos, neg = float(_rl.sum()), float(len(_rl) - _rl.sum())   # cân bằng lớp theo BẢN GHI
    pos_weight = neg / max(pos, 1.0)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device),
                                reduction=("none" if W_ALL is not None else "mean"))
    opt = torch.optim.AdamW(trainer.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched, total_steps = _make_scheduler(opt, max(1, len(dl_tr)))
    scaler = make_grad_scaler(cfg.amp and device.type == "cuda")
    eps = cfg.label_smoothing

    hist = defaultdict(list); n_skip = 0; t0 = time.time()
    # một "best" cho MỖI tiêu chí (cùng một lượt huấn luyện, không tốn thêm gì)
    crits = list(SELECTION_CRITERIA) if cfg.criteria_ablation else [cfg.select_metric]
    if cfg.select_metric not in crits:
        crits.append(cfg.select_metric)
    if "inner_logloss" not in crits:
        crits.append("inner_logloss")     # [V6] lưới an toàn: luôn có 1 tiêu chí không bão hoà
    best = {c: dict(score=-np.inf, epoch=-1, state=None,
                    p_out=None, y_out=None, i_out=None,
                    p_in=None, y_in=None, i_in=None) for c in crits}
    best_out = dict(auc=-np.inf, epoch=-1, p_out=None, y_out=None, i_out=None)
    scores_hist = {c: [] for c in crits}
    W = max(1, int(cfg.select_smooth_w))

    for ep in range(cfg.epochs):
        trainer.train(); run_loss, n_seen, n_used = 0.0, 0, 0
        for xb, yb, ib in dl_tr:
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            yb_s = yb * (1 - eps) + eps / 2
            opt.zero_grad(set_to_none=True)
            with amp_autocast(device, cfg.amp and device.type == "cuda"):
                out = trainer(xb)
                loss = crit(out[0] if MIL_ON else out, yb_s)
                if W_ALL is not None:
                    _ib = torch.as_tensor(ib, device=device).long()
                    loss = (loss * W_ALL.index_select(0, _ib)).mean()
            n_seen += xb.size(0)
            if not torch.isfinite(loss):
                n_skip += 1
                opt.zero_grad(set_to_none=True)
                continue                     # [V6] KHÔNG step scheduler cho batch bị bỏ
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainer.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item() * xb.size(0); n_used += xb.size(0)
        if n_used == 0:
            print(f"    !! [{model_name} f{fold}] toàn bộ batch epoch {ep} không hữu hạn - dừng.")
            break
        tr_loss = run_loss / n_used

        p_in, y_in, i_in = predict(model, dl_in, device)
        m_in = compute_metrics(y_in, p_in, 0.5)
        p_out, y_out, i_out = ((p_in, y_in, i_in) if not use_nested
                               else predict(model, dl_out, device))
        m_out = compute_metrics(y_out, p_out, 0.5)

        hist["epoch"].append(ep); hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(safe_bce(y_out, p_out))
        hist["inner_loss"].append(safe_bce(y_in, p_in))
        hist["inner_auc"].append(m_in["auc"]); hist["val_auc"].append(m_out["auc"])
        hist["val_acc"].append(m_out["accuracy"]); hist["lr"].append(opt.param_groups[0]["lr"])
        hist["inner_record_auc"].append(record_auc_from_frames(i_in, y_in, p_in))

        # --- cập nhật best cho MỌI tiêu chí (có làm mịn cửa sổ W) ---
        for c in crits:
            v = SELECTION_CRITERIA[c](i_in, y_in, p_in)
            scores_hist[c].append(v)
            # làm mịn CHỈ trên các epoch được phép chọn (không trộn điểm của pha warm-up)
            _e0 = max(SELECT_FROM_EPOCH, ep - W + 1)
            vv = np.asarray(scores_hist[c][_e0:ep + 1], dtype=float)
            vv = vv[np.isfinite(vv)]
            s = float(vv.mean()) if len(vv) else np.nan
            if ep >= SELECT_FROM_EPOCH and np.isfinite(s) and s > best[c]["score"] + 1e-6:
                best[c] = dict(score=s, epoch=ep, state=_snapshot(model),
                               p_out=p_out, y_out=y_out, i_out=i_out,
                               p_in=p_in, y_in=y_in, i_in=i_in)
        s_out = m_out["auc"] if m_out["auc"] == m_out["auc"] else m_out["accuracy"]
        if ep >= SELECT_FROM_EPOCH and s_out > best_out["auc"] + 1e-5:   # cùng ràng buộc -> so sánh công bằng
            best_out = dict(auc=s_out, epoch=ep, p_out=p_out, y_out=y_out, i_out=i_out)

        if verbose and (ep % max(1, cfg.epochs // 8) == 0 or ep == cfg.epochs - 1):
            print(f"    [{model_name} f{fold}@{device}] ep{ep:03d} loss {tr_loss:.4f} | "
                  f"trong: auc {m_in['auc']:.4f} loss {hist['inner_loss'][-1]:.4f} "
                  f"rec_auc {hist['inner_record_auc'][-1]:.4f} | ngoài auc {m_out['auc']:.4f} | "
                  f"lr {hist['lr'][-1]:.2e}")

        # --- dừng sớm theo TIÊU CHÍ CHÍNH, chỉ tính sau MIN_EPOCHS_EFF ---
        if best[cfg.select_metric]["epoch"] >= 0:
            primary = best[cfg.select_metric]["epoch"]
        elif best.get("inner_logloss", {}).get("epoch", -1) >= 0:
            primary = best["inner_logloss"]["epoch"]   # tiêu chí chính không hữu hạn
        else:
            primary = ep                              # chưa có gì để so -> không dừng sớm
        if ep >= MIN_EPOCHS_EFF and (ep - primary) >= cfg.early_stop_patience:
            if verbose:
                print(f"    early stop @ep{ep} (chọn ep{primary} theo '{cfg.select_metric}'; "
                      f"min_epochs hiệu dụng {MIN_EPOCHS_EFF}, đỉnh LR ep{LR_PEAK_EPOCH})")
            break

    dt = time.time() - t0
    if best[cfg.select_metric]["state"] is None:
        # [V6] Thoái lui an toàn: nếu tiêu chí chính không bao giờ có giá trị hữu hạn
        # (ví dụ AUC bản ghi bão hoà/không tính được), dùng tiêu chí log-loss.
        fallback = "inner_logloss" if best.get("inner_logloss", {}).get("state") is not None else None
        if fallback is None:
            raise RuntimeError(f"{model_name} fold {fold}: không có epoch hợp lệ nào. "
                               f"Giảm cfg.lr hoặc tăng cfg.epochs rồi chạy lại.")
        print(f"    !! [{model_name} f{fold}] tiêu chí '{cfg.select_metric}' không hữu hạn "
              f"-> dùng '{fallback}'.")
        cfg_select_used = fallback
    else:
        cfg_select_used = cfg.select_metric

    sel = best[cfg_select_used]
    # [V6-T9] Platt fit trên validation-TRONG của fold này (không đụng dữ liệu báo cáo)
    platt = platt_fit(sel["p_in"], sel["y_in"]) if cfg.calibrate_aggregation else (1.0, 0.0)
    if n_skip and verbose:
        print(f"    [{model_name} f{fold}] bỏ qua {n_skip} batch do loss không hữu hạn "
              f"(GradScaler/fp16) - đây là lý do KHÔNG tắt AMP toàn cục.")

    out = dict(model=model_name, fold=fold, history=dict(hist), nested=use_nested,
               best_epoch=sel["epoch"], select_by=cfg_select_used,
               mil=MIL_ON, platt=platt, n_skipped=int(n_skip),
               lr_peak_epoch=int(LR_PEAK_EPOCH),
               selected_after_lr_peak=bool(sel["epoch"] >= LR_PEAK_EPOCH),
               val_probs=sel["p_out"], val_y=sel["y_out"], val_idx=sel["i_out"],
               inner_probs=sel["p_in"], inner_y=sel["y_in"], inner_idx=sel["i_in"],
               val_probs_cal=platt_apply(sel["p_out"], platt),
               inner_probs_cal=platt_apply(sel["p_in"], platt),
               opt_probs=best_out["p_out"], opt_y=best_out["y_out"], opt_idx=best_out["i_out"],
               opt_epoch=best_out["epoch"], state=sel["state"], train_time_s=dt,
               score_history=dict(scores_hist), tag=tag,
               alt=({c: dict(epoch=best[c]["epoch"], score=best[c]["score"],
                             p_out=best[c]["p_out"], y_out=best[c]["y_out"],
                             i_out=best[c]["i_out"], p_in=best[c]["p_in"],
                             y_in=best[c]["y_in"], i_in=best[c]["i_in"]) for c in crits}))
    del model, trainer; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return out
print("Training engine OK (v6)")


In [ ]:
# =========================================================
# 7.4 Chạy cross-validation - hàng đợi fold trên các GPU (cân bằng tải thật)
#     [V6-P2] v5 gán cứng device = k % ngpu -> fold 0,2,4 dồn vào một GPU.
# =========================================================
import queue as _queue

def _prewarm():
    # Tải trọng số ImageNet MỘT LẦN trước nhánh song song, tránh hai thread
    # cùng ghi vào cache của torch hub.
    if "TransferCNN" in cfg.models:
        try:
            with _BUILD_LOCK:
                m = build_model("TransferCNN")
                if not getattr(m, "pretrained", False):
                    print("  !! TransferCNN không có trọng số ImageNet -> kết quả transfer "
                          "learning không hợp lệ (cần bật Internet trên Kaggle).")
                del m
        except Exception as e:
            print("  [prewarm TransferCNN]", type(e).__name__, e)

def _run_folds_parallel(model_name, splits, devices, seed_base):
    tasks = _queue.Queue()
    for k, (tr, va) in enumerate(splits):
        tasks.put((k, tr, va))
    out, lock = {}, threading.Lock()

    def worker(dev):
        while True:
            try:
                k, tr, va = tasks.get_nowait()
            except _queue.Empty:
                return
            try:
                r = train_one_fold(model_name, tr, va, k, dev, True, seed_base)
            except Exception as e:
                r = e
            with lock:
                out[k] = r
            tasks.task_done()

    ths = [threading.Thread(target=worker, args=(d,), daemon=True) for d in devices]
    for t in ths: t.start()
    for t in ths: t.join()
    return out

def run_cv(model_name, splits=None, seed_base=None, tag=""):
    splits = SPLITS if splits is None else splits
    seed_base = cfg.seed if seed_base is None else int(seed_base)
    ngpu = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
    par = cfg.parallel_folds and ngpu > 1 and len(splits) > 1
    print(f"\n=== {model_name} | {len(splits)} fold | seed {seed_base} | "
          f"{'song song ' + str(min(ngpu, cfg.max_parallel_folds)) + ' GPU' if par else 'tuần tự'} ===")
    if par:
        devices = [torch.device(f"cuda:{i}") for i in range(min(ngpu, cfg.max_parallel_folds))]
        res = _run_folds_parallel(model_name, splits, devices, seed_base)
        failed = {k: v for k, v in res.items() if isinstance(v, Exception)}
        if failed:
            for k, e in failed.items():
                print(f"  !! fold {k} lỗi song song: {type(e).__name__}: {e} -> chạy lại tuần tự")
            for k in sorted(failed):
                tr, va = splits[k]
                res[k] = train_one_fold(model_name, tr, va, k, DEVICE, True, seed_base)
        return [res[k] for k in sorted(res)]
    return [train_one_fold(model_name, tr, va, k, DEVICE, True, seed_base)
            for k, (tr, va) in enumerate(splits)]

def assemble_oof(results, key="val"):
    oof = np.full(len(frames), np.nan)
    for r in results:
        oof[r[f"{key}_idx"]] = r[f"{key}_probs"]
    return oof

def pool_inner(results):
    # Dự đoán trên validation-TRONG gộp qua các fold (chỉ số có thể lặp giữa các
    # fold - chấp nhận được vì chỉ dùng để CHỌN, không để báo cáo).
    y = np.concatenate([r["inner_y"] for r in results])
    p = np.concatenate([r["inner_probs"] for r in results])
    i = np.concatenate([r["inner_idx"] for r in results])
    f = np.concatenate([np.full(len(r["inner_y"]), r["fold"]) for r in results])
    return y, p, i, f

def pool_inner_cal(results):
    # [V6-T9] Như pool_inner nhưng trên xác suất ĐÃ hiệu chỉnh Platt của TỪNG fold
    # (mỗi fold có hệ số Platt riêng -> phải áp đúng hệ số của fold đó).
    y = np.concatenate([r["inner_y"] for r in results])
    p = np.concatenate([r["inner_probs_cal"] for r in results])
    i = np.concatenate([r["inner_idx"] for r in results])
    f = np.concatenate([np.full(len(r["inner_y"]), r["fold"]) for r in results])
    return y, p, i, f

def all_fold_predictions(model_name, splits=None, device=None):
    # [V6-T9] Ma trận [K fold x N frame]: mỗi fold-model dự đoán TOÀN BỘ frame.
    # Dùng để (i) mô phỏng ensemble "tệp hoàn toàn mới" một cách trung thực và
    # (ii) chọn ngưỡng cho ensemble thay vì dùng ngưỡng của mô hình đơn.
    splits = SPLITS if splits is None else splits
    device = device or DEVICE
    M = np.full((len(splits), len(frames)), np.nan, dtype=np.float32)
    for k in range(len(splits)):
        m = build_model(model_name)
        sd = torch.load(CKPT_DIR / f"{model_name}_fold{k}.pt", map_location="cpu")
        m.load_state_dict({kk: vv.float() for kk, vv in sd.items()})
        m.to(device).eval()
        for s0 in range(0, len(frames), 256):
            xb = torch.from_numpy(np.asarray(X_ALL[s0:s0+256])).float().to(device)
            with torch.no_grad(), amp_autocast(device, cfg.amp and device.type == "cuda"):
                pr = torch.sigmoid(m(xb).float())
            M[k, s0:s0+256] = pr.cpu().numpy()
        del m; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    return M

_prewarm()
print("CV runner OK (v6)")


In [ ]:
# =========================================================
# 7.4b ĐO TỐC ĐỘ (tuỳ chọn) - so hai đường dữ liệu trên chính phần cứng của bạn
# =========================================================
def benchmark_epoch(model_name="LeakCNN2D", fold=0, max_batches=30):
    if not GPU_RESIDENT:
        print("GPU_RESIDENT = False -> không có gì để so."); return None
    tr_idx, va_idx = SPLITS[fold]
    use_pol = model_uses_raw_waveform(model_name)
    rows = {}
    for tag in ["A. DataLoader + augment CPU", "B. thường trú GPU + augment GPU"]:
        set_seed(cfg.seed, deterministic=cfg.deterministic_run)
        model = build_model(model_name).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
        scaler = make_grad_scaler(cfg.amp and DEVICE.type == "cuda")
        crit = nn.BCEWithLogitsLoss()
        dl = (make_loaders(tr_idx, va_idx, seed=0, use_polarity=use_pol)[0] if tag.startswith("A")
              else DeviceBatches(tr_idx, DEVICE, cfg.batch_size, True, cfg.gpu_augment, 0,
                                 use_polarity=use_pol))
        model.train(); nb = 0
        for xb, yb, _ in dl:          # warm-up
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            with amp_autocast(DEVICE, cfg.amp and DEVICE.type == "cuda"):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True); nb += 1
            if nb >= 3: break
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0, nb, ns = time.time(), 0, 0
        for xb, yb, _ in dl:
            if nb >= max_batches: break
            xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE, non_blocking=True)
            with amp_autocast(DEVICE, cfg.amp and DEVICE.type == "cuda"):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True); nb += 1; ns += xb.size(0)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        dt = time.time() - t0
        rows[tag] = dict(batches=nb, ms_per_batch=round(dt / max(nb, 1) * 1000, 1),
                         samples_per_s=round(ns / max(dt, 1e-9)))
        del model, opt, dl; gc.collect(); torch.cuda.empty_cache()
    df = pd.DataFrame(rows).T
    display(Markdown(f"### Throughput benchmark — {model_name}, fold {fold}"))
    display(df)
    sp = df.loc["A. DataLoader + augment CPU", "ms_per_batch"] / \
         max(df.loc["B. thường trú GPU + augment GPU", "ms_per_batch"], 1e-9)
    print(f"Tăng tốc mỗi bước: {sp:.2f}x")
    return df

# benchmark_epoch()   # <- bỏ dấu # để chạy


### 7.5 Huấn luyện toàn bộ mô hình + CV lặp / Training with repeated cross-validation

**[V7-S1]** Với vài chục bản ghi, *một* phân hoạch 5-fold cho kết quả phụ thuộc đáng kể vào
việc bản ghi nào rơi vào fold nào. v6 chỉ chạy một phân hoạch và một seed, nên mọi chênh lệch
AUC giữa các kiến trúc đều lẫn với nhiễu phân hoạch.

Cell dưới đây chạy **`len(cfg.seeds)` lần CV**, mỗi lần một **phân hoạch mới** và một khởi tạo mới:

- **Lần lặp đầu** (seed `cfg.seed`) là lần chạy chính: checkpoint, ngưỡng, diễn giải (mục 10), triển khai (mục 13–15).
- **Bảng 5b** báo cáo trung bình ± độ lệch chuẩn qua các lần lặp. Chênh lệch giữa hai kiến trúc
  nhỏ hơn độ lệch chuẩn này **không diễn giải được**.
- **Trung bình xác suất OOF qua các lần lặp** là ước lượng ổn định nhất và **vẫn trung thực**:
  mỗi dự đoán của một frame đến từ các mô hình **chưa từng thấy bản ghi chứa frame đó**.

In [ ]:
# =========================================================
# 7.5 HUẤN LUYỆN TOÀN BỘ MÔ HÌNH + CV LẶP / TRAIN ALL MODELS WITH REPEATED CV
#     [V7-S1] Mỗi seed trong cfg.seeds = một PHÂN HOẠCH fold mới + khởi tạo mới.
#             Lần lặp 0 là lần chạy chính (checkpoint, diễn giải, triển khai).
# =========================================================
ALL_RESULTS, OOF, OOF_OPT, INNER, OOF_CAL = {}, {}, {}, {}, {}
INNER_CAL = {}
OOF_REPS = defaultdict(list)                     # OOF của từng lần lặp
SPLITS_REPS = [SPLITS] + [build_splits(s) for s in cfg.seeds[1:]]
for _sp in SPLITS_REPS[1:]:
    check_splits(_sp)
timing, rep_rows = [], []

for ri, sd in enumerate(cfg.seeds):
    print(f"\n######## CV LẶP {ri + 1}/{len(cfg.seeds)} (seed {sd}) ########")
    for name in cfg.models:
        t0 = time.time()
        res = run_cv(name, splits=SPLITS_REPS[ri], seed_base=sd, tag=f"rep{ri}")
        oo = assemble_oof(res, "val")
        OOF_REPS[name].append(oo)
        if ri == 0:
            ALL_RESULTS[name] = res
            OOF[name] = oo                                   # trung thực (epoch chọn bởi inner)
            OOF_OPT[name] = assemble_oof(res, "opt")         # thiên lệch (epoch tốt nhất trên tập báo cáo)
            INNER[name] = pool_inner(res)
            INNER_CAL[name] = pool_inner_cal(res)
            oof_c = np.full(len(frames), np.nan)
            for r in res:
                oof_c[r["val_idx"]] = r["val_probs_cal"]
            OOF_CAL[name] = oof_c
            if cfg.save_models:
                for r in res:
                    torch.save(r["state"], CKPT_DIR / f"{name}_fold{r['fold']}.pt")
            timing.append(dict(model=name, total_s=time.time() - t0,
                               per_fold_s=np.mean([r["train_time_s"] for r in res]),
                               mean_best_epoch=np.mean([r["best_epoch"] for r in res]),
                               min_best_epoch=int(np.min([r["best_epoch"] for r in res])),
                               frac_after_lr_peak=np.mean([r["selected_after_lr_peak"] for r in res]),
                               nested_folds=int(sum(r["nested"] for r in res)),
                               skipped_batches=int(sum(r["n_skipped"] for r in res))))
        mk = ~np.isnan(oo)
        auc_r, _ = record_auc_simple(oo)
        rep_rows.append(dict(model=name, repeat=ri, seed=sd,
                             auc_frame=float(roc_auc_score(y_all[mk], oo[mk])), auc_record=auc_r,
                             mean_best_epoch=float(np.mean([r["best_epoch"] for r in res]))))
        print(f"--> {name} (lặp {ri}, seed {sd}): {time.time()-t0:.1f}s | OOF phủ "
              f"{np.mean(mk)*100:.1f}% | AUC frame {rep_rows[-1]['auc_frame']:.4f} | "
              f"AUC bản ghi ({cfg.record_agg}) {auc_r:.4f}")
        if ri > 0:
            for r in res:
                r["state"] = None
            del res; gc.collect()
            if DEVICE.type == "cuda": torch.cuda.empty_cache()

timing_df = pd.DataFrame(timing).round(3)
display(Markdown("### Table 5. Training cost and checkpoint selection (repeat 0)"))
display(timing_df)
timing_df.to_csv(TAB_DIR / "table5_training_cost.csv", index=False)

if (timing_df.min_best_epoch < SELECT_FROM_EPOCH).any():
    print(f"!! LỖI: có checkpoint được chọn TRƯỚC epoch {SELECT_FROM_EPOCH} - ràng buộc [V7-B2] bị vi phạm.")
else:
    print(f"✓ [V7-B2] Mọi checkpoint được chọn từ epoch {SELECT_FROM_EPOCH} (đỉnh lịch LR) trở đi.")

# ---- Bảng 5b: CV lặp ---------------------------------------------------------
rep_df = pd.DataFrame(rep_rows)
rep_df.round(4).to_csv(TAB_DIR / "table5b_repeated_cv_raw.csv", index=False)
REPEAT_SUMMARY, OOF_ENS, _rows = {}, {}, []
for name in cfg.models:
    ens = np.nanmean(np.vstack(OOF_REPS[name]), axis=0)   # mỗi frame là OOF ở MỌI lần lặp
    OOF_ENS[name] = ens
    auc_e, g_e = record_auc_simple(ens)
    ci = cluster_bootstrap_ci(g_e.index.values, g_e.y.values, g_e.p.values, roc_auc_score,
                              n=cfg.bootstrap_n, seed=0)
    d = rep_df[rep_df.model == name]
    sd_r = float(d.auc_record.std(ddof=1)) if len(d) > 1 else float("nan")
    REPEAT_SUMMARY[name] = dict(n_repeats=int(len(d)), mean=float(d.auc_record.mean()), sd=sd_r,
                                ens_auc=auc_e, ens_lo=ci["lo"], ens_hi=ci["hi"])
    _rows.append(dict(model=name, n_repeats=int(len(d)),
                      AUC_record_mean=float(d.auc_record.mean()), AUC_record_sd=sd_r,
                      AUC_record_min=float(d.auc_record.min()), AUC_record_max=float(d.auc_record.max()),
                      AUC_frame_mean=float(d.auc_frame.mean()),
                      AUC_frame_sd=float(d.auc_frame.std(ddof=1)) if len(d) > 1 else float("nan"),
                      AUC_record_repeat_avg=auc_e, CI_lo=ci["lo"], CI_hi=ci["hi"]))
rep_sum = pd.DataFrame(_rows).sort_values("AUC_record_repeat_avg", ascending=False).reset_index(drop=True)
rep_sum.round(4).to_csv(TAB_DIR / "table5b_repeated_cv.csv", index=False)
display(Markdown(f"### Table 5b. Repeated cross-validation ({len(cfg.seeds)} partitions; "
                 f"record-level AUC with `{cfg.record_agg}` aggregation)"))
display(rep_sum.round(4))
if len(cfg.seeds) > 1:
    _sd = float(np.nanmean(rep_sum.AUC_record_sd))
    print(f"Độ lệch chuẩn trung bình giữa các lần lặp (mức bản ghi): {_sd:.4f}")
    print("=> Chênh lệch AUC giữa hai kiến trúc NHỎ HƠN con số này là nhiễu phân hoạch/khởi tạo,")
    print("   KHÔNG phải khác biệt kiến trúc.")
    print("Cột AUC_record_repeat_avg = AUC khi TRUNG BÌNH xác suất OOF qua các lần lặp: ổn định")
    print("nhất và vẫn trung thực (mỗi dự đoán đến từ mô hình chưa thấy bản ghi đó).")
else:
    print("LƯU Ý: chỉ 1 phân hoạch. Đặt cfg.seeds=(42, 43, 44) để đo độ bất định do phân hoạch.")


## 8. Kết quả và trực quan hoá / Results and visualisation

### Vì sao đánh giá hai cấp, và cấp nào mới là chính

Nhãn của tập dữ liệu này được gán ở **mức bản ghi** (mã `Rxxx`), không phải mức frame.
Điều đó có bốn hệ quả mà bài báo phải nói rõ:

**1. Mức bản ghi là đơn vị thống kê duy nhất sạch.** Mỗi `Rxxx` là một quan sát độc lập.
Số frame lớn không làm tăng cỡ mẫu hiệu dụng.

**2. Mọi khoảng tin cậy ở mức frame PHẢI bootstrap theo cụm bản ghi.** Mục 8.2 in cả CI
theo cụm (đúng) và CI theo frame (sai) cùng **hệ số chênh lệch đo được** giữa hai cách.

**3. Nhãn mức frame chỉ là nhãn mức bản ghi được "kế thừa" xuống frame.** Mục 8.7 **đo**
tỉ lệ thể hiện dương thay vì giả định nó, và kết luận theo ba mức: ủng hộ / không ủng hộ /
chưa kết luận được tiền đề "nhãn frame bị nhiễu".

**4. Mọi so sánh giữa các kiến trúc phải làm ở mức BẢN GHI** (McNemar mức bản ghi,
bootstrap ghép cặp theo bản ghi). McNemar ở mức frame là pseudo-replication: nó coi hàng nghìn
quyết định phụ thuộc lẫn nhau là các "mẫu độc lập".

**Quy tắc gộp mức bản ghi [V7-S2]:** `cfg.record_agg = "p_mean"` trên xác suất **thô**, được
**đăng ký trước**. Bảng 7 vẫn in kết quả của cả bốn quy tắc và của bản hiệu chỉnh Platt để minh bạch.


In [ ]:
# =========================================================
# 8.1 Hình 7: Đường cong học / Figure 7: learning curves
# =========================================================
n = len(cfg.models)
fig, axes = plt.subplots(2, n, figsize=(3.3*n, 6), squeeze=False)
for j, name in enumerate(cfg.models):
    for r in ALL_RESULTS[name]:
        h = r["history"]
        axes[0][j].plot(h["epoch"], h["train_loss"], lw=1, alpha=.75, color=PAL["grey"])
        axes[0][j].plot(h["epoch"], h["val_loss"],  lw=1.2, alpha=.85, color=MODEL_COLORS[name])
        axes[1][j].plot(h["epoch"], h["val_auc"],   lw=1.2, alpha=.85, color=MODEL_COLORS[name])
        axes[1][j].scatter([r["best_epoch"]], [h["val_auc"][r["best_epoch"]]],
                           s=18, color="k", zorder=5)
    axes[0][j].set_title(name, fontsize=10); axes[0][j].set_xlabel("epoch")
    axes[1][j].set_xlabel("epoch"); axes[1][j].set_ylim(.35, 1.02)
    axes[1][j].axhline(.5, color="k", lw=.7, ls="--")
axes[0][0].set_ylabel("loss (grey = train, colour = validation)")
axes[1][0].set_ylabel("validation AUC")
fig.suptitle("Learning curves across folds", y=1.02)
fig.tight_layout()
savefig(fig, "learning_curves",
        "Training and validation loss, and validation AUC, for every fold. The train-validation gap "
        "reflects the degree of overfitting under limited field data.")


In [ ]:
# =========================================================
# 8.2 Tổng hợp chỉ số mức frame / Frame-level metric aggregation
#     [V6-S1] CI bootstrap theo CỤM BẢN GHI (đúng) + in kèm CI bootstrap theo frame
#     (cách cũ) để ĐO mức hẹp giả tạo. NGƯỠNG vẫn chọn trên validation-TRONG (lồng).
# =========================================================
per_fold_rows, frame_summary_rows = [], []
THR, THR_BIASED, THR_CAL = {}, {}, {}

for name in cfg.models:
    res = ALL_RESULTS[name]
    fold_metrics = []
    for r in res:
        thr = (youden_threshold(r["inner_y"], r["inner_probs"]) if r["nested"]
               else youden_threshold(r["val_y"], r["val_probs"]))
        m = compute_metrics(r["val_y"], r["val_probs"], thr)
        m.update(model=name, fold=r["fold"], nested=r["nested"])
        fold_metrics.append(m); per_fold_rows.append(m)
    d = pd.DataFrame(fold_metrics)

    msk = ~np.isnan(OOF[name])
    y_oof = y_all[msk].astype(int); p_oof = OOF[name][msk]; rec_oof = groups_all[msk]
    y_in, p_in, _, _ = INNER[name]

    THR[name] = youden_threshold(y_in, p_in) if cfg.nested_selection else youden_threshold(y_oof, p_oof)
    THR_BIASED[name] = youden_threshold(y_oof, p_oof)
    y_in_cal, p_in_cal, _, _ = INNER_CAL[name]
    THR_CAL[name] = (youden_threshold(y_in_cal, p_in_cal)
                     if cfg.calibrate_aggregation else THR[name])

    m_oof = compute_metrics(y_oof, p_oof, THR[name])
    ci_c = cluster_bootstrap_ci(rec_oof, y_oof, p_oof, roc_auc_score, n=cfg.bootstrap_n, seed=0)
    ci_n = naive_frame_bootstrap_ci(y_oof, p_oof, n=min(2000, cfg.bootstrap_n), seed=0)
    vc = variance_components(np.where(msk)[0], y_oof, p_oof)
    de = design_effect(vc["icc"], vc["m_bar"])
    w_cl = ci_c["hi"] - ci_c["lo"]; w_nv = ci_n["hi"] - ci_n["lo"]

    frame_summary_rows.append(dict(
        model=name, AUC=m_oof["auc"], AUC_lo=ci_c["lo"], AUC_hi=ci_c["hi"],
        AUC_lo_naive=ci_n["lo"], AUC_hi_naive=ci_n["hi"],
        CI_width_cluster=w_cl, CI_width_naive=w_nv,
        CI_inflation=float(w_cl / w_nv) if w_nv > 0 else np.nan,
        AUC_cv_mean=d.auc.mean(), AUC_cv_std=d.auc.std(),
        ICC=vc["icc"], design_effect=de, n_clusters=ci_c["n_clusters"],
        n_eff=float(len(y_oof) / max(de, 1e-9)),
        Accuracy=m_oof["accuracy"], Acc_cv_std=d.accuracy.std(),
        Precision=m_oof["precision"], Recall=m_oof["recall"], F1=m_oof["f1"],
        Specificity=m_oof["specificity"], EER=m_oof["eer"], AP=m_oof["ap"],
        MCC=m_oof["mcc"], FPR=m_oof["fpr"], MissRate=m_oof["miss_rate"], thr=THR[name]))

per_fold_df = pd.DataFrame(per_fold_rows)
frame_df = pd.DataFrame(frame_summary_rows).sort_values("AUC", ascending=False).reset_index(drop=True)
per_fold_df.to_csv(TAB_DIR / "per_fold_metrics.csv", index=False)
frame_df.round(4).to_csv(TAB_DIR / "table6_frame_level.csv", index=False)

display(Markdown("### Table 6. Frame-level performance (out-of-fold, nested selection)"))
display(frame_df.round(4))
BEST_MODEL = frame_df.model.iloc[0]
print("Mô hình tốt nhất (frame-level AUC):", BEST_MODEL)
print("Ngưỡng chọn trên validation-trong:", {k: round(v, 3) for k, v in THR.items()})

# --- [V6-S1] Mức sai lệch của cách bootstrap cũ -------------------------
print("\n=== KIỂM TRA ĐƠN VỊ THỐNG KÊ ===")
print(f"{'model':<16}{'AUC':>8}{'CI cụm (đúng)':>24}{'CI frame (cũ)':>22}{'hệ số':>8}{'DE':>7}")
for _, r in frame_df.iterrows():
    print(f"{r.model:<16}{r.AUC:>8.4f}   [{r.AUC_lo:.4f}, {r.AUC_hi:.4f}]      "
          f"[{r.AUC_lo_naive:.4f}, {r.AUC_hi_naive:.4f}] {r.CI_inflation:>7.2f}x{r.design_effect:>7.1f}")
print(f"\nICC trung bình {frame_df.ICC.mean():.3f} | design effect trung bình "
      f"{frame_df.design_effect.mean():.1f} | n_eff trung bình {frame_df.n_eff.mean():.0f} "
      f"(trên {int(frame_df.n_clusters.mean())} bản ghi, {len(y_all)} frame)")
print("=> Cột 'CI frame (cũ)' CHỈ để chứng minh mức hẹp giả tạo. KHÔNG trích dẫn nó.")


In [ ]:
# =========================================================
# 8.3 Hình 8: ROC / PR / DET
# =========================================================
fig, ax = plt.subplots(1, 3, figsize=(14, 4.1))
for name in cfg.models:
    msk = ~np.isnan(OOF[name]); y, p = y_all[msk], OOF[name][msk]
    fpr, tpr, _ = roc_curve(y, p); auc = roc_auc_score(y, p)
    ax[0].plot(fpr, tpr, lw=1.7, color=MODEL_COLORS[name], label=f"{name} (AUC={auc:.3f})")
    pr, rc, _ = precision_recall_curve(y, p); apv = average_precision_score(y, p)
    ax[1].plot(rc, pr, lw=1.7, color=MODEL_COLORS[name], label=f"{name} (AP={apv:.3f})")
    fnr = 1 - tpr; eer = eer_from_roc(fpr, tpr)
    ax[2].plot(fpr*100, fnr*100, lw=1.7, color=MODEL_COLORS[name], label=f"{name} (EER={eer*100:.1f}%)")

ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set_xlabel("False positive rate")
ax[0].set_ylabel("True positive rate"); ax[0].set_title("(a) ROC — frame level"); ax[0].legend(fontsize=8)
base = float(np.nanmean(y_all))
ax[1].axhline(base, color="k", ls="--", lw=.8); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ax[1].set_title("(b) Precision–Recall"); ax[1].legend(fontsize=8)
ax[2].set_xscale("log"); ax[2].set_yscale("log"); ax[2].set_xlabel("False alarm rate (%)")
ax[2].set_ylabel("Miss rate (%)"); ax[2].set_title("(c) DET curve"); ax[2].legend(fontsize=8)
savefig(fig, "roc_pr_det",
        "Architecture comparison on out-of-fold predictions: (a) ROC, (b) precision-recall, (c) DET. "
        "The DET curve highlights the false-alarm versus missed-detection trade-off that "
        "governs operational usability.")


In [ ]:
# =========================================================
# 8.4 Hình 9: Ma trận nhầm lẫn & phân tán chỉ số theo fold
# =========================================================
fig = plt.figure(figsize=(3.1*len(cfg.models), 6.4))
gs = gridspec.GridSpec(2, len(cfg.models), height_ratios=[1.25, 1], hspace=.45)
for j, name in enumerate(cfg.models):
    msk = ~np.isnan(OOF[name]); y, p = y_all[msk], OOF[name][msk]
    cm = confusion_matrix(y, (p >= THR[name]).astype(int), labels=[0, 1])
    cmn = cm / cm.sum(1, keepdims=True)
    a = fig.add_subplot(gs[0, j]); a.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for r in range(2):
        for c in range(2):
            a.text(c, r, f"{cm[r,c]}\n{cmn[r,c]*100:.1f}%", ha="center", va="center",
                   color="white" if cmn[r, c] > .5 else "black", fontsize=9)
    a.set_xticks([0, 1]); a.set_xticklabels(["no-leak", "leak"])
    a.set_yticks([0, 1]); a.set_yticklabels(["no-leak", "leak"])
    a.set_title(f"{name}\nthr={THR[name]:.2f}", fontsize=9); a.grid(False)
    if j == 0: a.set_ylabel("true label")

a2 = fig.add_subplot(gs[1, :])
data = [per_fold_df.loc[per_fold_df.model == n, "auc"].values for n in cfg.models]
# [V7] matplotlib >= 3.9 đổi `labels` -> `tick_labels`; đặt nhãn trục riêng để chạy mọi phiên bản
bp = a2.boxplot(data, patch_artist=True, widths=.5)
a2.set_xticks(range(1, len(cfg.models) + 1)); a2.set_xticklabels(list(cfg.models))
for patch, n in zip(bp["boxes"], cfg.models):
    patch.set_facecolor(MODEL_COLORS[n]); patch.set_alpha(.55)
for i, n in enumerate(cfg.models):
    v = per_fold_df.loc[per_fold_df.model == n, "auc"].values
    a2.scatter(np.full(len(v), i+1) + np.random.default_rng(i).uniform(-.07, .07, len(v)), v,
               color="k", s=14, zorder=4)
a2.axhline(.5, color="k", ls="--", lw=.8); a2.set_ylabel("per-fold AUC")
a2.set_title("Between-fold variability as a measure of stability on field data")
savefig(fig, "confusion_and_fold_variance",
        "Confusion matrices at the Youden-optimal threshold (top) and the per-fold AUC distribution "
        "(bottom). A large between-fold spread indicates heterogeneity across field acquisition "
        "points.")


In [ ]:
# =========================================================
# 8.5 Đánh giá mức bản ghi / Record-level evaluation
#     PHÉP GỘP chọn trên validation-TRONG. [V6-T9] So sánh gộp trên xác suất THÔ vs
#     xác suất ĐÃ HIỆU CHỈNH PLATT (hiệu chỉnh fit trên validation-trong của từng fold).
# =========================================================
AGGS = ["p_mean", "p_med", "p_p90", "frac_pos"]
AGG_FN = dict(p_mean=np.mean, p_med=np.median,
              p_p90=lambda a: float(np.percentile(a, 90)),
              frac_pos=None)          # frac_pos cần ngưỡng, xử lý riêng

def record_scores(idx, y, p, thr_frame, folds=None):
    df = pd.DataFrame(dict(record_id=frames.record_id.values[np.asarray(idx)],
                           label=np.asarray(y), p=np.asarray(p)))
    keys = ["record_id"]
    if folds is not None:
        df["fold"] = np.asarray(folds); keys = ["fold", "record_id"]
    g = df.groupby(keys).agg(label=("label", "first"), p_mean=("p", "mean"),
                             p_med=("p", "median"),
                             p_p90=("p", lambda s: float(np.percentile(s, 90))),
                             frac_pos=("p", lambda s: float((s >= thr_frame).mean())),
                             n=("p", "size")).reset_index()
    return g

# --- (1) chọn quy tắc gộp trên validation-trong (xác suất ĐÃ hiệu chỉnh) ---
sel_rows = []
for name in cfg.models:
    if cfg.calibrate_aggregation:
        y_in, p_in_use, i_in, f_in = INNER_CAL[name]
    else:
        y_in, p_in, i_in, f_in = INNER[name]; p_in_use = p_in
    g_in = record_scores(i_in, y_in, p_in_use, THR_CAL[name], folds=f_in)
    for agg in AGGS:
        auc = (roc_auc_score(g_in.label, g_in[agg]) if len(set(g_in.label)) > 1 else np.nan)
        sel_rows.append(dict(model=name, aggregation=agg, inner_auc=auc))
agg_sel_df = pd.DataFrame(sel_rows)
AGG_INNER_BEST = agg_sel_df.groupby("aggregation").inner_auc.mean().idxmax()
# [V7-S2] quy tắc gộp ĐĂNG KÝ TRƯỚC (mặc định); "inner" = chọn trên validation-trong như v6
best_agg = cfg.record_agg if cfg.agg_selection == "fixed" else AGG_INNER_BEST
print(f"Quy tắc gộp dùng cho báo cáo: {best_agg} ({'đăng ký trước' if cfg.agg_selection == 'fixed' else 'chọn trên validation-trong'})"
      f" | quy tắc validation-trong sẽ chọn: {AGG_INNER_BEST}")

# --- (2) báo cáo trên fold NGOÀI ---
rec_rows, REC, REC_INNER, REC_RAW = [], {}, {}, {}
for name in cfg.models:
    if cfg.calibrate_aggregation:
        y_in, p_in_use, i_in, f_in = INNER_CAL[name]
    else:
        y_in, p_in, i_in, f_in = INNER[name]; p_in_use = p_in
    g_in = record_scores(i_in, y_in, p_in_use, THR_CAL[name], folds=f_in)
    REC_INNER[name] = g_in

    msk = ~np.isnan(OOF[name])
    p_oof = OOF[name][msk]
    p_use = (OOF_CAL[name][msk] if (cfg.calibrate_aggregation and not np.isnan(OOF_CAL[name][msk]).any())
             else p_oof)
    g = record_scores(np.where(msk)[0], y_all[msk], p_use, THR_CAL[name])
    REC[name] = g
    REC_RAW[name] = record_scores(np.where(msk)[0], y_all[msk], p_oof, THR[name])

    for agg in AGGS:
        thr_r = (youden_threshold(g_in.label.values, g_in[agg].values) if cfg.nested_selection
                 else youden_threshold(g.label.values, g[agg].values))
        m = compute_metrics(g.label.values, g[agg].values, thr_r)
        auc_raw = (roc_auc_score(REC_RAW[name].label, REC_RAW[name][agg])
                   if len(set(REC_RAW[name].label)) > 1 else np.nan)
        rec_rows.append(dict(model=name, aggregation=agg, auc_raw=auc_raw,
                             **{k: m[k] for k in ["auc", "accuracy", "balanced_acc", "precision",
                                                  "recall", "f1", "specificity", "mcc", "threshold"]}))
rec_df = pd.DataFrame(rec_rows)

# Khoảng tin cậy bootstrap Ở MỨC BẢN GHI (lấy mẫu lại BẢN GHI, không phải frame)
ci_rows = []
for name in cfg.models:
    g = REC[name]
    ci = cluster_bootstrap_ci(g.record_id.values, g.label.values, g[best_agg].values,
                              roc_auc_score, n=cfg.bootstrap_n, seed=1)
    # CI nhị thức cho recall/specificity ở ngưỡng vận hành
    yv = g.label.values.astype(int); sg = g[best_agg].values
    thr_r = (youden_threshold(REC_INNER[name].label.values, REC_INNER[name][best_agg].values)
             if cfg.nested_selection else youden_threshold(yv, sg))
    pred = (sg >= thr_r).astype(int)
    tp = int(((pred == 1) & (yv == 1)).sum()); fn_ = int(((pred == 0) & (yv == 1)).sum())
    tn = int(((pred == 0) & (yv == 0)).sum()); fp = int(((pred == 1) & (yv == 0)).sum())
    def _ci_ratio(k, n):
        if n == 0: return (np.nan, np.nan)
        lo, hi = spstats.binomtest(k, n).proportion_ci(confidence_level=0.95, method="wilson")
        return (float(lo), float(hi))
    r_lo, r_hi = _ci_ratio(tp, tp + fn_); s_lo, s_hi = _ci_ratio(tn, tn + fp)
    ci_rows.append(dict(model=name, auc_lo=ci["lo"], auc_hi=ci["hi"],
                        recall_ci_lo=r_lo, recall_ci_hi=r_hi,
                        spec_ci_lo=s_lo, spec_ci_hi=s_hi, n_records=int(len(g))))
ci_df = pd.DataFrame(ci_rows)
rec_df = rec_df.merge(ci_df, on="model", how="left")
rec_main = rec_df[rec_df.aggregation == best_agg].sort_values("auc", ascending=False).round(4)
rec_df.round(4).to_csv(TAB_DIR / "table7_record_level_all.csv", index=False)
rec_main.to_csv(TAB_DIR / "table7_record_level.csv", index=False)
agg_sel_df.round(4).to_csv(TAB_DIR / "table7b_aggregation_selection.csv", index=False)

display(Markdown(f"### Table 7. Record-level performance "
                 f"(aggregation rule `{best_agg}`, "
                 f"{'pre-registered' if cfg.agg_selection == 'fixed' else 'selected on the inner validation set'})"))
display(rec_main[["model", "auc", "auc_lo", "auc_hi", "accuracy", "precision", "recall",
                  "recall_ci_lo", "recall_ci_hi", "specificity", "spec_ci_lo", "spec_ci_hi",
                  "f1", "threshold", "auc_raw"]])
display(Markdown("**Inner-validation AUC of each aggregation rule (sensitivity analysis):**"))
display(agg_sel_df.pivot(index="model", columns="aggregation", values="inner_auc").round(4))
display(Markdown("**Results under all four aggregation rules (full transparency):**"))
display(rec_df.pivot(index="model", columns="aggregation", values="auc").round(4))
display(Markdown("**Effect of Platt calibration on record-level aggregation "
                 f"(`auc` = calibrated, `auc_raw` = raw posteriors):**"))
display(rec_df[rec_df.aggregation == best_agg][["model", "auc", "auc_raw"]].round(4))
print("Ghi chú: hiệu chỉnh Platt KHÔNG đổi quyết định ở mức frame (đơn điệu) nhưng CÓ thể")
print("đổi AUC mức bản ghi, vì trung bình của một biến đổi phi tuyến không bảo toàn thứ hạng.")

# ---------------------------------------------------------------------
# [V7-B5] DEPLOY_MODEL được mục 10.4 dùng TRƯỚC mục 13 (v6: NameError ở 10.4a).
# Xác định ngay tại đây bằng ĐÚNG tiêu chí của mục 13: AUC mức bản ghi trên validation-TRONG.
# ---------------------------------------------------------------------
_inner_auc = {}
for _n in cfg.models:
    _gi = REC_INNER[_n]
    _inner_auc[_n] = (float(roc_auc_score(_gi.label, _gi[best_agg])) if _gi.label.nunique() > 1
                      else -np.inf)
DEPLOY_MODEL = max(_inner_auc, key=_inner_auc.get)
print(f"\nMô hình triển khai (chọn trên validation-TRONG, cùng tiêu chí mục 13): {DEPLOY_MODEL}")


In [ ]:
# =========================================================
# 8.6 Hình 10: Kết quả mức bản ghi / Figure 10: record-level results
# =========================================================
fig, ax = plt.subplots(1, 3, figsize=(14, 4.0))
for name in cfg.models:
    g = REC[name]
    if len(set(g.label)) > 1:
        fpr, tpr, _ = roc_curve(g.label, g[best_agg]); auc = roc_auc_score(g.label, g[best_agg])
        ax[0].plot(fpr, tpr, lw=1.7, color=MODEL_COLORS[name], label=f"{name} ({auc:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set_title(f"(a) Record-level ROC ({best_agg})")
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend(fontsize=8)

g = REC[BEST_MODEL].sort_values(best_agg)
cols = [PAL["leak"] if l == 1 else PAL["noleak"] for l in g.label]
ax[1].scatter(range(len(g)), g[best_agg], c=cols, s=26)
# [V7] ngưỡng lấy từ validation-TRONG (v6 vẽ ngưỡng Youden tối ưu trên chính tập báo cáo)
thr_b = youden_threshold(REC_INNER[BEST_MODEL].label.values, REC_INNER[BEST_MODEL][best_agg].values)
ax[1].axhline(thr_b, color="k", ls="--", lw=1, label=f"threshold={thr_b:.2f}")
ax[1].set_xlabel("recordings (sorted)")
ax[1].set_ylabel(f"{best_agg} leak probability")
ax[1].set_title(f"(b) Per-recording score — {BEST_MODEL}"); ax[1].legend(fontsize=8)

w = .35; xs = np.arange(len(cfg.models))
fl = frame_df.set_index("model").loc[list(cfg.models), "AUC"].values
rl = rec_df[rec_df.aggregation == best_agg].set_index("model").loc[list(cfg.models), "auc"].values
ax[2].bar(xs - w/2, fl, w, label="frame-level", color=PAL["noleak"])
ax[2].bar(xs + w/2, rl, w, label="record-level", color=PAL["leak"])
for i, (a, b) in enumerate(zip(fl, rl)):
    ax[2].text(i - w/2, a, f"{a:.2f}", ha="center", va="bottom", fontsize=7)
    ax[2].text(i + w/2, b, f"{b:.2f}", ha="center", va="bottom", fontsize=7)
ax[2].set_xticks(xs); ax[2].set_xticklabels(cfg.models, rotation=20, ha="right")
ax[2].set_ylabel("AUC"); ax[2].axhline(.5, color="k", ls="--", lw=.8)
ax[2].set_title("(c) Frame-level vs record-level"); ax[2].legend(fontsize=8)
savefig(fig, "record_level_results",
        "Record-level performance: (a) ROC after aggregating frame posteriors; (b) per-recording "
        "score and decision threshold; (c) gain from moving from frame-level to record-level "
        "decisions.")


### 8.7 Ước lượng tỉ lệ thể hiện dương / Witness-rate estimation

Câu hỏi quyết định **toàn bộ khung lập luận MIL**: trong một bản ghi được gán nhãn `leak`,
**bao nhiêu phần trăm frame thật sự mang dấu hiệu rò rỉ?**

Cell này **đo** đại lượng đó (trung vị, trên các bản ghi leak, của tỉ lệ frame vượt ngưỡng)
và **tự rút ra kết luận theo ba mức** [V7]:

| Trung vị | Kết luận | Hệ quả |
|---|---|---|
| < 0,5 | Tiền đề MIL **được ủng hộ** | Nhãn frame nhiễu; có thể thử `mil_mode="bag"` |
| 0,5 – 0,9 | **Chưa kết luận được** | Không được mô tả nhãn frame là "nhiễu nặng", cũng không được khẳng định ngược lại |
| ≥ 0,9 | Tiền đề MIL **không được ủng hộ** | Điểm số gần như hằng số trong bản ghi → bắt buộc xem mục 9.6 (confound) |

(v6 dùng một ngưỡng duy nhất 0,5 và in "trung vị 0,81 ≈ 1" — phát biểu mạnh hơn số liệu.)

**Cảnh báo về tính vòng quanh:** ước lượng này dựa trên chính mô hình, nên nó đo "phần frame
mà mô hình cho là có rò rỉ", không phải chân lý. Muốn có chân lý phải nghe và dán nhãn thời
điểm rò rỉ trên một mẫu bản ghi (xem mục 16).


In [ ]:
# =========================================================
# 8.7 Tỉ lệ thể hiện dương trong túi dương / witness rate + KIỂM ĐỊNH TIỀN ĐỀ MIL
#     [V7] Kết luận BA MỨC (ủng hộ / chưa kết luận / không ủng hộ) thay cho một ngưỡng 0,5.
# =========================================================
name = BEST_MODEL
msk = ~np.isnan(OOF[name])
dfw = pd.DataFrame(dict(record_id=frames.record_id.values[msk],
                        label=y_all[msk], p=OOF[name][msk]))
thr_w = THR[name]
wit = (dfw.groupby("record_id")
       .agg(label=("label", "first"), n=("p", "size"),
            frac_above=("p", lambda s: float((s >= thr_w).mean())),
            p_mean=("p", "mean"), p_max=("p", "max"), p_std=("p", "std")).reset_index())
wit.to_csv(TAB_DIR / "table12_witness_rate.csv", index=False)

w_leak = wit.loc[wit.label == 1, "frac_above"]
w_nole = wit.loc[wit.label == 0, "frac_above"]
display(Markdown("### Table 12. Fraction of frames above threshold per recording"))
display(wit.groupby("label").frac_above.describe().round(3))

W_LEAK_MED = float(w_leak.median())
if W_LEAK_MED < 0.5:
    WITNESS_VERDICT = "supported"
elif W_LEAK_MED >= 0.9:
    WITNESS_VERDICT = "not_supported"
else:
    WITNESS_VERDICT = "inconclusive"
MIL_PREMISE_SUPPORTED = WITNESS_VERDICT == "supported"
print(f"Bản ghi LEAK  : trung vị {W_LEAK_MED:.2f} số frame vượt ngưỡng "
      f"(khoảng {w_leak.min():.2f}-{w_leak.max():.2f}, IQR {w_leak.quantile(.25):.2f}-{w_leak.quantile(.75):.2f})")
print(f"Bản ghi NO-LEAK: trung vị {w_nole.median():.2f}")

print()
if WITNESS_VERDICT == "supported":
    print(f"KẾT LUẬN: tiền đề MIL ĐƯỢC ỦNG HỘ (trung vị {W_LEAK_MED:.2f} < 0,5).")
    print("  -> Có cơ sở nói rằng nhãn mức frame bị nhiễu; có thể thử mil_mode='bag'.")
elif WITNESS_VERDICT == "not_supported":
    print(f"KẾT LUẬN: tiền đề MIL KHÔNG ĐƯỢC ủng hộ (trung vị {W_LEAK_MED:.2f} >= 0,9).")
    print("  -> Gần như MỌI frame trong bản ghi leak đều được mô hình coi là 'rò rỉ'.")
    print("  -> KHÔNG được mô tả nhãn mức frame là 'nhiễu nặng'; KHÔNG bật mil_mode='bag'.")
    print("  -> Điểm số gần như hằng số TRONG bản ghi: xem mục 8.8 (ICC) và 9.6 (confound).")
else:
    print(f"KẾT LUẬN: CHƯA KẾT LUẬN ĐƯỢC về tiền đề MIL (trung vị {W_LEAK_MED:.2f} nằm giữa 0,5 và 0,9).")
    print("  -> Phần lớn nhưng không phải mọi frame của bản ghi leak vượt ngưỡng; độ phân tán giữa")
    print("     các bản ghi lớn. KHÔNG khẳng định nhãn frame 'nhiễu nặng', cũng KHÔNG khẳng định ngược lại.")
    print("  -> Muốn phân định: dán nhãn thời điểm rò rỉ trên một mẫu bản ghi (mục 16).")

vc_w = variance_components(np.where(msk)[0], y_all[msk], OOF[name][msk])
print(f"\nPhân rã phương sai ({name}): ICC = {vc_w['icc']:.3f} -> "
      f"{vc_w['icc']*100:.0f}% phương sai nằm GIỮA các bản ghi, "
      f"{(1-vc_w['icc'])*100:.0f}% nằm giữa các frame trong cùng bản ghi.")
if vc_w["icc"] > 0.5 and WITNESS_VERDICT != "supported":
    print("=> Điểm số chủ yếu phân biệt BẢN GHI hơn là các đoạn thời gian trong bản ghi.")
    print("   Đây là dấu hiệu CẦN kiểm tra confound (mục 9.6), chưa phải bằng chứng confound.")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for lab, nm in [(0, "noleak"), (1, "leak")]:
    ax[0].hist(dfw.loc[dfw.label == lab, "p"], bins=40, alpha=.6, density=True,
               color=PAL[nm], label=f"frames in {nm} recordings")
ax[0].axvline(thr_w, color="k", ls="--", lw=1, label=f"threshold={thr_w:.2f}")
ax[0].set_xlabel("frame-level leak probability"); ax[0].set_ylabel("density")
ax[0].set_title("(a) Frame-probability distribution by bag label"); ax[0].legend(fontsize=7)

for lab, nm in [(0, "noleak"), (1, "leak")]:
    ax[1].hist(wit.loc[wit.label == lab, "frac_above"], bins=12, alpha=.65,
               color=PAL[nm], label=nm)
ax[1].axvline(W_LEAK_MED, color="k", ls=":", lw=1.2)
ax[1].axvspan(0.5, 0.9, color=PAL["warn"], alpha=.12, lw=0)
ax[1].set_xlabel("fraction of frames above threshold (per recording)")
ax[1].set_ylabel("number of recordings"); ax[1].legend(fontsize=8)
ax[1].set_title(f"(b) Witness rate per recording\nmedian of leak class = {W_LEAK_MED:.2f} "
                f"(shaded = inconclusive zone)")

w = wit.sort_values(["label", "frac_above"])
ax[2].barh(range(len(w)), w.frac_above,
           color=[PAL["leak"] if l else PAL["noleak"] for l in w.label])
ax[2].set_yticks(range(len(w))); ax[2].set_yticklabels(w.record_id, fontsize=6)
ax[2].set_xlabel("fraction of frames above threshold"); ax[2].set_title("(c) Individual recordings")
_cap_w = {"supported": "A median below 0.5 supports the multiple-instance premise that leak evidence "
                       "occupies only part of a leak recording.",
          "not_supported": "A median of 0.9 or above means almost every frame of a leak recording is "
                           "scored as leak: the noisy-frame-label premise is NOT supported.",
          "inconclusive": "A median between 0.5 and 0.9 does not resolve the multiple-instance premise; "
                          "frame-level annotation of leak onsets is required."}[WITNESS_VERDICT]
savefig(fig, "witness_rate", "Witness rate at record level (fraction of frames above the operating "
        f"threshold in each recording; median of leak recordings = {W_LEAK_MED:.2f}). " + _cap_w)


### 8.8 CHẨN ĐOÁN: vì sao mức bản ghi CAO HƠN mức frame?

Nhiều người kỳ vọng chỉ số mức frame cao hơn mức bản ghi. **Kỳ vọng đó đến từ các công bố
chia frame ngẫu nhiên**: khi frame của cùng một bản ghi nằm ở cả train lẫn test, chỉ số mức
frame bị thổi phồng tới mức vượt cả mức bản ghi. Với giao thức nhóm theo bản ghi, thứ tự tự
nhiên quay lại: **gộp làm giảm phương sai, nên mức bản ghi phải cao hơn**.

Cell này kiểm chứng điều đó **trên chính số liệu** bằng mô hình thành phần phương sai:

$$z_{ij} = \mu \cdot y_i + b_i + e_{ij}, \qquad b_i \sim N(0,\sigma_b^2),\ e_{ij} \sim N(0,\sigma_e^2)$$

với $z$ là điểm chuẩn tắc (van der Waerden) của frame $j$ trong bản ghi $i$. Từ đó:

$$\text{AUC}_{frame} = \Phi\!\left(\frac{\mu}{\sqrt{2(\sigma_b^2+\sigma_e^2)}}\right), \qquad
\text{AUC}_{rec} = \Phi\!\left(\frac{\mu}{\sqrt{2(\sigma_b^2+\sigma_e^2/K)}}\right)$$

Nếu hai giá trị **dự đoán** khớp hai giá trị **quan sát**, pipeline nhất quán nội tại. Nếu
lệch lớn, đó mới là dấu hiệu có lỗi.

Đại lượng then chốt là **ICC** $= \sigma_b^2/(\sigma_b^2+\sigma_e^2)$: tỉ lệ phương sai điểm số
nằm **giữa các bản ghi**. ICC cao vừa giải thích vì sao gộp frame chỉ lợi một chút, vừa là
**dấu hiệu cảnh báo confound**: phần lớn phương sai là khác biệt giữa các bản ghi, và nếu
bản ghi trùng với điểm khảo sát thì đó chính là thứ mô hình đang phân biệt.

> **Lưu ý thống kê [V6]:** kiểm định dấu ở cuối cell dùng **5 mô hình** làm 5 quan sát, nhưng
> 5 mô hình này **không độc lập** (cùng dữ liệu, cùng giao thức). Vì vậy p-value chỉ mang
> tính mô tả; bằng chứng chính là sai lệch |dự đoán − quan sát| và giá trị ICC.


In [ ]:
# =========================================================
# 8.8 Phân rã phương sai frame / bản ghi + kiểm tra nhất quán nội tại
#     (variance_components / design_effect đã được định nghĩa ở mục 7.2b)
# =========================================================
from scipy.stats import norm as _norm

rows = []
for name in cfg.models:
    msk = ~np.isnan(OOF[name])
    vc = variance_components(np.where(msk)[0], y_all[msk], OOF[name][msk])
    auc_f_obs = float(frame_df.set_index("model").AUC[name])
    auc_r_obs = float(rec_df[(rec_df.model == name) & (rec_df.aggregation == best_agg)].auc.iloc[0])
    tot = vc["sb2"] + vc["se2"]
    auc_f_pred = float(_norm.cdf(vc["mu"] / np.sqrt(2 * tot))) if tot > 0 else np.nan
    auc_r_pred = float(_norm.cdf(vc["mu"] / np.sqrt(2 * (vc["sb2"] + vc["se2"] / vc["K_eff"]))))
    rows.append(dict(model=name, ICC=vc["icc"], sb2=vc["sb2"], se2=vc["se2"],
                     K_eff=vc["K_eff"], m_bar=vc["m_bar"], DE=design_effect(vc["icc"], vc["m_bar"]),
                     p_saturated=vc["sat"],
                     AUC_frame_obs=auc_f_obs, AUC_frame_pred=auc_f_pred,
                     AUC_rec_obs=auc_r_obs, AUC_rec_pred=auc_r_pred,
                     delta_obs=auc_r_obs - auc_f_obs, delta_pred=auc_r_pred - auc_f_pred))
vc_df = pd.DataFrame(rows).round(4)
display(Markdown("### Table 13. Variance decomposition and internal consistency check"))
display(vc_df)
vc_df.to_csv(TAB_DIR / "table13_variance_components.csv", index=False)

n_up = int((vc_df.delta_obs > 0).sum())
p_sign = float(spstats.binomtest(n_up, len(vc_df), 0.5, alternative="greater").pvalue)
err_f = float(np.nanmean(np.abs(vc_df.AUC_frame_pred - vc_df.AUC_frame_obs)))
err_r = float(np.nanmean(np.abs(vc_df.AUC_rec_pred - vc_df.AUC_rec_obs)))
print(f"Mức bản ghi CAO HƠN mức frame ở {n_up}/{len(vc_df)} mô hình "
      f"(kiểm định dấu một phía, p = {p_sign:.3f}; 5 mô hình KHÔNG độc lập -> chỉ mô tả).")
print(f"Sai lệch |dự đoán - quan sát|: mức frame {err_f:.4f} | mức bản ghi {err_r:.4f}")
if max(err_f, err_r) > 0.05:
    print("!! Mô hình Gauss KHÔNG khớp tốt (sai lệch > 0.05). Nguyên nhân thường gặp:")
    print(f"   xác suất bão hoà ở 0/1 (tỉ lệ trung bình {vc_df.p_saturated.mean():.1%}),")
    print("   hoặc mô hình chưa hội tụ. Khi đó hãy dùng Hình 8.9 làm bằng chứng chính -")
    print("   đường AUC theo K là phi tham số, không phụ thuộc giả thiết phân phối.")
else:
    print("✓ Mô hình thành phần phương sai khớp cả hai mức -> pipeline nhất quán nội tại:")
    print("  chênh lệch frame/bản ghi giải thích được bằng việc gộp làm giảm phương sai,")
    print("  KHÔNG phải do lỗi chia dữ liệu.")
print(f"ICC trung bình: {vc_df.ICC.mean():.3f} | design effect trung bình {vc_df.DE.mean():.1f}")
print("DIỄN GIẢI:")
print(" - Gộp frame chỉ triệt được thành phần TRONG bản ghi. Với ICC cao, phần đó nhỏ.")
print(" - Phần sai số còn lại là thuộc tính của ĐIỂM KHẢO SÁT/BẢN GHI.")
if vc_df.ICC.mean() > 0.5:
    print(f" - ICC cao (+ tỉ lệ thể hiện dương trung vị {W_LEAK_MED:.2f}, mục 8.7) => điểm số phần lớn là")
    print("   thuộc tính của TỪNG BẢN GHI.")
    print("   Kết luận 'nhận dạng rò rỉ' CHỈ hợp lệ nếu đã loại trừ được")
    print("   khả năng mô hình đang phân biệt điểm khảo sát (xem mục 9.6).")


In [ ]:
# =========================================================
# 8.9 Hình: AUC mức bản ghi theo số frame được gộp (đường bão hoà)
#     [V6] Dùng ĐÚNG quy tắc gộp đang báo cáo (best_agg) thay vì luôn dùng median.
# =========================================================
def _agg_array(a, agg):
    a = np.asarray(a, dtype=float)
    if agg == "p_mean": return float(np.mean(a))
    if agg == "p_med":  return float(np.median(a))
    if agg == "p_p90":  return float(np.percentile(a, 90))
    return float(np.mean(a))

def record_auc_at_K(name, K, n_rep=20, seed=0, agg=None):
    agg = agg or best_agg
    rng = np.random.default_rng(seed)
    msk = np.where(~np.isnan(OOF[name]))[0]
    rec = frames.record_id.values[msk]; p = OOF[name][msk]; yv = y_all[msk]
    byrec = defaultdict(list)
    for r, pp, yy in zip(rec, p, yv): byrec[r].append((pp, yy))
    recs = list(byrec); ry = np.array([byrec[r][0][1] for r in recs])
    if len(np.unique(ry)) < 2: return np.nan, np.nan
    vals = []
    for _ in range(n_rep):
        sc = []
        for r in recs:
            arr = np.array([t[0] for t in byrec[r]])
            k = min(K, len(arr))
            sc.append(_agg_array(rng.choice(arr, k, replace=False), agg))
        vals.append(roc_auc_score(ry, np.array(sc)))
    return float(np.mean(vals)), float(np.std(vals))

Ks = [1, 2, 4, 8, 16, 32, 64, 9999]
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.0))
curve_rows = []
for name in cfg.models:
    ms, ss = [], []
    for K in Ks:
        m, sd = record_auc_at_K(name, K, n_rep=20 if K < 9999 else 1)
        ms.append(m); ss.append(sd if sd == sd else 0.0)
        curve_rows.append(dict(model=name, aggregation=best_agg, K=K, auc=m, sd=sd))
    xs = np.arange(len(Ks))
    ax[0].errorbar(xs, ms, yerr=ss, marker="o", ms=4, lw=1.5,
                   color=MODEL_COLORS[name], label=name, capsize=2)
    ax[0].axhline(float(frame_df.set_index("model").AUC[name]), color=MODEL_COLORS[name],
                  ls=":", lw=1, alpha=.6)
ax[0].set_xticks(np.arange(len(Ks)))
ax[0].set_xticklabels([str(k) if k < 9999 else "all" for k in Ks])
ax[0].set_xlabel("frames pooled per recording (K)")
ax[0].set_ylabel("record-level AUC"); ax[0].legend(fontsize=7)
ax[0].set_title(f"(a) Pooling more frames saturates quickly (aggregation = {best_agg})\n"
                f"(dotted line = frame-level AUC)")

ax[1].bar(np.arange(len(vc_df)), vc_df.ICC,
          color=[MODEL_COLORS[m] for m in vc_df.model], width=.6)
for i, v in enumerate(vc_df.ICC):
    ax[1].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax[1].set_xticks(np.arange(len(vc_df)))
ax[1].set_xticklabels(vc_df.model, rotation=20, ha="right")
ax[1].set_ylabel("ICC"); ax[1].set_ylim(0, 1.05)
ax[1].set_title("(b) Share of score variance between recordings")
savefig(fig, "pooling_saturation",
        "Why record-level evaluation exceeds frame level only slightly: (a) the benefit of pooling "
        "more frames saturates after about 8-16 frames; (b) a high ICC means most score variance "
        "lies between acquisition points, which frame pooling cannot remove.")
pd.DataFrame(curve_rows).round(4).to_csv(TAB_DIR / "table14_pooling_curve.csv", index=False)


## 9. Kiểm định thống kê / Statistical significance testing

Chỉ so sánh điểm trung bình là **chưa đủ** để tuyên bố một kiến trúc tốt hơn — nhưng dùng sai
đơn vị thống kê thì còn tệ hơn:

| Kiểm định | Lỗi thường gặp | Cách làm ở đây |
|---|---|---|
| **McNemar mức frame** | Hàng nghìn quyết định frame từ vài chục bản ghi → pseudo-replication; gần như **mọi cặp** đều `p ≈ 0`. | **McNemar ở MỨC BẢN GHI** là kiểm định chính (+ Holm). McNemar mức frame chỉ in để *mô tả*. |
| **Wilcoxon 5 fold** | Với k = 5, p nhỏ nhất **có thể** đạt là `2/2⁵ = 0,0625` → **không bao giờ** bác bỏ được ở 0,05. | In rõ **sàn p**; chỉ chạy khi số fold ≥ 8. |
| **CI cho AUC** | Bootstrap theo frame → hẹp giả tạo. | **Bootstrap theo cụm bản ghi** + **bootstrap ghép cặp theo bản ghi** cho ΔAUC. |

Ba kiểm định được dùng:

- **McNemar mức bản ghi** trên các quyết định ghép cặp — đúng đơn vị (n = số bản ghi).
- **Bootstrap ghép cặp theo cụm bản ghi** cho ΔAUC (A − B) — không phụ thuộc ngưỡng.
- **Wilcoxon signed-rank theo fold** — chỉ khi số fold đủ lớn; in kèm sàn p.

Độ lệch chuẩn giữa các lần lặp CV (Bảng 5b) là thước đo bổ sung: chênh lệch nhỏ hơn nó không diễn giải được.


In [ ]:
# =========================================================
# 9.1 So sánh cặp mô hình - ĐÚNG ĐƠN VỊ THỐNG KÊ
#     [V6-S2] Chính: McNemar mức BẢN GHI + bootstrap ghép cặp theo cụm.
#             Phụ (mô tả): McNemar mức frame, có cảnh báo pseudo-replication.
# =========================================================
def record_decisions(name, agg=None):
    agg = agg or best_agg
    g = REC[name]
    thr_r = (youden_threshold(REC_INNER[name].label.values, REC_INNER[name][agg].values)
             if cfg.nested_selection
             else youden_threshold(g.label.values, g[agg].values))
    d = pd.DataFrame(dict(record_id=g.record_id.values, label=g.label.values.astype(int),
                          score=g[agg].values, pred=(g[agg].values >= thr_r).astype(int)))
    return d.sort_values("record_id").reset_index(drop=True)

REC_DEC = {n: record_decisions(n) for n in cfg.models}

pairs = list(itertools.combinations(cfg.models, 2))
stat_rows = []
for a, b in pairs:
    da, db = REC_DEC[a], REC_DEC[b]
    assert (da.record_id.values == db.record_id.values).all(), "hai mô hình khác tập bản ghi"
    y_rec = da.label.values
    # (1) McNemar MỨC BẢN GHI - kiểm định chính
    mc_rec = mcnemar_test(y_rec, da.pred.values, db.pred.values)
    # (2) bootstrap ghép cặp theo cụm cho ΔAUC (mức bản ghi)
    auc_a, auc_b = roc_auc_score(da.label, da.score), roc_auc_score(db.label, db.score)
    d_rec = paired_cluster_delta_ci(da.record_id.values, y_rec, da.score.values, db.score.values,
                                    roc_auc_score, n=cfg.bootstrap_n, seed=0)
    # (3) bootstrap ghép cặp theo cụm cho ΔAUC (mức frame) - cùng tập frame
    msk = (~np.isnan(OOF[a])) & (~np.isnan(OOF[b]))
    d_frame = paired_cluster_delta_ci(groups_all[msk], y_all[msk].astype(int),
                                      OOF[a][msk], OOF[b][msk],
                                      roc_auc_score, n=cfg.bootstrap_n, seed=0)
    # (4) McNemar mức FRAME - CHỈ ĐỂ MÔ TẢ (pseudo-replication)
    yf = y_all[msk].astype(int)
    mc_fr = mcnemar_test(yf, (OOF[a][msk] >= THR[a]).astype(int), (OOF[b][msk] >= THR[b]).astype(int))
    # (5) Wilcoxon theo fold (kèm sàn p)
    va = per_fold_df.loc[per_fold_df.model == a].sort_values("fold").auc.values
    vb = per_fold_df.loc[per_fold_df.model == b].sort_values("fold").auc.values
    k_eff = min(len(va), len(vb))
    if k_eff >= 8:
        try: w = float(spstats.wilcoxon(va, vb, zero_method="zsplit").pvalue)
        except Exception: w = np.nan
    else:
        w = np.nan
    stat_rows.append(dict(
        model_A=a, model_B=b,
        AUC_A=auc_a, AUC_B=auc_b, dAUC=auc_a - auc_b,
        dAUC_lo=d_rec["lo"], dAUC_hi=d_rec["hi"], dAUC_p=d_rec["p_two_sided"],
        dAUC_frame_lo=d_frame["lo"], dAUC_frame_hi=d_frame["hi"],
        mcnemar_rec_n01=mc_rec["n01"], mcnemar_rec_n10=mc_rec["n10"],
        mcnemar_rec_p=mc_rec["p"],
        mcnemar_frame_p=mc_fr["p"], mcnemar_frame_n01=mc_fr["n01"], mcnemar_frame_n10=mc_fr["n10"],
        wilcoxon_p=w, wilcoxon_p_floor=wilcoxon_min_p(max(k_eff, 1)), n_folds=k_eff))
stat_df = pd.DataFrame(stat_rows)

# Hiệu chỉnh đa so sánh Holm-Bonferroni cho kiểm định CHÍNH (mức bản ghi)
stat_df["mcnemar_rec_p_holm"] = holm(stat_df.mcnemar_rec_p.fillna(1.0))
stat_df["mcnemar_p_holm"] = stat_df["mcnemar_rec_p_holm"]      # tương thích tên cũ
stat_df["significant_0.05"] = stat_df.mcnemar_rec_p_holm < .05
stat_df["dAUC_ci_excludes_0"] = (stat_df.dAUC_lo > 0) | (stat_df.dAUC_hi < 0)
display(Markdown("### Table 8. Pairwise model comparison: "
                 f"record-level McNemar (n = {len(records)}) + paired cluster bootstrap"))
display(stat_df[["model_A", "model_B", "AUC_A", "AUC_B", "dAUC", "dAUC_lo", "dAUC_hi",
                 "dAUC_p", "mcnemar_rec_n01", "mcnemar_rec_n10", "mcnemar_rec_p",
                 "mcnemar_rec_p_holm", "significant_0.05", "dAUC_ci_excludes_0"]].round(4))
stat_df.round(6).to_csv(TAB_DIR / "table8_statistical_tests.csv", index=False)

n_sig_rec = int(stat_df["significant_0.05"].sum())
n_ci = int(stat_df.dAUC_ci_excludes_0.sum())
print(f"\nMcNemar MỨC BẢN GHI (hiệu chỉnh Holm): {n_sig_rec}/{len(stat_df)} cặp khác biệt có ý nghĩa.")
print(f"Bootstrap ghép cặp: {n_ci}/{len(stat_df)} cặp có CI 95% của ΔAUC KHÔNG chứa 0.")
k_folds = int(per_fold_df.fold.nunique())
print(f"\nSÀN p CỦA WILCOXON: với {k_folds} fold, p nhỏ nhất có thể đạt là "
      f"{wilcoxon_min_p(k_folds):.4f} -> kiểm định này KHÔNG THỂ bác bỏ ở mức 0,05.")
if k_folds < 8:
    print("   => cột wilcoxon_p được ĐỂ TRỐNG (NaN) và không được trích dẫn.")
    print("   => muốn dùng Wilcoxon có ý nghĩa, cần lặp CV (ví dụ 5x5 = 25 cặp fold).")
print(f"\nMcNemar mức FRAME (mô tả, KHÔNG dùng để suy luận): "
      f"{int((stat_df.mcnemar_frame_p < 0.05).sum())}/{len(stat_df)} cặp có p < 0.05 - "
      f"con số này cao vì {len(frames)} frame chỉ đến từ {len(records)} bản ghi (pseudo-replication).")

ci_tab = frame_df[["model", "AUC", "AUC_lo", "AUC_hi", "AUC_cv_mean", "AUC_cv_std",
                   "ICC", "design_effect", "n_eff"]].round(4)
display(Markdown("**95% confidence intervals for frame-level AUC — cluster bootstrap over "
                 "recordings (with ICC and effective sample size):**"))
display(ci_tab)


In [ ]:
# =========================================================
# 9.2 Hình 11: Forest plot AUC + bản đồ p-value (McNemar mức BẢN GHI)
# =========================================================
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2),
                       gridspec_kw=dict(width_ratios=[1.15, 0.95, 0.85]))
d = frame_df.sort_values("AUC")
ys = np.arange(len(d))
ax[0].errorbar(d.AUC, ys, xerr=[d.AUC - d.AUC_lo, d.AUC_hi - d.AUC],
               fmt="o", color="k", capsize=4, ms=6, label="CI theo cụm bản ghi")
ax[0].errorbar(d.AUC, ys + .18, xerr=[d.AUC - d.AUC_lo_naive, d.AUC_hi_naive - d.AUC],
               fmt="x", color=PAL["grey"], capsize=3, ms=5, label="frame bootstrap CI (incorrect)")
for i, (m, v) in enumerate(zip(d.model, d.AUC)):
    ax[0].scatter([v], [i], s=70, color=MODEL_COLORS[m], zorder=5)
ax[0].set_yticks(ys); ax[0].set_yticklabels(d.model)
ax[0].axvline(.5, color="k", ls="--", lw=.8)
ax[0].set_xlabel("AUC (out-of-fold, 95% CI)"); ax[0].legend(fontsize=7, loc="lower right")
ax[0].set_title("(a) Forest plot — cluster CI vs frame CI")

P = np.ones((len(cfg.models), len(cfg.models)))
for _, r in stat_df.iterrows():
    i, j = list(cfg.models).index(r.model_A), list(cfg.models).index(r.model_B)
    P[i, j] = P[j, i] = r.mcnemar_rec_p_holm
im = ax[1].imshow(-np.log10(np.clip(P, 1e-12, 1)), cmap="viridis")
for i in range(len(cfg.models)):
    for j in range(len(cfg.models)):
        if i != j:
            ax[1].text(j, i, f"{P[i,j]:.3f}", ha="center", va="center", color="w", fontsize=7.5)
ax[1].set_xticks(range(len(cfg.models))); ax[1].set_xticklabels(cfg.models, rotation=45, ha="right", fontsize=8)
ax[1].set_yticks(range(len(cfg.models))); ax[1].set_yticklabels(cfg.models, fontsize=8)
ax[1].set_title(f"(b) Record-level McNemar (Holm)\nn = {len(records)}, the correct unit of analysis"); ax[1].grid(False)
fig.colorbar(im, ax=ax[1], label="$-\\log_{10}p$", pad=.02)

# (c) ΔAUC với CI ghép cặp - cho thấy phần lớn khác biệt không phân biệt được
sd = stat_df.sort_values("dAUC").reset_index(drop=True)
ys2 = np.arange(len(sd))
ax[2].errorbar(sd.dAUC, ys2, xerr=[sd.dAUC - sd.dAUC_lo, sd.dAUC_hi - sd.dAUC],
               fmt="o", color="k", capsize=3, ms=5)
ax[2].axvline(0, color=PAL["leak"], ls="--", lw=1.2)
ax[2].set_yticks(ys2)
ax[2].set_yticklabels([f"{r.model_A[:9]}−{r.model_B[:9]}" for _, r in sd.iterrows()], fontsize=6)
ax[2].set_xlabel("ΔAUC (A − B), 95% paired cluster CI")
ax[2].set_title("(c) ΔAUC: CI containing 0\n= no resolvable difference")
savefig(fig, "statistics",
        "Statistical testing: (a) out-of-fold AUC with cluster-bootstrap CIs over recordings versus "
        "the naive frame-level bootstrap (anti-conservatively narrow); (b) Holm-corrected McNemar "
        "p-values at RECORD level; (c) ΔAUC with paired cluster CIs — most pairs include 0, i.e. the "
        "data do not support ranking architectures.")


### 9.3 Thí nghiệm đối chứng rò rỉ dữ liệu / Data-leakage ablation

Đây là đóng góp phương pháp luận của bài báo: huấn luyện lại **đúng mô hình, đúng siêu tham số**,
chỉ thay giao thức chia dữ liệu:

- **(A) Nhóm theo bản ghi** — giao thức chuẩn của nghiên cứu này.
- **(B) Ngẫu nhiên mức frame** — giao thức phổ biến trong tài liệu.

**[V6-S3] Sửa lỗi quan trọng của v5:** ở v5, hai nhánh được chấm bằng **hai ngưỡng khác nhau**
(nhánh B tự chọn ngưỡng Youden **trên chính dữ liệu báo cáo**), nên chênh lệch accuracy/F1/EER
in ra là **hỗn hợp của rò rỉ dữ liệu và thiên lệch chọn ngưỡng** — đúng loại thiên lệch mà mục
9.4 đo được. Ở v6 **cả hai nhánh dùng cùng một ngưỡng** (ngưỡng lồng của nhánh A) và cùng một
quy tắc gộp mức bản ghi.

**[V6-S4] Đọc kết quả cho đúng sức mạnh của bằng chứng.** ΔAUC giữa hai giao thức là con số
**sạch** (AUC không phụ thuộc ngưỡng). Nhưng nếu ΔAUC **nhỏ**, thì không được dùng nó để tuyên
bố "các kết quả 96–99 % trong tài liệu bị thổi phồng" — phải nói đúng mức: *trên tập dữ liệu
này*, với ICC cao (điểm số gần như hằng số trong từng bản ghi), việc "nhớ" một nửa bản ghi
không thêm nhiều thông tin, nên mức thổi phồng **về AUC** là nhỏ, trong khi **về chỉ số ngưỡng**
(accuracy/FPR) lại lớn hơn.


In [ ]:
# =========================================================
# 9.3 Ablation: nhóm theo bản ghi vs ngẫu nhiên mức frame
#     [V6-S3] CÙNG NGƯỠNG, CÙNG QUY TẮC GỘP cho cả hai nhánh.
# =========================================================
RUN_LEAKAGE_ABLATION = bool(cfg.run_leakage_ablation)
ABLATION_MODEL = BEST_MODEL

if RUN_LEAKAGE_ABLATION:
    skf = StratifiedKFold(n_splits=len(SPLITS), shuffle=True, random_state=cfg.seed)
    naive_splits = list(skf.split(np.zeros(len(frames)), y_all))

    n_shared = float(np.mean([len(set(groups_all[tr]) & set(groups_all[va]))
                              for tr, va in naive_splits]))
    frac_shared = float(np.mean([len(set(groups_all[tr]) & set(groups_all[va])) /
                                 len(set(groups_all[va])) for tr, va in naive_splits]))
    print(f"Giao thức ngẫu nhiên: trung bình {n_shared:.1f} bản ghi xuất hiện ở CẢ train và "
          f"validation mỗi fold ({frac_shared*100:.0f}% số bản ghi của validation).")
    print("Giao thức nhóm theo bản ghi: 0. Đây là mức rò rỉ được tạo ra có chủ đích.")

    naive_res = [train_one_fold(ABLATION_MODEL, tr, va, k, verbose=False)
                 for k, (tr, va) in enumerate(naive_splits)]
    oof_naive = assemble_oof(naive_res, "val")

    mk = ~np.isnan(oof_naive); mg = ~np.isnan(OOF[ABLATION_MODEL])
    y_n = y_all[mk].astype(int); y_g = y_all[mg].astype(int)

    # [V6-S3] DÙNG CÙNG NGƯỠNG cho cả hai nhánh (ngưỡng lồng của nhánh A)
    thr_shared = THR[ABLATION_MODEL]
    # ngưỡng mức bản ghi cũng dùng chung: lấy từ validation-trong của nhánh A
    thr_rec_A = youden_threshold(REC_INNER[ABLATION_MODEL].label.values,
                                 REC_INNER[ABLATION_MODEL][best_agg].values)
    gA = REC[ABLATION_MODEL]
    gB = record_scores(np.where(mk)[0], y_n, oof_naive[mk], thr_shared)

    m_group = compute_metrics(y_g, OOF[ABLATION_MODEL][mg], thr_shared)
    m_naive = compute_metrics(y_n, oof_naive[mk], thr_shared)

    abl = pd.DataFrame([
        dict(protocol="A. Recording-grouped", **{k: m_group[k] for k in
             ["auc", "accuracy", "precision", "recall", "specificity", "f1", "fpr", "eer"]},
             auc_record=(float(roc_auc_score(gA.label, gA[best_agg]))
                         if gA.label.nunique() > 1 else np.nan)),
        dict(protocol="B. Random frame split", **{k: m_naive[k] for k in
             ["auc", "accuracy", "precision", "recall", "specificity", "f1", "fpr", "eer"]},
             auc_record=(float(roc_auc_score(gB.label, gB[best_agg]))
                         if gB.label.nunique() > 1 else np.nan)),
    ])
    abl.loc["delta"] = ["B − A (inflation)"] + list(abl.iloc[1, 1:] - abl.iloc[0, 1:])
    display(Markdown(f"### Table 9. Effect of data leakage — {ABLATION_MODEL} "
                     f"(both arms use the SAME threshold {thr_shared:.3f})"))
    display(abl.round(4))
    abl.round(4).to_csv(TAB_DIR / "table9_leakage_ablation.csv", index=False)

    d_auc = float(abl.loc["delta", "auc"]); d_acc = float(abl.loc["delta", "accuracy"])
    d_auc_rec = float(abl.loc["delta", "auc_record"])
    print(f"\nΔAUC mức frame  = {d_auc:+.4f}  <- con số SẠCH của hiệu ứng rò rỉ dữ liệu")
    print(f"ΔAUC mức bản ghi= {d_auc_rec:+.4f}")
    print(f"ΔAccuracy       = {d_acc:+.4f}  <- một phần là do lựa chọn ngưỡng, xem mục 9.4")
    icc_ab = float(vc_df.set_index('model').loc[ABLATION_MODEL, 'ICC'])
    print(f"\n[V6-S4] ĐỌC CHO ĐÚNG: ICC của mô hình này là {icc_ab:.3f} -> điểm số gần như HẰNG SỐ")
    print("  trong từng bản ghi. Vì vậy việc để lộ một nửa bản ghi ở train KHÔNG thêm nhiều")
    print("  thông tin, và mức thổi phồng về AUC là nhỏ.")
    if abs(d_auc) < 0.05:
        print("  !! ΔAUC < 0,05 -> KHÔNG được dùng con số này để khẳng định 'các kết quả 96-99%")
        print("     trong tài liệu là do rò rỉ dữ liệu'. Phát biểu đúng: trên tập dữ liệu NÀY,")
        print("     thổi phồng do chia ngẫu nhiên là nhỏ về AUC nhưng rõ hơn về FPR/accuracy.")
    else:
        print("  ΔAUC >= 0,05 -> có thể trích dẫn như bằng chứng định lượng về thổi phồng.")

    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.0))
    for lab, y_, p_, col in [("A. grouped", y_g, OOF[ABLATION_MODEL][mg], PAL["noleak"]),
                             ("B. random frames", y_n, oof_naive[mk], PAL["leak"])]:
        fpr, tpr, _ = roc_curve(y_, p_)
        ax[0].plot(fpr, tpr, lw=1.9, color=col, label=f"{lab} — AUC={roc_auc_score(y_, p_):.3f}")
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].legend(fontsize=8)
    ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].set_title("(a) ROC under the two protocols")

    keys = ["auc", "accuracy", "recall", "specificity", "f1"]
    xs = np.arange(len(keys)); w = .36
    ax[1].bar(xs - w/2, [m_group[k] for k in keys], w, color=PAL["noleak"], label="A. grouped")
    ax[1].bar(xs + w/2, [m_naive[k] for k in keys], w, color=PAL["leak"], label="B. random")
    for i, k in enumerate(keys):
        ax[1].text(i, max(m_group[k], m_naive[k]) + .01, f"{m_naive[k]-m_group[k]:+.3f}",
                   ha="center", fontsize=8)
    ax[1].set_xticks(xs); ax[1].set_xticklabels(keys, rotation=15)
    ax[1].set_ylim(0, 1.12); ax[1].legend(fontsize=8)
    ax[1].set_title("(b) Metrics under random splitting (same threshold)")
    savefig(fig, "leakage_ablation",
            f"Leakage-induced inflation on this dataset ({ABLATION_MODEL}), "
            f"both arms using the SAME threshold: ΔAUC {d_auc:+.3f}, Δaccuracy {d_acc:+.3f}. "
            f"ICC = {icc_ab:.2f} explains why the AUC inflation is small.")
else:
    print("Đã bỏ qua ablation rò rỉ dữ liệu (cfg.run_leakage_ablation=False).")


### 9.4 Thí nghiệm đối chứng LỰA CHỌN / Selection-bias ablation

Cùng logic với ablation rò rỉ dữ liệu ở 9.3, nhưng cho một kênh lạc quan tinh vi hơn — và
**không tốn thêm lần huấn luyện nào**, vì cả hai con số đã được ghi trong cùng một lần chạy:

- **(A) Lựa chọn lồng** — epoch dừng sớm và ngưỡng quyết định chọn trên validation-**trong**;
  fold ngoài chỉ để báo cáo. Đây là con số trung thực.
- **(B) Lựa chọn trên tập báo cáo** — lấy epoch có AUC cao nhất *trên chính fold ngoài*.
  Đây là cách làm ngầm định của phần lớn pipeline học sâu.

**[V6-T3] Thêm một ablation MIỄN PHÍ: tiêu chí chọn epoch.** Vì v6 lưu song song ba tiêu chí
(log-loss, AUC frame, AUC bản ghi) trong cùng một lượt huấn luyện, ta đo được **mức thay đổi
kết quả chỉ do đổi tiêu chí chọn** — nguồn lạc quan thứ ba, trước đây không đo được ở v5
(v5 chốt cứng tiêu chí AUC bản ghi dễ bão hoà).


In [ ]:
# =========================================================
# 9.4 Ablation lựa chọn: lồng (trung thực) vs trên tập báo cáo (thiên lệch)
#     + ablation TIÊU CHÍ CHỌN EPOCH (miễn phí, nhờ V6-T3)
# =========================================================
sel_rows = []
for name in cfg.models:
    mA = ~np.isnan(OOF[name]);     yA, pA = y_all[mA].astype(int), OOF[name][mA]
    mB = ~np.isnan(OOF_OPT[name]); yB, pB = y_all[mB].astype(int), OOF_OPT[name][mB]
    a = compute_metrics(yA, pA, THR[name])
    b = compute_metrics(yB, pB, THR_BIASED[name])
    ciA = cluster_bootstrap_ci(groups_all[mA], yA, pA, roc_auc_score, n=cfg.bootstrap_n, seed=0)
    ciB = cluster_bootstrap_ci(groups_all[mB], yB, pB, roc_auc_score, n=cfg.bootstrap_n, seed=0)
    sel_rows.append(dict(model=name,
                         AUC_nested=a["auc"], AUC_biased=b["auc"], dAUC=b["auc"] - a["auc"],
                         AUC_nested_lo=ciA["lo"], AUC_nested_hi=ciA["hi"],
                         AUC_biased_lo=ciB["lo"], AUC_biased_hi=ciB["hi"],
                         Acc_nested=a["accuracy"], Acc_biased=b["accuracy"],
                         dAcc=b["accuracy"] - a["accuracy"],
                         F1_nested=a["f1"], F1_biased=b["f1"],
                         epoch_nested=np.mean([r["best_epoch"] for r in ALL_RESULTS[name]]),
                         epoch_biased=np.mean([r["opt_epoch"] for r in ALL_RESULTS[name]])))
sel_abl = pd.DataFrame(sel_rows).round(4)
display(Markdown("### Table 9b. Selection-induced inflation (epoch + threshold)"))
display(sel_abl[["model", "AUC_nested", "AUC_biased", "dAUC", "Acc_nested", "Acc_biased",
                 "dAcc", "epoch_nested", "epoch_biased"]])
sel_abl.to_csv(TAB_DIR / "table9b_selection_ablation.csv", index=False)

# ---- [V6-T3] Ablation TIÊU CHÍ CHỌN EPOCH (không tốn thêm lượt huấn luyện) ----
crit_rows = []
for name in cfg.models:
    res = ALL_RESULTS[name]
    for c in SELECTION_CRITERIA:
        if not all(("alt" in r and c in r["alt"] and r["alt"][c]["i_out"] is not None) for r in res):
            continue
        oof = np.full(len(frames), np.nan)
        y_in_c, p_in_c = [], []
        for r in res:
            alt = r["alt"][c]
            oof[alt["i_out"]] = alt["p_out"]
            y_in_c.append(alt["y_in"]); p_in_c.append(alt["p_in"])
        mk = ~np.isnan(oof)
        yv = y_all[mk].astype(int)
        if len(np.unique(yv)) < 2: continue
        thr_c = youden_threshold(np.concatenate(y_in_c), np.concatenate(p_in_c))
        ci = cluster_bootstrap_ci(groups_all[mk], yv, oof[mk], roc_auc_score,
                                  n=min(cfg.bootstrap_n, 1000), seed=0)
        g_c = record_scores(np.where(mk)[0], yv, oof[mk], thr_c)
        auc_r = (float(roc_auc_score(g_c.label, g_c[best_agg])) if g_c.label.nunique() > 1 else np.nan)
        crit_rows.append(dict(model=name, criterion=c,
                              mean_epoch=float(np.mean([r["alt"][c]["epoch"] for r in res])),
                              AUC_frame=float(roc_auc_score(yv, oof[mk])),
                              AUC_lo=ci["lo"], AUC_hi=ci["hi"], AUC_record=auc_r))
crit_df = pd.DataFrame(crit_rows)
if len(crit_df):
    display(Markdown("### Table 9b-2. Effect of the epoch-selection criterion "
                     "(same training run, only the selection rule changes)"))
    piv = crit_df.pivot(index="model", columns="criterion", values="AUC_frame").round(4)
    display(piv)
    display(Markdown("**Mean epoch selected under each criterion:**"))
    display(crit_df.pivot(index="model", columns="criterion", values="mean_epoch").round(1))
    crit_df.round(4).to_csv(TAB_DIR / "table9b2_criterion_ablation.csv", index=False)
    _spread = (piv.max(axis=1) - piv.min(axis=1))
    print("Biên độ thay đổi AUC mức frame chỉ do ĐỔI TIÊU CHÍ CHỌN EPOCH:")
    print(_spread.round(4).to_string())
    print(f"Trung bình {_spread.mean():.4f} | lớn nhất {_spread.max():.4f}")
    if _spread.mean() > 0.02:
        print("!! Lớn hơn 0,02 -> tiêu chí chọn epoch là một NGUỒN LẠC QUAN ĐÁNG KỂ.")
        print("   Phải nêu trong Limitations và phải cố định tiêu chí TRƯỚC khi so sánh kiến trúc.")
    print(f"\nTiêu chí đang dùng cho kết quả chính: '{cfg.select_metric}' "
          f"({CRITERION_LABEL.get(cfg.select_metric, '')})")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
xs = np.arange(len(cfg.models)); w = .36
order = list(cfg.models)
A = [sel_abl.set_index("model").AUC_nested[n] for n in order]
B = [sel_abl.set_index("model").AUC_biased[n] for n in order]
loA = [sel_abl.set_index("model").AUC_nested_lo[n] for n in order]
hiA = [sel_abl.set_index("model").AUC_nested_hi[n] for n in order]
loB = [sel_abl.set_index("model").AUC_biased_lo[n] for n in order]
hiB = [sel_abl.set_index("model").AUC_biased_hi[n] for n in order]
ax[0].bar(xs - w/2, A, w, color=PAL["noleak"], label="A. lựa chọn lồng (trung thực)",
          yerr=[np.array(A)-np.array(loA), np.array(hiA)-np.array(A)], capsize=3)
ax[0].bar(xs + w/2, B, w, color=PAL["leak"], label="B. chọn trên tập báo cáo",
          yerr=[np.array(B)-np.array(loB), np.array(hiB)-np.array(B)], capsize=3)
for i, (a_, b_) in enumerate(zip(A, B)):
    ax[0].text(i, max(a_, b_) + .04, f"{b_-a_:+.3f}", ha="center", fontsize=8)
ax[0].set_xticks(xs); ax[0].set_xticklabels(order, rotation=20, ha="right")
ax[0].set_ylabel("AUC out-of-fold"); ax[0].axhline(.5, color="k", ls="--", lw=.8)
ax[0].set_ylim(0, 1.18); ax[0].legend(fontsize=8)
ax[0].set_title("(a) Selection-induced AUC inflation\n(error bars = 95% cluster CI over recordings)")

for name in cfg.models:
    h = ALL_RESULTS[name][0]["history"]
    if "inner_auc" in h and len(h["inner_auc"]) > 1:
        ax[1].plot(h["epoch"], [-v for v in h["inner_loss"]], lw=1.3,
                   color=MODEL_COLORS[name], label=f"{name} (-log-loss)")
        ax[1].plot(h["epoch"], h["val_auc"], lw=1.0, ls="--", alpha=.7, color=MODEL_COLORS[name])
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("criterion score")
ax[1].set_title("(b) Non-saturating criterion (solid) vs outer-fold AUC (dashed), fold 0")
ax[1].legend(fontsize=7)
savefig(fig, "selection_ablation",
        "Effect of selecting the epoch and threshold on the reported set: every metric is inflated "
        "although data and model are identical. This optimism channel is independent of the "
        "frame-level leakage quantified in Table 9.")
print("\nGhi chú cho bài báo: cột dAUC là mức thổi phồng do LỰA CHỌN, độc lập với")
print("mức thổi phồng do RÒ RỈ DỮ LIỆU ở Bảng 9. Hai nguồn này cộng dồn.")


### 9.5 BASELINE CỔ ĐIỂN — mạng CNN có thực sự thêm giá trị?

Đây là câu hỏi phản biện **đầu tiên**: *"Với vài chục bản ghi, một bộ phân loại cổ điển có làm
được gần bằng CNN không?"* Cell dưới đây trả lời bằng thực nghiệm, **cùng giao thức** (cùng fold
nhóm theo bản ghi, cùng bootstrap theo cụm bản ghi).

**Hai bộ đặc trưng** [V7-S3]:

- `spectral13`: năng lượng tương đối 7 dải + trọng tâm/độ trải/độ phẳng phổ + rolloff 85% + ZCR + RMS (như v6).
- `logmel_stats`: **trung bình và độ lệch chuẩn theo thời gian của 64 dải log-Mel** (128 chiều),
  đã trừ mức dB trung bình của frame → **bất biến với gain**, dùng *đúng* biểu diễn mà CNN nhìn thấy.
  Đây là đối thủ công bằng hơn nhiều so với 13 đặc trưng thủ công.

Ba bộ phân loại (LogReg, SVM-RBF, Gradient Boosting) × hai mức (frame → gộp `p_mean`, và
trực tiếp trên vector trung bình của bản ghi).

**So sánh GHÉP CẶP** [V7]: v6 kết luận "CNN vượt baseline" khi hai CI *riêng rẽ* không chồng lấn
(0,8092 vs 0,8076 — sát nhau tới 0,002). Cách đúng là **CI bootstrap ghép cặp của ΔAUC** trên
cùng tập bản ghi, hiệu chỉnh Holm theo số baseline. Mô hình CNN đại diện là `DEPLOY_MODEL` (chọn
trên validation-trong), không phải mô hình có AUC ngoài cao nhất — tránh thiên lệch có lợi cho CNN.

Ngoài ra in **baseline "chỉ dùng mức năng lượng"** (RMS trung bình bản ghi). Nếu riêng mức to/nhỏ
đã phân biệt tốt hai lớp, mọi kết luận về "học đặc trưng rò rỉ" phải được xem lại.


In [ ]:
# =========================================================
# 9.5 BASELINE CỔ ĐIỂN - cùng giao thức, so sánh GHÉP CẶP với CNN
#     [V7-S3] thêm bộ đặc trưng thống kê log-Mel (bất biến gain); kết luận dựa trên CI
#             bootstrap GHÉP CẶP của ΔAUC (+ Holm), không dựa trên "hai CI có chồng lấn".
# =========================================================
def frame_psd_matrix(X, n_fft=1024, hop=512, sr=None):
    # Ma trận PSD cho mọi frame, tính vector hoá (nhanh hơn vòng lặp welch).
    sr = sr or cfg.target_sr
    X = np.asarray(X, dtype=np.float32)
    win = np.hanning(n_fft).astype(np.float32)
    nseg = 1 + (X.shape[1] - n_fft) // hop
    idx = np.arange(n_fft)[None, :] + hop * np.arange(nseg)[:, None]
    freqs = np.fft.rfftfreq(n_fft, 1.0 / sr)
    out = np.empty((X.shape[0], len(freqs)), dtype=np.float32)
    for i in range(0, X.shape[0], 256):
        blk = X[i:i + 256]
        S = np.fft.rfft(blk[:, idx] * win, axis=2)
        out[i:i + 256] = (np.abs(S) ** 2).mean(axis=1)
    return out, freqs

def spectral_features(P, freqs, rms=None, zcr=None):
    tot = P.sum(axis=1) + 1e-20
    F_ = {}
    for (a, b), nm in zip(BANDS, BAND_NAMES):
        m = (freqs >= a) & (freqs < b)
        F_[f"band_{nm}"] = (P[:, m].sum(axis=1) / tot).astype(np.float32)
    cen = (P * freqs[None, :]).sum(axis=1) / tot
    F_["centroid"] = cen
    F_["spread"] = np.sqrt((P * (freqs[None, :] - cen[:, None]) ** 2).sum(axis=1) / tot)
    F_["flatness"] = np.exp(np.mean(np.log(P + 1e-12), axis=1)) / (P.mean(axis=1) + 1e-20)
    cs = np.cumsum(P, axis=1) / tot[:, None]
    F_["rolloff85"] = freqs[np.argmax(cs >= 0.85, axis=1)]
    if rms is not None: F_["rms"] = np.asarray(rms, dtype=np.float32)
    if zcr is not None: F_["zcr"] = np.asarray(zcr, dtype=np.float32)
    return pd.DataFrame(F_)

@torch.no_grad()
def logmel_stats_matrix(X, batch=256):
    # Mean + std theo thời gian của từng dải log-Mel, sau khi trừ mức dB trung bình của frame
    # -> bất biến với gain; đúng biểu diễn mà CNN nhìn thấy (trước chuẩn hoá theo mẫu).
    fe = LogMelFrontend().to(DEVICE).eval()
    out = []
    for s in range(0, len(X), batch):
        xb = torch.from_numpy(np.asarray(X[s:s + batch])).float().to(DEVICE)
        db = fe.wave_to_db(xb)[:, 0]
        db = db - db.mean(dim=(1, 2), keepdim=True)
        out.append(torch.cat([db.mean(2), db.std(2)], 1).float().cpu().numpy())
    del fe
    return np.concatenate(out)

print("Đang tính đặc trưng cho mọi frame...")
_t0 = time.time()
P_ALL, FREQS_BL = frame_psd_matrix(X_ALL)
FEAT_ALL = spectral_features(P_ALL, FREQS_BL, rms=frames.rms.values, zcr=frames.zcr.values)
FEAT_ALL.to_csv(TAB_DIR / "features_classical.csv", index=False)
FEATURE_NAMES = list(FEAT_ALL.columns)
FEATURE_SETS = {"spectral13": FEAT_ALL}
if "logmel_stats" in cfg.baseline_feature_sets:
    _lm = logmel_stats_matrix(X_ALL)
    FEATURE_SETS["logmel_stats"] = pd.DataFrame(
        _lm, columns=[f"mel{i:02d}_mean" for i in range(cfg.n_mels)] +
                     [f"mel{i:02d}_std" for i in range(cfg.n_mels)])
FEATURE_SETS = {k: v for k, v in FEATURE_SETS.items() if k in cfg.baseline_feature_sets}
print(f"  xong trong {time.time()-_t0:.1f}s | " +
      ", ".join(f"{k}: {v.shape[1]} chiều" for k, v in FEATURE_SETS.items()))

def _sk_model(kind, seed=0):
    if kind == "logreg":
        return LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced")
    if kind == "svm":
        return SVC(C=2.0, gamma="scale", probability=False, class_weight="balanced")
    if kind == "gbm":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                              max_depth=3, random_state=seed)
    raise ValueError(kind)

def _scores_from_model(mdl, Xv):
    if hasattr(mdl, "predict_proba"):
        return mdl.predict_proba(Xv)[:, 1]
    d = mdl.decision_function(Xv)
    return 1.0 / (1.0 + np.exp(-d))

def run_classical(feat_df, splits=None, seed_base=0):
    # Trả về các hàng kết quả + điểm OOF MỨC BẢN GHI của từng baseline (để so sánh ghép cặp).
    splits = SPLITS if splits is None else splits
    names = list(feat_df.columns)
    Xf = np.nan_to_num(feat_df.values.astype(np.float64))
    yf = y_all.astype(int); recf = groups_all
    fr = feat_df.copy(); fr["record_id"] = frames.record_id.values; fr["label"] = yf
    Xr = fr.groupby("record_id")[names].mean()
    yr = fr.groupby("record_id").label.first()
    rec_ids = Xr.index.values; pos = {r: i for i, r in enumerate(rec_ids)}
    rows, rec_scores = [], {}
    for kind in cfg.baseline_models:
        oof_f = np.full(len(frames), np.nan); oof_r = np.full(len(rec_ids), np.nan)
        for k, (tr, va) in enumerate(splits):
            sc = StandardScaler().fit(Xf[tr])
            mdl = _sk_model(kind, seed_base + k).fit(sc.transform(Xf[tr]), yf[tr])
            oof_f[va] = _scores_from_model(mdl, sc.transform(Xf[va]))
            itr = np.array([pos[r] for r in pd.unique(recf[tr])])
            iva = np.array([pos[r] for r in pd.unique(recf[va])])
            sc2 = StandardScaler().fit(Xr.values[itr])
            mdl2 = _sk_model(kind, seed_base + 100 + k).fit(sc2.transform(Xr.values[itr]), yr.values[itr])
            oof_r[iva] = _scores_from_model(mdl2, sc2.transform(Xr.values[iva]))
        mk = ~np.isnan(oof_f)
        ci = cluster_bootstrap_ci(recf[mk], yf[mk], oof_f[mk], roc_auc_score, n=cfg.bootstrap_n, seed=0)
        rows.append(dict(baseline=kind, level="frame", auc=float(roc_auc_score(yf[mk], oof_f[mk])),
                         lo=ci["lo"], hi=ci["hi"], n=int(mk.sum())))
        # (a) frame -> gộp p_mean theo bản ghi (cùng quy tắc với CNN)
        auc_fr, g_fr = record_auc_simple(oof_f)
        ci2 = cluster_bootstrap_ci(g_fr.index.values, g_fr.y.values, g_fr.p.values, roc_auc_score,
                                   n=cfg.bootstrap_n, seed=0)
        rows.append(dict(baseline=kind, level="frame->record", auc=auc_fr, lo=ci2["lo"], hi=ci2["hi"],
                         n=int(len(g_fr))))
        rec_scores[(kind, "frame->record")] = g_fr.p
        # (b) trực tiếp trên vector trung bình của bản ghi
        mr = ~np.isnan(oof_r)
        cir = cluster_bootstrap_ci(rec_ids[mr], yr.values[mr], oof_r[mr], roc_auc_score,
                                   n=cfg.bootstrap_n, seed=0)
        rows.append(dict(baseline=kind, level="record", auc=float(roc_auc_score(yr.values[mr], oof_r[mr])),
                         lo=cir["lo"], hi=cir["hi"], n=int(mr.sum())))
        rec_scores[(kind, "record")] = pd.Series(oof_r, index=rec_ids)
    return pd.DataFrame(rows), rec_scores

if cfg.run_baseline_models:
    _parts, BASE_REC_SCORES = [], {}
    for fs, fdf in FEATURE_SETS.items():
        _t0 = time.time()
        df_, sc_ = run_classical(fdf)
        df_.insert(0, "features", fs); _parts.append(df_)
        for (kind, lvl), s in sc_.items():
            BASE_REC_SCORES[(fs, kind, lvl)] = s
        print(f"  baseline [{fs}] xong trong {time.time()-_t0:.1f}s")
    base_df = pd.concat(_parts, ignore_index=True)

    # baseline "chỉ dùng mức năng lượng" (1 đặc trưng) - kiểm tra confound mức to/nhỏ
    lvl_rec = pd.DataFrame(dict(record_id=frames.record_id.values, rms=frames.rms.values,
                                label=y_all.astype(int))).groupby("record_id").agg(
        rms=("rms", "mean"), label=("label", "first"))
    lvl_auc = float(roc_auc_score(lvl_rec.label, lvl_rec.rms))
    lvl_ci = cluster_bootstrap_ci(lvl_rec.index.values, lvl_rec.label.values, lvl_rec.rms.values,
                                  roc_auc_score, n=cfg.bootstrap_n, seed=0)
    base_df = pd.concat([base_df, pd.DataFrame([dict(
        features="rms", baseline="Level only (RMS)", level="record", auc=lvl_auc,
        lo=lvl_ci["lo"], hi=lvl_ci["hi"], n=len(lvl_rec))])], ignore_index=True)
    display(Markdown("### Table 9c. Classical baselines (same folds, cluster-bootstrap CI)"))
    display(base_df.round(4))
    base_df.to_csv(TAB_DIR / "table9c_classical_baselines.csv", index=False)

    # ---- So sánh GHÉP CẶP: CNN (DEPLOY_MODEL, chọn trên validation-trong) vs từng baseline ----
    CNN_REF = DEPLOY_MODEL
    g_cnn = REC[CNN_REF].set_index("record_id")
    cmp_rows = []
    for (fs, kind, lvl), s in BASE_REC_SCORES.items():
        common = g_cnn.index.intersection(s.dropna().index)
        yv = g_cnn.loc[common, "label"].values.astype(int)
        pa = g_cnn.loc[common, best_agg].values; pb = s.loc[common].values
        d = paired_cluster_delta_ci(common.values, yv, pa, pb, roc_auc_score, n=cfg.bootstrap_n, seed=0)
        cmp_rows.append(dict(cnn=CNN_REF, features=fs, baseline=kind, level=lvl,
                             AUC_cnn=float(roc_auc_score(yv, pa)), AUC_base=float(roc_auc_score(yv, pb)),
                             dAUC=float(roc_auc_score(yv, pa) - roc_auc_score(yv, pb)),
                             dAUC_lo=d["lo"], dAUC_hi=d["hi"], p=d["p_two_sided"], n=int(len(common))))
    BASE_CMP = pd.DataFrame(cmp_rows)
    BASE_CMP["p_holm"] = holm(BASE_CMP.p.fillna(1.0).values)
    BASE_CMP["cnn_better"] = BASE_CMP.dAUC_lo > 0
    BASE_CMP = BASE_CMP.sort_values("AUC_base", ascending=False).reset_index(drop=True)
    display(Markdown(f"### Table 9c-2. Paired comparison at record level: {CNN_REF} (selected on inner "
                     f"validation) vs each baseline (paired cluster bootstrap, Holm over {len(BASE_CMP)} tests)"))
    display(BASE_CMP.round(4))
    BASE_CMP.round(6).to_csv(TAB_DIR / "table9c2_cnn_vs_baselines_paired.csv", index=False)

    _strong = BASE_CMP.iloc[0]
    BEST_BASE_TXT = (f"{_strong.baseline} on {_strong.features} ({_strong.level}, AUC {_strong.AUC_base:.3f})")
    _n_beat = int((BASE_CMP.cnn_better & (BASE_CMP.p_holm < 0.05)).sum())
    print(f"\nCNN tham chiếu: {CNN_REF} (AUC bản ghi {BASE_CMP.AUC_cnn.iloc[0]:.4f})")
    print(f"Baseline mạnh nhất: {BEST_BASE_TXT}")
    print(f"CNN vượt baseline có ý nghĩa (CI ghép cặp > 0 VÀ p_holm < 0.05): {_n_beat}/{len(BASE_CMP)}")
    if _n_beat == len(BASE_CMP):
        BASELINE_VERDICT_VI = ("CNN vượt MỌI baseline cổ điển với CI ghép cặp của ΔAUC nằm hoàn toàn trên 0 "
                               "(hiệu chỉnh Holm)")
        BASELINE_VERDICT_EN = ("the CNN outperformed EVERY classical baseline, with paired-bootstrap CIs "
                               "for dAUC entirely above zero (Holm-corrected)")
    elif bool(BASE_CMP.cnn_better.iloc[0]) and _n_beat > 0:
        BASELINE_VERDICT_VI = (f"CNN vượt baseline mạnh nhất theo CI ghép cặp nhưng chỉ {_n_beat}/{len(BASE_CMP)} "
                               "so sánh còn có ý nghĩa sau hiệu chỉnh Holm - lợi thế của học sâu cần nêu thận trọng")
        BASELINE_VERDICT_EN = (f"the CNN exceeded the strongest baseline by paired CI, but only {_n_beat}/"
                               f"{len(BASE_CMP)} comparisons survived Holm correction; the advantage of deep "
                               "learning must be stated cautiously")
    else:
        BASELINE_VERDICT_VI = ("dữ liệu KHÔNG chứng minh được CNN vượt baseline cổ điển mạnh nhất (CI ghép cặp "
                               "của ΔAUC chứa 0); đóng góp nên đặt ở giao thức đánh giá, không ở kiến trúc")
        BASELINE_VERDICT_EN = ("the data do NOT show that the CNN outperforms the strongest classical "
                               "baseline (paired CI for dAUC includes zero)")
    print("=> " + BASELINE_VERDICT_VI)
    print(f"\nBaseline 'CHỈ mức năng lượng': AUC bản ghi {lvl_auc:.4f} [{lvl_ci['lo']:.4f}, {lvl_ci['hi']:.4f}]")
    if lvl_auc > 0.75:
        print("!! Mức to/nhỏ của tín hiệu ĐÃ phân biệt được hai lớp ở mức cao. Đây là confound")
        print("   nghiêm trọng: mô hình có thể chỉ đang học 'độ ồn của vị trí thu'. Phải nêu rõ.")
    # tương thích tên cũ cho các cell phía sau
    best_base_r = base_df[(base_df.level == "record") & (base_df.baseline != "Level only (RMS)")] \
        .sort_values("auc", ascending=False).iloc[0]
else:
    BEST_BASE_TXT, BASELINE_VERDICT_VI, BASELINE_VERDICT_EN = "không chạy", "chưa đánh giá", "not evaluated"
    print("cfg.run_baseline_models=False -> bỏ qua baseline cổ điển (KHÔNG nên bỏ khi viết bài).")


### 9.6 KIỂM TRA CONFOUND theo điểm khảo sát / site-level confounding

Các dấu hiệu trong chính kết quả của notebook có thể chỉ về cùng một khả năng: **mô hình phân
biệt điểm khảo sát (nơi thu âm) chứ không phân biệt hiện tượng rò rỉ**:

1. **ICC cao** (mục 8.8): phần lớn phương sai điểm số nằm *giữa các bản ghi*.
2. **Tỉ lệ thể hiện dương cao** (mục 8.7): điểm số gần như hằng số trong từng bản ghi.
3. **Mức to/nhỏ tín hiệu khác nhau giữa hai lớp** (mục 9.5).

Cell này **đo** confound ở mức có thể đo được, theo thứ tự mạnh → yếu:

| Phép kiểm tra | Cần gì | Sức mạnh |
|---|---|---|
| **Leave-one-site-out** (huấn luyện lại, bỏ hẳn 1 điểm khảo sát) | `cfg.site_source` | Mạnh nhất — trả lời trực tiếp câu hỏi |
| **AUC trong cùng một điểm** (chỉ dùng điểm có cả hai lớp) | `cfg.site_source` | Mạnh — loại bỏ khác biệt giữa các vị trí |
| **Baseline mức năng lượng / đặc trưng phổ** | không cần gì | Trung bình — phát hiện confound "chỉ học độ to" |
| **Phân bố điểm theo bản ghi** | không cần gì | Yếu — chỉ là dấu hiệu cảnh báo |

> Nếu **không** có thông tin điểm khảo sát, notebook in **cảnh báo rõ ràng** rằng confound chưa
> được loại trừ và **[V7] tự sinh tệp `site_map_template.csv`** để điền cột `site`.


In [ ]:
# =========================================================
# 9.6 KIỂM TRA CONFOUND - đo được gì thì đo, không được thì nói rõ là không được
# =========================================================
CONFOUND = dict(has_site=bool(HAS_SITE), loso_auc=None, within_site_auc=None,
                level_auc=None, verdict="chưa xác định")

def _record_score_for(name, agg=None):
    agg = agg or best_agg
    return REC[name][["record_id", "label", agg]].rename(columns={agg: "score"})

def within_site_auc(name, agg=None, min_pairs=4):
    # AUC tính BÊN TRONG từng điểm khảo sát (chỉ điểm có cả 2 lớp) rồi gộp theo trọng số.
    if not HAS_SITE: return np.nan, 0
    g = _record_score_for(name, agg).merge(records[["record_id", "site"]], on="record_id")
    num = den = 0.0; used = 0
    for s, d in g.groupby("site"):
        if d.label.nunique() < 2 or len(d) < min_pairs: continue
        try:
            a = roc_auc_score(d.label, d.score)
        except Exception:
            continue
        w = len(d)
        num += a * w; den += w; used += 1
    return (float(num / den) if den > 0 else np.nan), used

print("=== 9.6 KIỂM TRA CONFOUND THEO ĐIỂM KHẢO SÁT ===")
if cfg.run_confound_checks:
    # (1) mức năng lượng - đã in ở 9.5, in lại để có một chỗ duy nhất
    _lr = pd.DataFrame(dict(record_id=frames.record_id.values, rms=frames.rms.values,
                            label=y_all.astype(int))).groupby("record_id").agg(
        rms=("rms", "mean"), label=("label", "first"))
    CONFOUND["level_auc"] = float(roc_auc_score(_lr.label, _lr.rms))
    print(f"(1) Chỉ dùng MỨC NĂNG LƯỢNG trung bình bản ghi: AUC = {CONFOUND['level_auc']:.4f}")

    # (2) trong cùng điểm khảo sát
    if HAS_SITE:
        wa, n_site = within_site_auc(BEST_MODEL)
        CONFOUND["within_site_auc"] = wa
        print(f"(2) AUC TRONG cùng điểm khảo sát ({n_site} điểm có đủ 2 lớp): "
              f"{'n/a' if wa != wa else f'{wa:.4f}'}")
        if wa == wa:
            print(f"    So sánh: AUC toàn cục mức bản ghi = "
                  f"{float(rec_main.set_index('model').auc[BEST_MODEL]):.4f}")
            if wa < 0.7:
                print("    => Khi loại bỏ khác biệt giữa các vị trí, khả năng phân biệt GIẢM MẠNH.")
                print("       Đây là bằng chứng trực tiếp của confound theo điểm khảo sát.")
    else:
        print("(2) BỎ QUA: chưa có thông tin điểm khảo sát (cfg.site_source='none').")
        # [V7-C1] Sinh sẵn tệp mẫu để người thu dữ liệu chỉ việc điền cột `site`.
        _tpl = records[["record_id", "cls", "n_files", "duration_s"]].copy()
        _tpl["site"] = ""
        _tpl.to_csv(OUT / "site_map_template.csv", index=False, encoding="utf-8-sig")
        print(f"    -> Đã tạo {OUT / 'site_map_template.csv'}: điền cột `site` (vị trí/tuyến ống/ngày thu)")
        print("       rồi đặt cfg.site_source='csv', cfg.site_csv=<đường dẫn>, cfg.run_loso_full=True.")

    # (3) leave-one-site-out đầy đủ (tuỳ chọn vì tốn thêm một chu kỳ huấn luyện)
    if HAS_SITE and getattr(cfg, "run_loso_full", False):
        site_of = records.set_index("record_id").site.to_dict()
        gsite = np.array([site_of[r] for r in REC_IDS])
        n_site = len(np.unique(gsite))
        if n_site >= 3:
            loso = []
            for s in np.unique(gsite):
                va = _expand(REC_IDS[gsite == s]); tr = _expand(REC_IDS[gsite != s])
                loso.append((tr, va))
            print(f"    Đang chạy leave-one-site-out ({n_site} fold) cho {BEST_MODEL}...")
            res_loso = run_cv(BEST_MODEL, splits=loso, tag="loso")
            oof_l = assemble_oof(res_loso, "val")
            mk = ~np.isnan(oof_l)
            CONFOUND["loso_auc"] = float(roc_auc_score(y_all[mk].astype(int), oof_l[mk]))
            ci = cluster_bootstrap_ci(groups_all[mk], y_all[mk].astype(int), oof_l[mk],
                                      roc_auc_score, n=cfg.bootstrap_n, seed=0)
            print(f"(3) Leave-one-site-out: AUC = {CONFOUND['loso_auc']:.4f} "
                  f"[{ci['lo']:.4f}, {ci['hi']:.4f}]  (so với "
                  f"{float(rec_main.set_index('model').auc[BEST_MODEL]):.4f} khi chỉ bỏ theo bản ghi)")
    elif HAS_SITE:
        print("(3) Leave-one-site-out: BỎ QUA (đặt cfg.run_loso_full=True để chạy; rất nên chạy).")

# ---- KẾT LUẬN TỰ ĐỘNG, dựa trên số liệu ---------------------------------
_lvl = CONFOUND["level_auc"]
_hi_conf = (_lvl is not None and _lvl > 0.75)
_ws = CONFOUND["within_site_auc"]
_ws_bad = (_ws is not None and _ws == _ws and _ws < 0.7)
if _ws_bad or _hi_conf:
    CONFOUND["verdict"] = "CÓ DẤU HIỆU CONFOUND"
    print("\n!!! KẾT LUẬN: CÓ DẤU HIỆU CONFOUND THEO ĐIỂM KHẢO SÁT / MỨC NĂNG LƯỢNG.")
    print("    Trong bài báo PHẢI:")
    print("      - nói rõ các bản ghi leak và noleak không được thu ở cùng một tập điểm,")
    print("      - hạ mức kết luận từ 'nhận dạng rò rỉ' xuống 'phân biệt hai nhóm bản ghi',")
    print("      - và nêu leave-one-site-out là thí nghiệm bắt buộc cho nghiên cứu tiếp theo.")
elif not HAS_SITE:
    CONFOUND["verdict"] = "CHƯA LOẠI TRỪ ĐƯỢC CONFOUND"
    print("\n!! KẾT LUẬN: CHƯA LOẠI TRỪ ĐƯỢC CONFOUND (thiếu thông tin điểm khảo sát).")
    print("   Số liệu hiện có KHÔNG đủ để phân biệt 'mô hình học rò rỉ' với")
    print("   'mô hình học vị trí thu'. Phải phát biểu đúng mức độ này trong bài báo.")
else:
    CONFOUND["verdict"] = "Không thấy dấu hiệu confound rõ"
    print("\n✓ Không thấy dấu hiệu confound rõ rệt ở các kiểm tra gián tiếp.")
pd.DataFrame([CONFOUND]).to_csv(TAB_DIR / "table9d_confound_checks.csv", index=False)


### 9.7 ABLATION CẤU HÌNH TÍN HIỆU & HUẤN LUYỆN (bật mặc định) [V7-A1]

Huấn luyện lại **cùng mô hình (`cfg.ablation_model`, cố định trước), cùng phân hoạch fold, cùng
seed**, chỉ đổi đúng một yếu tố:

| Biến thể | Thay đổi | Câu hỏi |
|---|---|---|
| `no_fir` | tắt augmentation đáp ứng kênh ngẫu nhiên | FIR ngẫu nhiên có xoá mất đặc trưng *độ dốc phổ* — thứ phân biệt hai lớp? |
| `rec_balanced` | trọng số 1/n_frame cho mỗi bản ghi | bản ghi dài có đang chi phối huấn luyện? |
| `sr16k` | lấy mẫu 16 kHz (giữ dải 4–8 kHz), n_fft 2048 | dải > 4 kHz có mang thông tin? |
| `frame4s` / `frame1s` | độ dài frame 4 s / 1 s | vì sao chọn 2 s? |
| `no_notch` / `no_highpass` | bỏ notch 50 Hz / bỏ lọc thông cao | các bộ lọc có cần thiết? |

**Lựa chọn trung thực:** mỗi biến thể có AUC mức bản ghi trên **validation-trong** (dùng để
*chọn*) và trên fold ngoài (dùng để *báo cáo*), cùng **ΔAUC ghép cặp** so với cấu hình gốc. Biến
thể được đề xuất là biến thể có AUC validation-trong cao nhất. Muốn dùng nó làm cấu hình chính,
hãy đổi `cfg` rồi chạy lại toàn bộ notebook; khi đó con số báo cáo vẫn không thiên lệch.


In [ ]:
# =========================================================
# 9.7 ABLATION CẤU HÌNH TÍN HIỆU & HUẤN LUYỆN [V7-A1]
#     Cùng mô hình (cố định trước), cùng phân hoạch, cùng seed; chỉ đổi MỘT yếu tố.
#     Chọn biến thể bằng AUC mức bản ghi trên validation-TRONG -> không thiên lệch.
# =========================================================
def record_stratified_splits(frames_df, K_, seed):
    rl = frames_df.groupby("record_id").label.first()
    r2f = frames_df.groupby("record_id").indices
    ids, yy = rl.index.values, rl.values.astype(int)
    K_ = int(min(K_, pd.Series(yy).value_counts().min()))
    skf = StratifiedKFold(n_splits=K_, shuffle=True, random_state=seed)
    ex = lambda sub: np.sort(np.concatenate([r2f[r] for r in sub]))
    return [(ex(ids[a]), ex(ids[b])) for a, b in skf.split(np.zeros(len(ids)), yy)], K_

class _FramesSwap:
    # Đổi tạm bộ frame toàn cục để tái dùng nguyên vẹn bộ máy huấn luyện.
    # Phải tính lại MỌI cấu trúc dẫn xuất từ chỉ số frame.
    _DERIVED = ("X_ALL", "frames", "y_all", "groups_all", "rec_lab", "rec_to_frames",
                "REC_IDS", "REC_Y", "FRAMES_BY_REC")
    def __init__(self, Xv, Fv):
        self.Xv, self.Fv = Xv, Fv
    def __enter__(self):
        g = globals()
        self.saved = {k: g.get(k) for k in self._DERIVED}
        _RESIDENT.clear()
        g["X_ALL"] = self.Xv; g["frames"] = self.Fv
        g["y_all"] = self.Fv.label.values.astype(np.float32)
        g["groups_all"] = self.Fv.record_id.values
        g["rec_lab"] = self.Fv.groupby("record_id").label.first()
        g["rec_to_frames"] = self.Fv.groupby("record_id").indices
        g["REC_IDS"] = g["rec_lab"].index.values
        g["REC_Y"] = g["rec_lab"].values.astype(int)
        g["FRAMES_BY_REC"] = self.Fv.groupby("record_id").indices
        return self.Fv
    def __exit__(self, *exc):
        g = globals()
        for k, v in self.saved.items():
            g[k] = v
        _RESIDENT.clear()
        return False

class _CfgSwap:
    # Đổi tạm các thuộc tính cfg (khôi phục kể cả khi có lỗi).
    def __init__(self, **kw):
        self.kw = kw
    def __enter__(self):
        self.saved = {k: getattr(cfg, k) for k in self.kw}
        for k, v in self.kw.items():
            setattr(cfg, k, v)
        return cfg
    def __exit__(self, *exc):
        for k, v in self.saved.items():
            setattr(cfg, k, v)
        return False

ABL_SPECS = {
    "no_fir":       dict(cfg=dict(aug_fir_p=0.0)),
    "rec_balanced": dict(cfg=dict(record_balanced_loss=True)),
    "sr16k":        dict(cfg=dict(target_sr=16000, n_fft=2048, hop_length=512, f_max=8000.0),
                         frames=dict()),
    "frame4s":      dict(frames=dict(frame_sec=4.0, hop_sec=2.0)),
    "frame1s":      dict(frames=dict(frame_sec=1.0, hop_sec=0.5)),
    "no_notch":     dict(frames=dict(notch=False)),
    "no_highpass":  dict(frames=dict(highpass=0.0)),
}

def _abl_scores(res):
    # điểm mức bản ghi (fold ngoài) + AUC mức bản ghi gộp trên validation-trong
    oof = assemble_oof(res, "val"); mk = ~np.isnan(oof)
    g = record_scores(np.where(mk)[0], y_all[mk], oof[mk], 0.5)[["record_id", "label", best_agg]]
    y_in, p_in, i_in, f_in = pool_inner(res)
    g_in = record_scores(i_in, y_in, p_in, 0.5, folds=f_in)
    auc_in = float(roc_auc_score(g_in.label, g_in[best_agg])) if g_in.label.nunique() > 1 else np.nan
    auc_fr = float(roc_auc_score(y_all[mk].astype(int), oof[mk]))
    return g.set_index("record_id"), auc_in, auc_fr, int(len(frames))

def run_variant(tag, model_name):
    spec = ABL_SPECS[tag]
    with _CfgSwap(**spec.get("cfg", {})):
        if "frames" in spec:
            fkw = spec["frames"]
            Xv, Fv = get_frames(tag, frame_sec=fkw.get("frame_sec"), hop_sec=fkw.get("hop_sec"),
                                highpass=fkw.get("highpass"), notch=fkw.get("notch"))
            with _FramesSwap(Xv, Fv):
                sp, _ = record_stratified_splits(frames, cfg.n_splits, cfg.seed)
                res = run_cv(model_name, splits=sp, seed_base=cfg.seed, tag=tag)
                out = _abl_scores(res)
            del Xv, Fv
        else:
            res = run_cv(model_name, splits=SPLITS, seed_base=cfg.seed, tag=tag)
            out = _abl_scores(res)
    for r in res:
        r["state"] = None
    del res; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return out

if cfg.run_signal_ablation and cfg.ablation_variants:
    ABL_MODEL = cfg.ablation_model if cfg.ablation_model in cfg.models else BEST_MODEL
    print(f"Ablation trên {ABL_MODEL} | biến thể: {list(cfg.ablation_variants)}")
    # cấu hình gốc: TÁI DÙNG lần chạy chính (cùng phân hoạch SPLITS, cùng seed)
    _mk = ~np.isnan(OOF[ABL_MODEL])
    g_base = record_scores(np.where(_mk)[0], y_all[_mk], OOF[ABL_MODEL][_mk], 0.5) \
        .set_index("record_id")[["label", best_agg]]
    _gi = record_scores(*[INNER[ABL_MODEL][j] for j in (2, 0, 1)], 0.5, folds=INNER[ABL_MODEL][3])
    base_in = float(roc_auc_score(_gi.label, _gi[best_agg]))
    abl_rows = [dict(variant="base", n_frames=int(len(frames)), inner_auc_record=base_in,
                     outer_auc_record=float(roc_auc_score(g_base.label, g_base[best_agg])),
                     outer_auc_frame=float(roc_auc_score(y_all[_mk].astype(int), OOF[ABL_MODEL][_mk])),
                     dAUC_vs_base=0.0, dAUC_lo=np.nan, dAUC_hi=np.nan, p=np.nan)]
    for tag in cfg.ablation_variants:
        if tag not in ABL_SPECS:
            print(f"  !! bỏ qua biến thể không xác định: {tag}"); continue
        t0 = time.time()
        g_v, auc_in, auc_fr, n_fr = run_variant(tag, ABL_MODEL)
        common = g_base.index.intersection(g_v.index)
        yv = g_base.loc[common, "label"].values.astype(int)
        pv_, pb_ = g_v.loc[common, best_agg].values, g_base.loc[common, best_agg].values
        d = paired_cluster_delta_ci(common.values, yv, pv_, pb_, roc_auc_score,
                                    n=min(cfg.bootstrap_n, 1000), seed=0)
        abl_rows.append(dict(variant=tag, n_frames=n_fr, inner_auc_record=auc_in,
                             outer_auc_record=float(roc_auc_score(yv, pv_)), outer_auc_frame=auc_fr,
                             dAUC_vs_base=float(roc_auc_score(yv, pv_) - roc_auc_score(yv, pb_)),
                             dAUC_lo=d["lo"], dAUC_hi=d["hi"], p=d["p_two_sided"]))
        print(f"  [{tag}] {time.time()-t0:.0f}s | inner {auc_in:.4f} | outer {abl_rows[-1]['outer_auc_record']:.4f} "
              f"| Δ vs base {abl_rows[-1]['dAUC_vs_base']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]")
    abl_sig = pd.DataFrame(abl_rows)
    abl_sig["p_holm"] = np.r_[np.nan, holm(abl_sig.p.iloc[1:].fillna(1.0).values)] if len(abl_sig) > 1 else np.nan
    ABL_SELECTED = str(abl_sig.loc[abl_sig.inner_auc_record.idxmax(), "variant"])
    abl_sig["selected_by_inner"] = np.where(abl_sig.variant == ABL_SELECTED, "<<<", "")
    display(Markdown(f"### Table 9e. Signal/training ablation — {ABL_MODEL} (same partition and seed; "
                     f"record-level `{best_agg}`; Δ = paired cluster bootstrap vs base)"))
    display(abl_sig.round(4))
    abl_sig.round(6).to_csv(TAB_DIR / "table9e_signal_ablation.csv", index=False)
    print(f"\nBiến thể được validation-TRONG chọn: {ABL_SELECTED}")
    if ABL_SELECTED == "base":
        print("=> Cấu hình gốc vẫn là lựa chọn tốt nhất theo tiêu chí không thiên lệch.")
    else:
        _r = abl_sig.set_index("variant").loc[ABL_SELECTED]
        if _r.dAUC_lo > 0:
            print(f"=> Đề xuất: dùng '{ABL_SELECTED}' (ΔAUC ngoài {_r.dAUC_vs_base:+.4f}, CI ghép cặp "
                  f"[{_r.dAUC_lo:+.4f}, {_r.dAUC_hi:+.4f}] > 0). Đổi cfg tương ứng rồi chạy lại TOÀN BỘ notebook;")
            print("   con số báo cáo của lần chạy đó vẫn trung thực vì lựa chọn chỉ dựa trên validation-trong.")
        else:
            print(f"=> Validation-trong nghiêng về '{ABL_SELECTED}', nhưng ΔAUC ngoài {_r.dAUC_vs_base:+.4f} có CI "
                  f"ghép cặp [{_r.dAUC_lo:+.4f}, {_r.dAUC_hi:+.4f}] CHỨA 0: chênh lệch CHƯA phân giải được.")
            print("   Lựa chọn thận trọng: GIỮ cấu hình gốc; chỉ chuyển khi có thêm bản ghi/lần lặp ủng hộ.")
    _sig = abl_sig[(abl_sig.variant != "base") & ((abl_sig.dAUC_lo > 0) | (abl_sig.dAUC_hi < 0))]
    if len(_sig):
        print("Biến thể có ΔAUC ghép cặp KHÁC 0 (CI không chứa 0): " +
              ", ".join(f"{r.variant} ({r.dAUC_vs_base:+.3f})" for _, r in _sig.iterrows()))
    else:
        print("Không biến thể nào có ΔAUC ghép cặp khác 0 -> kết quả KHÔNG nhạy với các lựa chọn tiền xử lý"
              " này (đây là bằng chứng về độ vững, nên đưa vào bài báo).")
else:
    abl_sig, ABL_SELECTED = None, None
    print("Bỏ qua ablation (cfg.run_signal_ablation=False).")


## 10. Diễn giải mô hình / Model interpretability

Ba phân tích trả lời câu hỏi *"mô hình đã học đúng vật lý hay chỉ học nhiễu nền?"*:

1. **Grad-CAM** trên log-Mel: vùng thời gian–tần số nào chi phối quyết định.
2. **Occlusion theo dải tần**: che từng nhóm dải Mel **trước khi lên dB và trước chuẩn hoá**, đo
   mức sụt AUC ⇒ tầm quan trọng nhân quả của mỗi dải.
3. **t-SNE** trên embedding, **tô màu theo BẢN GHI** để phơi ra cấu trúc theo điểm khảo sát, kèm
   các chỉ số đo trong không gian gốc (không đọc khoảng cách trên bản đồ 2D).

**Nguyên tắc:** hình và caption KHÔNG được khẳng định kết luận trước khi đo — mọi nhận định được
**in ra từ số đo**, kèm cả trường hợp số đo phản bác giả thuyết. Grad-CAM/occlusion chạy trên
**fold cố định (fold 0)**, không chọn fold có AUC cao nhất.

**[V7-B1]** Ở v6, Grad-CAM của `CRNN` gây lỗi *"cudnn RNN backward can only be called in training
mode"* và **dừng toàn bộ notebook** — mục 10.2 → 16 chưa từng chạy. v7 tắt cuDNN cục bộ khi tính
gradient cho Grad-CAM, giữ mô hình ở chế độ eval.


In [ ]:
# =========================================================
# 10.1 Grad-CAM - fold CỐ ĐỊNH, mô hình đang báo cáo, kết luận SINH TỪ SỐ ĐO
# =========================================================
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0]))
    def __call__(self, x):
        self.model.zero_grad(set_to_none=True)
        # [V7-B1] cuDNN không cho backward của RNN ở chế độ eval -> v6 dừng ở đây với CRNN
        #         ("cudnn RNN backward can only be called in training mode"). Tắt cuDNN cục bộ
        #         giữ nguyên chế độ eval (BatchNorm/Dropout đúng như lúc suy luận).
        with torch.backends.cudnn.flags(enabled=False):
            logit = self.model(x)
            logit.sum().backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = cam - cam.amin(dim=(2, 3), keepdim=True)
        cam = cam / (cam.amax(dim=(2, 3), keepdim=True) + 1e-8)
        return cam.detach(), torch.sigmoid(logit.detach())

def load_best(name, fold=0):
    m = build_model(name)
    sd = torch.load(CKPT_DIR / f"{name}_fold{fold}.pt", map_location="cpu")
    m.load_state_dict({k: v.float() for k, v in sd.items()})
    return m.to(DEVICE).eval()

CAM_FOLD = 0        # [V6-M4] CỐ ĐỊNH, không chọn fold tốt nhất
_cam_order = [BEST_MODEL] + [n for n in cfg.cam_preference] + list(cfg.models)
CAM_MODELS = []
for n in _cam_order:
    if (n in cfg.models and n not in CAM_MODELS
            and getattr(MODEL_ZOO[n], "uses_mel_cam", False)
            and getattr(MODEL_ZOO[n], "cam_usable", False)):
        CAM_MODELS.append(n)
    if len(CAM_MODELS) >= 2:
        break
if not CAM_MODELS:
    print("Không có mô hình nào phù hợp để Grad-CAM -> bỏ qua mục 10.")
else:
    print(f"Mô hình dùng cho Grad-CAM (fold {CAM_FOLD} cố định): {CAM_MODELS}")
    if BEST_MODEL not in CAM_MODELS:
        print(f"  ({BEST_MODEL} không nằm trong danh sách: bản đồ đặc trưng quá thô để CAM "
              f"có ý nghĩa - phải nêu rõ điều này trong bài báo.)")
    mel_f = librosa.mel_frequencies(n_mels=cfg.n_mels, fmin=cfg.f_min, fmax=cfg.f_max)
    CAM_MASS = {}
    for cam_name in CAM_MODELS:
        mdl = load_best(cam_name, CAM_FOLD)
        cam_engine = GradCAM(mdl, mdl.cam_layer())
        va_idx = SPLITS[CAM_FOLD][1]
        pv = OOF[cam_name][va_idx]
        sel_leak = va_idx[np.argsort(np.where(y_all[va_idx] == 1, pv, -1))[-3:]]
        sel_noleak = va_idx[np.argsort(np.where(y_all[va_idx] == 0, -pv, -1))[-3:]]
        sel = np.concatenate([sel_noleak, sel_leak])
        xb = torch.from_numpy(np.asarray(X_ALL[sel])).float().to(DEVICE)
        xb.requires_grad_(False)
        with torch.no_grad():
            spec = mdl.spec(xb).cpu().numpy()[:, 0]
        cam, prob = cam_engine(xb)
        cam = F.interpolate(cam, size=spec.shape[1:], mode="bilinear", align_corners=False)
        cam = cam.cpu().numpy()[:, 0]; prob = prob.cpu().numpy().ravel()

        # [V6-M3] ĐO vùng nóng theo dải Mel để sinh caption từ số liệu
        mass = cam.sum(axis=2)                       # [N, n_mels]
        mass = mass / (mass.sum(axis=1, keepdims=True) + 1e-12)
        is_leak = np.array([1 if y_all[s] == 1 else 0 for s in sel])
        m_leak = mass[is_leak == 1].mean(0) if (is_leak == 1).any() else np.zeros(cfg.n_mels)
        if (not np.isfinite(m_leak).all()) or float(m_leak.sum()) <= 1e-9:
            # [V7] bản đồ toàn 0 (ReLU triệt tiêu hết) -> argmax vô nghĩa, KHÔNG đưa vào kết luận
            print(f"\n[{cam_name}] !! Bản đồ Grad-CAM SUY BIẾN (toàn 0) -> loại khỏi kết luận.")
            del mdl; gc.collect()
            continue
        top = np.argsort(m_leak)[-3:][::-1]
        lo_share = float(m_leak[mel_f <= 800].sum())
        CAM_MASS[cam_name] = dict(top_bands=[float(mel_f[i]) for i in top],
                                  top_shares=[float(m_leak[i]) for i in top],
                                  share_below_800=lo_share,
                                  peak_hz=float(mel_f[int(np.argmax(m_leak))]))
        print(f"\n[{cam_name}] Khối lượng Grad-CAM (frame leak, fold {CAM_FOLD}) tập trung nhất ở: "
              + ", ".join(f"{mel_f[i]:.0f} Hz ({m_leak[i]*100:.0f}%)" for i in top))
        print(f"  Tỉ lệ khối lượng CAM ở dải <=800 Hz: {lo_share*100:.0f}% | "
              f"đỉnh phổ CAM: {CAM_MASS[cam_name]['peak_hz']:.0f} Hz")

        fig, ax = plt.subplots(2, 6, figsize=(17, 5.0))
        for i in range(len(sel)):
            lab = "leak" if y_all[sel[i]] == 1 else "no-leak"
            ax[0][i].imshow(spec[i], aspect="auto", origin="lower", cmap="magma")
            ax[0][i].set_title(f"{lab} | p={prob[i]:.2f}", fontsize=9)
            ax[1][i].imshow(spec[i], aspect="auto", origin="lower", cmap="gray")
            imc = ax[1][i].imshow(cam[i], aspect="auto", origin="lower", cmap="jet",
                                  alpha=.5, vmin=0, vmax=1)
            yt = np.linspace(0, cfg.n_mels-1, 5).astype(int)
            for a in (ax[0][i], ax[1][i]):
                a.set_yticks(yt); a.set_yticklabels([f"{mel_f[k]:.0f}" for k in yt], fontsize=7)
                a.grid(False)
            ax[1][i].set_xlabel("time frame", fontsize=8)
        ax[0][0].set_ylabel("Hz"); ax[1][0].set_ylabel("Hz (Grad-CAM)")
        fig.colorbar(imc, ax=ax[1, :].tolist(), fraction=.02, pad=.01, label="CAM (normalised)")
        fig.suptitle(f"Grad-CAM - {cam_name} (fold {CAM_FOLD})", y=1.02)
        _cap = (f"Grad-CAM of {cam_name}, fold {CAM_FOLD} (fixed, not the best-performing fold): "
                f"top row = input log-Mel, bottom row = overlaid activation map. "
                f"The CAM mass of leak frames concentrates at "
                + ", ".join(f"{mel_f[i]:.0f} Hz ({m_leak[i]*100:.0f}%)" for i in top)
                + f"; {lo_share*100:.0f}% of the mass lies below 800 Hz.")
        if lo_share > 0.5:
            _cap += (" The model therefore concentrates its attention in the low-frequency range.")
        else:
            _cap += (" The model does NOT concentrate its attention in the low-frequency range, contrary "
                     "to the hypothesis usually invoked for plastic pipes; this must be stated "
                     "explicitly in the Discussion.")
        savefig(fig, "gradcam_" + cam_name, _cap)
        del mdl; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    if not CAM_MASS:
        print("!! Không mô hình nào cho bản đồ Grad-CAM hợp lệ -> không rút kết luận từ Grad-CAM.")
    _peak = np.mean([v["peak_hz"] for v in CAM_MASS.values()]) if CAM_MASS else np.nan
    _share = np.mean([v["share_below_800"] for v in CAM_MASS.values()]) if CAM_MASS else np.nan
    print(f"\nKẾT LUẬN Grad-CAM (sinh từ số đo): đỉnh chú ý trung bình {_peak:.0f} Hz | "
          f"{_share*100:.0f}% khối lượng ở dải <= 800 Hz.")
    if _share < 0.5 and BAND_NARRATIVE == "low_band_not_supported":
        print("  -> Hai phân tích ĐỘC LẬP (§3.5 đo trên dữ liệu, §10.1 đo trên mô hình) cùng")
        print("     cho thấy thông tin KHÔNG nằm ở dải rất thấp. Phải viết đúng như vậy.")
    elif _share < 0.5:
        print("  -> Grad-CAM không ủng hộ giả thuyết dải thấp; xem thêm §3.5 và §10.2.")


In [ ]:
# =========================================================
# 10.2 Occlusion theo dải tần - che TRÊN BIỂU DIỄN dB TRƯỚC CHUẨN HOÁ
#      [V6] v5 che sau chuẩn hoá theo từng mẫu nên "che" = gán về giá trị trung bình.
# =========================================================
def band_occlusion(model, idx, n_groups=8, batch=128, model_name=""):
    if not CAM_MODELS:
        return None
    mel_f = librosa.mel_frequencies(n_mels=cfg.n_mels, fmin=cfg.f_min, fmax=cfg.f_max)
    # [V6] Che ĐÚNG chỗ: hook vào chính module biến đổi thời gian-tần số (MelSpectrogram /
    # Spectrogram) để dải bị che biến mất TRƯỚC khi lên dB và trước khi chuẩn hoá.
    front = getattr(model, "f_mel", None) or getattr(model, "front", None)
    if front is None:
        return None
    target = getattr(front, "mel", None) or getattr(front, "spec", None) or front
    edges = np.linspace(0, cfg.n_mels, n_groups + 1).astype(int)
    y = y_all[idx]

    def run(mask_slice):
        h = None
        if mask_slice is not None:
            def hook(m, i, o):
                oo = o.clone()
                lo, hi = mask_slice
                if oo.dim() == 4:        # [B, C, F, T]
                    oo[:, :, lo:hi, :] = 0.0
                elif oo.dim() == 3:      # [B, F, T]
                    oo[:, lo:hi, :] = 0.0
                else:
                    oo[..., lo:hi, :] = 0.0
                return oo
            h = target.register_forward_hook(hook)
        P = []
        with torch.no_grad():
            for s in range(0, len(idx), batch):
                xb = torch.from_numpy(np.asarray(X_ALL[idx[s:s+batch]])).float().to(DEVICE)
                P.append(torch.sigmoid(model(xb).float()).cpu().numpy())
        if h is not None: h.remove()
        return np.concatenate(P)

    base = roc_auc_score(y, run(None))
    rows = []
    for gidx in range(n_groups):
        lo, hi = edges[gidx], edges[gidx+1]
        a = roc_auc_score(y, run((lo, hi)))
        rows.append(dict(group=gidx, mel_lo=lo, mel_hi=hi,
                         f_lo=float(mel_f[lo]), f_hi=float(mel_f[min(hi, cfg.n_mels-1)]),
                         auc_masked=a, auc_drop=base - a))
    return base, pd.DataFrame(rows)

if CAM_MODELS:
    va_idx = SPLITS[CAM_FOLD][1]
    sub_idx = va_idx if len(va_idx) <= 2000 else np.random.default_rng(0).choice(va_idx, 2000, replace=False)
    occ_all = {}
    for cam_name in CAM_MODELS:
        mdl = load_best(cam_name, CAM_FOLD)
        out = band_occlusion(mdl, sub_idx, n_groups=8, model_name=cam_name)
        del mdl; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
        if not out:
            continue
        base_auc, occ = out
        occ_all[cam_name] = (base_auc, occ)
        occ.to_csv(TAB_DIR / f"table9_band_occlusion_{cam_name}.csv", index=False)
        fig, ax = plt.subplots(figsize=(8.5, 3.6))
        lbl = [f"{r.f_lo:.0f}-{r.f_hi:.0f}" for _, r in occ.iterrows()]
        cols = [PAL["leak"] if v > 0 else PAL["grey"] for v in occ.auc_drop]
        ax.bar(range(len(occ)), occ.auc_drop, color=cols, width=.62)
        ax.set_xticks(range(len(occ))); ax.set_xticklabels(lbl, rotation=40, ha="right")
        ax.set_xlabel("occluded band (Hz)"); ax.set_ylabel("ΔAUC")
        ax.axhline(0, color="k", lw=.8)
        _mx = occ.loc[occ.auc_drop.idxmax()]
        _tiny = float(occ.auc_drop.max()) < 0.05
        ax.set_title(f"Causal band importance — {cam_name} "
                     f"(baseline AUC = {base_auc:.3f}, fold {CAM_FOLD})")
        _cap = (f"AUC drop when each group of Mel bands is occluded, applied to the dB representation "
                f"BEFORE normalisation ({cam_name}, fold {CAM_FOLD}). Largest drop: "
                f"{_mx.f_lo:.0f}-{_mx.f_hi:.0f} Hz (ΔAUC = {_mx.auc_drop:.4f}).")
        if base_auc < 0.6:
            # [V7] mô hình gần/kém hơn ngẫu nhiên trên fold này -> ΔAUC khi che không diễn giải được
            _cap += (f" The baseline AUC on this fold ({base_auc:.2f}) is below 0.6, so the occlusion "
                     "profile is NOT interpretable and no band-importance claim is made.")
        elif _tiny:
            _cap += (" The largest drop is below 0.05, so NO single band carries the decision; this "
                     "indicates that the model relies on a DISTRIBUTED cue (spectral slope, "
                     "noise floor) rather than on a narrowband leak signature, and must be "
                     "stated explicitly in the Discussion.")
        savefig(fig, "band_occlusion_" + cam_name, _cap)
        display(Markdown(f"**{cam_name}** — band occlusion (baseline AUC {base_auc:.4f}):"))
        display(occ.round(4))

    if occ_all:
        # [V6] Kiểm tra tính nhất quán giữa occlusion và sức phân biệt đo trên dữ liệu (§3.5)
        n0, (b0, o0) = list(occ_all.items())[0]
        ba = band_auc.copy()
        ba["f_mid"] = (ba.f_lo + ba.f_hi) / 2.0
        mids = ((o0.f_lo + o0.f_hi) / 2.0).values
        info = np.interp(mids, ba.f_mid.values, ba.suc_phan_biet.values)
        r_sp = (float(spstats.spearmanr(o0.auc_drop.values, info).statistic)
                if (len(o0) > 4 and b0 >= 0.6) else np.nan)
        if b0 < 0.6:
            print(f"\n[{n0}] AUC nền của fold {CAM_FOLD} = {b0:.3f} < 0.6 -> không so sánh occlusion với §3.5.")
        print(f"\nTương quan hạng giữa ΔAUC khi che (mô hình) và sức phân biệt của dải "
              f"(dữ liệu, §3.5): rho = {r_sp:.3f}")
        if r_sp == r_sp and r_sp > 0.5:
            print("  -> Hai cách đo ĐỒNG THUẬN: mô hình chú ý đúng vào các dải mà dữ liệu cho là")
            print("     có thông tin. Đây là bằng chứng ủng hộ mô hình học đặc trưng vật lý.")
        elif r_sp == r_sp:
            print("  -> Hai cách đo KHÔNG đồng thuận (rho thấp): mô hình dùng dải khác với dải")
            print("     phân biệt được ở mức năng lượng đơn biến. Phải nêu và giải thích trong bài.")


In [ ]:
# =========================================================
# 10.3 t-SNE trên không gian embedding - [V6-M5] TÔ MÀU THEO BẢN GHI trước tiên
#      Nếu các cụm t-SNE trùng với BẢN GHI/ĐIỂM KHẢO SÁT chứ không trùng với NHÃN,
#      đó là bằng chứng trực quan mạnh nhất của confound.
# =========================================================
if CAM_MODELS:
    tsne_name = CAM_MODELS[0]
    mdl = load_best(tsne_name, CAM_FOLD)
    va_idx = SPLITS[CAM_FOLD][1]
    n_show = min(1500, len(va_idx))
    sel = np.random.default_rng(1).choice(va_idx, n_show, replace=False)
    Z = []
    with torch.no_grad():
        for s in range(0, len(sel), 128):
            xb = torch.from_numpy(np.asarray(X_ALL[sel[s:s+128]])).float().to(DEVICE)
            Z.append(mdl.embed(xb).float().cpu().numpy())
    Z = np.concatenate(Z)
    if n_show > 30:
        emb = TSNE(n_components=2, perplexity=min(30, max(5, n_show // 20)),
                   init="pca", random_state=0, learning_rate="auto").fit_transform(Z)
        rec_sel = frames.record_id.values[sel]
        fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))
        # (a) theo BẢN GHI - hình quan trọng nhất
        urec = pd.unique(rec_sel)
        cmap = plt.get_cmap("tab20")
        for i, r in enumerate(urec):
            m = rec_sel == r
            ax[0].scatter(emb[m, 0], emb[m, 1], s=8, alpha=.7,
                          color=cmap(i % 20), label=(r if len(urec) <= 20 else None))
        ax[0].set_title(f"(a) t-SNE coloured by RECORDING ({len(urec)} recordings)\ntight clusters "
                        f"per recording = site-confound signature")
        if len(urec) <= 20: ax[0].legend(fontsize=6, ncol=2)
        # (b) theo nhãn
        for lab, nm in [(0, "noleak"), (1, "leak")]:
            m = y_all[sel] == lab
            ax[1].scatter(emb[m, 0], emb[m, 1], s=10, alpha=.65, color=PAL[nm], label=nm)
        ax[1].legend(); ax[1].set_title("(b) t-SNE coloured by label")
        # (c) theo xác suất dự đoán
        sc = ax[2].scatter(emb[:, 0], emb[:, 1], s=10, alpha=.75,
                           c=OOF[tsne_name][sel], cmap="coolwarm", vmin=0, vmax=1)
        fig.colorbar(sc, ax=ax[2], label="predicted leak probability")
        ax[2].set_title("(c) t-SNE coloured by predicted probability")
        for a in ax: a.set_xticks([]); a.set_yticks([]); a.grid(False)
        fig.suptitle(f"Learned feature space — {tsne_name} (fold {CAM_FOLD})", y=1.03)
        savefig(fig, "tsne_embedding",
                f"t-SNE projection of the embedding before the classifier ({tsne_name}, fold {CAM_FOLD}). "
                f"Panel (a) is coloured by RECORDING: if points from the same recording form separate "
                f"clusters, then most of the embedding variance is RECORDING/ACQUISITION-SITE specific, "
                f"and any claim of leak identification must be downgraded accordingly.")
        # định lượng mức "cụm theo bản ghi": tỉ lệ hàng xóm gần nhất cùng bản ghi
        from sklearn.neighbors import NearestNeighbors
        # TÊN BIẾN không được là `nn`: sẽ che `torch.nn` và làm hỏng mọi cell phía sau.
        nnbr = NearestNeighbors(n_neighbors=11).fit(emb)
        _, ind = nnbr.kneighbors(emb)
        same_rec = float(np.mean([(rec_sel[ind[i, 1:]] == rec_sel[i]).mean() for i in range(len(emb))]))
        same_lab = float(np.mean([(y_all[sel][ind[i, 1:]] == y_all[sel][i]).mean() for i in range(len(emb))]))
        print(f"\nt-SNE: tỉ lệ 10 hàng xóm gần nhất CÙNG BẢN GHI = {same_rec:.3f} "
              f"| CÙNG NHÃN = {same_lab:.3f}")
        base_rec = float(np.mean([np.mean(rec_sel == r) for r in rec_sel]))
        print(f"  (mức nền ngẫu nhiên cho 'cùng bản ghi' ≈ {base_rec:.3f})")
        if same_rec > 3 * base_rec:
            print("  => Embedding bị CHI PHỐI BỞI BẢN GHI: các frame cùng bản ghi nằm sát nhau")
            print("     hơn hẳn mức ngẫu nhiên. Đây là bằng chứng trực quan của confound và")
            print("     phải được nêu trong Discussion, kèm mục 9.6.")
    del mdl; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()


In [ ]:
# =========================================================
# 10.4 Phân tích lỗi theo bản ghi / Per-recording error analysis
#      [V6-M8] snr_proxy_db -> level_db (đó là MỨC RMS, không phải SNR).
# =========================================================
g = REC[BEST_MODEL].copy()
thr_b = (youden_threshold(REC_INNER[BEST_MODEL].label.values, REC_INNER[BEST_MODEL][best_agg].values)
         if cfg.nested_selection
         else youden_threshold(g.label.values, g[best_agg].values))
g["pred"] = (g[best_agg] >= thr_b).astype(int)
g["correct"] = g.pred == g.label
rec_meta = records.set_index("record_id")[["duration_s", "n_files"]]
g = g.join(rec_meta, on="record_id")
rms_rec = frames.groupby("record_id").rms.mean().rename("mean_rms")
g = g.join(rms_rec, on="record_id")
g["level_db"] = 20*np.log10(g.mean_rms + 1e-9)      # MỨC tín hiệu, KHÔNG phải SNR

err = g[~g.correct].sort_values(best_agg)
display(Markdown(f"### Table 10. Misclassified recordings — {BEST_MODEL} "
                 f"({len(err)}/{len(g)} recordings, threshold {thr_b:.3f})"))
_show = [c for c in ["record_id", "label", best_agg, "n", "n_files", "duration_s", "level_db"] if c in g.columns]
display(err[_show].round(3))
g.round(4).to_csv(TAB_DIR / "table10_record_predictions.csv", index=False)

# [V6] Sai "tự tin" hay sai "sát ngưỡng"? Hai loại này có ý nghĩa rất khác nhau.
marg = (g[best_agg] - thr_b).abs()
print(f"\nKhoảng cách tới ngưỡng: trung vị {marg.median():.3f} | "
      f"số bản ghi sai có |score-thr| > 0.20: {int((marg[~g.correct] > 0.20).sum())}/{len(err)}")
if len(err) and (marg[~g.correct] > 0.20).mean() > 0.5:
    print("=> Phần lớn lỗi là SAI TỰ TIN (không phải sát ngưỡng). Kiểu lỗi này thường nghĩa là")
    print("   đặc trưng quyết định KHÔNG gắn với hiện tượng cần đo (ví dụ đặc trưng của vị trí")
    print("   thu), chứ không phải mô hình thiếu độ nhạy. Xem mục 9.6.")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
ax[0].scatter(g.level_db, g[best_agg], c=[PAL["leak"] if l else PAL["noleak"] for l in g.label],
              s=34, edgecolor=["k" if not c else "none" for c in g.correct], linewidth=.8)
ax[0].axhline(thr_b, color="k", ls="--", lw=1)
ax[0].set_xlabel("recording signal level / mean frame RMS (dB)"); ax[0].set_ylabel(f"{best_agg} leak probability")
ax[0].set_title("(a) Score vs signal level (black edge = misclassified)")

for lab, nm in [(0, "noleak"), (1, "leak")]:
    ax[1].hist(g.loc[g.label == lab, best_agg], bins=12, alpha=.65, color=PAL[nm], label=nm)
ax[1].axvline(thr_b, color="k", ls="--", lw=1)
ax[1].set_xlabel(f"{best_agg} leak probability"); ax[1].set_ylabel("number of recordings")
ax[1].set_title("(b) Class separation at record level"); ax[1].legend()
savefig(fig, "error_analysis",
        "Record-level error analysis: (a) relationship between the signal LEVEL (not SNR) and the "
        "predicted probability, with class separation at the operating threshold; (b) score "
        "distribution of the two classes.")


In [ ]:
# =========================================================
# 10.4a Dựng các KHÔNG GIAN BIỂU DIỄN và đo trong KHÔNG GIAN GỐC
#       Mọi chỉ số ở cell này tính trên PCA-50 của không gian GỐC, KHÔNG trên bản đồ 2D
#       -> kết luận không phụ thuộc perplexity (xem nguyên tắc 3).
# =========================================================
from sklearn.decomposition import PCA as _PCA
from sklearn.neighbors import NearestNeighbors as _NNbr
from sklearn.manifold import trustworthiness as _trustworthiness
from sklearn.metrics import silhouette_score as _silhouette

TSNE_FOLD      = int(CAM_FOLD)          # fold CỐ ĐỊNH (không chọn fold tốt nhất)
TSNE_K         = 10                     # số láng giềng cho mọi chỉ số độ thuần
TSNE_PCA_DIM   = 50                     # giảm chiều trước t-SNE (công bằng giữa các không gian)
TSNE_PERP      = 30
TSNE_N_SHOW    = 200 if cfg.smoke_test else 1200
TSNE_XREC_POOL = 150                    # bể láng giềng để lọc ra láng giềng KHÁC bản ghi
TSNE_SEED      = 11

# --- Tập frame: validation của fold cố định -> embedding là OUT-OF-FOLD, không thiên lệch ---
_va_t = SPLITS[TSNE_FOLD][1]
_rng_t = np.random.default_rng(TSNE_SEED)
TSNE_IDX = np.sort(_rng_t.choice(_va_t, min(TSNE_N_SHOW, len(_va_t)), replace=False))
TSNE_Y   = y_all[TSNE_IDX].astype(int)
TSNE_REC = frames.record_id.values[TSNE_IDX]
TSNE_P   = OOF[DEPLOY_MODEL][TSNE_IDX] if DEPLOY_MODEL in OOF else np.full(len(TSNE_IDX), np.nan)
print(f"Tập so sánh: {len(TSNE_IDX)} frame | {len(np.unique(TSNE_REC))} bản ghi | "
      f"{int(TSNE_Y.sum())} leak / {int((1 - TSNE_Y).sum())} noleak | fold {TSNE_FOLD} (validation)")


# ---------------------------------------------------------------------
# Tiện ích đo trong không gian gốc
# ---------------------------------------------------------------------
def _standardise(X):
    X = np.nan_to_num(np.asarray(X, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    return (X - X.mean(0)) / (X.std(0) + 1e-9)

def pca_reduce(X, d=None, seed=0):
    X = _standardise(X)
    d = int(min(d or TSNE_PCA_DIM, X.shape[1], X.shape[0] - 1))
    if d >= X.shape[1]:
        return X
    return _PCA(n_components=d, random_state=seed).fit_transform(X)

def _knn_index(X, k):
    k = int(min(k + 1, len(X)))
    _, ind = _NNbr(n_neighbors=k).fit(X).kneighbors(X)
    return ind[:, 1:]                      # bỏ chính điểm đó

def knn_purity(X, labels, k=TSNE_K):
    labels = np.asarray(labels)
    ind = _knn_index(X, k)
    return float(np.mean([(labels[ind[i]] == labels[i]).mean() for i in range(len(X))]))

def chance_purity(labels):
    # Xác suất một điểm KHÁC lấy ngẫu nhiên cùng nhãn -> mức nền đúng của độ thuần kNN.
    labels = np.asarray(labels); N = len(labels)
    _, cnt = np.unique(labels, return_counts=True)
    return float(np.sum(cnt * (cnt - 1)) / max(N * (N - 1), 1))

def norm_purity(p, chance):
    # 0 = đúng mức ngẫu nhiên, 1 = hoàn hảo. So sánh được giữa các không gian có số lớp khác nhau.
    if not np.isfinite(p) or not np.isfinite(chance) or chance >= 1:
        return np.nan
    return float((p - chance) / (1 - chance))

def knn_purity_crossgroup(X, labels, groups, k=TSNE_K, pool=TSNE_XREC_POOL):
    # CHỈ SỐ QUYẾT ĐỊNH: độ thuần nhãn chỉ tính trên láng giềng thuộc BẢN GHI KHÁC.
    labels = np.asarray(labels); groups = np.asarray(groups)
    ind = _knn_index(X, pool)
    vals = []
    for i in range(len(X)):
        nb = ind[i][groups[ind[i]] != groups[i]][:k]
        if len(nb) >= max(3, k // 3):
            vals.append(float((labels[nb] == labels[i]).mean()))
    return (float(np.mean(vals)) if vals else np.nan), int(len(vals))

def _safe_sil(X, labels):
    labels = np.asarray(labels)
    if len(np.unique(labels)) < 2 or len(np.unique(labels)) >= len(X):
        return np.nan
    try:
        return float(_silhouette(X, labels))
    except Exception:
        return np.nan


# ---------------------------------------------------------------------
# Trích các không gian biểu diễn
# ---------------------------------------------------------------------
@torch.no_grad()
def embed_frames(model, idx, batch=128):
    # fp32 (TẮT AMP) để hình học không bị lượng tử hoá bởi fp16.
    Z = []
    for s in range(0, len(idx), batch):
        xb = torch.from_numpy(np.asarray(X_ALL[idx[s:s + batch]])).float().to(DEVICE)
        with amp_autocast(DEVICE, False):
            z = model.embed(xb)
        Z.append(torch.nan_to_num(z.float()).cpu().numpy())
    return np.concatenate(Z)

@torch.no_grad()
def logmel_matrix(idx, batch=128):
    fe = LogMelFrontend().to(DEVICE).eval()      # specaug=None -> KHÔNG che gì
    out = []
    for s in range(0, len(idx), batch):
        xb = torch.from_numpy(np.asarray(X_ALL[idx[s:s + batch]])).float().to(DEVICE)
        with amp_autocast(DEVICE, False):
            m = fe(xb)
        out.append(torch.nan_to_num(m.float()).flatten(1).cpu().numpy())
    del fe; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return np.concatenate(out)

REP_RAW, REP_KIND = {}, {}

REP_RAW["log-Mel (input)"] = logmel_matrix(TSNE_IDX)
REP_KIND["log-Mel (input)"] = "untrained"
print(f"  log-Mel (input)              : {REP_RAW['log-Mel (input)'].shape[1]} chiều")

if "FEAT_ALL" in globals() and FEAT_ALL is not None and len(FEAT_ALL) == len(frames):
    REP_RAW["Spectral features"] = FEAT_ALL.values[TSNE_IDX]
    REP_KIND["Spectral features"] = "untrained"
    print(f"  Spectral features (§9.5)     : {REP_RAW['Spectral features'].shape[1]} chiều")
else:
    print("  [bỏ qua] đặc trưng phổ cổ điển - cần cfg.run_baseline_models=True (mục 9.5)")

for _name in cfg.models:
    _ck = CKPT_DIR / f"{_name}_fold{TSNE_FOLD}.pt"
    if not _ck.exists():
        print(f"  [bỏ qua] {_name}: thiếu checkpoint fold {TSNE_FOLD}")
        continue
    _m = load_best(_name, TSNE_FOLD)
    REP_RAW[_name] = embed_frames(_m, TSNE_IDX)
    REP_KIND[_name] = "learned"
    print(f"  {_name:28s}: {REP_RAW[_name].shape[1]} chiều (embedding)")
    del _m; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

assert len(REP_RAW) >= 2, "Cần ít nhất hai không gian biểu diễn để so sánh."

# PCA-50 -> không gian dùng cho MỌI chỉ số và làm đầu vào t-SNE
REP_Z = {k: pca_reduce(v) for k, v in REP_RAW.items()}


# ---------------------------------------------------------------------
# Bảng chỉ số (đo trong KHÔNG GIAN GỐC)
# ---------------------------------------------------------------------
_lab_ch = chance_purity(TSNE_Y)
_rec_ch = chance_purity(TSNE_REC)
print(f"\nMức nền ngẫu nhiên: độ thuần NHÃN {_lab_ch:.4f} | độ thuần BẢN GHI {_rec_ch:.4f}")

_rows = []
for k, Z in REP_Z.items():
    lab_p = knn_purity(Z, TSNE_Y)
    rec_p = knn_purity(Z, TSNE_REC)
    xrec_p, n_used = knn_purity_crossgroup(Z, TSNE_Y, TSNE_REC)
    _rows.append(dict(
        representation=k, kind=REP_KIND[k],
        dim_raw=int(REP_RAW[k].shape[1]), dim_used=int(Z.shape[1]),
        label_purity=lab_p, label_purity_norm=norm_purity(lab_p, _lab_ch),
        record_purity=rec_p, record_purity_norm=norm_purity(rec_p, _rec_ch),
        label_purity_xrecord=xrec_p,
        label_purity_xrecord_norm=norm_purity(xrec_p, _lab_ch),
        n_frames_xrecord=n_used,
        silhouette_label=_safe_sil(Z, TSNE_Y),
        silhouette_record=_safe_sil(Z, TSNE_REC)))
TSNE_METRICS = pd.DataFrame(_rows)

display(Markdown(f"### Table 14b. Neighbourhood structure of each representation "
                 f"(measured in the ORIGINAL PCA-{TSNE_PCA_DIM} space, k={TSNE_K}, "
                 f"n={len(TSNE_IDX)} out-of-fold frames)"))
display(TSNE_METRICS.round(4))
TSNE_METRICS.round(4).to_csv(TAB_DIR / "table14b_representation_structure.csv", index=False)
print("Chuẩn hoá: 0 = đúng mức ngẫu nhiên, 1 = hoàn hảo. "
      "'xrecord' = chỉ xét láng giềng thuộc BẢN GHI KHÁC.")

In [ ]:
# =========================================================
# 10.4b Lưới t-SNE so sánh: hàng = không gian biểu diễn, cột = cách tô màu
#       Kèm trustworthiness để biết bản đồ 2D đáng tin đến đâu.
# =========================================================
def run_tsne(X, perplexity=TSNE_PERP, seed=0):
    p = float(np.clip(perplexity, 5.0, max(5.0, (len(X) - 1) / 3.0)))
    # Không truyền n_iter/max_iter: tên tham số này đổi giữa các phiên bản scikit-learn.
    return TSNE(n_components=2, perplexity=p, init="pca",
                learning_rate="auto", random_state=seed).fit_transform(X)

REP_EMB, _trust = {}, {}
for k, Z in REP_Z.items():
    REP_EMB[k] = run_tsne(Z)
    try:
        _trust[k] = float(_trustworthiness(Z, REP_EMB[k], n_neighbors=TSNE_K))
    except Exception:
        _trust[k] = np.nan
    print(f"  t-SNE {k:28s} trustworthiness {_trust[k]:.3f}")
TSNE_METRICS["trustworthiness_2d"] = TSNE_METRICS.representation.map(_trust)
TSNE_METRICS.round(4).to_csv(TAB_DIR / "table14b_representation_structure.csv", index=False)

_names = list(REP_EMB)
_nrow = len(_names)
_rec_u = pd.unique(TSNE_REC)
_cmap20 = plt.get_cmap("tab20")
_rec_col = {r: _cmap20(i % 20) for i, r in enumerate(_rec_u)}

fig, axes = plt.subplots(_nrow, 3, figsize=(11.5, 2.85 * _nrow), squeeze=False)
for i, k in enumerate(_names):
    E = REP_EMB[k]
    tag = "UNTRAINED" if REP_KIND[k] == "untrained" else "learned"

    # (a) tô theo NHÃN
    a = axes[i][0]
    for lab, nm in [(0, "noleak"), (1, "leak")]:
        m = TSNE_Y == lab
        a.scatter(E[m, 0], E[m, 1], s=7, alpha=.65, color=PAL[nm], label=nm, linewidths=0)
    if i == 0:
        a.legend(fontsize=7, markerscale=2, loc="best")
    a.set_ylabel(f"{k}\n[{tag}]", fontsize=8)
    a.set_title("coloured by LABEL" if i == 0 else "", fontsize=9)

    # (b) tô theo BẢN GHI  <- hình quan trọng nhất
    a = axes[i][1]
    a.scatter(E[:, 0], E[:, 1], s=7, alpha=.7, linewidths=0,
              color=[_rec_col[r] for r in TSNE_REC])
    a.set_title(f"coloured by RECORDING ({len(_rec_u)})" if i == 0 else "", fontsize=9)

    # (c) tô theo xác suất của mô hình triển khai
    a = axes[i][2]
    sc = a.scatter(E[:, 0], E[:, 1], s=7, alpha=.75, linewidths=0,
                   c=TSNE_P, cmap="coolwarm", vmin=0, vmax=1)
    a.set_title(f"coloured by p({DEPLOY_MODEL})" if i == 0 else "", fontsize=9)

    for a in axes[i]:
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
    axes[i][2].text(.98, .02, f"trust={_trust[k]:.2f}", transform=axes[i][2].transAxes,
                    ha="right", va="bottom", fontsize=7, color=PAL["grey"])
fig.colorbar(sc, ax=axes[:, 2].tolist(), fraction=.02, pad=.01, label="leak probability")
fig.suptitle("Comparative t-SNE of frame representations "
             f"(fold {TSNE_FOLD} validation frames, out-of-fold)", y=1.005, fontsize=11)
savefig(fig, "tsne_comparative_grid",
        "Comparative t-SNE across representation spaces. Rows: two UNTRAINED reference spaces "
        "(input log-Mel, classical spectral features) followed by the learned embedding of each "
        "architecture; columns: the same projection coloured by label, by recording, and by the "
        "deployment model's predicted probability. Cluster distances and cluster sizes in a t-SNE "
        "map are not interpretable - all quantitative statements come from Table 14b, which is "
        f"computed in the original PCA-{TSNE_PCA_DIM} space. `trust` is the trustworthiness of "
        "each 2-D projection.")

In [ ]:
# =========================================================
# 10.4c t-SNE MỨC BẢN GHI - mỗi bản ghi một điểm, đơn vị thống kê đúng
#       CẢNH BÁO PHƯƠNG PHÁP: embedding của các fold KHÁC NHAU nằm ở các không gian
#       KHÁC NHAU nên KHÔNG thể gộp chung vào một phép chiếu. Vì vậy dùng MỘT mô hình
#       (fold cố định) áp lên TOÀN BỘ bản ghi, và ĐÁNH DẤU bản ghi nào thuộc validation
#       của fold đó. Nếu bản ghi train và validation phân bố giống nhau thì hình học này
#       KHÔNG phải do ghi nhớ tập huấn luyện.
# =========================================================
TSNE_REC_FRAMES = 12 if cfg.smoke_test else 40      # số frame lấy mẫu mỗi bản ghi

_rng_r = np.random.default_rng(TSNE_SEED + 1)
_pick = {}
for _rid, _fidx in frames.groupby("record_id").indices.items():
    _fidx = np.asarray(_fidx)
    _pick[_rid] = (_fidx if len(_fidx) <= TSNE_REC_FRAMES
                   else np.sort(_rng_r.choice(_fidx, TSNE_REC_FRAMES, replace=False)))
_rid_all = np.array(sorted(_pick))
_flat = np.concatenate([_pick[r] for r in _rid_all])
_owner = np.concatenate([np.full(len(_pick[r]), i) for i, r in enumerate(_rid_all)])
_y_rec = np.array([int(rec_lab[r]) for r in _rid_all])
_val_rec = set(pd.unique(groups_all[SPLITS[TSNE_FOLD][1]]).tolist())
_is_val = np.array([r in _val_rec for r in _rid_all])
print(f"Mức bản ghi: {len(_rid_all)} bản ghi x <= {TSNE_REC_FRAMES} frame = {len(_flat)} frame "
      f"| {int(_is_val.sum())} bản ghi thuộc validation của fold {TSNE_FOLD}")

_model_names = [k for k in REP_Z if REP_KIND[k] == "learned"]
REC_EMB, _rec_rows = {}, []
for k in _model_names:
    _m = load_best(k, TSNE_FOLD)
    Zf = embed_frames(_m, _flat)
    del _m; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    # gộp TRUNG BÌNH embedding trong từng bản ghi -> một điểm / bản ghi
    Zr = np.stack([Zf[_owner == i].mean(0) for i in range(len(_rid_all))])
    Zr = pca_reduce(Zr, d=min(30, len(_rid_all) - 2))
    REC_EMB[k] = run_tsne(Zr, perplexity=min(15, max(5, (len(_rid_all) - 1) // 3)))
    _p_lab = knn_purity(Zr, _y_rec, k=min(5, len(_rid_all) - 1))
    _ch = chance_purity(_y_rec)
    _rec_rows.append(dict(model=k, n_records=int(len(_rid_all)),
                          record_label_purity_k5=_p_lab,
                          record_label_purity_norm=norm_purity(_p_lab, _ch),
                          silhouette_label=_safe_sil(Zr, _y_rec)))
    print(f"  {k:28s} độ thuần nhãn mức bản ghi (k=5) {_p_lab:.4f} "
          f"(nền {_ch:.4f}, chuẩn hoá {norm_purity(_p_lab, _ch):+.4f})")

TSNE_REC_METRICS = pd.DataFrame(_rec_rows)
display(Markdown("### Table 14c. Record-level neighbourhood structure "
                 "(one point per recording - the correct unit of analysis)"))
display(TSNE_REC_METRICS.round(4))
TSNE_REC_METRICS.round(4).to_csv(TAB_DIR / "table14c_record_level_structure.csv", index=False)

_nc = min(3, max(1, len(REC_EMB)))
_nr = int(np.ceil(len(REC_EMB) / _nc))
fig, axes = plt.subplots(_nr, _nc, figsize=(4.2 * _nc, 4.0 * _nr), squeeze=False)
for j, k in enumerate(REC_EMB):
    a = axes[j // _nc][j % _nc]
    E = REC_EMB[k]
    # dự đoán mức bản ghi của chính mô hình đó -> đánh dấu bản ghi bị phân loại sai
    _g = REC[k].set_index("record_id")
    _thr = (youden_threshold(REC_INNER[k].label.values, REC_INNER[k][best_agg].values)
            if cfg.nested_selection else
            youden_threshold(_g.label.values, _g[best_agg].values))
    _sc = np.array([float(_g[best_agg].get(r, np.nan)) for r in _rid_all])
    _wrong = np.isfinite(_sc) & ((_sc >= _thr).astype(int) != _y_rec)
    for lab, nm in [(0, "noleak"), (1, "leak")]:
        m = _y_rec == lab
        a.scatter(E[m, 0], E[m, 1], s=46, alpha=.85, color=PAL[nm], label=nm,
                  edgecolor="none")
    a.scatter(E[_is_val, 0], E[_is_val, 1], s=110, facecolors="none",
              edgecolors="k", linewidths=.9, label=f"in fold-{TSNE_FOLD} validation")
    if _wrong.any():
        a.scatter(E[_wrong, 0], E[_wrong, 1], s=150, marker="x", color="k",
                  linewidths=1.3, label="misclassified")
    a.set_title(f"{k}\nrecord-level label purity (k=5) = "
                f"{float(TSNE_REC_METRICS.set_index('model').record_label_purity_k5[k]):.3f}",
                fontsize=9)
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
    if j == 0:
        a.legend(fontsize=7, markerscale=.8, loc="best")
for j in range(len(REC_EMB), _nr * _nc):
    axes[j // _nc][j % _nc].axis("off")
fig.suptitle("Record-level t-SNE: one point per recording (mean-pooled embedding), "
             f"single model from fold {TSNE_FOLD} applied to all recordings", y=1.01, fontsize=11)
savefig(fig, "tsne_record_level",
        "Record-level t-SNE - one point per recording, embeddings mean-pooled within a recording, "
        f"using the fold-{TSNE_FOLD} model applied to ALL recordings. Rings mark the recordings "
        f"that were in that fold's validation set (unseen); crosses mark recordings the model "
        "misclassifies at its operating threshold. Because embeddings from different folds live "
        "in different spaces, they cannot be pooled into one projection - hence a single fold "
        "model with the seen/unseen distinction made explicit.")

In [ ]:
# =========================================================
# 10.4d Tương đồng giữa các không gian + quét perplexity + KẾT LUẬN TỰ ĐỘNG
# =========================================================
# ---- (1) Độ trùng lặp k-láng-giềng giữa mọi cặp không gian (đo trong không gian GỐC) ----
def knn_sets(X, k=TSNE_K):
    return _knn_index(X, k)

_sets = {k: knn_sets(Z) for k, Z in REP_Z.items()}
_keys = list(_sets)
_OV = np.eye(len(_keys))
for i in range(len(_keys)):
    for j in range(i + 1, len(_keys)):
        A, B = _sets[_keys[i]], _sets[_keys[j]]
        v = float(np.mean([len(set(A[t]) & set(B[t])) / A.shape[1] for t in range(len(A))]))
        _OV[i, j] = _OV[j, i] = v
_ov_chance = TSNE_K / max(len(TSNE_IDX) - 1, 1)

# ---- (2) Tương quan hạng của MA TRẬN KHOẢNG CÁCH (cấu trúc toàn cục, không chỉ cục bộ) ----
from scipy.spatial.distance import pdist as _pdist
_sub = np.sort(np.random.default_rng(3).choice(len(TSNE_IDX), min(400, len(TSNE_IDX)),
                                               replace=False))
_pdv = {k: _pdist(REP_Z[k][_sub]) for k in _keys}
_RS = np.eye(len(_keys))
for i in range(len(_keys)):
    for j in range(i + 1, len(_keys)):
        r = float(spstats.spearmanr(_pdv[_keys[i]], _pdv[_keys[j]])[0])
        _RS[i, j] = _RS[j, i] = r

_lbl = [k if len(k) <= 13 else k[:12] + "." for k in _keys]
fig, ax = plt.subplots(1, 3, figsize=(17.5, 4.6),
                       gridspec_kw=dict(width_ratios=[1, 1, 1.05], wspace=0.55))
for a, M, ttl, vlo, cm in [
        (ax[0], _OV, f"(a) shared {TSNE_K}-NN fraction\n(chance = {_ov_chance:.3f})", 0, "viridis"),
        (ax[1], _RS, "(b) Spearman correlation of\npairwise distances", -1, "coolwarm")]:
    im = a.imshow(M, cmap=cm, vmin=vlo, vmax=1)
    for i in range(len(_keys)):
        for j in range(len(_keys)):
            a.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=6.5,
                   color="w" if (cm == "viridis" and M[i, j] < .55) else "k")
    a.set_xticks(range(len(_keys))); a.set_xticklabels(_lbl, rotation=45, ha="right", fontsize=7)
    a.set_yticks(range(len(_keys))); a.set_yticklabels(_lbl, fontsize=7)
    a.set_title(ttl, fontsize=9); a.grid(False)
    fig.colorbar(im, ax=a, fraction=.046, pad=.02)

# ---- (3) Quét perplexity: kết luận có phụ thuộc siêu tham số không? ----
_perps = [5, 15, 30, 50]
_pr = []
for p in _perps:
    E = run_tsne(REP_Z[DEPLOY_MODEL], perplexity=p)
    _pr.append(dict(perplexity=p,
                    label_purity_2d=knn_purity(E, TSNE_Y),
                    record_purity_2d=knn_purity(E, TSNE_REC),
                    trustworthiness=float(_trustworthiness(REP_Z[DEPLOY_MODEL], E,
                                                           n_neighbors=TSNE_K))))
TSNE_PERP_SCAN = pd.DataFrame(_pr)
a = ax[2]
a.plot(TSNE_PERP_SCAN.perplexity, TSNE_PERP_SCAN.record_purity_2d, marker="o",
       color=PAL["warn"], label="record purity (2-D map)")
a.plot(TSNE_PERP_SCAN.perplexity, TSNE_PERP_SCAN.label_purity_2d, marker="s",
       color=PAL["leak"], label="label purity (2-D map)")
a.plot(TSNE_PERP_SCAN.perplexity, TSNE_PERP_SCAN.trustworthiness, marker="^",
       color=PAL["noleak"], label="trustworthiness")
_rp = float(TSNE_METRICS.set_index("representation").record_purity[DEPLOY_MODEL])
_lp = float(TSNE_METRICS.set_index("representation").label_purity[DEPLOY_MODEL])
a.axhline(_rp, color=PAL["warn"], ls=":", lw=1)
a.axhline(_lp, color=PAL["leak"], ls=":", lw=1)
a.set_xlabel("t-SNE perplexity"); a.set_ylabel("value"); a.set_ylim(0, 1.05)
a.legend(fontsize=7); a.grid(True, alpha=.3)
a.set_title(f"(c) Perplexity sensitivity - {DEPLOY_MODEL}\n"
            "(dotted = value in the original space)", fontsize=9)
savefig(fig, "tsne_representation_similarity",
        "Do the architectures learn the same representation, and do the conclusions depend on "
        f"t-SNE settings? (a) fraction of shared {TSNE_K} nearest neighbours between every pair of "
        "representation spaces, measured in the original space; (b) Spearman correlation between "
        "their pairwise-distance matrices; (c) neighbourhood purity and trustworthiness of the "
        "2-D map across perplexity values, with the original-space values as dotted references.")
TSNE_PERP_SCAN.round(4).to_csv(TAB_DIR / "table14d_perplexity_scan.csv", index=False)
display(Markdown("### Table 14d. Perplexity sensitivity"))
display(TSNE_PERP_SCAN.round(4))

_ovdf = pd.DataFrame(_OV, index=_keys, columns=_keys).round(3)
_ovdf.to_csv(TAB_DIR / "table14e_knn_overlap.csv")
pd.DataFrame(_RS, index=_keys, columns=_keys).round(3).to_csv(TAB_DIR / "table14f_distance_rank_corr.csv")

In [ ]:
# =========================================================
# 10.4e KẾT LUẬN - sinh TỪ SỐ ĐO, không viết trước
# =========================================================
_M = TSNE_METRICS.set_index("representation")
_un = [k for k in _M.index if REP_KIND[k] == "untrained"]
_le = [k for k in _M.index if REP_KIND[k] == "learned"]

def _mean(keys, col):
    v = _M.loc[keys, col].astype(float).values if keys else np.array([])
    v = v[np.isfinite(v)]
    return float(v.mean()) if len(v) else np.nan

rec_un, rec_le = _mean(_un, "record_purity_norm"), _mean(_le, "record_purity_norm")
lab_un, lab_le = _mean(_un, "label_purity_norm"), _mean(_le, "label_purity_norm")
xr_un, xr_le = _mean(_un, "label_purity_xrecord_norm"), _mean(_le, "label_purity_xrecord_norm")
xr_dep = float(_M.label_purity_xrecord_norm.get(DEPLOY_MODEL, np.nan))
rec_dep = float(_M.record_purity_norm.get(DEPLOY_MODEL, np.nan))
lab_dep = float(_M.label_purity_norm.get(DEPLOY_MODEL, np.nan))

_ov_models = [ _OV[_keys.index(a), _keys.index(b)]
               for i, a in enumerate(_le) for b in _le[i + 1:] ]
ov_models = float(np.mean(_ov_models)) if _ov_models else np.nan
_ov_ul = [ _OV[_keys.index(a), _keys.index(b)] for a in _un for b in _le ]
ov_un_le = float(np.mean(_ov_ul)) if _ov_ul else np.nan

print("=" * 82)
print("KẾT LUẬN MỤC 10.4 (mọi con số đo trong KHÔNG GIAN GỐC, không phải trên hình t-SNE)")
print("=" * 82)
print(f"Độ thuần chuẩn hoá (0 = ngẫu nhiên, 1 = hoàn hảo), trung bình theo nhóm:")
print(f"  theo BẢN GHI : chưa huấn luyện {rec_un:+.3f} | đã huấn luyện {rec_le:+.3f}")
print(f"  theo NHÃN    : chưa huấn luyện {lab_un:+.3f} | đã huấn luyện {lab_le:+.3f}")
print(f"  theo NHÃN, XUYÊN BẢN GHI: chưa huấn luyện {xr_un:+.3f} | đã huấn luyện {xr_le:+.3f}")
print(f"  mô hình triển khai ({DEPLOY_MODEL}): bản ghi {rec_dep:+.3f} | nhãn {lab_dep:+.3f} "
      f"| nhãn xuyên bản ghi {xr_dep:+.3f}")

TSNE_VERDICT = {}

# --- (1) Cụm theo bản ghi: do mô hình hay do dữ liệu? ---
print("\n[1] CỤM THEO BẢN GHI ĐẾN TỪ ĐÂU?")
if np.isfinite(rec_un) and rec_un >= 0.5:
    if np.isfinite(rec_le) and rec_le <= rec_un + 0.10:
        TSNE_VERDICT["record_clustering_origin"] = "data"
        print(f"    Ngay cả không gian CHƯA HUẤN LUYỆN đã cụm theo bản ghi ({rec_un:+.3f}),")
        print(f"    và mô hình không làm mạnh thêm đáng kể ({rec_le:+.3f}).")
        print("    => Cấu trúc theo bản ghi là THUỘC TÍNH CỦA DỮ LIỆU, không phải mô hình ghi nhớ.")
        print("       Cách viết đúng trong bài báo: các frame chồng lấn trong cùng một lần thu")
        print("       gần như trùng nhau; ĐÂY KHÔNG PHẢI bằng chứng mô hình ghi nhớ, NHƯNG vẫn")
        print("       là lý do khiến frame không thể dùng làm đơn vị thống kê (xem ICC ở 8.2).")
    else:
        TSNE_VERDICT["record_clustering_origin"] = "data+amplified"
        print(f"    Dữ liệu đã cụm theo bản ghi ({rec_un:+.3f}) và huấn luyện LÀM MẠNH THÊM "
              f"({rec_le:+.3f}).")
        print("    => Mô hình khuếch đại đặc trưng riêng của từng bản ghi. Phải nêu rõ.")
elif np.isfinite(rec_le) and rec_le >= 0.5:
    TSNE_VERDICT["record_clustering_origin"] = "model"
    print(f"    Không gian chưa huấn luyện KHÔNG cụm theo bản ghi ({rec_un:+.3f}) nhưng không")
    print(f"    gian đã huấn luyện thì có ({rec_le:+.3f}).")
    print("    => Chính MÔ HÌNH tạo ra cấu trúc theo bản ghi: dấu hiệu ghi nhớ bản ghi /")
    print("       học 'dấu vân tay' điểm khảo sát. Đây là phát hiện phải đưa vào Discussion.")
else:
    TSNE_VERDICT["record_clustering_origin"] = "weak"
    print("    Không thấy cụm theo bản ghi mạnh ở cả hai nhóm.")

# --- (2) Chỉ số quyết định: nhãn có chuyển giao được giữa các bản ghi? ---
print("\n[2] ĐẶC TRƯNG CÓ CHUYỂN GIAO ĐƯỢC GIỮA CÁC BẢN GHI? (chỉ số quyết định)")
if not np.isfinite(xr_dep):
    TSNE_VERDICT["cross_record_transfer"] = "unknown"
    print("    Không đủ láng giềng khác bản ghi để tính - tăng TSNE_N_SHOW rồi chạy lại.")
elif xr_dep >= 0.40:
    TSNE_VERDICT["cross_record_transfer"] = "strong"
    print(f"    Độ thuần nhãn xuyên bản ghi (chuẩn hoá) = {xr_dep:+.3f} -> MẠNH.")
    print("    => Biểu diễn đặt frame của CÁC BẢN GHI KHÁC NHAU, CÙNG NHÃN gần nhau.")
    print("       Đây là bằng chứng ủng hộ mô hình học đặc trưng rò rỉ chuyển giao được,")
    print("       và là lập luận ĐỐI TRỌNG mạnh nhất với lo ngại confound ở mục 9.6.")
elif xr_dep >= 0.15:
    TSNE_VERDICT["cross_record_transfer"] = "moderate"
    print(f"    Độ thuần nhãn xuyên bản ghi (chuẩn hoá) = {xr_dep:+.3f} -> TRUNG BÌNH.")
    print("    => Có thành phần chuyển giao được, nhưng yếu hơn cấu trúc theo bản ghi.")
    print("       Phát biểu đúng mức: mô hình học được một phần đặc trưng chung, phần còn lại")
    print("       gắn với từng lần thu.")
else:
    TSNE_VERDICT["cross_record_transfer"] = "weak"
    print(f"    Độ thuần nhãn xuyên bản ghi (chuẩn hoá) = {xr_dep:+.3f} -> GẦN MỨC NGẪU NHIÊN.")
    print("    => CẢNH BÁO NGHIÊM TRỌNG: biểu diễn gom frame theo BẢN GHI chứ không theo")
    print("       đặc trưng rò rỉ chung. Khả năng phân biệt ở Bảng 6/7 có thể chỉ là hệ quả của")
    print("       việc nhận ra bản ghi. Kết luận của bài báo PHẢI hạ xuống mức 'phân biệt hai")
    print("       nhóm bản ghi', và leave-one-site-out (mục 9.6) trở thành thí nghiệm bắt buộc.")

# --- (3) Huấn luyện có thêm được gì cho tác vụ phân lớp? ---
print("\n[3] HUẤN LUYỆN THÊM ĐƯỢC GÌ SO VỚI KHÔNG GIAN CHƯA HUẤN LUYỆN?")
_d_lab = lab_le - lab_un if (np.isfinite(lab_le) and np.isfinite(lab_un)) else np.nan
_d_xr = xr_le - xr_un if (np.isfinite(xr_le) and np.isfinite(xr_un)) else np.nan
print(f"    Δ độ thuần nhãn            = {_d_lab:+.3f}")
print(f"    Δ độ thuần nhãn xuyên bản ghi = {_d_xr:+.3f}  <- đây là phần THỰC SỰ có giá trị")
TSNE_VERDICT["learning_gain_label"] = float(_d_lab) if np.isfinite(_d_lab) else None
TSNE_VERDICT["learning_gain_xrecord"] = float(_d_xr) if np.isfinite(_d_xr) else None
if np.isfinite(_d_xr) and _d_xr <= 0.05:
    print("    => Huấn luyện KHÔNG cải thiện đáng kể cấu trúc chuyển giao được giữa các bản ghi.")
    print("       Kết hợp với mục 9.5 (so sánh GHÉP CẶP CNN vs baseline cổ điển), đây là bằng chứng")
    print("       thứ hai, độc lập, cho thấy lợi thế của học sâu trên tập dữ liệu này chưa")
    print("       được chứng minh.")
elif np.isfinite(_d_xr):
    print("    => Huấn luyện có cải thiện cấu trúc chuyển giao được. Nên trích dẫn con số này")
    print("       cùng với Bảng 9c như hai bằng chứng độc lập.")

# --- (4) Các kiến trúc có học ra cùng một biểu diễn? ---
print(f"\n[4] {len(_le)} KIẾN TRÚC CÓ HỌC RA CÙNG MỘT BIỂU DIỄN?")
print(f"    Trùng lặp {TSNE_K}-láng-giềng: giữa các mô hình {ov_models:.3f} | "
      f"mô hình vs chưa huấn luyện {ov_un_le:.3f} | mức ngẫu nhiên {_ov_chance:.3f}")
TSNE_VERDICT["knn_overlap_between_models"] = ov_models
if np.isfinite(ov_models) and ov_models >= 0.35:
    TSNE_VERDICT["architectures_converge"] = True
    print("    => Các kiến trúc HỘI TỤ về cùng một cấu trúc láng giềng, dù khác nhau hoàn toàn")
    print("       về thiết kế và số tham số (0,66M - 11,3M). Điều này KHỚP với Bảng 8: AUC của")
    print("       chúng không phân biệt được. Diễn giải hợp lý: hiệu năng bị chặn bởi DỮ LIỆU")
    print(f"       ({len(records)} bản ghi, ICC cao), không bởi dung lượng mô hình -> thêm tham số hay đổi")
    print("       kiến trúc sẽ không giúp; thêm ĐIỂM KHẢO SÁT mới giúp.")
else:
    TSNE_VERDICT["architectures_converge"] = False
    print("    => Các kiến trúc học ra cấu trúc KHÁC nhau dù AUC tương đương. Khi đó ensemble")
    print("       nhiều kiến trúc có triển vọng thật sự (đa dạng hoá sai số).")

# --- (5) Độ tin cậy của phép chiếu ---
print("\n[5] HÌNH t-SNE CÓ ĐÁNG TIN?")
_tr = TSNE_METRICS.trustworthiness_2d.astype(float)
print(f"    trustworthiness: {_tr.min():.3f} - {_tr.max():.3f} (trung bình {_tr.mean():.3f})")
_spread = float(TSNE_PERP_SCAN.record_purity_2d.max() - TSNE_PERP_SCAN.record_purity_2d.min())
print(f"    biến thiên độ thuần bản ghi trên bản đồ 2D khi perplexity đổi 5->50: {_spread:.3f}")
TSNE_VERDICT["trustworthiness_mean"] = float(_tr.mean())
TSNE_VERDICT["perplexity_spread_record_purity"] = _spread
if _spread <= 0.10:
    print("    => Kết luận KHÔNG phụ thuộc perplexity.")
else:
    print("    => Bản đồ 2D nhạy với perplexity -> CHỈ trích dẫn số từ không gian gốc (Bảng 14b).")

print("\n" + "=" * 82)
print("CÂU NÊN VIẾT TRONG BÀI BÁO (ghép từ 5 kết luận trên):")
_origin = {"data": "một thuộc tính của dữ liệu chứ không phải do mô hình ghi nhớ",
           "data+amplified": "vốn có trong dữ liệu và được mô hình khuếch đại",
           "model": "do chính mô hình tạo ra",
           "weak": "không rõ rệt"}[TSNE_VERDICT["record_clustering_origin"]]
_trans = {"strong": "và đặc trưng học được CHUYỂN GIAO tốt giữa các bản ghi",
          "moderate": "và đặc trưng học được chuyển giao ở mức trung bình giữa các bản ghi",
          "weak": "nhưng đặc trưng học được GẦN NHƯ KHÔNG chuyển giao giữa các bản ghi",
          "unknown": "còn khả năng chuyển giao giữa các bản ghi chưa đo được"}[
              TSNE_VERDICT["cross_record_transfer"]]
print(f"  Cấu trúc cụm theo bản ghi trong không gian đặc trưng là {_origin}, {_trans}; "
      f"{len(_le)} kiến trúc "
      f"{'HỘI TỤ về cùng' if TSNE_VERDICT['architectures_converge'] else 'KHÔNG hội tụ về cùng'} "
      f"một cấu trúc láng giềng (trùng lặp {TSNE_K}-NN = {ov_models:.2f} so với mức ngẫu nhiên "
      f"{_ov_chance:.3f}).")
print("=" * 82)

with open(TAB_DIR / "tsne_comparative_verdict.json", "w", encoding="utf-8") as _f:
    json.dump(dict(fold=TSNE_FOLD, k=TSNE_K, n_frames=int(len(TSNE_IDX)),
                   pca_dim=TSNE_PCA_DIM, perplexity=TSNE_PERP,
                   deploy_model=DEPLOY_MODEL,
                   chance_label=_lab_ch, chance_record=_rec_ch,
                   group_means=dict(record_norm_untrained=rec_un, record_norm_learned=rec_le,
                                    label_norm_untrained=lab_un, label_norm_learned=lab_le,
                                    xrecord_norm_untrained=xr_un, xrecord_norm_learned=xr_le),
                   knn_overlap=dict(between_models=ov_models, untrained_vs_learned=ov_un_le,
                                    chance=_ov_chance),
                   verdict=TSNE_VERDICT),
              _f, indent=2, ensure_ascii=False, default=str)
print(f"Đã lưu: {TAB_DIR / 'tsne_comparative_verdict.json'}")

## 11. Xuất kết quả cho bài báo / Export artefacts for the manuscript

Sinh bảng LaTeX/Markdown và **đoạn Results tự động**, nhưng khác v5 ở một điểm quan trọng:

> **[V6] Đoạn văn tự sinh KHÔNG được chứa khẳng định viết trước.** v5 có sẵn câu
> *"confirming that leak evidence in low-pressure plastic pipes is distributed across the
> recording rather than concentrated in isolated frames"* — một kết luận khoa học được viết
> cứng vào template, bất kể số liệu ra sao. Ở v6, mọi câu khẳng định đều là **hệ quả của số đo**
> (kèm cả trường hợp số đo phản bác giả thuyết), và luôn có một mục **"Điều KHÔNG được khẳng
> định từ dữ liệu này"**.


In [ ]:
# =========================================================
# 11.1 Sinh bảng LaTeX + Markdown / LaTeX and Markdown tables
# =========================================================
def to_latex(df, caption, label, float_fmt="%.3f"):
    body = df.to_latex(index=False, float_format=lambda v: float_fmt % v, escape=True)
    return ("\\begin{table}[htbp]\n\\centering\n\\caption{" + caption + "}\n\\label{tab:" +
            label + "}\n" + body + "\\end{table}\n")

t6 = frame_df[["model", "AUC", "AUC_lo", "AUC_hi", "AUC_lo_naive", "AUC_hi_naive", "ICC",
               "design_effect", "n_eff", "Accuracy", "Precision", "Recall", "F1",
               "Specificity", "EER"]].round(3)
t7 = rec_main[["model", "auc", "auc_lo", "auc_hi", "accuracy", "precision", "recall",
               "recall_ci_lo", "recall_ci_hi", "specificity", "spec_ci_lo", "spec_ci_hi",
               "f1", "auc_raw"]].round(3)
t8 = stat_df[["model_A", "model_B", "dAUC", "dAUC_lo", "dAUC_hi", "mcnemar_rec_p_holm",
              "wilcoxon_p_floor", "n_folds"]].round(4)

tex = "\n".join([
    to_latex(t6, "Frame-level classification performance (out-of-fold, recording-grouped "
                 f"cross-validation). Confidence intervals are CLUSTER bootstraps over the {len(records)} "
                 "independent recordings; the naive frame-level CIs are shown only to "
                 "demonstrate their anti-conservative width.", "frame_level"),
    to_latex(t7, f"Record-level classification performance using {best_agg} aggregation of "
                 f"{'calibrated' if cfg.calibrate_aggregation else 'raw'} frame posteriors; "
                 f"Wilson 95% intervals for recall/specificity.",
             "record_level"),
    to_latex(t8, "Pairwise comparison between architectures: Holm-corrected McNemar at the "
                 f"RECORD level (n = {len(records)}) and paired cluster bootstrap for dAUC.", "stats")])
(TAB_DIR / "manuscript_tables.tex").write_text(tex, encoding="utf-8")

def _md_table(df):
    # [V6] to_markdown() cần gói tuỳ chọn 'tabulate'; thiếu gói thì lùi về CSV
    # thay vì làm chết cả notebook.
    try:
        return df.to_markdown(index=False)
    except Exception as e:
        print(f"  !! không tạo được bảng Markdown ({type(e).__name__}: {e}) -> xuất CSV thay thế.")
        return "```\n" + df.to_csv(index=False) + "```"

md_tables = ("## Table 6. Frame-level (cluster-bootstrap CI)\n\n" + _md_table(t6) +
             f"\n\n## Table 7. Record-level ({best_agg})\n\n" + _md_table(t7) +
             "\n\n## Table 8. Statistical tests (record-level McNemar + paired bootstrap)\n\n" +
             _md_table(t8) + "\n")
(TAB_DIR / "manuscript_tables.md").write_text(md_tables, encoding="utf-8")
print("Đã xuất bảng LaTeX/Markdown vào", TAB_DIR)
print(tex[:700] + "\n...")


In [ ]:
# =========================================================
# 11.2 Đoạn Results tự sinh - CHỈ phát biểu điều số liệu cho phép
#      [V6] Không còn câu khẳng định viết trước; thêm mục "KHÔNG được khẳng định".
# =========================================================
def _gv(name, default=None):
    return globals().get(name, default)

b = frame_df.iloc[0]
rb = rec_main.iloc[0]
d_rec = float(rb.auc) - float(frame_df.set_index("model").AUC[rb.model])   # [V7-B9] cùng một mô hình
n_rec_tot = int(len(records))
n_frames_tot = int(len(frames))
icc_mean = float(frame_df.ICC.mean())
de_mean = float(frame_df.DE.mean()) if "DE" in frame_df.columns else float(frame_df.design_effect.mean())
mil_ok = bool(_gv("MIL_PREMISE_SUPPORTED", False))
w_med = float(_gv("W_LEAK_MED", np.nan))
conf = _gv("CONFOUND", {}) or {}
conf_verdict = conf.get("verdict", "chưa xác định")
narr = _gv("BAND_NARRATIVE", "unknown")

sig = stat_df[stat_df["significant_0.05"]]
n_ci_excl = int(stat_df.dAUC_ci_excludes_0.sum()) if "dAUC_ci_excludes_0" in stat_df.columns else -1
abl_df = _gv("abl")
d_auc_leak = float(abl_df.loc["delta", "auc"]) if abl_df is not None and "delta" in abl_df.index else np.nan
d_acc_leak = float(abl_df.loc["delta", "accuracy"]) if abl_df is not None and "delta" in abl_df.index else np.nan
sel_d = sel_abl.dAUC.mean()
# [V7-B7] v6 lọc baseline theo tên "CHỈ mức năng lượng (RMS)" trong khi bảng dùng tên
#         "Level only (RMS)" -> baseline 1 đặc trưng có thể bị chọn nhầm làm "tốt nhất".
#         v7 lấy kết luận trực tiếp từ phép so sánh GHÉP CẶP ở mục 9.5.
base_txt = _gv("BEST_BASE_TXT", "không chạy")
base_verdict = _gv("BASELINE_VERDICT_VI", "chưa đánh giá được")
base_verdict_en = _gv("BASELINE_VERDICT_EN", "not evaluated")

_cf_up = str(conf_verdict).upper()
_conf_en = {"CÓ DẤU HIỆU CONFOUND": "signs of site/level confounding were found",
            "CHƯA LOẠI TRỪ ĐƯỢC CONFOUND": "site confounding could not be excluded (no site information)",
            "Không thấy dấu hiệu confound rõ": "no clear sign of confounding in the indirect checks"
            }.get(str(conf_verdict), str(conf_verdict))
if "CONFOUND" in _cf_up or "CHƯA" in _cf_up:
    claim_txt = (" that the model identifies the leak event itself, because leak and no-leak "
                 "recordings were not collected at the same acquisition sites and the "
                 f"per-recording score is largely recording-specific (ICC = {icc_mean:.2f}) - it may be "
                 "discriminating acquisition sites; nor ")
else:
    claim_txt = " "
_band_txt = {"low_band_supported": "consistent with a low-frequency-dominated leak signature",
             "low_band_not_supported": "NOT concentrated in the very-low-frequency band (< 800 Hz) "
                                       "that is usually assumed for plastic pipes",
             }.get(narr, "not resolvable between the low (< 800 Hz) and upper bands (the bootstrap "
                         "CI of the difference includes zero)")
_band_txt_vi = {"low_band_supported": "nằm ở vùng tần số thấp",
                "low_band_not_supported": "KHÔNG nằm ở vùng tần số rất thấp (< 800 Hz) như thường "
                                          "được giả định cho ống nhựa",
                }.get(narr, "CHƯA phân định được giữa dải thấp và dải cao (CI của hiệu số chứa 0)")
_wv = _gv("WITNESS_VERDICT", "not_supported")
_mil_txt = {"supported": "supports", "not_supported": "does not support"}.get(_wv, "is inconclusive about")
_mil_txt_vi = {"supported": "ủng hộ", "not_supported": "KHÔNG ủng hộ"}.get(_wv, "chưa đủ để kết luận về")
_rs = (_gv("REPEAT_SUMMARY", {}) or {}).get(rb.model)
if _rs and _rs.get("n_repeats", 1) > 1:
    rep_txt_en = (f"Across {_rs['n_repeats']} repetitions of the cross-validation with different "
                  f"partitions, the record-level AUC of {rb.model} was {_rs['mean']:.3f} +/- {_rs['sd']:.3f}; "
                  f"averaging the out-of-fold posteriors over repetitions gave AUC = {_rs['ens_auc']:.3f} "
                  f"(CI {_rs['ens_lo']:.3f}-{_rs['ens_hi']:.3f}).")
    rep_txt_vi = (f"Qua {_rs['n_repeats']} lần lặp CV với phân hoạch khác nhau, AUC mức bản ghi của "
                  f"{rb.model} là {_rs['mean']:.3f} ± {_rs['sd']:.3f}; trung bình xác suất out-of-fold qua "
                  f"các lần lặp cho AUC = {_rs['ens_auc']:.3f} (CI {_rs['ens_lo']:.3f}-{_rs['ens_hi']:.3f}).")
else:
    rep_txt_en = rep_txt_vi = ""
_dep = _gv("DEPLOY_MODEL", rb.model)

para_en = f'''
We evaluated five deep architectures on {n_rec_tot} field recordings ({n_frames_tot} overlapping
{cfg.frame_sec}-s frames) acquired from low-pressure PVC water pipes (1-2 bar), using
recording-grouped cross-validation with a nested inner-validation protocol that selects the
stopping epoch and the decision threshold without touching the reported folds; the record-level
aggregation rule ({best_agg}) was {'fixed in advance' if cfg.agg_selection == 'fixed' else 'selected on the inner validation sets'}. Frame-level confidence intervals are cluster bootstraps over recordings, because
only the {n_rec_tot} recordings are statistically independent: the mean intraclass correlation was
ICC = {icc_mean:.2f}, giving an estimated design effect of {de_mean:.0f} and an effective sample
size of about {b.n_eff:.0f} frames rather than {n_frames_tot}.

{b.model} achieved the highest frame-level discrimination (AUC = {b.AUC:.3f}, 95% cluster
bootstrap CI {b.AUC_lo:.3f}-{b.AUC_hi:.3f}; per-fold mean {b.AUC_cv_mean:.3f} +/- {b.AUC_cv_std:.3f}),
while {rb.model} achieved the highest record-level discrimination after aggregating frame
posteriors with {best_agg} (AUC = {rb.auc:.3f}, CI {rb.auc_lo:.3f}-{rb.auc_hi:.3f}; recall
{rb.recall:.3f}, specificity {rb.specificity:.3f}). For {rb.model}, aggregation changed the AUC by
{d_rec:+.3f} relative to its own frame-level value. At the record level, {len(sig)} of {len(stat_df)} architecture pairs differed
significantly under a Holm-corrected McNemar test performed at the recording level (n = {n_rec_tot}),
and {n_ci_excl} of {len(stat_df)} pairs had a paired cluster-bootstrap CI for dAUC that excluded
zero; the remaining pairs are statistically indistinguishable in this dataset. {rep_txt_en}
The deployment model, chosen on the inner validation sets only, is {_dep}.

Diagnostics. (i) The fraction of frames exceeding the operating threshold inside leak recordings
had a median of {w_med:.2f}, which {_mil_txt} the multiple-instance premise that leak evidence
occupies only isolated segments of a recording. (ii) The measured single-band informativeness was
{_band_txt}. (iii) Confound screening: {_conf_en}; the mean signal level alone reached
AUC = {conf.get('level_auc', float('nan')):.3f} at the record level. (iv) Classical baselines
(logistic regression / SVM / gradient boosting on hand-crafted spectral features and on gain-invariant
log-Mel statistics, same splits, paired comparison):
best = {base_txt}; {base_verdict_en}.

What this dataset does NOT allow us to claim:{claim_txt}that any architecture is superior, since
most pairwise CIs for dAUC include zero; nor that state-of-the-art accuracies reported in the
literature are inflated by data leakage, since the measured leakage inflation on this dataset was
dAUC = {d_auc_leak:+.3f}.
'''

para_vi = f'''
Nghiên cứu đánh giá năm kiến trúc học sâu trên {n_rec_tot} bản ghi hiện trường ({n_frames_tot} frame
{cfg.frame_sec} giây, chồng lấn 50%) thu trên ống PVC áp lực thấp (1-2 bar), dùng cross-validation
nhóm theo bản ghi kèm giao thức lựa chọn LỒNG. Khoảng tin cậy ở mức frame được bootstrap theo CỤM
bản ghi vì chỉ {n_rec_tot} bản ghi là độc lập: ICC trung bình {icc_mean:.2f} -> hệ số phóng đại
phương sai {de_mean:.0f}, cỡ mẫu hiệu dụng ~{b.n_eff:.0f} frame thay vì {n_frames_tot}.

Ở mức frame, {b.model} đạt AUC = {b.AUC:.3f} (CI 95% theo cụm {b.AUC_lo:.3f}-{b.AUC_hi:.3f}).
Ở mức bản ghi (gộp bằng {best_agg}), {rb.model} đạt AUC = {rb.auc:.3f} (CI {rb.auc_lo:.3f}-{rb.auc_hi:.3f}),
độ nhạy {rb.recall:.3f} [CI Wilson {rb.recall_ci_lo:.3f}-{rb.recall_ci_hi:.3f}], độ đặc hiệu
{rb.specificity:.3f}. Kiểm định McNemar MỨC BẢN GHI (hiệu chỉnh Holm) cho thấy {len(sig)}/{len(stat_df)}
cặp kiến trúc khác biệt có ý nghĩa; {n_ci_excl}/{len(stat_df)} cặp có CI ghép cặp của dAUC không
chứa 0. {rep_txt_vi} Mô hình triển khai (chọn chỉ trên validation-trong): {_dep}.

CHẨN ĐOÁN: (i) tỉ lệ thể hiện dương trung vị trong bản ghi leak là {w_med:.2f} ->
{_mil_txt_vi} tiền đề học đa thể hiện; (ii) dải tần mang thông tin
{_band_txt_vi};
(iii) kiểm tra confound: {conf_verdict}; chỉ riêng mức năng lượng trung bình đã đạt AUC
{conf.get('level_auc', float('nan')):.3f} ở mức bản ghi; (iv) baseline cổ điển: {base_txt} - {base_verdict}.
'''
(OUT / "results_paragraph.md").write_text("## English\n" + para_en + "\n## Tiếng Việt\n" + para_vi,
                                          encoding="utf-8")
display(Markdown("### Auto-generated Results paragraph (statements are limited to what the data support)"))
display(Markdown("**EN** " + para_en))
display(Markdown("**VI** " + para_vi))


In [ ]:
def _fresh_dir(p, tag=""):
    # [V6-P2] Tạo thư mục đích "sạch" mà KHÔNG làm chết notebook nếu không xoá được
    # thư mục cũ (file đang bị giữ / OneDrive / chạy lại trong cùng phiên Kaggle).
    p = Path(p)
    if p.exists():
        try:
            shutil.rmtree(p)
        except Exception as e:
            p = Path(str(p) + "_" + time.strftime("%H%M%S"))
            print(f"  !! không xoá được {tag or p.name} cũ ({type(e).__name__}) -> dùng {p.name}")
    p.mkdir(parents=True, exist_ok=True)
    return p

# =========================================================
# 11.3 Đóng gói toàn bộ kết quả / Package every artefact
# =========================================================
index_rows = [dict(figure=k, caption=v) for k, v in FIG_INDEX.items()]
pd.DataFrame(index_rows).to_csv(FIG_DIR / "figure_index.csv", index=False)

run_meta = dict(
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
    synthetic_demo=bool(SYNTHETIC), data_root=str(DATA_ROOT),
    n_records=int(manifest.record_id.nunique()), n_frames=int(len(frames)),
    total_minutes=float(manifest.duration_s.sum()/60),
    config={k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cfg).items()},
    n_folds=len(SPLITS), best_model_frame=BEST_MODEL,
    nested_selection=bool(cfg.nested_selection), nested_folds=[bool(x) for x in NESTED_OK],
    thresholds={k: float(v) for k, v in THR.items()},
    best_model_record=str(rec_main.model.iloc[0]), best_aggregation=best_agg,
    # [V6] Ghi kèm các kết luận chẩn đoán để người đọc sau biết phạm vi áp dụng:
    band_hypothesis=BAND_NARRATIVE, mil_premise_supported=bool(MIL_PREMISE_SUPPORTED),
    confound_verdict=(CONFOUND or {}).get("verdict", "n/a"),
    statistics=dict(unit="recording", ci="cluster bootstrap over recordings",
                    mcnemar="record-level", n_records=int(len(records)),
                    wilcoxon_min_p=wilcoxon_min_p(len(SPLITS))),
    gpu=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    torch=torch.__version__)
(OUT / "run_metadata.json").write_text(json.dumps(run_meta, indent=2, ensure_ascii=False,
                                                  default=str), encoding="utf-8")

stamp = time.strftime("%Y%m%d_%H%M")
bundle = _fresh_dir(OUT / f"leak_cnn_results_{stamp}", "gói kết quả")
for src in [FIG_DIR, TAB_DIR]:
    shutil.copytree(src, bundle / src.name, dirs_exist_ok=True)
for f in ["run_metadata.json", "results_paragraph.md", "manifest_records.csv"]:
    if (OUT / f).exists(): shutil.copy(OUT / f, bundle / f)
try:
    shutil.make_archive(str(bundle), "zip", bundle)
    print("Gói kết quả / results bundle:", str(bundle) + ".zip")
except Exception as e:
    print(f"!! không nén được gói kết quả ({type(e).__name__}: {e}) - thư mục vẫn dùng được: {bundle}")
print("\nHình đã tạo / figures:")
for k, v in FIG_INDEX.items(): print(" -", k)


## 12. Kết luận, hạn chế và hướng phát triển / Conclusions, limitations, future work

**Đóng góp của pipeline này (phần Discussion):**

1. **Chế độ vật lý chưa được khảo sát**: ống PVC ở 1–2 bar — suy giảm mạnh, SNR thấp. Đa số công
   bố WoS/Scopus dùng ống kim loại ở áp suất ≥ 5 bar nên kết quả không chuyển giao trực tiếp được.
2. **Giao thức đánh giá chống rò rỉ + chống lạc quan do lựa chọn**, kèm **đo** mức thổi phồng do
   rò rỉ dữ liệu và do lựa chọn epoch/ngưỡng/tiêu chí (mục 9.3, 9.4).
3. **Đơn vị thống kê đúng**: CI bootstrap theo cụm bản ghi, McNemar mức bản ghi, so sánh ghép cặp,
   hiệu chỉnh Holm, công khai sàn p của Wilcoxon, CV lặp.
4. **Đo thay vì giả định**: tỉ lệ thể hiện dương (8.7), ICC/design effect (8.2, 8.8), dải tần mang
   thông tin (3.5, 3.5b), tầm quan trọng nhân quả theo dải bằng occlusion (10.2).
5. **Kiểm tra confound** (9.6), **baseline cổ điển ghép cặp** (9.5) và **ablation** cấu hình (9.7).

**Hạn chế phải nêu trung thực** (số liệu cụ thể lấy từ output của chính lần chạy):

- **Số bản ghi hạn chế** → CI rộng; cỡ mẫu hiệu dụng (mục 8.2) chỉ cỡ vài lần số bản ghi. Cần
  **thêm điểm khảo sát**, không phải kéo dài thời lượng thu tại chỗ.
- **Thiếu thông tin điểm khảo sát** → **chưa loại trừ được confound** "mô hình học vị trí thu".
  Đây là hạn chế nghiêm trọng nhất.
- **Nhãn ở mức bản ghi**, không có nhãn thời điểm rò rỉ → không kiểm chứng trực tiếp được tiền đề MIL.
- **Một chuỗi thiết bị thu, một mạng ống** → hiệu lực ngoại suy thấp.
- **Tiền xử lý không nhân quả** (chuẩn hoá đỉnh theo toàn bộ tệp, quyết định ở mức bản ghi)
  → pipeline chạy offline, chưa phải thời gian thực.


In [ ]:
# =========================================================
# 12.1 Bảng so sánh với tài liệu / Comparison against the literature
#      [V6-S7] SỬA LỖI: v5 trộn hai mô hình trong cùng một dòng - frame AUC của TransferCNN
#      nhưng record AUC của MultiBranchCNN. Nay mọi con số của "THIS WORK" lấy từ CÙNG một
#      mô hình (mô hình đang báo cáo ở mức bản ghi) và ghi rõ mô hình đó.
# =========================================================
REPORT_MODEL = str(rb.model)
_req = rec_main[rec_main.model == REPORT_MODEL].iloc[0]
_frq = frame_df[frame_df.model == REPORT_MODEL].iloc[0]

lit = pd.DataFrame([
    dict(verified="? chua xac minh", study="Shukla & Piratla (2020), Autom. Constr.",
         material="PVC/cast iron", pressure="lab", representation="raw acceleration",
         model="1D-CNN", protocol="random split", unit="not reported",
         metric="Acc 0.95-0.99"),
    dict(verified="? chua xac minh", study="Cody et al. (2020), J. Comput. Civ. Eng.",
         material="live network", pressure="operational", representation="hydroacoustic spectrogram",
         model="deep autoencoder", protocol="random split", unit="not reported", metric="AUC ~0.99"),
    dict(verified="? chua xac minh", study="Guo et al. (2021), JWRPM", material="metal",
         pressure="n/a", representation="time-frequency map", model="TF-CNN",
         protocol="random split", unit="not reported", metric="Acc ~0.97"),
    dict(verified="V da doc ban goc", study="Ahmad et al. (2022), Sensors 22:1562",
         material="steel", pressure="7 & 13 bar", representation="Morse CWT scalogram",
         model="CNN + conv-autoencoder", protocol="70/30 random x10", unit="not reported",
         metric="Acc 0.966-0.984"),
    dict(verified="? chua xac minh", study="Ma et al. (2024), JWRPM 150(6)", material="network",
         pressure="operational", representation="log spectrogram", model="CNN",
         protocol="random split", unit="not reported", metric="Acc >0.95"),
    dict(verified="V da doc ban goc", study="Li et al. (2025), Water 17:1115",
         material="steel DN50", pressure="controlled", representation="STFT + 1D signal",
         model="MobileNetV3 + Mamba", protocol="6:2:2 split", unit="not reported",
         metric="Acc 0.983"),
    dict(verified="V da doc ban goc", study="Zhang et al. (2026), Water 18:1528",
         material="5 cities", pressure="operational", representation="STFT 224x224",
         model="Incep-ResNet + SE", protocol="90/10 stratified", unit="not reported",
         metric="Acc 0.966"),
    dict(verified="-", study=f"THIS WORK ({REPORT_MODEL})", material="PVC",
         pressure="1-2 bar (low)",
         representation="log-Mel / MFCC / low-band STFT / waveform",
         model=f"{REPORT_MODEL} (5 architectures compared)",
         protocol=f"record-grouped CV (K={len(SPLITS)}), nested selection",
         unit=f"{len(records)} recordings (effective n ~ {_frq.n_eff:.0f} frames)",
         metric=(f"AUC {_frq.AUC:.3f} frame [{_frq.AUC_lo:.3f}-{_frq.AUC_hi:.3f}] / "
                 f"{_req.auc:.3f} record [{_req.auc_lo:.3f}-{_req.auc_hi:.3f}]")),
])
display(Markdown("### Table 11. Positioning against the indexed literature (WoS/Scopus)"))
display(lit)
lit.to_csv(TAB_DIR / "table11_literature_comparison.csv", index=False)
print("CỘT 'verified': '?' = tiêu đề/tạp chí đúng nhưng TÁC GIẢ và SỐ LIỆU chưa mở bản gốc.")
print("Phải xác minh trên Scopus trước khi đưa vào danh mục tài liệu tham khảo.")
print("\nCột 'unit' là cột quan trọng nhất khi so sánh: nghiên cứu này báo cáo SỐ BẢN GHI ĐỘC LẬP")
print("và cỡ mẫu hiệu dụng, không chỉ số frame. Nếu công bố khác không nêu đơn vị thống kê thì")
print("việc so sánh trực tiếp là không hợp lệ.")
print(f"\nMọi con số của dòng 'THIS WORK' đều lấy từ CÙNG một mô hình ({REPORT_MODEL}).")


## 13. Chọn mô hình triển khai / Deployment model selection

Đến đây có năm kiến trúc và phải chọn **một** để triển khai. Cách làm hiển nhiên — lấy mô hình có AUC cao nhất trong Bảng 7 — **lại chính là lỗi mà notebook này được xây để tránh**: chọn trên tập báo cáo thì con số của mô hình thắng cuộc bị thổi phồng, đúng như đã đo ở mục 9.4 cho việc chọn epoch và ngưỡng.

Vì vậy mô hình triển khai được chọn theo **AUC mức bản ghi trên validation-TRONG** — dữ liệu chưa từng dùng để báo cáo. Fold ngoài chỉ dùng để *báo cáo* hiệu năng của mô hình đã chọn.

Notebook in ra cả hai lựa chọn (theo validation-trong và theo tập báo cáo). Nếu chúng khác nhau, chênh lệch AUC chính là **mức thổi phồng do chọn mô hình** — nguồn lạc quan thứ ba, độc lập với hai nguồn ở mục 9.3 và 9.4.

Ngoài AUC, bảng lựa chọn còn có **số tham số** và **tốc độ suy luận** — hai yếu tố quyết định khi triển khai trên thiết bị khảo sát cầm tay.


In [ ]:
# =========================================================
# 13.1 Bảng lựa chọn mô hình triển khai
#      Tiêu chí: AUC mức bản ghi trên VALIDATION-TRONG.
#      [V6-S5] v5 kết luận "không có thổi phồng do chọn mô hình" chỉ vì hai argmax trùng nhau,
#      trong khi tiêu chí chọn đã BÃO HOÀ gần 1.0 trên vài chục bản ghi/fold và
#      chênh lệch nằm trong sai số chuẩn. Nay in KÈM CI của chính tiêu chí chọn để thấy rõ
#      mức độ phân giải được.
# =========================================================
def _bench_ms_per_frame(name, n=256, reps=3):
    m = build_model(name).to(DEVICE).eval()
    xb = torch.from_numpy(np.asarray(X_ALL[:n])).float().to(DEVICE)
    with torch.no_grad():
        for _ in range(2): m(xb)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(reps): m(xb)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        dt = (time.time() - t0) / reps
    n_par = sum(p.numel() for p in m.parameters() if p.requires_grad)
    del m; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return dt / n * 1000, n_par

sel_rows = []
for name in cfg.models:
    g_in = REC_INNER[name]
    if g_in.label.nunique() > 1:
        auc_in = float(roc_auc_score(g_in.label, g_in[best_agg]))
        ci_in = cluster_bootstrap_ci(g_in.record_id.values, g_in.label.values,
                                     g_in[best_agg].values, roc_auc_score,
                                     n=min(cfg.bootstrap_n, 1000), seed=0)
    else:
        auc_in, ci_in = np.nan, dict(lo=np.nan, hi=np.nan)
    r_out = rec_df[(rec_df.model == name) & (rec_df.aggregation == best_agg)].iloc[0]
    ms, npar = _bench_ms_per_frame(name)
    sel_rows.append(dict(
        model=name,
        AUC_rec_inner=auc_in, AUC_inner_lo=ci_in["lo"], AUC_inner_hi=ci_in["hi"],
        AUC_rec_outer=float(r_out.auc), CI_lo=float(r_out.auc_lo), CI_hi=float(r_out.auc_hi),
        AUC_frame=float(frame_df.set_index("model").AUC[name]),
        recall=float(r_out.recall), specificity=float(r_out.specificity),
        params_M=round(npar / 1e6, 3), ms_per_frame=round(ms, 4),
        realtime_factor=round(cfg.frame_sec * 1000 / max(ms, 1e-9))))
dep_sel_df = pd.DataFrame(sel_rows)

DEPLOY_MODEL = str(dep_sel_df.loc[dep_sel_df.AUC_rec_inner.idxmax(), "model"])
ARGMAX_OUTER = str(dep_sel_df.loc[dep_sel_df.AUC_rec_outer.idxmax(), "model"])
dep_sel_df["selected"] = np.where(dep_sel_df.model == DEPLOY_MODEL, "<<< DEPLOY", "")
dep_sel_df = dep_sel_df.sort_values("AUC_rec_inner", ascending=False)
dep_sel_df.round(4).to_csv(TAB_DIR / "table15_model_selection.csv", index=False)

display(Markdown("### Table 15. Deployment model selection "
                 "(selection criterion and its own confidence interval)"))
display(dep_sel_df.round(4))

_s = dep_sel_df.set_index("model")
print(f"Selected on the INNER validation set : {DEPLOY_MODEL} (inner AUC "
      f"{_s.AUC_rec_inner[DEPLOY_MODEL]:.4f} "
      f"[{_s.AUC_inner_lo[DEPLOY_MODEL]:.4f}, {_s.AUC_inner_hi[DEPLOY_MODEL]:.4f}])")
print(f"If selected on the REPORTED set      : {ARGMAX_OUTER} (outer AUC "
      f"{_s.AUC_rec_outer[ARGMAX_OUTER]:.4f})")
if DEPLOY_MODEL != ARGMAX_OUTER:
    infl = _s.AUC_rec_outer[ARGMAX_OUTER] - _s.AUC_rec_outer[DEPLOY_MODEL]
    print(f"\n!! Hai cách chọn KHÁC NHAU -> chênh lệch {infl:+.4f} chính là mức thổi phồng do")
    print(f"   chọn mô hình. Con số phải báo cáo là {_s.AUC_rec_outer[DEPLOY_MODEL]:.4f} for "
          f"{DEPLOY_MODEL}.")
else:
    print("\nHai cách chọn trùng nhau. LƯU Ý [V6-S5]: điều này KHÔNG tự động nghĩa là 'không có")
    print("thổi phồng', vì tiêu chí chọn có thể không đủ độ phân giải để phân biệt các mô hình.")

# [V6-S5] Tiêu chí chọn có phân giải được không?
_top2 = dep_sel_df.head(2)
_ov = not (_top2.AUC_inner_lo.iloc[1] > _top2.AUC_inner_hi.iloc[0] or
           _top2.AUC_inner_hi.iloc[1] < _top2.AUC_inner_lo.iloc[0])
print(f"\nTop two models by the selection criterion: {_top2.model.iloc[0]} "
      f"({_top2.AUC_rec_inner.iloc[0]:.4f}) vs {_top2.model.iloc[1]} ({_top2.AUC_rec_inner.iloc[1]:.4f})")
if _ov:
    print("=> CI CỦA CHÍNH TIÊU CHÍ CHỌN CHỒNG LẤN: việc chọn mô hình triển khai KHÔNG được dữ liệu")
    print("   phân giải. Trong bài báo phải viết đúng như vậy; nên chọn theo tiêu chí ghép")
    print("   (hiệu năng + chi phí suy luận + số tham số) và nói rõ đó là quyết định KỸ THUẬT.")
    print("   Bảng dưới đây đã có ms/frame và số tham số để làm căn cứ đó:")
    display(dep_sel_df[["model", "AUC_rec_inner", "AUC_inner_lo", "AUC_inner_hi",
                    "params_M", "ms_per_frame", "realtime_factor"]].round(4))
else:
    print("=> CI không chồng lấn: tiêu chí chọn phân giải được hai mô hình dẫn đầu.")


In [ ]:
# =========================================================
# 13.2 Hình: cơ sở lựa chọn (hiệu năng vs chi phí)
# =========================================================
fig, ax = plt.subplots(1, 3, figsize=(14.5, 4.0))
order = dep_sel_df.model.tolist()
xs = np.arange(len(order)); w = .36

ax[0].bar(xs - w/2, dep_sel_df.AUC_rec_inner, w, color=PAL["noleak"], label="inner validation (selection criterion)")
ax[0].bar(xs + w/2, dep_sel_df.AUC_rec_outer, w, color=PAL["leak"], label="outer fold (reported)")
for i, m in enumerate(order):
    if m == DEPLOY_MODEL:
        ax[0].annotate("selected", (i, max(dep_sel_df.AUC_rec_inner.iloc[i], dep_sel_df.AUC_rec_outer.iloc[i]) + .015),
                       ha="center", fontsize=9, fontweight="bold", color="k")
ax[0].set_xticks(xs); ax[0].set_xticklabels(order, rotation=20, ha="right")
ax[0].set_ylabel("record-level AUC"); ax[0].set_ylim(.5, 1.05)
ax[0].axhline(.5, color="k", ls="--", lw=.8); ax[0].legend(fontsize=8)
ax[0].set_title("(a) Selection criterion vs reported value")

for i, r in dep_sel_df.reset_index(drop=True).iterrows():
    ax[1].errorbar(r.AUC_rec_outer, i, xerr=[[r.AUC_rec_outer - r.CI_lo], [r.CI_hi - r.AUC_rec_outer]],
                   fmt="o", ms=8, capsize=4, color=MODEL_COLORS[r.model])
ax[1].set_yticks(range(len(dep_sel_df))); ax[1].set_yticklabels(dep_sel_df.model)
ax[1].axvline(.5, color="k", ls="--", lw=.8)
ax[1].set_xlabel("record-level AUC (outer fold, 95% CI)")
ax[1].set_title("(b) Widely overlapping confidence intervals")

sc = ax[2].scatter(dep_sel_df.ms_per_frame, dep_sel_df.AUC_rec_outer,
                   s=60 + 400 * dep_sel_df.params_M / max(dep_sel_df.params_M.max(), 1e-9),
                   c=[MODEL_COLORS[m] for m in dep_sel_df.model], alpha=.85)
for _, r in dep_sel_df.iterrows():
    ax[2].annotate(r.model, (r.ms_per_frame, r.AUC_rec_outer), fontsize=7,
                   xytext=(4, 4), textcoords="offset points")
ax[2].set_xscale("log"); ax[2].set_xlabel("ms per frame (log scale)")
ax[2].set_ylabel("record-level AUC")
ax[2].set_title("(c) Performance vs cost\n(marker size = number of parameters)")
savefig(fig, "model_selection",
        "Basis for the deployment-model choice: (a) the selection criterion is the inner-validation "
        "record-level AUC, shown next to the reported outer-fold value; (b) record-level "
        "bootstrap confidence intervals; (c) performance versus inference cost.")


In [ ]:
# =========================================================
# 13.3 Bảng điểm vận hành - kèm CI Wilson và cảnh báo về ngưỡng ensemble
# =========================================================
g_dep = REC[DEPLOY_MODEL]
y_dep, s_dep = g_dep.label.values.astype(int), g_dep[best_agg].values
fpr_c, tpr_c, thr_c = roc_curve(y_dep, s_dep)
thr_c = np.where(np.isfinite(thr_c), thr_c, 1.0)

def _wilson(k, n):
    if n == 0: return (np.nan, np.nan)
    lo, hi = spstats.binomtest(int(k), int(n)).proportion_ci(confidence_level=0.95, method="wilson")
    return float(lo), float(hi)

n_pos, n_neg = int(y_dep.sum()), int((1 - y_dep).sum())
op_rows = []
for target in [0.02, 0.05, 0.10, 0.20, 0.30]:
    i = max(int(np.searchsorted(fpr_c, target, side="right") - 1), 0)
    tp = tpr_c[i] * n_pos; fp = fpr_c[i] * n_neg
    r_lo, r_hi = _wilson(tp, n_pos); f_lo, f_hi = _wilson(fp, n_neg)
    op_rows.append(dict(target_FPR=target, threshold=float(thr_c[i]),
                        recall=float(tpr_c[i]), recall_lo=r_lo, recall_hi=r_hi,
                        FPR_actual=float(fpr_c[i]), fpr_lo=f_lo, fpr_hi=f_hi,
                        specificity=1 - float(fpr_c[i]), miss_rate=1 - float(tpr_c[i])))
j = int(np.argmax(tpr_c - fpr_c))
tp = tpr_c[j] * n_pos; fp = fpr_c[j] * n_neg
r_lo, r_hi = _wilson(tp, n_pos); f_lo, f_hi = _wilson(fp, n_neg)
op_rows.append(dict(target_FPR=np.nan, threshold=float(thr_c[j]), recall=float(tpr_c[j]),
                    recall_lo=r_lo, recall_hi=r_hi, FPR_actual=float(fpr_c[j]),
                    fpr_lo=f_lo, fpr_hi=f_hi, specificity=1 - float(fpr_c[j]), miss_rate=1 - float(tpr_c[j])))
op_df = pd.DataFrame(op_rows)
op_df.index = [f"FPR<={t:.0%}" for t in op_df.target_FPR[:-1]] + ["Youden"]
op_df.round(4).to_csv(TAB_DIR / "table16_operating_points.csv")
display(Markdown(f"### Table 16. Operating points — {DEPLOY_MODEL} "
                 f"(with 95% Wilson intervals)"))
display(op_df.drop(columns=["target_FPR"]).round(4))

print(f"\nDiễn giải trên {len(g_dep)} bản ghi ({n_pos} có rò rỉ / {n_neg} không):")
for lbl, r in op_df.iterrows():
    print(f"  {lbl:10s} threshold {r.threshold:.3f} -> detected {r.recall*n_pos:.0f}/{n_pos} "
          f"[{r.recall_lo*n_pos:.0f}-{r.recall_hi*n_pos:.0f}], "
          f"false alarms {r.FPR_actual*n_neg:.0f}/{n_neg} "
          f"[{r.fpr_lo*n_neg:.0f}-{r.fpr_hi*n_neg:.0f}]")
print("\nNếu chi phí đào nhầm THẤP hơn chi phí thất thoát nước - thường đúng với mạng cấp nước -")
print("hãy chọn một hàng có FPR cao hơn Youden để giảm bỏ sót.")
print("\n!! LƯU Ý VỀ NGƯỠNG KHI TRIỂN KHAI (mục 14): ngưỡng này được fit trên phân bố điểm của")
print("   MỘT mô hình. Khi suy luận tệp mới, xác suất được lấy trung bình của K fold -> phân bố")
print("   điểm dịch chuyển, nên ngưỡng cần được hiệu chỉnh lại trên dữ liệu hiện trường mới.")
print("   Mục 14 sẽ in ra mức dịch chuyển phân bố để cảnh báo hướng lệch.")


## 14. Kiểm thử một tệp âm thanh hiện trường / Single-file field inference

Chạy trọn tuyến quyết định trên **một tệp `.wav` thu trực tiếp ngoài hiện trường, chưa qua xử lý**, đúng như khi triển khai thật.

**Nếu tệp thuộc một bản ghi đã có trong tập dữ liệu**, notebook chỉ dùng mô hình của **fold mà bản ghi đó nằm ở validation** — mô hình chưa từng thấy nó. Tệp hoàn toàn mới thì dùng trung bình xác suất của cả K fold. Trường hợp nào cũng in rõ.

> Demo chạy trên bản ghi mà mô hình đã được huấn luyện trên đó thì không chứng minh được gì. Cell này chặn sẵn khả năng đó.


In [ ]:
# =========================================================
# 14.1 Cấu hình kiểm thử hiện trường / Single-file test configuration
#      [V6-T9] Dùng xác suất ĐÃ hiệu chỉnh Platt và ngưỡng mức bản ghi tương ứng.
# =========================================================
TEST_FILE  = ""              # <- ĐIỀN đường dẫn tệp .wav hiện trường
TEST_MODEL = DEPLOY_MODEL    # mô hình đã chọn ở mục 13
SAVE_PREFIX = "field_test"

REC_THR = float(youden_threshold(REC_INNER[TEST_MODEL].label.values,
                                 REC_INNER[TEST_MODEL][best_agg].values))
FRAME_THR = float(THR[TEST_MODEL])
print(f"Mô hình              : {TEST_MODEL} (chọn ở mục 13)")
print(f"  AUC mức bản ghi    : {float(rec_main.set_index('model').auc[TEST_MODEL]):.4f}")
print(f"  Quy tắc gộp        : {best_agg}")
print(f"  Ngưỡng bản ghi/frame: {REC_THR:.4f} / {FRAME_THR:.4f}")
print(f"  Hiệu chỉnh Platt   : {'BẬT' if cfg.calibrate_aggregation else 'TẮT'}")

if not TEST_FILE:
    _r = records.sample(1, random_state=cfg.seed).iloc[0]
    TEST_FILE = files[files.record_id == _r.record_id].path.iloc[0]
    print(f"\n!! Chưa điền TEST_FILE -> tự chọn {Path(TEST_FILE).name} "
          f"(bản ghi {_r.record_id}, nhãn thật = {_r.cls}).")
assert Path(TEST_FILE).exists(), f"Không tìm thấy tệp: {TEST_FILE}"


In [ ]:
# =========================================================
# 14.2 Chọn bộ trọng số KHÔNG bị nhiễm theo bản ghi của tệp
# =========================================================
test_rid, _how = extract_record_id(Path(TEST_FILE))
in_corpus = test_rid in set(REC_IDS)
true_label = None
if in_corpus:
    true_label = int(rec_lab[test_rid])
    use_folds = [k for k, (_tr, va) in enumerate(SPLITS) if test_rid in set(groups_all[va])]
    mode = f"out-of-fold (fold {use_folds[0]})" if use_folds else "ensemble"
    if not use_folds: use_folds = list(range(len(SPLITS)))
    print(f"Bản ghi {test_rid} CÓ trong tập dữ liệu (nhãn thật: "
          f"{'leak' if true_label else 'noleak'}) -> dùng {mode}, ước lượng KHÔNG thiên lệch.")
else:
    use_folds = list(range(len(SPLITS)))
    mode = f"ensemble {len(use_folds)} fold"
    print(f"Bản ghi {test_rid} KHÔNG có trong tập dữ liệu -> dùng {mode}.")

def load_fold_model(name, fold):
    m = build_model(name)
    sd = torch.load(CKPT_DIR / f"{name}_fold{fold}.pt", map_location="cpu")
    m.load_state_dict({k: v.float() for k, v in sd.items()})
    return m.to(DEVICE).eval()

TEST_MODELS = [load_fold_model(TEST_MODEL, k) for k in use_folds]
print(f"Đã nạp {len(TEST_MODELS)} bộ trọng số.")


In [ ]:
# =========================================================
# 14.3 Suy luận trọn tuyến trên tệp thô
#      [V6] Áp Platt của đúng fold đang dùng; ghi rõ đây là xử lý OFFLINE (không nhân quả).
# =========================================================
@torch.no_grad()
def predict_waveform(x, models, batch=128, platt_list=None):
    if len(x) < FRAME_LEN:
        x = np.pad(x, (0, FRAME_LEN - len(x)))
    starts = list(range(0, len(x) - FRAME_LEN + 1, HOP_LEN)) or [0]
    X = np.stack([x[s:s + FRAME_LEN] for s in starts]).astype(np.float32)
    acc = np.zeros(len(X), dtype=np.float64)
    for j, m in enumerate(models):
        P = []
        for s0 in range(0, len(X), batch):
            xb = torch.from_numpy(X[s0:s0 + batch]).float().to(DEVICE)
            with amp_autocast(DEVICE, cfg.amp and DEVICE.type == "cuda"):
                lg = m(xb)
            pr = torch.sigmoid(lg.float())
            P.append(torch.nan_to_num(pr, nan=.5).clamp(0, 1).cpu().numpy())
        p_one = np.concatenate(P)
        if platt_list is not None and cfg.calibrate_aggregation:
            p_one = platt_apply(p_one, platt_list[j])
        acc += p_one
    return np.asarray(starts) / cfg.target_sr, acc / len(models), X

t_raw0 = time.time()
x_raw, sr_raw = librosa.load(TEST_FILE, sr=None, mono=True)
x_proc = load_audio(TEST_FILE)
_platts = [ALL_RESULTS[TEST_MODEL][k]["platt"] for k in use_folds] if cfg.calibrate_aggregation else None
t_frames, p_frames, X_test = predict_waveform(x_proc, TEST_MODELS, platt_list=_platts)
dt = time.time() - t_raw0

agg_fn = dict(p_mean=np.mean, p_med=np.median,
              p_p90=lambda a: float(np.percentile(a, 90)),
              frac_pos=lambda a: float((a >= FRAME_THR).mean()))[best_agg]
score = float(agg_fn(p_frames))
decision = "LEAK DETECTED" if score >= REC_THR else "NO LEAK"

display(Markdown(f'''### Field-test result — `{Path(TEST_FILE).name}`

| Item | Value |
|---|---|
| Original sampling rate | {sr_raw} Hz -> {cfg.target_sr} Hz |
| Duration | {len(x_raw)/sr_raw:.1f} s |
| Analysis windows | {len(p_frames)} ({cfg.frame_sec} s, hop {cfg.hop_sec} s) |
| Model | {TEST_MODEL} ({mode}) |
| Record-level score (`{best_agg}`, calibrated) | **{score:.4f}** (threshold {REC_THR:.4f}) |
| **DECISION** | **{decision}** |
| Windows above threshold | {float((p_frames >= FRAME_THR).mean()):.2%} |
| Processing time | {dt:.2f} s ({len(x_raw)/sr_raw/max(dt,1e-9):.0f}x file duration) |
'''))
if true_label is not None:
    ok = (score >= REC_THR) == bool(true_label)
    print(("✓ ĐÚNG" if ok else "✗ SAI") + f" - nhãn thật: {'leak' if true_label else 'noleak'}")

print("\n!! PHẠM VI ÁP DỤNG CỦA CON SỐ THỜI GIAN: đây là thông lượng xử lý theo LÔ trên, KHÔNG")
print("   phải độ trễ của hệ thống thời gian thực. Pipeline chuẩn hoá đỉnh theo TOÀN BỘ bản ghi")
print("   và ra quyết định ở mức bản ghi, tức là xử lý OFFLINE. Muốn chạy trực tuyến phải thay")
print("   chuẩn hoá bằng phương án nhân quả (RMS trượt / hiệu chuẩn cố định) và đánh giá lại.")


In [ ]:
# =========================================================
# 14.4 Hình: dòng thời gian quyết định + phổ đồ có tô vùng phát hiện
# =========================================================
fig = plt.figure(figsize=(13.5, 7.2))
gs = gridspec.GridSpec(3, 1, height_ratios=[1, 1.1, 1.5], hspace=.42)

a0 = fig.add_subplot(gs[0])
tt = np.arange(len(x_proc)) / cfg.target_sr
a0.plot(tt, x_proc, lw=.3, color=PAL["grey"])
a0.set_ylabel("amplitude"); a0.set_xlim(0, tt[-1])
a0.set_title(f"{Path(TEST_FILE).name} — pre-processed waveform")
plt.setp(a0.get_xticklabels(), visible=False)

a1 = fig.add_subplot(gs[1], sharex=a0)
col = PAL["leak"] if score >= REC_THR else PAL["noleak"]
a1.plot(t_frames + cfg.frame_sec / 2, p_frames, lw=1.6, color=col, marker="o", ms=2.5)
a1.axhline(FRAME_THR, color="k", ls="--", lw=1, label=f"frame threshold={FRAME_THR:.2f}")
a1.axhline(score, color=col, ls="-.", lw=1.2, label=f"{best_agg}={score:.3f}")
a1.fill_between(t_frames + cfg.frame_sec / 2, FRAME_THR, p_frames,
                where=p_frames >= FRAME_THR, color=PAL["leak"], alpha=.25)
a1.set_ylim(0, 1.02); a1.set_ylabel("leak probability")
a1.legend(fontsize=8, ncol=2); a1.set_title("Leak probability over time (shaded = above threshold)")
plt.setp(a1.get_xticklabels(), visible=False)

a2 = fig.add_subplot(gs[2], sharex=a0)
S = librosa.amplitude_to_db(np.abs(librosa.stft(x_proc, n_fft=cfg.n_fft,
                                                hop_length=cfg.hop_length)), ref=np.max)
im = librosa.display.specshow(S, sr=cfg.target_sr, hop_length=cfg.hop_length,
                              x_axis="time", y_axis="linear", ax=a2, cmap="magma")
a2.set_ylim(0, 1500)
for ts, pp in zip(t_frames, p_frames):
    if pp >= FRAME_THR:
        a2.axvspan(ts, ts + cfg.frame_sec, color="cyan", alpha=.10, lw=0)
a2.set_title("Spectrogram 0-1500 Hz — cyan spans = windows flagged as leak")
fig.colorbar(im, ax=a2, format="%+2.0f dB", pad=.01)
savefig(fig, f"{SAVE_PREFIX}_timeline",
        f"Field test on {Path(TEST_FILE).name}: waveform, leak probability over time, and spectrogram "
        f"with the detected regions.")


In [ ]:
# =========================================================
# 14.5 Xuất kết quả chi tiết theo cửa sổ
# =========================================================
out = pd.DataFrame(dict(window=np.arange(len(p_frames)),
                        t_start_s=np.round(t_frames, 3),
                        t_end_s=np.round(t_frames + cfg.frame_sec, 3),
                        probability=np.round(p_frames, 4),
                        above_threshold=(p_frames >= FRAME_THR)))
csv_path = OUT / f"{SAVE_PREFIX}_{Path(TEST_FILE).stem}.csv"
out.to_csv(csv_path, index=False, encoding="utf-8-sig")
display(Markdown("**Five highest-probability windows:**"))
display(out.nlargest(5, "probability"))
print("Đã lưu:", csv_path)
if score >= REC_THR:
    seg = out[out.above_threshold]
    if len(seg):
        print(f"Segment worth listening to: {seg.t_start_s.min():.1f}s - "
              f"{seg.t_end_s.max():.1f}s ({len(seg)} windows).")


## 15. Xuất gói mô hình triển khai / Export the deployment bundle

Đóng gói mô hình đã chọn ở mục 13 thành **gói chạy độc lập**:

| Tệp | Vai trò |
|---|---|
| `fold*.ts` | **TorchScript** — chạy được mà không cần mã nguồn notebook này, không cần dựng lại kiến trúc, không phụ thuộc phiên bản thư viện |
| `manifest.json` | Toàn bộ tham số tiền xử lý, ngưỡng, quy tắc gộp, **bảng điểm vận hành**, hiệu năng out-of-fold |
| `state_dicts.pt` | Trọng số thuần, dự phòng |
| `README.txt` | Tóm tắt và cảnh báo phạm vi áp dụng |

Gói này dùng với notebook `leak_detector_inference.ipynb`. Vì notebook suy luận đọc mọi tham số **từ manifest**, không thể xảy ra tình trạng suy luận dùng tiền xử lý khác lúc huấn luyện — lỗi triển khai âm thầm và phổ biến nhất với mô hình âm thanh.

Quá trình xuất **tự kiểm tra** TorchScript cho kết quả trùng với mô hình gốc (sai lệch < 1e-5) trước khi ghi ra đĩa; fold nào không đạt sẽ bị loại và báo rõ.


In [ ]:
# =========================================================
# 15.1 Đóng gói mô hình triển khai
#     [V6-P2] Dùng _fresh_dir() (định nghĩa ở mục 11.3) để không chết ở bước đóng gói
#     nếu thư mục của lần chạy trước chưa xoá được.
# =========================================================
EXPORT_MODEL = DEPLOY_MODEL            # đã chọn trên validation-TRONG ở mục 13
BUNDLE_NAME  = f"leak_detector_{EXPORT_MODEL}_{time.strftime('%Y%m%d')}"
BUNDLE_DIR   = _fresh_dir(OUT / BUNDLE_NAME, "gói mô hình")

_rm = rec_main.set_index("model"); _fm = frame_df.set_index("model")
print(f"Mô hình xuất: {EXPORT_MODEL}")
print(f"  AUC mức bản ghi : {float(_rm.auc[EXPORT_MODEL]):.4f} "
      f"[{float(_rm.auc_lo[EXPORT_MODEL]):.4f}, {float(_rm.auc_hi[EXPORT_MODEL]):.4f}]")
print(f"  AUC mức frame   : {float(_fm.AUC[EXPORT_MODEL]):.4f}")

example = torch.randn(2, FRAME_LEN)
exported, failed = [], []
for k in range(len(SPLITS)):
    ckpt = CKPT_DIR / f"{EXPORT_MODEL}_fold{k}.pt"
    if not ckpt.exists():
        failed.append((k, "thiếu checkpoint")); continue
    m = build_model(EXPORT_MODEL)
    m.load_state_dict({kk: vv.float() for kk, vv in torch.load(ckpt, map_location="cpu").items()})
    m.eval().cpu()
    try:
        with torch.no_grad():
            ts = torch.jit.trace(m, example, strict=False)
            d = float((m(example) - ts(example)).abs().max())
        assert d < 1e-5, f"TorchScript lệch {d:.2e}"
        torch.jit.save(ts, str(BUNDLE_DIR / f"fold{k}.ts"))
        exported.append(k)
        print(f"  fold {k}: OK (sai lệch {d:.1e}, "
              f"{(BUNDLE_DIR/f'fold{k}.ts').stat().st_size/1024**2:.1f} MB)")
    except Exception as e:
        failed.append((k, f"{type(e).__name__}: {e}")); print(f"  fold {k}: THẤT BẠI")
    del m
assert exported, "Không xuất được fold nào."

torch.save({f"fold{k}": torch.load(CKPT_DIR / f"{EXPORT_MODEL}_fold{k}.pt", map_location="cpu")
            for k in exported}, BUNDLE_DIR / "state_dicts.pt")


In [ ]:
# =========================================================
# 15.2 Manifest: mọi tham số cần để tái lập ĐÚNG quyết định + GIỚI HẠN đã đo được
# =========================================================
_perf = dict(
    record_auc=float(_rm.auc[EXPORT_MODEL]),
    record_auc_ci95=[float(_rm.auc_lo[EXPORT_MODEL]), float(_rm.auc_hi[EXPORT_MODEL])],
    record_accuracy=float(_rm.accuracy[EXPORT_MODEL]),
    record_recall=float(_rm.recall[EXPORT_MODEL]),
    record_specificity=float(_rm.specificity[EXPORT_MODEL]),
    frame_auc=float(_fm.AUC[EXPORT_MODEL]),
    frame_auc_ci95_cluster=[float(_fm.AUC_lo[EXPORT_MODEL]), float(_fm.AUC_hi[EXPORT_MODEL])],
    frame_auc_ci95_naive=[float(_fm.AUC_lo_naive[EXPORT_MODEL]), float(_fm.AUC_hi_naive[EXPORT_MODEL])],
    n_independent_recordings=int(len(records)),
    icc=float(_fm.ICC[EXPORT_MODEL]),
    design_effect=float(_fm.design_effect[EXPORT_MODEL]),
    effective_n_frames=float(_fm.n_eff[EXPORT_MODEL]))

_limitations = []
if not bool(_gv("HAS_SITE", False)):
    _limitations.append("Không có thông tin điểm khảo sát: chưa loại trừ được khả năng mô hình "
                        "phân biệt vị trí thu thay vì nhận dạng rò rỉ.")
else:
    _limitations.append("Đã có thông tin điểm khảo sát; xem bảng 9d để biết kết quả kiểm định.")
_wv = _gv("WITNESS_VERDICT", "not_supported")
if _wv != "supported":
    _limitations.append(f"Tỉ lệ thể hiện dương trung vị {float(_gv('W_LEAK_MED', np.nan)):.2f} "
                        f"({'không ủng hộ' if _wv == 'not_supported' else 'chưa đủ để kết luận về'}) "
                        "tiền đề 'nhãn mức frame bị nhiễu'; không dùng mô hình này để định vị thời điểm rò rỉ.")
if _gv("BAND_NARRATIVE", "") == "inconclusive":
    _limitations.append("Chưa phân định được dải tần mang thông tin (CI của hiệu số chứa 0); xem mục 3.5b.")
if _gv("BAND_NARRATIVE", "") == "low_band_not_supported":
    _limitations.append("Dấu hiệu phân biệt KHÔNG tập trung ở dải < 800 Hz như giả thuyết cho "
                        "ống nhựa; xem mục 3.5.")
if len(cfg.seeds) < 2:  # CV lặp
    _limitations.append("Chỉ huấn luyện 1 seed: độ bất định giữa các lần huấn luyện chưa được "
                        "định lượng đầy đủ.")
_limitations.append("Xử lý OFFLINE (chuẩn hoá đỉnh theo toàn bộ bản ghi, quyết định ở mức bản "
                    "ghi): không dùng cho hệ thống thời gian thực khi chưa đánh giá lại.")

# [V6-P2] Đặt tên `deploy_manifest` thay vì `manifest`: `manifest` đã là DataFrame
# mức TỆP (mục 2.3c); ghi đè nó bằng dict ở đây sẽ làm hỏng mọi cell phía sau
# dùng lại tên đó - lỗi chỉ lộ ra khi chạy và rất khó truy.
deploy_manifest = {
    "format_version": 4,
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_name": EXPORT_MODEL,
    "selection": {
        "criterion": "AUC mức bản ghi trên validation-TRONG (CI của chính tiêu chí có trong bảng 15)",
        "inner_auc": float(dep_sel_df.set_index("model").AUC_rec_inner[EXPORT_MODEL]),
        "inner_auc_ci95": [float(dep_sel_df.set_index("model").AUC_inner_lo[EXPORT_MODEL]),
                           float(dep_sel_df.set_index("model").AUC_inner_hi[EXPORT_MODEL])],
        "argmax_on_reported_data": ARGMAX_OUTER,
        "candidates": {r.model: float(r.AUC_rec_inner) for _, r in dep_sel_df.iterrows()},
        "note": "Các mô hình dẫn đầu có CI CHỒNG LẤN -> lựa chọn không được dữ liệu phân giải; "
                "đây là quyết định kỹ thuật (hiệu năng + chi phí suy luận)."},
    "folds": exported, "n_folds_total": len(SPLITS),
    "framework": {"torch": torch.__version__},
    "preprocessing": {
        "target_sr": int(cfg.target_sr), "highpass_hz": float(cfg.highpass_hz),
        "notch_mains": bool(cfg.notch_mains), "mains_hz": float(cfg.mains_hz),
        "mains_harmonics": int(cfg.mains_harmonics), "notch_q": float(cfg.notch_q),
        "peak_normalise": True, "peak_norm_pct": float(cfg.peak_norm_pct),
        "peak_norm_unit": "file", "causal_online": False,
        "frame_sec": float(cfg.frame_sec), "hop_sec": float(cfg.hop_sec),
        "frame_len": int(FRAME_LEN), "hop_len": int(HOP_LEN)},
    "representation": {
        "n_fft": int(cfg.n_fft), "hop_length": int(cfg.hop_length),
        "n_mels": int(cfg.n_mels), "f_min": float(cfg.f_min), "f_max": float(cfg.f_max),
        "low_band_hz": float(cfg.low_band_hz), "mfcc_norm": str(cfg.mfcc_norm)},
    "decision": {
        "frame_threshold": float(THR[EXPORT_MODEL]),
        "aggregation": str(best_agg),
        "record_threshold": float(_rm.threshold[EXPORT_MODEL]),
        "calibrated": bool(cfg.calibrate_aggregation),
        "ensemble": "trung bình xác suất đã hiệu chỉnh của các fold",
        "ensemble_threshold_warning": "Ngưỡng fit trên phân bố điểm của MỘT mô hình; khi dùng "
                                      "ensemble K fold, phân bố điểm dịch chuyển -> phải hiệu "
                                      "chỉnh lại ngưỡng trên dữ liệu hiện trường mới.",
        "operating_points": json.loads(op_df.reset_index()
                                       .rename(columns={"index": "ten"})
                                       .to_json(orient="records"))},
    "performance_out_of_fold": _perf,
    "statistics": {
        "unit_of_analysis": "recording (Rxxx)",
        "ci_method": "cluster bootstrap over recordings",
        "pairwise_test": "McNemar at recording level (n=%d), Holm-corrected" % len(records),
        "wilcoxon_min_p_with_%d_folds" % len(SPLITS): wilcoxon_min_p(len(SPLITS))},
    "confound_checks": {k: (v if not isinstance(v, (np.floating, np.integer)) else float(v))
                        for k, v in (_gv("CONFOUND", {}) or {}).items()},
    "limitations": _limitations,
    "training_data": {
        "n_records": int(len(records)), "n_files": int(len(files)), "n_frames": int(len(frames)),
        "n_leak_records": int((records.label == 1).sum()),
        "n_noleak_records": int((records.label == 0).sum()),
        "total_minutes": float(records.duration_s.sum() / 60),
        "pipe": "PVC, 1-2 bar", "note": "nhãn ở mức bản ghi Rxxx"}}
(BUNDLE_DIR / "manifest.json").write_text(
    json.dumps(deploy_manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

(BUNDLE_DIR / "README.txt").write_text(f'''GÓI MÔ HÌNH PHÁT HIỆN RÒ RỈ - {EXPORT_MODEL}
Tạo lúc: {deploy_manifest["created"]}

Nội dung:
  fold*.ts        TorchScript, chạy độc lập, không cần mã nguồn gốc
  state_dicts.pt  trọng số thuần (dự phòng)
  manifest.json   tham số tiền xử lý + ngưỡng + điểm vận hành + hiệu năng + GIỚI HẠN

Cách dùng: mở notebook "leak_detector_inference.ipynb", trỏ BUNDLE vào thư mục
hoặc tệp .zip này và AUDIO_FILE vào tệp .wav hiện trường, rồi Run All.

Cách chọn mô hình: trên validation-TRONG, có kèm CI của chính tiêu chí chọn.
Các ứng viên: {", ".join(f"{k} {v:.3f}" for k, v in deploy_manifest["selection"]["candidates"].items())}

Hiệu năng out-of-fold trên {deploy_manifest["training_data"]["n_records"]} bản ghi:
  AUC mức bản ghi {_perf["record_auc"]:.4f} (CI 95%: {_perf["record_auc_ci95"][0]:.4f}-{_perf["record_auc_ci95"][1]:.4f})
  Độ chính xác {_perf["record_accuracy"]:.4f} | Độ nhạy {_perf["record_recall"]:.4f} | Độ đặc hiệu {_perf["record_specificity"]:.4f}
  AUC mức frame  {_perf["frame_auc"]:.4f} (CI theo CỤM bản ghi: {_perf["frame_auc_ci95_cluster"][0]:.4f}-{_perf["frame_auc_ci95_cluster"][1]:.4f})
  ICC = {_perf["icc"]:.3f} | design effect = {_perf["design_effect"]:.1f} | n hiệu dụng = {_perf["effective_n_frames"]:.0f} frame

GIỚI HẠN ĐÃ ĐO ĐƯỢC (đọc trước khi dùng):
{chr(10).join("  - " + s for s in _limitations)}

CẢNH BÁO PHẠM VI: huấn luyện trên ống PVC áp lực 1-2 bar, lấy mẫu {int(cfg.target_sr)} Hz,
một chuỗi thiết bị thu duy nhất. Vật liệu, áp suất hoặc cảm biến khác thì hiệu năng trên
KHÔNG còn hiệu lực và cần đánh giá lại. Ngưỡng cần hiệu chỉnh lại cho dữ liệu mới.
''', encoding="utf-8")

zip_path = shutil.make_archive(str(BUNDLE_DIR), "zip", BUNDLE_DIR)
print(f"\n✓ Đã xuất: {zip_path}  ({Path(zip_path).stat().st_size/1024**2:.1f} MB)")
print(f"  {len(exported)} fold TorchScript + manifest + dự phòng")
if failed: print("  Fold không xuất được:", failed)


In [ ]:
# =========================================================
# 15.3 Tự kiểm tra gói + [V6-T8] KIỂM CHỨNG fp16: tính lại AUC từ chính checkpoint đã giao
#      v5 chỉ kiểm tra việc ghi/đọc file (đường vẫn đúng), nhưng KHÔNG kiểm tra rằng trọng số
#      fp16 đã lưu cho ra đúng chỉ số đã báo cáo (chỉ số vốn tính từ mô hình fp32 trong RAM).
# =========================================================
_probe = torch.from_numpy(np.asarray(X_ALL[:64])).float()
ok_all = True
for k in exported:
    m = build_model(EXPORT_MODEL)
    m.load_state_dict({kk: vv.float() for kk, vv in
                       torch.load(CKPT_DIR / f"{EXPORT_MODEL}_fold{k}.pt", map_location="cpu").items()})
    m.eval().cpu()
    ts = torch.jit.load(str(BUNDLE_DIR / f"fold{k}.ts"), map_location="cpu").eval()
    with torch.no_grad():
        d = float((torch.sigmoid(m(_probe)) - torch.sigmoid(ts(_probe))).abs().max())
    ok = d < 1e-5; ok_all &= ok
    print(f"  fold {k}: sai lệch xác suất tối đa {d:.2e}  {'✓' if ok else '✗'}")
    del m, ts

_man = json.loads((BUNDLE_DIR / "manifest.json").read_text(encoding="utf-8"))
need = [("target_sr", cfg.target_sr), ("frame_len", FRAME_LEN), ("hop_len", HOP_LEN)]
for kk, vv in need:
    got = _man["preprocessing"][kk]
    ok = int(got) == int(vv); ok_all &= ok
    print(f"  manifest.{kk}: {got} {'✓' if ok else '✗ (mong đợi ' + str(vv) + ')'}")

# --- [V6-T8] AUC tính lại từ checkpoint fp16 ĐÃ GIAO --------------------
print("\n[V6-T8] Kiểm chứng fp16: tính lại AUC out-of-fold từ chính checkpoint đã giao...")
oof_ship = np.full(len(frames), np.nan)
for k in exported:
    m = build_model(EXPORT_MODEL)
    m.load_state_dict({kk: vv.float() for kk, vv in
                       torch.load(CKPT_DIR / f"{EXPORT_MODEL}_fold{k}.pt", map_location="cpu").items()})
    m.to(DEVICE).eval()
    va = SPLITS[k][1]
    P = []
    with torch.no_grad():
        for s0 in range(0, len(va), 256):
            xb = torch.from_numpy(np.asarray(X_ALL[va[s0:s0+256]])).float().to(DEVICE)
            with amp_autocast(DEVICE, cfg.amp and DEVICE.type == "cuda"):
                P.append(torch.sigmoid(m(xb).float()).cpu().numpy())
    oof_ship[va] = np.concatenate(P)
    del m; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

mk_ship = ~np.isnan(oof_ship)
auc_ship = float(roc_auc_score(y_all[mk_ship].astype(int), oof_ship[mk_ship]))
auc_rep = float(frame_df.set_index("model").AUC[EXPORT_MODEL])
_d = abs(auc_ship - auc_rep)
print(f"  AUC báo cáo            : {auc_rep:.4f}")
print(f"  AUC từ checkpoint giao : {auc_ship:.4f}  (lệch {_d:.4f})")
if _d < 0.002:
    print("  ✓ Trọng số fp16 đã giao TÁI LẬP được đúng chỉ số đã báo cáo.")
else:
    ok_all = False
    print("  ✗ Lệch quá 0.002 -> con số báo cáo KHÔNG khớp mô hình đã giao. Phải điều tra")
    print("    (thường do lưu checkpoint fp16 hoặc sai fold).")
print("\n" + ("✓ GÓI ĐẠT - sẵn sàng giao cho notebook suy luận."
              if ok_all else "✗ GÓI KHÔNG ĐẠT - đừng dùng, hãy xem lại mục 15."))


---

## 16. CHANGELOG v6 → v7 và những gì CÒN PHẢI LÀM

Phần này liệt kê **chính xác** những gì đã sửa so với `leak-cnn-pvc-kaggle-v6.ipynb`, để người
đọc (và phản biện) truy vết được. Nhóm mã: **B** = lỗi mã, **S** = thống kê, **A** = ablation,
**C** = confound, **D** = dữ liệu. Toàn bộ v7 được sinh tái lập từ v6 bằng `tools/build_v7.py`.


In [ ]:
# =========================================================
# 16.1 Bảng đối chiếu v6 -> v7
# =========================================================
changes = pd.DataFrame([
    ("B1", "Grad-CAM của CRNN",
     "RuntimeError 'cudnn RNN backward can only be called in training mode' -> notebook DỪNG ở 10.1;"
     " mục 10.2-16 (occlusion, t-SNE, bảng LaTeX, đoạn Results, gói triển khai) chưa từng chạy",
     "tắt cuDNN cục bộ khi tính gradient Grad-CAM, mô hình vẫn ở chế độ eval",
     "toàn bộ phần sau của pipeline phải chạy được"),
    ("B2", "Chọn checkpoint trong warm-up",
     "cấu hình khai báo min_epochs_after_lr_peak nhưng best được cập nhật từ epoch 0; "
     "log: 'chọn ep1', LeakCNN1D chọn trung bình epoch 0.8 (gần như chưa huấn luyện)",
     "chỉ cập nhật best khi ep >= SELECT_FROM_EPOCH (= đỉnh LR); làm mịn chỉ trên các epoch hợp lệ; "
     "Bảng 5 kiểm tra ràng buộc",
     "so sánh kiến trúc công bằng; LeakCNN1D không còn bị 'xử thua' do chọn epoch 0"),
    ("B3", "Tự chỉnh low_band_hz",
     "hạ 2800 -> 1000 Hz theo MỘT dải (283-413 Hz, p=0.034, mất ý nghĩa sau Holm), cắt dải 1.3-2.7 kHz "
     "mà Grad-CAM LeakCNN2D dùng (~2.5 kHz)",
     "giá trị đăng ký trước (2800 Hz); auto (tuỳ chọn) chỉ MỞ RỘNG để phủ top-3 dải",
     "không để một kiểm định nhiễu quyết định kiến trúc"),
    ("B4", "Mô tả SpecAugment của MultiBranchCNN",
     "văn bản: dùng chung cho 3 nhánh; mã: chỉ nhánh Mel. Mặt nạ chung cho cả batch",
     "mỗi nhánh một SpecAugment (MFCC: chỉ che thời gian); mặt nạ riêng cho từng mẫu",
     "mã và mô tả phải khớp"),
    ("B5", "DEPLOY_MODEL dùng trước khi định nghĩa",
     "mục 10.4a-e dùng DEPLOY_MODEL, chỉ được gán ở mục 13 -> NameError (bị che bởi B1)",
     "xác định ngay sau mục 8.5 bằng đúng tiêu chí của mục 13 (AUC bản ghi trên validation-trong)",
     "tránh lỗi âm thầm"),
    ("B6", "Ghi đè biến sel_df",
     "mục 13 ghi đè sel_df của mục 8.5", "đổi tên agg_sel_df / dep_sel_df", "vệ sinh mã"),
    ("B7", "Lọc baseline RMS trong đoạn Results",
     "lọc theo tên 'CHỈ mức năng lượng (RMS)' trong khi bảng dùng 'Level only (RMS)'",
     "kết luận lấy trực tiếp từ so sánh ghép cặp mục 9.5", "baseline 1 đặc trưng không bị chọn nhầm"),
    ("B8", "TransferCNN / LeakCNN1D",
     "ResNet18 nhận ảnh 64x63 -> bản đồ 2x2; LeakCNN1D nhận biên độ tuyệt đối (dùng được 'độ to')",
     "phóng log-Mel lên >=128 (bản đồ 4x4); LeakCNN1D chuẩn hoá theo mẫu như mô hình phổ",
     "so sánh kiến trúc công bằng, bớt kênh confound mức năng lượng"),
    ("B9", "Đoạn Results tự sinh",
     "'Aggregation changed AUC by ...' lấy AUC bản ghi của mô hình A trừ AUC frame của mô hình B; "
     "câu tiếng Anh chèn chuỗi tiếng Việt; câu 'NOT allow us to claim' gãy ngữ pháp",
     "cùng MỘT mô hình; ánh xạ kết luận sang tiếng Anh; nêu rõ quy tắc gộp đăng ký trước",
     "đoạn văn đi thẳng vào bản thảo"),
    ("B10", "Diễn giải trong trường hợp suy biến",
     "Grad-CAM toàn 0 vẫn cho 'đỉnh 20 Hz'; occlusion kết luận 'đặc trưng phân tán' cả khi AUC nền < 0.5",
     "loại bản đồ suy biến khỏi kết luận; occlusion chỉ diễn giải khi AUC nền >= 0.6",
     "không rút kết luận từ mô hình chưa học được gì"),
    ("B11", "Tương thích thư viện", "boxplot(labels=...) lỗi trên matplotlib >= 3.9",
     "đặt nhãn trục riêng", "chạy được trên mọi phiên bản"),
    ("S1", "Một phân hoạch, một seed",
     "mọi chênh lệch kiến trúc lẫn với nhiễu phân hoạch",
     "CV lặp (mặc định 3 phân hoạch); Bảng 5b: mean±sd và AUC của OOF trung bình qua lần lặp",
     "đo độ bất định mà phản biện chắc chắn hỏi"),
    ("S2", "Ngã rẽ phân tích ở mức bản ghi",
     "quy tắc gộp chọn trên validation-trong + Platt (làm AUC MultiBranch giảm 0.856 -> 0.801)",
     "p_mean trên xác suất thô, ĐĂNG KÝ TRƯỚC; các lựa chọn khác chỉ là phân tích độ nhạy",
     "giảm số quyết định dựa trên tập nhỏ"),
    ("S3", "CNN vs baseline cổ điển",
     "13 đặc trưng thủ công; kết luận bằng 'hai CI không chồng lấn' (0.8092 vs 0.8076)",
     "thêm thống kê log-Mel 128 chiều; CI bootstrap GHÉP CẶP của ΔAUC + Holm; CNN tham chiếu = mô hình "
     "chọn trên validation-trong",
     "kiểm định đúng câu hỏi 'CNN có thêm giá trị?'"),
    ("S4", "Kiểm định theo dải tần",
     "7 + 14 kiểm định không hiệu chỉnh; giả thuyết dải thấp kết luận từ hai số điểm (0.164 vs 0.194)",
     "Holm cho mọi họ kiểm định; CI bootstrap của hiệu số; kết luận 3 mức (có 'chưa phân định')",
     "phát biểu khớp sức mạnh bằng chứng"),
    ("S5", "Tỉ lệ thể hiện dương",
     "ngưỡng duy nhất 0.5; in 'trung vị 0.81 ~ 1'",
     "3 mức: <0.5 ủng hộ / 0.5-0.9 chưa kết luận / >=0.9 không ủng hộ", "không phóng đại"),
    ("A1", "Ablation",
     "tắt mặc định; chỉ độ dài frame + bộ lọc",
     "bật mặc định: no_fir, rec_balanced, sr16k, frame1s/4s, no_notch, no_highpass; chọn theo "
     "validation-trong; ΔAUC ghép cặp so với gốc",
     "trả lời 'vì sao chọn cấu hình này' bằng số liệu, không thiên lệch"),
    ("C1", "Thông tin điểm khảo sát",
     "chỉ cảnh báo", "tự sinh site_map_template.csv để điền", "hạ rào cản cho thí nghiệm leave-one-site-out"),
    ("D1", "Gán nhãn & dò dữ liệu",
     "khớp chuỗi con (khoá 'rr' khớp mọi tên có 'rr'); data_root sai -> âm thầm quét cả /kaggle/input/datasets",
     "khớp theo từ + tự kiểm tra; chọn thư mục sâu nhất chứa leak/noleak, cảnh báo rõ ràng",
     "tránh trộn dữ liệu / gán nhãn sai"),
    ("D2", "Số liệu cũ viết cứng",
     "văn bản còn '116 bản ghi', '8 801 frame', 'ICC 0,75', 'trung vị 0,99', 'mở rộng 2800 Hz'",
     "mọi con số in từ lần chạy; build_v7.py chặn tự động nếu còn số cũ",
     "không để số liệu sai lọt vào bản thảo"),
], columns=["id", "item", "v6 (defect)", "v7 (fixed)", "reason"])
display(Markdown("### Table 17 (internal). v6 to v7 changelog — not intended for the manuscript"))
display(changes)

# ---- Trạng thái tự kiểm tra của lần chạy này ----
print("\n=== TRẠNG THÁI LẦN CHẠY NÀY ===")
print(f"  - Dữ liệu                    : {'MÔ PHỎNG (synthetic)' if SYNTHETIC else DATA_ROOT}")
print(f"  - Đơn vị thống kê            : BẢN GHI (n={len(records)}); CI theo cụm")
print(f"  - Tiêu chí chọn epoch        : {cfg.select_metric}, chỉ từ epoch {SELECT_FROM_EPOCH}")
print(f"  - CV lặp                     : {len(cfg.seeds)} phân hoạch")
print(f"  - Quy tắc gộp mức bản ghi    : {best_agg} ({cfg.agg_selection}); Platt: {cfg.calibrate_aggregation}")
print(f"  - Mô hình triển khai         : {DEPLOY_MODEL} (chọn trên validation-trong)")
print(f"  - Baseline cổ điển           : {BASELINE_VERDICT_VI if cfg.run_baseline_models else 'không chạy'}")
print(f"  - Ablation                   : {'biến thể được chọn = ' + str(ABL_SELECTED) if ABL_SELECTED else 'không chạy'}")
print(f"  - Kiểm tra confound          : có site: {bool(_gv('HAS_SITE', False))} | "
      f"{(_gv('CONFOUND', {}) or {}).get('verdict', 'n/a')}")
print(f"  - Tiền đề MIL                : {_gv('WITNESS_VERDICT', 'n/a')} (trung vị {float(_gv('W_LEAK_MED', np.nan)):.2f})")
print(f"  - Giả thuyết dải thấp        : {_gv('BAND_NARRATIVE', 'n/a')} | CI hiệu số {BAND_TEST['ci']}")

print("\n=== VIỆC CÒN PHẢI LÀM TRƯỚC KHI NỘP BÀI (không thể tự động hoá) ===")
todo = [
 "1. Điền site_map_template.csv (vị trí/tuyến ống/ngày thu cho từng Rxxx), đặt cfg.site_source='csv' và "
 "cfg.run_loso_full=True: đây là thí nghiệm quan trọng nhất còn thiếu.",
 "2. Nghe và dán nhãn THỜI ĐIỂM rò rỉ trên ~10 bản ghi (2 người độc lập, báo cáo thoả thuận) để kiểm "
 "chứng trực tiếp tiền đề MIL.",
 "3. Nếu Bảng 9e đề xuất biến thể khác 'base': đổi cfg tương ứng và chạy lại toàn bộ notebook.",
 "4. Xác minh các tài liệu trong Bảng 11 trên Scopus (cột 'verified').",
 "5. Nếu định triển khai thời gian thực: thay chuẩn hoá đỉnh theo tệp bằng phương án NHÂN QUẢ và đánh giá lại.",
 "6. Notebook suy luận riêng (leak_detector_inference.ipynb) phải đọc manifest format_version 4 "
 "(trường mới: preprocessing.peak_norm_pct).",
]
for t in todo: print("  " + t)
print("\nGhi chú: mọi con số trong notebook này chỉ được trích dẫn kèm ĐƠN VỊ THỐNG KÊ và "
      "KHOẢNG TIN CẬY THEO CỤM.")
